# BOREAS — borehole array storage with a conducting formation

Latent-heat storage in repurposed oil-and-gas wells. **A new series, not a new
version of P2H2P/THUMS** — that model is frozen at v0.16 and still runs; its
notebook is untouched.

*Notebook **v2.25**, results pack v2.25 · 2026-11-21.*

**What this version is.** Section 2 holds every input: the design point
(2.1), every list the paper sweeps over (2.2), and a switch (2.3) between
the published results and re-marching everything from your own values.
Sections 12–13 reproduce all 39 of the paper's result files and rebuild the
macro pack, and section 15 checks that the notebook itself runs.

---

## What changed, and why the two changes are one change

### 1. The geometry

THUMS put $n$ **hairpins** in the hole — U-tubes, down one leg and back up the
other, both exchanging with the PCM. That turns out to be self-contradictory,
and the contradiction is quantitative.

Charge water enters at $T_m + 10$ K and glides 55 K down, so the **return leg
runs about 45 K below the melting point**. Graded by depth it would refreeze
whatever the down leg had just melted. So THUMS graded the cascade along the
**tube path** instead — which works, but then two different melting points
have to coexist at the same depth, centimeters apart, in contiguous PCM with
no wall between them. Conduction across that gap is

| hairpins | leg-to-leg short circuit | % of a ~120 kW duty |
|---|---|---|
| 1 | 43 kW | 36 % |
| 2 | 62 kW | 52 % |
| 3 | **92 kW** | **77 %** |

a short circuit straight across the cascade the model exists to represent —
and a term THUMS never had.

**BOREAS separates the two jobs.** `n_ft` fin tubes all run *downward* and all
exchange; one **insulated downcomer** returns the whole flow without
exchanging. They meet in a manifold at the bottom of the same hole — no
lateral, no new drilling, the same class of object as the U-bend it replaces.

### 2. The formation

THUMS hypothesis **H6** made the borehole wall adiabatic, so $\eta_{storage}$
was identically 1 — not approximately, *exactly*, by construction. BOREAS
conducts to the rock.

### Why the second needed the first

With a hairpin, **every depth is occupied by two tube segments**, so one rock
temperature would have had two PCM segments claiming it. Only when the return
stops exchanging does $z$ become a depth. In BOREAS `L_tube = L_well`, not
twice it, and the map is one-to-one.

Three things follow for free:

- the cascade runs **hot at the bottom**, which *aligns* it with the
  geothermal gradient instead of fighting it
- the PCM becomes **pourable in layers** (path-graded PCM was never
  manufacturable)
- the downcomer's leak goes **into** the store on charge — energy misplaced,
  not energy lost

---
## 1. Setup

`CoolProp` is not in a stock Colab image. The model is written to disk by the
next cell, so this notebook needs nothing from the repository.

In [ ]:
!pip install -q CoolProp

In [ ]:
%%writefile boreas.py
"""BOREAS -- BOREhole Array Storage.

Latent-heat storage in repurposed wellbores, with a DOWNCOMER geometry and a
conducting formation. The live model, v1.3.

BOREAS is a new series, not a new version of THUMS. The P2H2P/THUMS line stays
exactly where it was at v0.16 and remains runnable; nothing here edits it. Two
things differ, and they are not independent:

  THE GEOMETRY. THUMS put n hairpins in the hole. A hairpin needs its cascade
  graded along the TUBE PATH (the return leg runs ~45 K below the melting
  point and would otherwise refreeze what the down leg melted), which puts two
  melting points at the same depth in contiguous PCM. Conduction across that
  gap is 43 kW at one hairpin and 92 kW at three, against a ~120 kW duty --
  a short circuit straight across the cascade the model exists to represent,
  and a term THUMS never had. BOREAS runs n_ft fin tubes DOWNWARD and returns
  the whole flow through ONE INSULATED DOWNCOMER. Same pressure drop, same PCM
  volume, no short circuit.

  THE FORMATION. THUMS hypothesis H6 made the borehole wall adiabatic, so
  eta_storage was identically 1 -- not approximately, exactly, by construction.
  BOREAS conducts to the rock through casing, cement and ground, with the
  undisturbed formation temperature set by an INDEPENDENT geothermal gradient.

The second needed the first. With a hairpin, every depth is occupied by two
tube segments, so one rock temperature would have had two PCM segments
claiming it. Only when the return stops exchanging does z become a depth.

Setting T_rock=None in `march_h` recovers the adiabatic wall exactly, which is
how the v0.16 reproduction test is run. One formulation (enthalpy, "Formulation C"), one sizing
framing (specify the hardware and march to cyclic steady state), no root
finding anywhere.

    plant      cycle_state_points   ->  state points and the two efficiencies
               energy_budget        ->  how much energy must move per cycle

    borehole   pcm_capacities       ->  E'_lat, C_s, C_l  per unit tube length
               pcm_state            ->  E'  ->  (A_melt, T_pcm)
               advance_segment      ->  branchwise-exact step of E'
               march_h              ->  one half-cycle over the whole exchanger

    cycle      simulate_css         ->  repeat until the state returns to itself
               simulate_css_corrected  ->  ... with one ORC correction pass
               css_report           ->  print it

Everything is driven by one frozen dataclass, `Case`. To study a parameter,
copy `CASE` with the field changed -- `CASE.with_(N_lay=12)` -- and re-run.
`validate_case` checks a case for the inconsistencies that are easy to create
and hard to see.

Superseded material is not here. The first-cycle sizing chain, the closed-form
Stefan front and the area-based melt front were removed; the reasoning that
removed them is in DESIGN_NOTES.md and in the project report, and the code is
in version control.
"""
import numpy as np
import CoolProp.CoolProp as CP
from CoolProp.CoolProp import PropsSI
from math import log, pi
import pandas as pd


# ==========================================================================
# properties
# ==========================================================================


########################################
########################################
########################################


# ==========================================================================
# cycles
# ==========================================================================


def double_stage_rankine(fluid, T_1e, T_3e, eta_t=1.0, eta_p=1.0,
                         x_int=None):
  """
  Calculates parameters for a two-stage Rankine cycle with regeneration/reheating

  Args:
    fluid: The name of the working fluid.
    T_1e: Temperature of state 1e in Kelvin.
    T_3e: Temperature of state 3e in Kelvin.

  Returns:
    A dictionary containing calculated Rankine cycle parameters,
    including enthalpy, entropy, temperature, and pressure for each state,
    the Rankine efficiency, and saturation pressures.
  """
  # State 3e: Saturated vapor at the 1st stage (turbine) inlet
  p_evape = CP.PropsSI('P', 'T', T_3e, 'Q', 1, fluid) # Condenser pressure (same as 3e)
  p_3e = p_evape
  h_3e = CP.PropsSI('H', 'T', T_3e, 'Q', 1, fluid)/1000.
  s_3e = CP.PropsSI('S', 'T', T_3e, 'Q', 1, fluid)

  # State 1e: Saturated liquid at condenser outlet
  p_conde = CP.PropsSI('P', 'T', T_1e, 'Q', 0, fluid) # Evaporator pressure
  p_1e = p_conde
  h_1e = CP.PropsSI('H', 'T', T_1e, 'Q', 0, fluid)/1000.
  s_1e = CP.PropsSI('S', 'T', T_1e, 'Q', 0, fluid)

  # Intermediate pressure.
  #
  # The paper's choice is the ARITHMETIC mean, x_int = 0.5 on the linear
  # interpolation below; the classical work-minimising choice for two-stage
  # compression is the GEOMETRIC mean, which is what the heat pump uses.
  # x_int makes the choice a parameter so the optimum can be found rather
  # than asserted:  p_int = p_low + x (p_high - p_low).
  if x_int is None:
      p_inte = (p_evape + p_conde)*(1./2.)
  else:
      p_lo, p_hi = min(p_evape, p_conde), max(p_evape, p_conde)
      p_inte = p_lo + float(x_int)*(p_hi - p_lo)

  # State 5e: 1st stage turbine outlet
  p_5e = p_inte
  # IRREVERSIBLE from v0.12. h_5e_s is where an isentropic machine would
  # land; the real one recovers only eta_t of that drop and therefore leaves
  # HOTTER, with more entropy. Everything downstream of 5e -- the reheat duty,
  # the regenerator split y, the evaporator inlet 10e -- moves with it, which
  # is exactly why a downstream multiplier on the work could not reproduce it.
  h_5e_s = CP.PropsSI('H', 'S', s_3e, 'P', p_5e, fluid)/1000.
  h_5e = h_3e - eta_t * (h_3e - h_5e_s)
  s_5e = CP.PropsSI('S', 'H', h_5e*1000., 'P', p_5e, fluid)
  T_5e = CP.PropsSI('T', 'H', h_5e*1000., 'P', p_5e, fluid)

  # State 6e: 2nd stage turbine inlet
  p_6e = p_5e
  T_6e = T_3e #ideal reheat
  h_6e = CP.PropsSI('H', 'T', T_6e, 'P', p_6e, fluid)/1000.
  s_6e = CP.PropsSI('S', 'T', T_6e, 'P', p_6e, fluid)

  # State 4e: 2nd stage turbine outlet
  p_4e = p_conde
  h_4e_s = CP.PropsSI('H', 'S', s_6e, 'P', p_4e, fluid)/1000.
  h_4e = h_6e - eta_t * (h_6e - h_4e_s)
  s_4e = CP.PropsSI('S', 'H', h_4e*1000., 'P', p_4e, fluid)
  T_4e = CP.PropsSI('T', 'H', h_4e*1000., 'P', p_4e, fluid)

  # State 2e: Pump 1 outlet
  p_2e = p_evape
  # a pump absorbs MORE than isentropic, so the divide goes the other way
  h_2e_s = CP.PropsSI('H', 'S', s_1e, 'P', p_2e, fluid)/1000.
  h_2e = h_1e + (h_2e_s - h_1e) / eta_p
  s_2e = CP.PropsSI('S', 'H', h_2e*1000., 'P', p_2e, fluid)
  T_2e = CP.PropsSI('T', 'H', h_2e*1000., 'P', p_2e, fluid)

  # State 7e: Pump 2 inlet
  p_7e = p_inte
  h_7e = CP.PropsSI('H', 'P', p_7e, 'Q', 0, fluid)/1000.
  s_7e = CP.PropsSI('S', 'P', p_7e, 'Q', 0, fluid)
  T_7e = CP.PropsSI('T', 'P', p_7e, 'Q', 0, fluid)

  # State 8e: Pump 2 outlet
  p_8e = p_evape
  h_8e_s = CP.PropsSI('H', 'P', p_8e, 'S', s_7e, fluid)/1000.
  h_8e = h_7e + (h_8e_s - h_7e) / eta_p
  s_8e = CP.PropsSI('S', 'H', h_8e*1000., 'P', p_8e, fluid)
  T_8e = CP.PropsSI('T', 'H', h_8e*1000., 'P', p_8e, fluid)

  # State 9e:
  p_9e = p_evape
  T_9e = T_7e
  h_9e = CP.PropsSI('H', 'T', T_9e, 'P', p_9e, fluid)/1000.
  s_9e = CP.PropsSI('S', 'T', T_9e, 'P', p_9e, fluid)

  # Energy balance in the regenerator
  y_frac = (h_5e - h_7e) / ((h_9e - h_2e) + (h_5e - h_7e))

  # State 10e:
  p_10e = p_evape
  h_10e = y_frac * h_9e + (1. - y_frac) * h_8e
  s_10e = CP.PropsSI('S', 'H', h_10e*1000, 'P', p_10e, fluid)
  T_10e = CP.PropsSI('T', 'H', h_10e*1000, 'P', p_10e, fluid)

  # cycle thermal efficiency
  numerator_eff = (h_3e - h_10e) + y_frac * (h_6e - h_5e) - y_frac * (h_4e - h_1e)
  denominator_eff = (h_3e - h_10e) + y_frac * (h_6e - h_5e)
  if abs(denominator_eff) > 1e-9:
      rank_eff = numerator_eff / denominator_eff
  else:
      rank_eff = float('inf') # Assign large value if denominator is zero or near zero

  return {
      'T_1e': T_1e, 'p_1e': p_1e, 'h_1e': h_1e, 's_1e': s_1e,
      'T_2e': T_2e, 'p_2e': p_2e, 'h_2e': h_2e, 's_2e': s_2e,
      'T_3e': T_3e, 'p_3e': p_3e, 'h_3e': h_3e, 's_3e': s_3e,
      'T_4e': T_4e, 'p_4e': p_4e, 'h_4e': h_4e, 's_4e': s_4e,
      'T_5e': T_5e, 'p_5e': p_5e, 'h_5e': h_5e, 's_5e': s_5e,
      'T_6e': T_6e, 'p_6e': p_6e, 'h_6e': h_6e, 's_6e': s_6e,
      'T_7e': T_7e, 'p_7e': p_7e, 'h_7e': h_7e, 's_7e': s_7e,
      'T_8e': T_8e, 'p_8e': p_8e, 'h_8e': h_8e, 's_8e': s_8e,
      'T_9e': T_9e, 'p_9e': p_9e, 'h_9e': h_9e, 's_9e': s_9e,
      'T_10e': T_10e, 'p_10e': p_10e, 'h_10e': h_10e, 's_10e': s_10e,
      'rank_eff': rank_eff,
      'p_evape': p_evape,
      'p_conde': p_conde,
      'p_inte': p_inte,
      'y_frac': y_frac # Added y_frac to the return dictionary
  }

# @title


def two_stage_htheatpump_2regs(refrig, T_13h, T_2h, DT_sub, eta_c=1.0,
                               x_int=None):
  """
  Calculates parameters for a two-stage high-temp heat pump with a liquid separator and 2 regenerators

  Args:
    refrig: The name of the refrigerant.
    T_13h: Temperature of state 13h in Kelvin.
    T_2h: Temperature of state 2h in Kelvin.
    DT_sub: Refrigerant subcooling degree at condenser outlet.

  Returns:
    A dictionary containing calculated Rankine cycle parameters.
  """
  # State 2h: Saturated Vapor at T_2h (Condenser Outlet)
  p_condh = CP.PropsSI('P', 'T', T_2h, 'Q', 0, refrig) # Condensing pressure
  p_2h = p_condh
  h_2h = CP.PropsSI('H', 'T', T_2h, 'Q', 1, refrig)/1000.
  s_2h = CP.PropsSI('S', 'T', T_2h, 'Q', 1, refrig)

  # State 3h: Subcooled Liquid at (T_2h - DT_sub) and p_condh (Condenser Outlet with subcooling)
  T_3h = T_2h - DT_sub
  p_3h = p_condh
  h_3h = CP.PropsSI('H', 'T', T_3h, 'P', p_condh, refrig)/1000.
  s_3h = CP.PropsSI('S', 'T', T_3h, 'P', p_condh, refrig)

  # State 13h: Saturated vapor at evaporating pressure
  p_evaph = CP.PropsSI('P', 'T', T_13h, 'Q', 1, refrig) # Evaporating pressure
  p_13h = p_evaph
  h_13h = CP.PropsSI('H', 'T', T_13h, 'Q', 1, refrig)/1000.
  s_13h = CP.PropsSI('S', 'T', T_13h, 'Q', 1, refrig)


  # Intermediate pressure. Default is the GEOMETRIC mean -- equal stage
  # pressure ratios, the classical work-minimising choice for two-stage
  # compression. x_int overrides it with the same linear parameterisation
  # the ORC uses, so the two cycles are swept on one scale.
  if x_int is None:
      p_inth = (p_evaph * p_condh)**(1./2.)
  else:
      p_lo, p_hi = min(p_evaph, p_condh), max(p_evaph, p_condh)
      p_inth = p_lo + float(x_int)*(p_hi - p_lo)

  # --- the compressions, IRREVERSIBLE from v0.12 ------------------------
  # The closure runs BACKWARDS: state 2h is fixed on the dew line at T_2h
  # (the condensing temperature the water demands), and the suction state is
  # whatever lands there. With eta_c = 1 that inverts in closed form, which
  # is what this used to do. With eta_c < 1 the same compression ends HOTTER,
  # so to still land on the dew point the suction must carry LESS superheat,
  # and the inversion is implicit -- h_2h_s depends on s_1h, which depends on
  # h_1h. Two bisections, outer stage first.
  #
  # The design intent is unchanged and worth restating: IHX-2 is sized to
  # deliver exactly the suction superheat that puts the discharge on the dew
  # point, so the condenser has no desuperheating duty. epsilon_IHX_2 remains
  # a reported OUTPUT, and it is what moves when eta_c does.
  p_1h = p_evaph
  p_11h = p_inth

  def _suction_for(h_target, p_lo, p_hi):
      """Enthalpy at p_lo whose eta_c compression to p_hi reaches h_target."""
      lo = CP.PropsSI('H', 'P', p_lo, 'Q', 1, refrig)/1000. - 40.0
      hi = h_target
      for _ in range(90):
          mid = 0.5*(lo + hi)
          s_mid = CP.PropsSI('S', 'H', mid*1000., 'P', p_lo, refrig)
          h_s = CP.PropsSI('H', 'S', s_mid, 'P', p_hi, refrig)/1000.
          if mid + (h_s - mid)/eta_c < h_target:
              lo = mid
          else:
              hi = mid
      return 0.5*(lo + hi)

  if abs(eta_c - 1.0) < 1e-12:
      s_1h = s_2h                      # closed form, the pre-v0.12 path
      T_1h = CP.PropsSI('T', 'P', p_1h, 'S', s_1h, refrig)
      h_1h = CP.PropsSI('H', 'T', T_1h, 'P', p_evaph, refrig)/1000.
      s_11h = s_1h
      T_11h = CP.PropsSI('T', 'S', s_11h, 'P', p_11h, refrig)
      h_11h = CP.PropsSI('H', 'S', s_11h, 'P', p_11h, refrig)/1000.
  else:
      h_11h = _suction_for(h_2h, p_inth, p_condh)      # HP stage inlet = 6h
      h_1h = _suction_for(h_11h, p_evaph, p_inth)      # LP stage inlet
      T_1h = CP.PropsSI('T', 'H', h_1h*1000., 'P', p_evaph, refrig)
      s_1h = CP.PropsSI('S', 'H', h_1h*1000., 'P', p_evaph, refrig)
      T_11h = CP.PropsSI('T', 'H', h_11h*1000., 'P', p_11h, refrig)
      s_11h = CP.PropsSI('S', 'H', h_11h*1000., 'P', p_11h, refrig)

  # State 5h: Low Pressure Compressor outlet
  T_5h = T_11h
  p_5h = p_11h
  h_5h = h_11h
  s_5h = s_11h

  # State 6h: High Pressure Compressor inlet
  T_6h = T_11h
  p_6h = p_11h
  h_6h = h_11h
  s_6h = s_11h

  # State 9h
  p_9h = p_inth
  T_9h = CP.PropsSI('T', 'P', p_9h, 'Q', 0, refrig) # Saturated liquid temp at p_inth
  h_9h = CP.PropsSI('H', 'P', p_9h, 'Q', 0, refrig)/1000.
  s_9h = CP.PropsSI('S', 'P', p_9h, 'Q', 0, refrig)

  # State 10h
  p_10h = p_inth
  T_10h = CP.PropsSI('T', 'P', p_10h, 'Q', 1, refrig) # Saturated vapor temp at p_inth
  h_10h = CP.PropsSI('H', 'P', p_10h, 'Q', 1, refrig)/1000.
  s_10h = CP.PropsSI('S', 'P', p_10h, 'Q', 1, refrig)

  # State 14h
  p_14h = p_evaph
  T_14h = CP.PropsSI('T', 'P', p_14h, 'Q', 0, refrig) # Saturated liquid temp at p_evaph
  h_14h = CP.PropsSI('H', 'P', p_14h, 'Q', 0, refrig)/1000.
  s_14h = CP.PropsSI('S', 'P', p_14h, 'Q', 0, refrig)

  # IHX-1 effectiveness (based on T_3h and T_10h)
  # Ensure denominator is non-zero
  denominator_epsilon1 = (T_3h - T_10h)
  if abs(denominator_epsilon1) > 1e-9:
      epsilon_IHX_1 = (T_11h - T_10h) / denominator_epsilon1
  else:
      epsilon_IHX_1 = 1.0 # Assume 100% effectiveness if temperature difference is zero

  # CP_ratio_1:
  try:
      cpf_1 = CP.PropsSI('C', 'P', p_condh, 'Q', 0, refrig)
      cpv_1 = CP.PropsSI('C', 'P', p_inth, 'Q', 1, refrig)
      # Avoid division by zero
      if abs(cpf_1) > 1e-9:
           cpr_1 = cpv_1 / cpf_1
      else:
           cpr_1 = 1.0 # Assume ratio is 1 if cpf is zero
  except ValueError:
      cpr_1 = 1.0

  # --- the IHX network, closed by an exact enthalpy balance ---------------
  # The two regenerators are closed by ENERGY, not by effectiveness-times-
  # cp-ratio correlations. Which stream passes through each one matters:
  #
  #   IHX-1   liquid 3h -> 12h  (1 kg)   heats the FLASHED VAPOUR, x_8h,
  #                                      from 10h to 11h
  #   IHX-2   liquid 12h -> 7h  (1 kg)   heats the SUCTION stream, which is
  #                                      what left the separator as LIQUID,
  #                                      (1 - x_8h), from 13h to 1h
  #
  # x_8h appears on both sides, so the three relations are solved as a fixed
  # point. The map is linear and strongly contracting here, so it converges
  # in a handful of steps.
  #
  # Until v0.9 IHX-2 used x_4h -- the vapour QUALITY at the evaporator inlet
  # -- where the flow SPLIT (1 - x_8h) belongs. A quality is not a flow
  # fraction, and the cycle did not close: IHX-2 took 9.19 kJ/kg out of the
  # liquid and put 21.76 kJ/kg into the vapour, and the condenser reported
  # 271.43 kJ/kg against 258.33 kJ/kg of work plus evaporator heat. That is
  # a 4.8 % creation of energy, and it inflated the COP.
  p_7h = p_condh
  dh_IHX1 = h_11h - h_10h            # per kg of flashed vapour
  dh_IHX2 = h_1h - h_13h             # per kg of suction (low-stage) flow
  x_8h = 0.3
  for _n_ihx in range(200):
      h_12h = h_3h - x_8h * dh_IHX1
      h_7h = h_12h - (1.0 - x_8h) * dh_IHX2
      x_new = (h_7h - h_9h) / (h_10h - h_9h)          # h_8h = h_7h
      converged = abs(x_new - x_8h) < 1e-12
      x_8h = x_new
      if converged:
          break
  else:
      raise RuntimeError(
          "two_stage_htheatpump_2regs: the IHX enthalpy balance did not "
          f"converge in 200 iterations (x_8h = {x_8h!r}). The pressure "
          "levels or the subcooling are probably inconsistent.")
  if not np.isfinite(x_8h) or not (0.0 <= x_8h <= 1.0):
      raise RuntimeError(
          f"two_stage_htheatpump_2regs: separator vapour fraction "
          f"x_8h = {x_8h:.6g} is outside [0, 1], so there is no physical "
          "flash split. Check T_2h, T_13h and DT_sub.")
  h_12h = h_3h - x_8h * dh_IHX1
  h_7h = h_12h - (1.0 - x_8h) * dh_IHX2
  h_8h = h_7h                        # isenthalpic expansion into the separator
  h_4h = h_9h                        # isenthalpic expansion into the evaporator
  T_7h = CP.PropsSI('T', 'H', h_7h * 1000., 'P', p_condh, refrig)
  T_12h = CP.PropsSI('T', 'H', h_12h * 1000., 'P', p_condh, refrig)

  # Both of these are now DIAGNOSTICS rather than inputs. State 1h is fixed
  # by the isentropic compression back from 2h, so the effectiveness IHX-2
  # would need in order to deliver it is an OUTPUT of the cycle, and worth
  # reading: a value above 1 would mean the regenerator cannot exist.
  x_4h = (h_4h - h_14h) / (h_13h - h_14h)
  epsilon_IHX_2 = (T_1h - T_13h) / (T_12h - T_13h)
  cpr_2 = (CP.PropsSI('C', 'P', p_evaph, 'Q', 1, refrig)
           / CP.PropsSI('C', 'P', p_condh, 'Q', 0, refrig))


  # State 7h (using the converged T_7h)
  # h_7h and s_7h were calculated inside the loop, re-calculate if needed outside
  try:
       h_7h = CP.PropsSI('H', 'T', T_7h, 'P', p_7h, refrig)/1000.
       s_7h = CP.PropsSI('S', 'T', T_7h, 'P', p_7h, refrig)
  except ValueError:
       h_7h, s_7h = np.nan, np.nan


  # State 12h (using the converged T_12h)
  # T_12h was calculated inside the loop, re-calculate if needed outside
  try:
       p_12h = p_condh
       h_12h = CP.PropsSI('H', 'T', T_12h, 'P', p_12h, refrig)/1000.
       s_12h = CP.PropsSI('S', 'T', T_12h, 'P', p_12h, refrig)
  except ValueError:
       h_12h, s_12h = np.nan, np.nan


  # State 8h (using the converged h_8h=h_7h and p_8h=p_inth)
  # h_8h was calculated inside the loop, re-calculate if needed outside
  try:
       p_8h = p_inth
       T_8h = CP.PropsSI('T', 'P', p_8h, 'H', h_8h*1000., refrig)
       s_8h = CP.PropsSI('S', 'P', p_8h, 'H', h_8h*1000., refrig)
  except ValueError:
       T_8h, s_8h = np.nan, np.nan


  # State 4h (using the converged h_4h=h_9h and p_4h=p_evaph)
  # h_4h was calculated inside the loop, re-calculate if needed outside
  try:
       p_4h = p_evaph
       T_4h = CP.PropsSI('T', 'P', p_4h, 'H', h_4h*1000., refrig)
       s_4h = CP.PropsSI('S', 'P', p_4h, 'H', h_4h*1000., refrig)
  except ValueError:
       T_4h, s_4h = np.nan, np.nan


  # HP COP
  # Ensure denominator is non-zero before division and check for NaN values
  # Need to use the final converged x_8h and h_5h, h_1h
  denominator_cop = (h_2h - h_1h) - x_8h * (h_5h - h_1h)
  if abs(denominator_cop) > 1e-9 and not np.isnan(h_2h) and not np.isnan(h_1h) and not np.isnan(x_8h) and not np.isnan(h_5h):
      hp_cop = (h_2h - h_3h) / denominator_cop
  else:
      hp_cop = float('inf') # Assign a large value if denominator is zero or near zero or NaN


  return {
      'T_1h': T_1h, 'p_1h': p_1h, 'h_1h': h_1h, 's_1h': s_1h,
      'T_2h': T_2h, 'p_2h': p_2h, 'h_2h': h_2h, 's_2h': s_2h,
      'T_3h': T_3h, 'p_3h': p_3h, 'h_3h': h_3h, 's_3h': s_3h,
      'T_4h': T_4h, 'p_4h': p_4h, 'h_4h': h_4h, 's_4h': s_4h,
      'T_5h': T_5h, 'p_5h': p_5h, 'h_5h': h_5h, 's_5h': s_5h,
      'T_6h': T_6h, 'p_6h': p_6h, 'h_6h': h_6h, 's_6h': s_6h,
      'T_7h': T_7h, 'p_7h': p_7h, 'h_7h': h_7h, 's_7h': s_7h,
      'T_8h': T_8h, 'p_8h': p_8h, 'h_8h': h_8h, 's_8h': s_8h,
      'T_9h': T_9h, 'p_9h': p_9h, 'h_9h': h_9h, 's_9h': s_9h,
      'T_10h': T_10h, 'p_10h': p_10h, 'h_10h': h_10h, 's_10h': s_10h,
      'T_11h': T_11h, 'p_11h': p_11h, 'h_11h': h_11h, 's_11h': s_11h,
      'T_12h': T_12h, 'p_12h': p_12h, 'h_12h': h_12h, 's_12h': s_12h,
      'T_13h': T_13h, 'p_13h': p_13h, 'h_13h': h_13h, 's_13h': s_13h,
      'hp_cop': hp_cop,
      'p_evaph': p_evaph,
      'p_condh': p_condh, # Changed from p_evaph to p_condh
      'p_inth': p_inth,
      'epsilon_IHX_1': epsilon_IHX_1,
      'epsilon_IHX_2': epsilon_IHX_2,
      'x_8h': x_8h,
  }

# @title
#import numpy as np
#import CoolProp.CoolProp as CP
#import matplotlib.pyplot as plt


# ==========================================================================
# network
# ==========================================================================


########################################
########################################
########################################


def compute_U_i(h_i, r_i, r_e, k_w, Rf_i_prime, k_m, delta, L_tube, fin_t, fin_L, num_fins):
    """
    Computes the overall heat transfer coefficient based on internal surface area,
    including thermal resistances for internal convection, wall conduction, fouling,
    and the outer finned surface with a melt layer and fin efficiency.

    Args:
        h_i: Internal convective heat transfer coefficient [W/m²·K].
        r_i: Internal radius of the tube [m].
        r_e: External radius of the tube [m].
        k_w: Casing wall thermal conductivity (also used for fin conductivity) [W/m·K]. # Updated docstring
        Rf_i_prime: Internal fouling resistance [m²·K/W].
        k_m: PCM thermal conductivity [W/m.K].
        delta: Melt layer thickness [m].
        L_tube: Length of the tube [m].
        fin_t: Thickness of each fin [m].
        fin_L: Length of each fin extending radially outwards [m].
        num_fins: The number of fins.

    Returns:
        Overall heat transfer coefficient based on internal surface area [W/m²·K].
    """
    R_conv_internal = 1.0 / h_i
    R_wall = (r_i / k_w) * log(r_e / r_i)
    R_fouling = Rf_i_prime

    # Calculate h_e (heat transfer coefficient for the outer layer)
    if delta > 1e-9 and r_e > 1e-9: # Check for non-zero delta and r_e
        log_term = np.log(1 + delta / r_e)
        if log_term > 1e-9: # Check for non-zero log term
            h_e = k_m / r_e / log_term
        else:
             # If log_term is near zero, delta/r_e is very small, meaning very thin layer.
             # This would imply a large h_e, but setting it too large might cause issues.
             # Assign a large, but finite value. Or, consider the limit as delta approaches 0.
             # As delta -> 0, log(1 + delta/r_e) ~ delta/r_e. h_e ~ k_m / r_e / (delta/r_e) = k_m/delta.
             # So the previous definition is consistent with the limit.
             # Let's revert to the previous definition for small delta, or handle it as an infinite h_e (zero resistance).
             # Given the error handling downstream, a large finite number for h_e is safer than infinity here.
             h_e = k_m / delta # Use the previous definition for very small delta
    else:
        # If delta is zero, there is no melt layer, so resistance is infinite
        h_e = 1e9 # Assign a small value, leading to large resistance

    # Calculate fin parameters for efficiency
    fin_Lc = fin_L + fin_t / 2.0
    # Ensure h_e is non-zero or negative before taking sqrt, use k_w for fin conductivity
    if h_e > 1e-9 and k_w > 1e-9 and fin_t > 1e-9:
         fin_m = np.sqrt(2 * h_e / k_w / fin_t) # Used k_w instead of KW
    else:
         fin_m = 1e9 # Assign a large value if denominator is near zero


    # Calculate fin efficiency
    fin_m_Lc = fin_m * fin_Lc
    if fin_m_Lc != 0:
        fin_eff = np.tanh(fin_m_Lc) / fin_m_Lc
    else:
        fin_eff = 1.0 # Efficiency is 100% if fin_m * fin_Lc is zero

    # Calculate fin surface area
    A_f = L_tube * (2 * fin_L + fin_t)

    # Calculate total outer surface area
    A_t = 2 * L_tube * (np.pi * r_e + num_fins * fin_L)

    # Calculate overall surface efficiency
    # Ensure A_t is not zero to avoid division error
    if A_t > 0:
        eta_o = 1 - (num_fins * A_f / A_t) * (1 - fin_eff)
    else:
        eta_o = 1.0 # If total area is zero, efficiency is irrelevant, assume 1

    # Calculate the outer perimeter of the finned tube (used in R_outer_layer calculation)
    perim_T = 2 * pi * r_e + 2 * num_fins * fin_L

    if eta_o > 1e-9 and h_e > 1e-9: # Check if eta_o and h_e are effectively non-zero
         R_outer_layer = (2 * pi * r_i) / (perim_T * eta_o * h_e)
    else:
        R_outer_layer = float('inf') # Assign infinite resistance if heat transfer is blocked

    R_total = R_conv_internal + R_wall + R_fouling + R_outer_layer

    if R_total != 0 and not np.isinf(R_total):
        U_i = 1.0 / R_total
    else:
        U_i = 0.0 # If total resistance is infinite or zero, U_i is 0.0 (no heat transfer)

    return U_i


########################################
########################################
########################################


# ==========================================================================
# front_closed_form
# ==========================================================================


#####################################
#####################################
#####################################


# ==========================================================================
# march_legacy
# ==========================================================================


########################################
########################################
########################################


# ==========================================================================
# hydraulics
# ==========================================================================


def calculate_pressure_drop(geom_par_vector, m_dot_well_tube, fluid, T_inlet,
                            T_outlet, P_inlet, n_bends=2):
    """
    Calculates the pressure drop and pumping power in a single tube of the hairpin heat exchanger.

    Args:
        geom_par_vector: An array or list containing [L_well, L_tube, D_well, r_i, r_e, D_i_tube, D_e_tube, fin_t, fin_L, num_fins, num_tubes].
                         We will use L_tube and D_i_tube (2*r_i) from this vector.
        m_dot_well_tube: Mass flow rate of the fluid through a single tube [kg/s].
        fluid: Name of the working fluid.
        T_inlet: Inlet temperature of the fluid [K].
        T_outlet: Outlet temperature of the fluid [K].
        P_inlet: Inlet pressure of the fluid [Pa].


    Returns:
        A tuple containing:
            - Pressure drop in the tube [Pa].
            - Pumping power [W]. # Added pumping power to return values
    """
    L_well, L_tube, D_well, r_i, r_e, D_i_tube, D_e_tube, fin_t, fin_L, num_fins, num_tubes = geom_par_vector
    L = L_tube # Length of the tube
    D_i = D_i_tube # Inner diameter of the tube
    r_i_tube = D_i / 2.0 # Inner radius

    # Average temperature and pressure for fluid properties
    T_avg = 0.5 * (T_inlet + T_outlet)
    # Assume average pressure is close to inlet pressure for property calculation for now
    # A more rigorous approach might iterate, but this is a good starting point.
    P_avg = P_inlet # Approximation


    try:
        # Get fluid properties at average conditions
        rho = CP.PropsSI("D", "T", T_avg, "P", P_avg, fluid)
        mu = CP.PropsSI("V", "T", T_avg, "P", P_avg, fluid) # Dynamic viscosity
        # You might also need Specific Heat (Cp) and Thermal Conductivity (k) for friction factor correlations,
        # but for basic pressure drop (Darcy-Weisbach), rho and mu are key.

    except ValueError:
        print(f"Warning: Could not retrieve fluid properties for {fluid} at T={T_avg:.2f}K, P={P_avg:.2f}Pa.")
        return float('nan'), float('nan') # Return NaN for both if properties fail


    # Calculate flow velocity
    A_flow = np.pi * r_i_tube**2
    if rho * A_flow > 1e-9:
        u = m_dot_well_tube / (rho * A_flow)
    else:
        return 0.0, 0.0 # No flow, no pressure drop and no pumping power

    # Calculate Reynolds number
    if mu > 1e-9 and D_i > 1e-9:
        Re = rho * u * D_i / mu
    else:
        return 0.0, 0.0

    # Calculate friction factor (Darcy friction factor)
    f = 0.0
    if Re < 2000: # Laminar flow
        f = 64.0 / Re
    else: # Turbulent flow (using Haaland correlation as an example)
        # More accurate correlations like Colebrook-White could be used if roughness is known.
        # For simplicity, using an approximation that doesn't require iteration for now.
        # If pipe roughness (epsilon) is available, a better correlation can be used.
        # Assuming smooth pipe for now (epsilon = 0).
        # The Haaland correlation: 1/sqrt(f) = -1.8 * log10((epsilon/D_i)/3.7 + (6.9/Re))
        # With epsilon=0: 1/sqrt(f) = -1.8 * log10(6.9/Re)
        # f = (1 / (-1.8 * log10(6.9/Re)))**2

        # Using a simpler turbulent correlation like Blasius for smooth pipes (Re < 100,000)
        # or a transition correlation for wider Re range.
        # Let's use a common approximation for smooth pipes for simplicity:
        if Re <= 1e5: # Blasius correlation
             f = 0.3164 / (Re**0.25)
        else: # For higher Re, use a different correlation, e.g., from Moody chart or a more general formula
             # Swamee–Jain equation (explicit approximation of Colebrook–White)
             # Requires pipe roughness (epsilon). Let's assume a typical value for smooth drawn tubing.
             # Example epsilon for drawn tubing might be 0.0015 mm = 1.5e-6 m
             # If epsilon is not a parameter, we have to stick to correlations that don't need it,
             # or make an assumption. Let's assume smooth pipe and use Blasius/extend it.
             # A more robust approach would take epsilon as an input parameter.
             # For now, extending Blasius or using a simple fit:
             # Let's use a piecewise approach based on Re.
             # For very high Re, f approaches a constant for smooth pipes.
             # A simple fit across a wider range might be better.
             # Let's stick to the Blasius for Re <= 100,000 and
             # a rough approximation for higher Re if needed, or add roughness as parameter.

             # For now, let's refine the Blasius range and consider a higher Re approximation if necessary.
             # The Blasius correlation is typically for Re up to 10^5.
             # For 10^5 < Re < 10^6, a common approximation is f = 0.0032 + 0.221 / Re^0.237.
             # For Re >= 10^6, f ~ 0.018 (fully turbulent smooth pipe).
             # Let's add a simple transition:
             # NIKURADSE (1932), f = 0.0032 + 0.221 Re^-0.237, quoted as
             # valid to about 3e6 for smooth pipe. It used to hand over at
             # 1e6 to a constant 0.018 -- a 55 % JUMP, since the formula
             # gives 0.0116 there. Nothing in the packs reaches 1e6 (the
             # reference legs are at 1.2e5 and 2.1e5, and the worst case
             # in the study, one fin tube at the shoe, is still below it),
             # but an n_ft = 1 point or a larger downcomer could cross it
             # and the pumping would step. The formula is extended to its
             # stated range instead, and beyond that the run FAILS rather
             # than quietly using a constant.
             if Re <= 3e6:
                  f = 0.0032 + 0.221 / Re**0.237
             else:
                  raise ValueError(
                      f'Re = {Re:.3g} is beyond the smooth-pipe '
                      f'correlations in this model (valid to 3e6). The '
                      f'friction factor would be an extrapolation; set a '
                      f'larger flow area or add a rough-pipe form.')


    # Frictional pressure drop (Darcy-Weisbach equation)
    # Ensure D_i is non-zero before division
    if D_i > 1e-9:
        delta_P_friction = f * (L / D_i) * (0.5 * rho * u**2)
    else:
         delta_P_friction = 0.0


    # Minor losses (e.g., bends, entrance/exit)
    # These depend on fittings and geometry. Assuming a simple hairpin with two 180-degree bends
    # and entrance/exit losses. Need minor loss coefficients (K).
    # For simplicity, assuming K_bend for 180 deg return bend ~ 2.2 (for flanged or welded)
    # K_entrance ~ 0.5, K_exit ~ 1.0
    # Total minor loss coefficient K_total = K_entrance + 2 * K_bend + K_exit
    # K_total = 0.5 + 2 * 2.2 + 1.0 = 5.9 (example values)
    # Minor loss pressure drop: delta_P_minor = K_total * 0.5 * rho * u**2

    # Let's include minor losses as a parameter or use assumed values.
    # For this function, let's assume typical minor loss coefficients for a hairpin.
    K_entrance = 0.5 # Sharp-edged entrance
    K_bend_180 = 2.2 # 180 degree standard elbow/bend
    K_exit = 1.0 # Exit into a large reservoir/pipe

    # For one HAIRPIN: two tubes and two bends. BOREAS is not a hairpin --
    # its fin tube and its downcomer are two legs sharing ONE bottom
    # manifold -- so borehole_pumping asks for the bend allowance on the
    # fin-tube leg only and passes n_bends = 0 for the downcomer.
    # Measured at the reference case: the duplicate was worth 0.06 points
    # of f_pump and nothing on eta_RTE or e' to four decimals, friction
    # over L/D ~ 3e4 being three orders larger than any minor loss.
    K_total = K_entrance + n_bends * K_bend_180 + K_exit

    delta_P_minor = K_total * 0.5 * rho * u**2

    # Total pressure drop
    delta_P_total = delta_P_friction + delta_P_minor

    # Calculate pumping power [W]
    # Ensure rho is non-zero before division
    if rho > 1e-9:
        pumping_power = delta_P_total * m_dot_well_tube / rho
    else:
        pumping_power = 0.0


    return delta_P_total, pumping_power # Return both pressure drop and pumping power

# @title

# Function to evaluate the energy ratio for a given value of N_wells (Charging)


# ==========================================================================
# solvers
# ==========================================================================


#########################################
#########################################
#########################################

# Function to evaluate the energy ratio for a given value of m_dot_d_well1 (Discharging)


#########################################
#########################################
#########################################


#########################################
#########################################
#########################################


#############################################
#############################################
#############################################


# ==========================================================================
# config
# ==========================================================================
# One flat Case object. v0.1 passed sixteen positional arguments, which is how
# the PCM conductivity ended up in the slot meant for the tube wall for months
# without anything contradicting it. Named fields make that mistake visible.

from dataclasses import dataclass, replace

FT = 0.3048
IN = 0.0254


@dataclass(frozen=True)
class Case:
    # ---- PCM (Saher 2024 trimodal) ----
    T_m_C: float = 150.0          # melting temperature            [C]
    rho_s: float = 1550.0         # solid density                  [kg/m3]
    rho_l: float = 1450.0         # liquid density                 [kg/m3]
    cp_s: float = 1280.0          # solid specific heat            [J/kg/K]
    cp_l: float = 1800.0          # liquid specific heat           [J/kg/K]
    k_s: float = 0.60             # solid conductivity             [W/m/K]
    k_l: float = 0.45             # liquid conductivity            [W/m/K]
    h_m: float = 380_000.0        # latent heat of fusion          [J/kg]
    cost_per_kWh: float = 20.0

    # ---- borehole and tube geometry ----
    L_ft: float = 5000.0          # well depth                     [ft]
    D_in: float = 7.0             # borehole diameter              [in]
    r_i: float = 0.01623          # tube inner radius              [m]
    r_e: float = 0.02108          # tube outer radius              [m]
    fin_t: float = 0.0015         # fin thickness                  [m]
    fin_L: float = 0.0035         # fin radial length              [m]
    num_fins: int = 24            # fins per fin tube
    # ---- THE DOWNCOMER GEOMETRY. This is what BOREAS is for. ------------
    # THUMS put n hairpins in the hole: each a U-tube, down one leg and back
    # up the other, both legs exchanging with the PCM. That geometry is
    # self-contradictory and the contradiction is quantitative.
    #
    # Charge water enters at T_m + DT_4C_M and glides DT_3C_2C down, so the
    # RETURN leg runs ~45 K BELOW the melting point. Graded by depth it
    # refreezes what the down leg just melted. So THUMS graded the cascade
    # along the TUBE PATH instead -- which works, but requires two different
    # melting points to coexist at the same depth, centimeters apart, with
    # contiguous PCM and no wall between them. Conduction between those legs
    # is 43 kW at one hairpin and 92 kW at three, against a ~120 kW duty, and
    # it is a short circuit straight across the cascade the model exists to
    # represent. THUMS never had that term.
    #
    # BOREAS separates the two jobs. n_ft FIN TUBES all run DOWNWARD and all
    # exchange; ONE INSULATED DOWNCOMER returns the whole flow without
    # exchanging. They meet in a manifold at the bottom of the same hole --
    # no lateral, no new drilling, the same class of object as the U-bend it
    # replaces. Consequences:
    #
    #   z becomes DEPTH, one-to-one (L_tube = L_well, not twice it)
    #   the legs are PARALLEL, not series
    #   the cascade grades with depth, so the PCM is POURABLE in layers
    #   the cascade runs HOT AT THE BOTTOM, which aligns it with the
    #     geothermal gradient instead of fighting it -- see cascade_hot_bottom
    #
    # Sizing the downcomer for equal velocity keeps the pressure drop
    # identical to the hairpin's: same velocity everywhere, same total path
    # 2*L_well. The nested-coaxial alternative costs 3.3x on f_pump, which is
    # the one quantity Study 2 showed governs efficiency below ground.
    n_ft: int = 3                 # FIN TUBES per borehole (all downward)
    t_ins: float = 0.008          # downcomer insulation thickness [m]
    k_ins: float = 0.007          # downcomer insulation, VIT       [W/m/K]
    k_wall: float = 45.0          # steel conductivity             [W/m/K]
    Rf_i: float = 1e-3            # internal fouling resistance    [m2K/W]

    # ---- the formation. THUMS hypothesis H6 said the wall was adiabatic,
    # which made eta_storage identically 1 -- a number no reviewer of a
    # borehole storage paper will accept. These are what break it.
    T_surface_C: float = 18.0     # ground surface / mean annual air [C]
    grad_K_per_km: float = 45.0   # geothermal gradient              [K/km]
    k_rock: float = 2.5           # formation conductivity        [W/m/K]
    rho_cp_rock: float = 2.34e6   # formation volumetric heat cap [J/m3/K]
    # The annulus is ONE conductivity over the whole length: there is no
    # top-of-cement position in the model, only k_cement in
    # R_cem = ln(r_bore/r_casing_o)/(2 pi k_cement). The documented
    # completion is Class G cement below a top of cement with mud above,
    # and both are 0.7 W/m/K, so the annulus is thermally uniform and the
    # top-of-cement position drops out. 0.7 is the upper end of the
    # 0.5-0.7 reported for Class G (Won 2015; Ichim 2018).
    k_cement: float = 0.7         # annulus, uniform              [W/m/K]
    r_casing_o: float = 0.098     # casing outer radius             [m]
    r_bore: float = 0.110         # drilled hole radius             [m]
    t_operation_yr: float = 10.0  # age of the field when evaluated  [yr]
    # The gradient and the source temperature are specified INDEPENDENTLY and
    # checked against each other, never derived one from the other. 60 C at
    # the wellhead is a LOWER BOUND on the formation temperature -- the water
    # cools coming up -- and the producing wells may be deeper than these.
    # Writing that bound as an equality is what an earlier draft did, and it
    # threw away exactly the physics the deficit represents.
    L_producing_ft: float = 8000.0   # depth the geothermal water comes from

    # ---- THE WELL FIELD IS A CONE, NOT AN ARRAY -------------------------
    # THUMS wellheads sit on 6 ft centers with 12 ft between double rows,
    # and over 1200 wells are drilled DIRECTIONALLY from four islands with
    # drift angles to 84 deg, reaching 6500 acres. So the spacing is not a
    # number, it is a function of depth: meters apart at the wellhead, of
    # order 150 m at reservoir depth.
    #
    # That matters more than any other single parameter here. Neighbors
    # merge thermally in r^2/alpha, which at 6 ft is 36 DAYS and at 160 m is
    # eight centuries. The shallow section is one thermal body within a month
    # and barely loses at all; the deep section is a field of isolated wells.
    #
    # And the two effects run opposite ways with depth. Shallow rock is cold,
    # so the driving difference is largest exactly where the shielding is
    # strongest; deep rock is hot, so the driving difference is smallest
    # exactly where the wells stand alone. Neither a single spacing nor a
    # single resistance can represent that, which is why B is B(z).
    #
    # n_wells_field = None recovers the isolated well, which is the
    # conservative bound and the v1.0 default behavior.
    n_wells_field: float = None   # wells sharing the formation; None = alone
    B_surface_m: float = 1.83     # wellhead spacing, 6 ft                [m]
    z_kickoff_m: float = 300.0    # vertical above the kickoff point      [m]
    fan_angle_deg: float = 45.0   # effective half-angle of the fan     [deg]
    # ---- HOW THE WELLHEADS ARE PACKED, WHICH IS NOT THE SAME AS HOW FAR
    # APART THEY ARE.  v1.0-v1.2 assumed a compact disc, R0 = B0 sqrt(N/pi),
    # because that is the obvious way to turn a count and a spacing into a
    # footprint. It is also the most flattering way: a disc MINIMISES
    # perimeter for a given count, and the shielded resistance goes as
    # 1/perimeter.
    #
    # The THUMS islands are nothing like a disc. Wellheads sit in LINEAR
    # DOUBLE ROWS along the island rim -- 6 ft center to center, 12 ft
    # between the rows -- and the field is split across FOUR islands, each
    # with its own perimeter. That presents 1.86 m of perimeter per well
    # against 0.205 m for the disc: 9.1x more, hence 9.1x the loss.
    #
    # The spacing was never wrong. The packing was. Reported island-case
    # losses of ~1 % become ~9 %.
    packing: str = "disc"         # 'disc' | 'rows' | 'ring'
    n_islands: int = 1            # clusters the field is split across
    row_gap_m: float = 3.66       # 12 ft between the double rows        [m]

    # ---- power block and operating conditions ----
    fluid: str = "cyclopentane"   # ORC working fluid
    fluid2: str = "Water"         # borehole secondary fluid
    refrig: str = "cyclopentane"  # heat-pump refrigerant
    P: float = 1e6                # secondary-fluid pressure       [Pa]
    W_dot_el_out: float = 1000.0  # ORC net electrical output      [kW]
    # ---- machine efficiencies -------------------------------------------
    # ISENTROPIC efficiencies, applied INSIDE the cycles from v0.12: the
    # expansions, compressions and pumpings move the state points themselves.
    # Before v0.12 the state points were isentropic and 0.85 was applied as a
    # multiplier on the work downstream. That is accurate for the efficiency
    # -- 0.6 % at the design point -- but it leaves every state point ideal,
    # so the turbine exit, the regenerator split, the composite curves and
    # the evaporator pinch were all the reversible cycle's. It also makes a
    # component-wise exergy balance impossible: an isentropic machine
    # destroys nothing, so the lost work appears nowhere. See DN-21.
    #
    # Setting all three to 1.0 recovers the pre-v0.12 cycles exactly.
    eta_turb_s: float = 0.85      # ORC turbines, isentropic
    eta_pump_s: float = 0.85      # ORC pumps, isentropic
    eta_comp_s: float = 0.85      # heat-pump compressors, isentropic
    # ELECTRICAL / MECHANICAL, and correctly applied outside the working
    # fluid: these are not thermodynamic irreversibilities of the cycle.
    ElG_eff: float = 0.95         # ORC generator
    ElH_eff: float = 0.95         # compressor motor
    # RETIRED at v0.12. These were the isentropic efficiencies applied as
    # downstream multipliers; keeping them would double-count. validate_case
    # raises if a Case still names either.
    Turb_eff: float = None
    Comp_eff: float = None
    T_sink_C: float = 20.0
    # Approach at the ORC condenser, measured to the sink OUTLET. With
    # DT_sink_glide = 0 the sink is an infinite reservoir and outlet = inlet,
    # which is the model as built. Referencing it to the inlet -- as this did
    # before v0.9b -- is the same wrong-end pattern that made DT_3A_13H
    # unsafe; it was merely masked by the reservoir assumption. See DN-18.
    DT_E_sink: float = 5.0
    # Temperature rise of the sink stream through the ORC condenser. Zero is
    # the reservoir idealisation. Give it a value and the condenser acquires
    # an INTERIOR pinch at the desuperheat corner, ~93 % of the duty, which
    # crosses at about 5.4 K -- the ORC condenser is not structurally safe
    # the way the HTHP condenser is.
    DT_sink_glide: float = 0.0
    DT_2D_3E: float = 12.0
    # Minimum water-to-working-fluid gap ANYWHERE in the ORC evaporator, not
    # just at its hot end. DT_2D_3E alone cannot keep the two composite curves
    # apart, because the ORC takes most of its heat at one temperature while
    # the water glides; see `orc_pinch`. Raise this and the ORC boils lower
    # and yields less; lower it and the exchanger grows. See DN-15.
    DT_pinch_ORC: float = 5.0
    # Discharge-inlet subcooling below the COLDEST cascade layer, state 4D
    # in the plant diagram; the symmetric partner of DT_4C_M. See DN-8.
    #
    # RENAMED from DT_M_1D at round 25. Figure 1 now numbers the water loop
    # so that 1 is the FIELD OUTLET and 4 the FIELD INLET in both
    # half-cycles, and this approach is at the discharge field inlet. The
    # old name is still accepted (see __post_init__) and warns, so cached
    # runs keep loading; it is NOT part of the cache key, which is built
    # from the canonical name only.
    DT_M_4D: float = 10.0
    DT_M_1D: float = None       # deprecated alias for DT_M_4D
    T_source_C: float = 60.0
    DT_4C_M: float = 10.0

    # ---- THE HEAT-PUMP SOURCE LOOP --------------------------------------
    # The source is no longer a hypothetical geothermal producer, and it is
    # no longer the island's co-produced water. Both were rejected on the
    # same ground: they are SCARCE. A scarce source forces a large
    # DT_3A_4A to hold the flow down, and that drags the evaporator down by
    # exactly as much as a cold source would -- 60 C water across 15 K
    # evaporates at 40 C, while 17 C seawater across 5 K evaporates at 7 C.
    # The warm source buys 27 % on COP and, for the produced water, costs
    # 70 % of the island's well slots to get it.
    #
    # Seawater is unlimited, so DT_3A_4A is free to be small. It is NOT
    # free of pumping: the intake moves ~3.8 m3/s at the reference, and
    # that parasitic belongs inside eta_RTE rather than in a footnote.
    # ---- INTERMEDIATE PRESSURES of the two cycles -----------------------
    # p_int = p_low + x (p_high - p_low). None keeps each cycle's own rule:
    # the ORC the ARITHMETIC mean (x = 0.5), the HTHP the GEOMETRIC mean,
    # which on this linear scale is NOT 0.5. Set either to sweep it.
    # ---- THE IDLE PERIODS BETWEEN THE HALF-CYCLES -----------------------
    # The day is t_ch + t_dc + whatever is left, and what was left used to
    # be skipped: the store jumped from the end of charge to the start of
    # discharge with no time elapsed, so the loss over those hours was
    # missing from Q_loss and from every indicator built on it.
    #
    # t_idle = None means "split whatever the day has left", one period
    # after charge and one after discharge. idle_mode='none' reproduces
    # v2.8 exactly and is what the regression check uses.
    t_idle: float = None          # h, EACH period; None -> auto from the day
    idle_mode: str = 'simulate'   # 'simulate' | 'none'
    day_h: float = 24.0

    x_int_orc: float = None
    x_int_hp: float = None

    source: str = 'seawater'      # 'seawater'|'produced_water'|'geothermal'
    rho_source: float = 1025.0    # kg/m3, seawater
    source_loop_dp_bar: float = 1.0   # intake, screens, evaporator, outfall
    source_pump_eff: float = 0.75     # wire-to-water

    # ---- THE ORC SINK LOOP, on the same seawater ------------------------
    # The condenser was an INFINITE RESERVOIR: DT_sink_glide = 0, outlet =
    # inlet, so T_1e = T_sink + DT_E_sink and the sink cost nothing. That
    # is only tenable while the pump is free. Once it is charged, a zero
    # glide means an infinite flow, and the design has the same trade the
    # source side has: a small glide keeps the condenser cold and costs
    # pumping, a large one saves pumping and raises T_1e by the same
    # kelvin it adds. DT_sink_glide is now a real design variable.
    sink_loop_dp_bar: float = 1.0
    sink_pump_eff: float = 0.75
    rho_sink: float = 1025.0

    # ---- LINE LOSSES between the plant and the field --------------------
    # Figure 1 draws a coil on the charge line 3c -> 4c and on the discharge
    # line 1d -> 2d. Each stands for the heat loss and the pumping loss of
    # the run between the plant and the wellhead.
    #
    # NOT COMPUTED YET. They are placeholders and default to ZERO, so that
    # T_3c = T_4c and T_2d = T_1d exactly and every number in the packs is
    # what it was before the renumbering. V19 asserts those identities.
    line_ch_dT: float = 0.0       # 3c -> 4c temperature drop          [K]
    line_ch_dp: float = 0.0       # 3c -> 4c pressure drop            [Pa]
    line_dc_dT: float = 0.0       # 1d -> 2d temperature drop          [K]
    line_dc_dp: float = 0.0       # 1d -> 2d pressure drop            [Pa]
    # How far the SOURCE stream is cooled in the heat-pump evaporator. This is
    # the specification: it sets the source flow needed for a given duty.
    DT_3A_4A: float = 10.0
    # Minimum approach in the heat-pump evaporator, at its COLD end -- where
    # the source leaves and the refrigerant enters. The evaporating
    # temperature is DERIVED from it,
    #     T_13h = T_source_C - DT_3A_4A - DT_pinch_HPE,
    # so the approach can no longer be negative by arithmetic. Unlike the ORC
    # evaporator the cold stream here is isothermal with NO preheat kink, so
    # the gap is monotonic in Q and an end approach is genuinely sufficient:
    # no composite-curve search is needed. See DN-17.
    DT_pinch_HPE: float = 5.0
    # RETIRED at v0.9. The evaporating temperature used to be set by
    # T_13h = T_source_C - DT_3A_13H, independently of DT_3A_4A, so the
    # approach was the DIFFERENCE of two free fields and nothing checked its
    # sign. Left here only so that an old Case naming it fails loudly in
    # validate_case rather than being silently ignored.
    DT_3A_13H: float = None
    DT_2H_3C: float = 10.0
    DT_sub: float = 2.0
    # The CHARGING water glide, across the HTHP condenser. Because
    # T_3c = T_4c -- the water leaves the condenser and enters the borehole
    # with no state change between -- this is identically the borehole
    # charging glide. That equality is energy conservation on a closed loop,
    # not an assumption, and cannot be relaxed.
    DT_3C_2C: float = 55.0        # charging water glide           [K]
    # The DISCHARGING water glide, across the ORC evaporator, and likewise
    # identically the borehole discharging glide. Until v0.9b this was not a
    # field at all: both mass flows were divided by DT_3C_2C, which silently
    # asserted that the two half-cycles glide by the same amount. They need
    # not. None keeps the old behavior exactly. See DN-18.
    DT_3D_2D: float = None        # discharging water glide        [K]
    # The cascade grading parameter. Layer spacing is DT_cascade / N_lay and
    # the top-to-bottom SPAN is DT_cascade * (N_lay - 1) / N_lay, one layer
    # short of DT_cascade itself.
    #
    # Setting this equal to the charging glide -- which None does -- is not
    # arbitrary: it is the unique choice that makes the approach between the
    # water and the layer it is melting equal to DT_4C_M at the LEADING FACE
    # of every layer, decaying to DT_4C_M - DT_cascade/N_lay at each trailing
    # face. That sawtooth is the cascade.
    #
    # Prescribing it independently is legitimate, but it is bounded BELOW,
    # not above: the approach at the far end of the well is
    #     charging : span - (glide_ch - DT_4C_M)
    #     discharge: span - (glide_dc - DT_M_4D)
    # so a span SMALLER than glide minus the approach inverts the driving
    # difference at that end. At the default the margin is only 3.889 K.
    DT_cascade: float = None      # cascade grading parameter      [K]
    t_ch: float = 10.0            # charging duration              [h]
    t_dc: float = 10.0            # discharging duration           [h]

    # ---- the geothermal resource (new at v0.13) --------------------------
    # Until v0.13 the 60 C source was effectively unlimited: the model drew
    # whatever flow the duty needed and no output recorded how much that was.
    # The exergy audit makes it a priced resource, so the flow is now reported
    # and converted into a well count.
    #
    # m_geo is NOT an input. It is DERIVED, exactly as N_wells is:
    #     m_geo = Q_src / (h(T_source) - h(T_source - DT_3A_4A))
    # and the uncertainty is carried by the per-well yield below, which is a
    # field-measured quantity, rather than by a plant-level flow that nobody
    # has ever measured. See DN-22.
    #
    # PROVISIONAL DEFAULT. 12.5 kg/s is the producer in the 4500 m / 206 C
    # repurposed-well case study, which is a hotter and deeper well than ours;
    # the published range for repurposed oil and gas wells runs from about
    # 1 kg/s up to that figure, and co-produced water from depleted fields can
    # exceed it. Treat N_geo as proportional to this number, not as a result.
    m_dot_geo_well: float = 12.5  # per producing well             [kg/s]
    # Lower bound on the reinjection temperature, T_4a = T_source - DT_3A_4A.
    # This is what bounds DT_3A_4A from ABOVE, and the second law wants to run
    # against that bound: raising DT_3A_4A cuts m_geo far faster than it cuts
    # COP. The default is deliberately slack -- it is a placeholder for a
    # formation constraint (silica scaling, injectivity) that the project does
    # not yet have a number for. validate_case warns, it does not raise.
    T_reinject_min_C: float = 25.0
    # Which exergy is charged for the geothermal stream.
    #   'resource'  everything the well lifts, relative to T_sink, with the
    #               reinjected stream booked as a named LOSS attributed to no
    #               component. The right convention for a dedicated well, which
    #               is paid for whether or not its exergy is used.
    #   'stripped'  only what the evaporator removes; reinjection is free.
    # The two give OPPOSITE guidance on DT_3A_4A: 'stripped' falls monotonically
    # and pushes the design toward the largest possible well count, 'resource'
    # has an interior optimum near 20 K. See DN-23.
    exergy_convention: str = "resource"
    loss_surplus: float = 0.05    # assumed storage loss, lambda
    N_lay: int = 9                # PCM layers along the well

    # ---- numerics ----
    n_segments: int = 100
    n_times: int = 40
    delta_max: float = 0.5
    delta_tol: float = 1e-5
    delta_maxiter: int = 20
    N_wells_bracket: tuple = (3.0, 120.0)
    ratio_bracket: tuple = (0.2, 2.0)
    tol_Q_ratio: float = 1e-3
    max_iterations: int = 40

    # ---- switches ----
    # The ONLY remaining switch. True carries PCM enthalpy (latent + sensible in
    # both phases), so a fully melted segment superheats and a fully solid one
    # subcools. False pins the PCM at T_m, which is retained not as a model but
    # as the control case for the self-levelling comparison: it isolates what
    # the sensible branches do, at a fixed well count.
    sensible_heat: bool = True

    # Which material lies between the tube and the phase front, and therefore
    # which shell the heat has to cross. See `conduction_shell` and DN-13.
    #   'directional'  melting -> liquid shell (k_l); freezing -> frozen
    #                  shell (k_s). The physical arrangement in both directions.
    #   'melt_side'    the melted thickness always, with k_l whenever any melt
    #                  exists. Correct while melting, INVERTED while freezing.
    #                  Retained only to reproduce results published before the
    #                  correction; it is not a model.
    front_geometry: str = "directional"

    # ---- derived ----
    @property
    def glide_dc_spec(self):
        """Specified discharging glide; falls back to the charging glide."""
        return self.DT_3C_2C if self.DT_3D_2D is None else self.DT_3D_2D

    @property
    def cascade_param(self):
        """Cascade grading parameter; falls back to the charging glide."""
        return self.DT_3C_2C if self.DT_cascade is None else self.DT_cascade

    @property
    def cascade_span(self):
        """Top-to-bottom melting range, one layer short of cascade_param."""
        return self.cascade_param * (self.N_lay - 1) / self.N_lay

    @property
    def T_m(self):
        return self.T_m_C + 273.15

    @property
    def L_well(self):
        return self.L_ft * FT

    @property
    def L_tube(self):
        """Developed length of one fin tube.

        EQUAL TO L_well, where THUMS had twice it. The fin tubes only go
        down; the return is the downcomer and does not exchange. This single
        line is what makes z a depth rather than a path length, and so what
        lets a depth-dependent formation temperature be attached at all.
        """
        return self.L_well

    @property
    def r_id_bore(self):
        """Downcomer bore, sized so its velocity matches the fin tubes'.

        The downcomer carries the WHOLE flow that the n_ft tubes carry
        between them, so equal velocity means equal total area. That is the
        condition under which this geometry has the same pressure drop as the
        hairpin it replaces -- same velocity everywhere, same total path.
        """
        return self.r_i * np.sqrt(float(self.n_ft))

    @property
    def r_id_outer(self):
        """Downcomer outside radius: bore + wall + insulation."""
        wall = self.r_e - self.r_i
        return self.r_id_bore + wall + self.t_ins

    @property
    def D_well(self):
        return self.D_in * IN

    @property
    def V_borehole(self):
        return np.pi * self.D_well ** 2 / 4.0 * self.L_well

    @property
    def V_well(self):
        """PCM volume in one borehole: the hole, less tubes and fins."""
        A_tube = np.pi * self.r_e ** 2
        A_fins = self.num_fins * self.fin_t * self.fin_L
        A_id = np.pi * self.r_id_outer ** 2
        occupied = self.n_ft * (A_tube + A_fins) + A_id
        return self.V_borehole - occupied * self.L_well

    @property
    def rho_latent(self):
        """Density for the latent inventory, the SAME in both directions.

        The front is carried as a solid-equivalent area, so V_melt/V_well is a
        melted MASS fraction and the energy banked on charging equals the energy
        available on discharging. Using rho_l to melt and rho_s to freeze -- the
        directional choice that is correct for the conductivity k_m -- would
        return rho_s/rho_l = 1.069 times more energy than went in.
        """
        return self.rho_s

    @property
    def r_cell(self):
        """Radius of the equivalent PCM cell owned by ONE tube leg.

        The n_ft fin tubes share the PCM-bearing cross-section equally. The
        downcomer occupies area but exchanges no heat with the PCM it
        displaces, so it is SUBTRACTED before the division rather than
        counted as a participant -- a fin tube's cell is bounded by its
        neighbors and by the downcomer's outer wall, not by a share of it.

            pi r_cell^2 = (pi R^2 - pi r_id_outer^2) / n_ft

        It is NOT the borehole wall. With three legs in a 7 in hole r_cell is
        about 40 mm while the wall is at 88.90 mm, so a melt-layer limit
        imposed at the wall would be a factor 2 too permissive and therefore
        silent at every design point.
        """
        R = self.D_well / 2.0
        A_pcm_annulus = np.pi * (R ** 2 - self.r_id_outer ** 2)
        return np.sqrt(A_pcm_annulus / (np.pi * self.n_ft))

    @property
    def delta_merge(self):
        """Melt-layer thickness at which neighboring fronts touch.

        Hypothesis H3 (concentric annulus, no interaction between legs) is a
        statement about delta < delta_merge. At and beyond this thickness the
        remaining solid sits in the corners between legs, reached through a
        constricted path that the annular resistance ln(1+delta/r_e)/(2 pi k)
        does not represent.

        Under Formulation C this is not a bound that can be VIOLATED --
        area_from_delta(delta_merge) equals A_avail identically, and the
        enthalpy cap holds A_melt <= A_avail -- so delta <= delta_merge always.
        It is a bound the solution can SIT ON, which is what it does here.
        """
        return self.r_cell - self.r_e

    def geom_vector(self):
        """The eleven positional geometry arguments the geometry helpers expect."""
        return [self.L_well, self.L_tube, self.D_well, self.r_i, self.r_e,
                2 * self.r_i, 2 * self.r_e, self.fin_t, self.fin_L,
                self.num_fins, self.n_ft]

    def stefan_number(self, dT):
        """Ste = cp dT / h_m. The quasi-steady melt layer degrades as this grows."""
        return self.cp_l * dT / self.h_m

    def __post_init__(self):
        # The deprecated DT_M_1D, if someone still sets it, wins over the
        # default of DT_M_4D but never over an explicit DT_M_4D. It is
        # cleared afterwards so it can never reach a cache key or a
        # dataclasses.asdict() round trip as a second source of truth.
        if self.DT_M_1D is not None:
            import warnings as _w
            _w.warn('DT_M_1D is renamed DT_M_4D (round 25: the water loop '
                    'is numbered 1 = field outlet, 4 = field inlet). The '
                    'old name still works for now.', DeprecationWarning,
                    stacklevel=2)
            if self.DT_M_4D == 10.0:        # untouched default
                object.__setattr__(self, 'DT_M_4D', self.DT_M_1D)
            object.__setattr__(self, 'DT_M_1D', None)

    def with_(self, **kw):
        return replace(self, **kw)


CASE = Case()          # the design point


# ==========================================================================
# front_energy_balance   (the v0.2 correction)
# ==========================================================================


def delta_from_area(A_melt, r_e, n_f=0, t_f=0.0, L_f=0.0):
    """Invert `area_from_delta`. Closed form; the fins only shift the quadratic.

    For delta <= L_f the fin metal grows with the front:

        pi d^2 + (2 pi r_e - n_f t_f) d - A = 0

    and for delta > L_f it is constant at n_f t_f L_f:

        pi d^2 + 2 pi r_e d - (A + n_f t_f L_f) = 0

    With n_f = 0 both collapse to  d = sqrt(r_e^2 + A/pi) - r_e,  the bare
    concentric annulus used before this correction.
    """
    A = np.maximum(np.asarray(A_melt, dtype=float), 0.0)
    # branch 1: front still inside the fins
    b = 2.0 * np.pi * r_e - n_f * t_f
    d_in = (-b + np.sqrt(b * b + 4.0 * np.pi * A)) / (2.0 * np.pi)
    # branch 2: front beyond the fin tips
    d_out = np.sqrt(r_e ** 2 + (A + n_f * t_f * L_f) / np.pi) - r_e
    return np.where(d_in <= L_f, d_in, d_out)


# ==========================================================================
# march   (segment-by-segment integration down the tube)
# ==========================================================================

_PROP_CACHE = {}


def fluid_props(fluid, T, P):
    """Density, viscosity, conductivity, cp. Cached on 0.05 K bins."""
    key = (fluid, round(T * 20.0), round(P))
    hit = _PROP_CACHE.get(key)
    if hit is not None:
        return hit
    rho = CP.PropsSI("D", "T", T, "P", P, fluid)
    mu = CP.PropsSI("V", "T", T, "P", P, fluid)
    k = CP.PropsSI("L", "T", T, "P", P, fluid)
    cp = CP.PropsSI("C", "T", T, "P", P, fluid)
    _PROP_CACHE[key] = (rho, mu, k, cp)
    return rho, mu, k, cp


def h_internal(fluid, T, P, r_i, m_dot):
    """Internal convective coefficient. Gnielinski if turbulent, else laminar.

        Re = 4 m_dot / (pi D mu)
        f  = (0.79 ln Re - 1.64)^-2
        Nu = (f/8)(Re-1000) Pr / [1 + 12.7 sqrt(f/8) (Pr^(2/3) - 1)]
    """
    rho, mu, k, cp = fluid_props(fluid, T, P)
    D = 2 * r_i
    Re = 4.0 * m_dot / (np.pi * D * mu)
    Pr = cp * mu / k
    if Re < 2300:
        Nu = 3.66                      # fully developed, isothermal wall
    else:
        f = (0.79 * np.log(Re) - 1.64) ** -2
        Nu = (f / 8.0) * (Re - 1000) * Pr / (
            1 + 12.7 * np.sqrt(f / 8.0) * (Pr ** (2.0 / 3.0) - 1))
    return Nu * k / D, cp, Re, Pr


def layer_map(T_m_lay, n_segments, N_lay):
    """Melting temperature per segment, on the v0.1 node convention."""
    z = np.linspace(0.0, 1.0, n_segments + 1)
    z_lay = np.linspace(0.0, 1.0, N_lay + 1)
    T_m = np.empty(n_segments + 1)
    for j in range(N_lay):
        T_m[(z >= z_lay[j]) & (z < z_lay[j + 1])] = T_m_lay[j]
    T_m[-1] = T_m_lay[-1]
    return T_m[1:]


# ==========================================================================
# enthalpy state   (v0.4: sensible heat in both phases)
# ==========================================================================
# Formulation B tracks A_melt and nothing else, so a segment that runs out of
# PCM has nowhere to put further heat: the march clips it and discards the
# remainder. At the design point the discarded heat on discharge was 148% of
# the heat delivered -- i.e. most of what the network asked for.
#
# The fix is to carry ENTHALPY per unit tube length instead of melted area, with
# the melted area recovered from it. One scalar per segment, measured from the
# fully-solid-at-T_m datum:
#
#     E' < 0            solid, SUBCOOLED     T = T_m + E'/C_s
#     0 <= E' <= E'_lat two-phase            T = T_m,  A_melt = E'/(rho h_m)
#     E' > E'_lat       liquid, SUPERHEATED  T = T_m + (E'-E'_lat)/C_l
#
# Three things follow, and the third is the reason to do it:
#
#   1. no clipping anywhere -- heat always has somewhere to go;
#   2. the melt fraction is bounded in [0,1] BY CONSTRUCTION, so the
#      eps_local > 1 that the latent-only model produced cannot occur;
#   3. desuperheating and subcooling during discharge are recovered, which is
#      the energy the latent-only model threw away.
#
# delta still follows from A_melt exactly as before, so the heat-transfer
# coefficient is unchanged in form -- only its driving temperature is now
# T_pcm rather than T_m.


def pcm_capacities(case):
    """Per unit tube length: latent capacity and the two sensible capacities."""
    A_avail = case.V_well / (case.n_ft * case.L_tube)   # PCM area per tube [m2]
    E_lat = case.rho_latent * case.h_m * A_avail             # J/m to melt it all
    C_s = case.rho_latent * case.cp_s * A_avail              # J/m/K, solid
    C_l = case.rho_latent * case.cp_l * A_avail              # J/m/K, liquid
    return A_avail, E_lat, C_s, C_l


def pcm_state(E, T_m_seg, case):
    """Enthalpy per unit tube length -> (melted area, PCM temperature).

    The inverse of the enthalpy curve above. `A_melt` saturates at the segment's
    own PCM content, which is what makes eps_local <= 1 structural.
    """
    A_avail, E_lat, C_s, C_l = pcm_capacities(case)
    T_m_seg = np.asarray(T_m_seg, dtype=float)
    A = np.clip(E, 0.0, E_lat) / (case.rho_latent * case.h_m)
    if not case.sensible_heat:                    # v0.3 behavior: PCM pinned at T_m
        return A, T_m_seg.copy()
    T = np.where(E < 0.0, T_m_seg + E / C_s,
                 np.where(E > E_lat, T_m_seg + (E - E_lat) / C_l, T_m_seg))
    return A, T


def advance_segment(E, T_m, T0, K, dt, E_lat, C_s, C_l, sensible):
    """Integrate ONE segment's enthalpy over dt, exactly, branch by branch.

    The segment obeys   dE'/dt = K (T0 - T_pcm(E'))   with T0 held over the step.
    That is linear inside each branch of the enthalpy curve, so it can be solved
    in closed form rather than stepped:

      plateau  T_pcm = T_m constant           -> dE'/dt constant, E' linear in t
      solid    T_pcm = T_m + E'/C_s           -> exponential relaxation to
      liquid   T_pcm = T_m + (E'-E_lat)/C_l      E'_eq = C (T0 - T_m) [+E_lat]

    Explicit Euler is NOT usable here. On the latent plateau the segment has
    effectively infinite heat capacity, so any step is stable; the moment
    sensible heat is added the capacity becomes finite and the stability limit
    is dt < C/K, about 620 s at the design point. The time grid's last step is
    8491 s, fourteen times that, and an explicit update diverges -- it drove the
    secondary fluid to 261 K on the first attempt.

    Returns (E_new, q_avg) with q_avg = (E_new - E)/dt, so the heat taken from
    the fluid is exactly the enthalpy the PCM gained: closure stays structural.
    """
    if K <= 0.0 or dt <= 0.0:
        return E, 0.0
    E0, t_left = E, dt
    for _ in range(6):   # at most a few branch crossings
        if t_left <= 0.0:
            break
        if not sensible:                     # v0.3: PCM pinned at T_m
            E = E + K * (T0 - T_m) * t_left
            break
        if 0.0 <= E <= E_lat:                # ---- latent plateau: linear ----
            rate = K * (T0 - T_m)
            if rate == 0.0:
                break
            t_edge = ((E_lat - E) / rate) if rate > 0 else (E / -rate)
            if t_edge >= t_left:
                E = E + rate * t_left
                break
            # Land on the boundary and step just PAST it. Without the nudge the
            # next iteration re-enters the plateau at zero rate and the segment
            # sticks at E_lat for ever -- which showed up as every segment
            # reporting a melt fraction of exactly 1.000 and no superheat.
            eps = 1e-9 * max(E_lat, 1.0)
            E = (E_lat + eps) if rate > 0 else -eps
            t_left -= t_edge
        else:                                # ---- sensible: exponential ----
            if E < 0.0:
                C, E_ref, lo, hi = C_s, 0.0, -np.inf, 0.0
            else:
                C, E_ref, lo, hi = C_l, E_lat, E_lat, np.inf
            E_eq = E_ref + C * (T0 - T_m)
            E_new = E_eq + (E - E_eq) * np.exp(-K * t_left / C)
            if lo <= E_new <= hi:
                E = E_new
                break
            edge = hi if E_new > hi else lo   # crosses back onto the plateau
            num, den = E_eq - edge, E_eq - E
            if den == 0.0 or num / den <= 0.0:
                E = edge
                break
            t_edge = -C / K * np.log(num / den)
            if t_edge >= t_left or t_edge <= 0.0:
                E = E_new
                break
            E = edge
            t_left -= t_edge
    return E, (E - E0) / dt


def segment_profile(case, T_inlet, T_m_seg, m_dot, k_wall, E, n_segments=None):
    """Instantaneous fluid and conductance profile for a frozen PCM state.

    The inner loop of `march_h` without the time advance: it evaluates the
    resistance network, the segment conductance and the fluid temperature
    profile for the state `E` as it stands, and returns the instantaneous
    q' = K (T_f - T_pcm).

    It exists so that the last recorded frame of a march can carry a fluid
    profile belonging to the *end* state rather than to the state one step
    earlier. Everything it computes is already computed inside `march_h`;
    this is one extra sweep per recorded march, not one per time level.
    """
    n_segments = n_segments or len(E)
    dz = case.L_tube / n_segments
    A, T_pcm = pcm_state(E, T_m_seg, case)
    delta = delta_from_area(A, case.r_e, case.num_fins, case.fin_t, case.fin_L)
    A_avail, E_lat = pcm_capacities(case)[:2]
    d_melt, k_melt = conduction_shell(case, A, A_avail, True, E, E_lat)
    d_froz, k_froz = conduction_shell(case, A, A_avail, False, E, E_lat)

    T_prof = np.empty(n_segments + 1)
    T_prof[0] = T_inlet
    NTU_prof = np.empty(n_segments)
    U_prof = np.empty(n_segments)
    q_prime = np.empty(n_segments)

    T0 = T_inlet
    for i in range(n_segments):
        melting = T0 > T_pcm[i]
        d_path = float(d_melt[i] if melting else d_froz[i])
        k_m = float(k_melt[i] if melting else k_froz[i])
        h_i, cp_d, _, _ = h_internal(case.fluid2, T0, case.P, case.r_i, m_dot)
        U_i = compute_U_i(h_i, case.r_i, case.r_e, k_wall, case.Rf_i, k_m,
                          d_path, case.L_tube, case.fin_t,
                          case.fin_L, case.num_fins)
        NTU = float(np.clip((2 * np.pi * case.r_i * U_i * dz) / (m_dot * cp_d),
                            -50.0, 50.0))
        K = m_dot * cp_d * (1.0 - np.exp(-NTU)) / dz
        q_prime[i] = K * (T0 - T_pcm[i])
        T0 = T0 - q_prime[i] * dz / (m_dot * cp_d)
        T_prof[i + 1] = T0
        NTU_prof[i] = NTU
        U_prof[i] = U_i

    return dict(T_fluid=T_prof, NTU=NTU_prof, U_i=U_prof, q_prime=q_prime,
                A_melt=A, T_pcm=T_pcm, delta=delta, E=np.array(E, float))


def bulk_shape_factor(case):
    """Dimensionless resistance of a SINGLE-PHASE cell, mean to tube surface.

    A cell that is entirely one phase has no front. Heat entering at the tube is
    stored throughout its volume, so the temperature varies with radius and the
    lumped state T_pcm is the VOLUME MEAN. The resistance that belongs with that
    mean is not the surface-to-surface annulus value.

    Take the cell as an annulus r_e <= r <= r_o, adiabatic at r_o (its neighbor
    is identical, so the boundary is a symmetry plane), storing sensible heat at
    a uniform volumetric rate s = rho c dT/dt. Quasi-steady, the heat crossing
    radius r is what the material beyond it stores,

        -k 2 pi r dT/dr = s pi (r_o^2 - r^2),

    which integrates to

        T(r_e) - T(r) = (s/2k) [ r_o^2 ln(r/r_e) - (r^2 - r_e^2)/2 ].

    Averaging over the volume and dividing by the total Q' = s pi (r_o^2-r_e^2)
    gives R'_bulk = S/(2 pi k) with, writing beta = r_o/r_e,

        S = [ beta^4 ln(beta) - beta^4/2 + beta^2/2 - (beta^2-1)^2/4 ]
            / (beta^2 - 1)^2 .

    S depends on GEOMETRY ALONE -- k cancels -- so one shape factor serves both
    phases and only the conductivity changes between them. For the default cell,
    beta = 2.1086 and S = 0.34672, against ln(beta) = 0.74604 for the
    surface-to-surface annulus: the mean-to-surface resistance is smaller by a
    factor 2.152, the cylindrical analogue of the familiar 1/3 for a slab with
    uniform generation.

    Verified against numerical quadrature of the same integral to six decimals.
    """
    beta = case.r_cell / case.r_e
    b2 = beta * beta
    return ((b2 * b2 * np.log(beta) - b2 * b2 / 2.0 + b2 / 2.0
             - (b2 - 1.0) ** 2 / 4.0) / (b2 - 1.0) ** 2)


def bulk_equivalent_delta(case):
    """The shell thickness that reproduces the bulk resistance.

    The network in `compute_U_i` takes a thickness, not a resistance, and wraps
    it in the fin efficiency and the finned-area ratio. Feeding it an equivalent
    thickness therefore keeps the single-phase branches inside exactly the same
    machinery as the two-phase ones, rather than bolting a second path onto it:

        ln(1 + delta_eq/r_e) = S    =>    delta_eq = r_e (e^S - 1).

    Like S it is a property of the geometry alone -- the same delta_eq serves
    subcooled solid and superheated liquid, and only k changes. For the default
    cell it is 8.74 mm, against 0 and 23.37 mm, which are the two values the
    model used before this was worked out.
    """
    return case.r_e * (np.exp(bulk_shape_factor(case)) - 1.0)


def conduction_shell(case, A_melt, A_avail, melting, E=None, E_lat=None):
    """Thickness and conductivity of the shell between the tube and the front.

    The PCM-side resistance in `compute_U_i` is that of an annulus growing
    OUTWARD FROM THE TUBE WALL,

        R' = ln(1 + delta/r_e) / (2 pi k),

    so the question this function answers is: which material is that annulus
    made of, and how thick is it?

    It depends on the direction of the phase change, because the front always
    grows away from the tube -- the tube is the driven boundary in both
    half-cycles.

      MELTING (fluid hotter than the PCM). Melting begins at the tube wall and
      the front moves outward, so the shell is LIQUID and its thickness is that
      of the melted area. Heat crosses the melt to reach the remaining solid.

      FREEZING (fluid colder). Solidification also begins at the tube wall, so
      the shell is the FROZEN material and its thickness is that of the solid
      area, A_avail - A_melt. The liquid is displaced outward, beyond the front,
      and is no longer in the conduction path at all.

    All four limits come out right without special-casing:

      fully solid, melting     A_melt = 0            -> delta = 0, no resistance
      fully melted, melting    A_melt = A_avail      -> the full liquid annulus
      fully melted, freezing   A_avail - A_melt = 0  -> delta = 0, no resistance
      fully solid, freezing    A_avail - A_melt = A  -> the full frozen annulus

    The last of those is the one that matters. Until v0.7 the melted thickness
    was used in BOTH directions, which put the conduction path on the wrong
    side of the front during discharge: a segment that had frozen solid was
    given ZERO PCM-side resistance, exactly where the physical resistance is
    largest. The consequence was that the modelled exchanger IMPROVED as it
    froze, when it should degrade. See DN-13 for what that was worth.

    LIMIT OF ANY LUMPED FRONT. One thickness can describe one front. After the
    first half-cycle from a fully solid store the geometry is generally
    three-region -- at cyclic steady state the charge begins with residual
    liquid left in the OUTER part of the cell, so melting produces liquid at the
    tube, solid in the middle and liquid outside. Neither this function nor the
    retired one can represent that; this one is the better approximation, not a
    correct treatment. It is hypothesis H5, stated honestly.
    """
    if case.front_geometry not in ("directional", "melt_side"):
        raise ValueError(f"front_geometry must be 'directional' or "
                         f"'melt_side', not {case.front_geometry!r}")

    A_melt = np.asarray(A_melt, float)
    if case.front_geometry == "melt_side":          # retired, see above
        return (delta_from_area(A_melt, case.r_e, case.num_fins, case.fin_t,
                                case.fin_L),
                np.full(A_melt.shape, case.k_l))

    # ---- two-phase: the shell between the tube and the front --------------
    A_path = A_melt if melting else (A_avail - A_melt)
    delta = delta_from_area(A_path, case.r_e, case.num_fins, case.fin_t,
                            case.fin_L)
    k_path = np.full(A_melt.shape, case.k_l if melting else case.k_s)

    # ---- single phase: there is no front, so there is no side ------------
    # `E` is optional only so that callers which do not have it (none, now)
    # keep the two-phase behavior; when it is given, the single-phase cells
    # get the bulk resistance of `bulk_shape_factor`, the SAME in both
    # directions, because a one-phase cell has no front and cannot care which
    # way the heat is going.
    if E is not None:
        E = np.asarray(E, float)
        d_bulk = bulk_equivalent_delta(case)
        solid = E < 0.0                              # subcooled solid
        liquid = E > E_lat                           # superheated liquid
        delta = np.where(solid | liquid, d_bulk, delta)
        k_path = np.where(solid, case.k_s,
                          np.where(liquid, case.k_l, k_path))
    return delta, k_path


def mixed_mean_outlet(res):
    """Flow-mixed mean outlet temperature of a march [K].

    The temperature at which the stream returns to the plant, averaged over the
    half-cycle. At constant mass flow and c_p the mixing average reduces to the
    time average, which on a logarithmic grid still has to be weighted by the
    step -- the plain mean of `T_out` is dominated by the early levels and is
    wrong by several kelvin.

    This is the quantity the ORC actually sees, and since v0.6 it is a RESULT
    rather than a prescribed state point (`cycle_state_points`).
    """
    t = np.asarray(res["t"], float)
    dt = np.diff(np.concatenate(([0.0], t)))
    return float(np.sum(np.asarray(res["T_out"], float) * dt) / np.sum(dt))


def unmirror_march(res):
    """Re-index a reversed-flow march result into charge (depth) coordinates.

    Discharge marches from the opposite end of the tube, so its segment index
    runs backwards relative to the charge. `run_cycle` and `simulate_css` hand
    it a mirrored cascade and a mirrored initial state and un-mirror the result
    when they close the loop; anything that *plots* a discharge result has to
    do the same or it draws the well upside down.

    Returns a shallow copy with every per-segment array reversed, including the
    recorded history. `z`, `t` and `t_hist` are coordinates, not fields, and
    are left alone. Scalars are left alone. The cascade in depth coordinates is
    the same for both half-cycles, which is the point: after this call a
    discharge result is plotted against `T_m_seg`, not against `T_m_seg[::-1]`.
    """
    n = len(res["E"])
    out = dict(res)
    for k in ("E", "A_melt", "T_pcm", "delta", "eps_local", "merge_proximity"):
        if k in out and out[k] is not None:
            out[k] = np.asarray(out[k])[::-1].copy()
    h = res.get("history")
    if h:
        hh = {}
        for k, v in h.items():
            v = np.asarray(v)
            # T_fluid has n+1 nodes (inlet plus one per segment); the rest have n
            hh[k] = v[:, ::-1].copy() if v.ndim == 2 and v.shape[1] in (n, n + 1) else v
        out["history"] = hh
    return out


# ==========================================================================
# the formation   (BOREAS v1.0 -- THUMS hypothesis H6 said this was adiabatic)
# ==========================================================================
# H6 made eta_storage identically 1. Not approximately 1: EXACTLY 1, by
# construction, because nothing could leave. That is an indefensible headline
# for a borehole storage paper and it is the first thing a reviewer attacks.
#
# The resistances are wildly unequal and it is worth knowing which one is the
# model. At the design point, per meter of borehole:
#
#     casing steel        0.0003 m K/W     0.1 %
#     cement              0.018            5 %
#     PCM to casing       ~0.12            25 %
#     GROUND              0.32             ~70 %
#
# Everything near the well is a rounding error. The ground resistance is the
# physics, and unlike the others it GROWS WITH TIME as the thermal front
# spreads -- which is why `t_operation_yr` is a Case parameter and why a
# single quoted loss figure is meaningless without the age that goes with it.
#
# THE SIGN IS NOT ALWAYS NEGATIVE. With a 45 K/km gradient the rock at 8000 ft
# is near 128 C, and at the Wilmington gradient of 56.5 K/km it is near 156 C
# -- hotter than the coldest cascade layers. Those segments GAIN heat from the
# formation. A model that assumes loss and takes an absolute value somewhere
# would hide that, so nothing here does.


def ground_resistance(case):
    """Quasi-steady line-source resistance of the formation, per meter.

    At time t the disturbance has spread to a radius of order sqrt(alpha t),
    and the resistance is the log of that over the borehole radius. It is
    quasi-steady in the sense that the DAILY cycle is far faster
    than the formation's response, so within one charge-discharge the ground
    looks like a fixed resistance to a fixed far-field temperature. The cycle
    time is 10 h and the formation time constant is years; the separation is
    four orders of magnitude, so this is a good approximation rather than a
    convenient one.

    The prefactor is the exact large-time line-source constant
    2 exp(-gamma/2); see the derivation inline below.

    SINGLE WELL, NO FIELD SHIELDING. This is deliberate and it is the
    conservative choice: an isolated well is the worst case, because in a
    field the neighbors hold the ground between them warm and only the
    perimeter leaks. The shielding factor is 29x to 42x, essentially
    independent of the driving temperature and only weakly dependent on rock
    conductivity, so a field result can be obtained by dividing. Nothing here
    should be read as the loss a real cluster would see.
    """
    if case.k_rock <= 0.0:
        return np.inf                      # no formation: adiabatic wall
    alpha = case.k_rock / case.rho_cp_rock
    t = case.t_operation_yr * 365.25 * 24.0 * 3600.0
    # THE PREFACTOR IS DERIVED, NOT QUOTED.  v1.0 used 2.5, attributed in this
    # docstring to "Eskilson's form". That attribution was wrong: no source for
    # 2.5 could be found, and it appears to have been invented. The correct
    # constant follows in three lines from the large-time Carslaw-Jaeger
    # infinite line source,
    #
    #     R' = [ln(4 a t / rb^2) - gamma] / (4 pi k)
    #        = (1/2 pi k) [ln(2 sqrt(a t)/rb) - gamma/2]
    #        = (1/2 pi k)  ln( 2 exp(-gamma/2) sqrt(a t) / rb ),
    #
    # so the prefactor is 2 exp(-gamma/2) = 1.49861, not 2.5. The difference is
    # a CONSTANT additive offset of ln(2.5/1.49861)/(2 pi k) = 0.0326 m K/W at
    # the reference case -- 9.3 % of R_g -- which does not decay with time and
    # so cannot be dismissed as a transient.
    #
    # It bites only where the GROUND term binds. Under the max() rule of
    # `formation_resistance`, the perimeter term dominates at every depth for
    # the 1000-well fan and the island array, so the realistic results are
    # unchanged; the isolated-well bound moves from 75.6 % to 79.0 % loss and
    # the 100-well fan from 54.9 % to 56.7 %.
    #
    # The asymptotic form is safe here: rb^2/(4 a t) = 9e-6 at the reference
    # case, so the E1 expansion is exact to five decimal places.
    C_ILS = 2.0 * np.exp(-0.5772156649015329 / 2.0)      # = 1.498612
    arg = C_ILS * np.sqrt(alpha * t) / case.r_bore
    R = np.log(arg) / (2.0 * np.pi * case.k_rock) if arg > 0.0 else -1.0
    if not np.isfinite(R) or R <= 0.0:
        # THE LINE SOURCE HAS BROKEN DOWN, and the branch matters.
        # An earlier version returned 0.0 here, meaning the front "has not
        # cleared the hole yet". Zero RESISTANCE is infinite CONDUCTANCE: it
        # says the rock is a perfect heat sink, which is the opposite of what
        # was meant and the most damaging possible failure of this function.
        # It was caught the first time the adiabatic limit was watched -- a
        # test that drives k_rock to zero and expects eta_storage = 1 got
        # eta_storage = 0.20 and a LARGER loss than the physical case.
        #
        # Infinity is the right fallback: outside its validity the line
        # source is not evidence of a loss, and `validate_case` warns rather
        # than letting a silent number through.
        return np.inf
    return R


def cluster_R0(case):
    """Surface footprint radius of the well cluster [m], disc-equivalent."""
    if case.n_wells_field is None:
        return None
    return case.B_surface_m * np.sqrt(case.n_wells_field / np.pi)


def field_perimeter(case, depth):
    """Perimeter the cluster presents to the formation at depth [m].

    THIS, NOT 2*pi*R, IS WHAT THE SHIELDING ACTUALLY USES. The perimeter
    model says a cluster loses heat across its boundary and the wells inside
    share it, so the quantity that matters is meters of boundary per well.
    Expressing that as a radius silently assumes the wells fill a circle.

        disc   P = 2 pi R0                    compact, minimum perimeter
        rows   P = n_isl * 2 (L_row + gap)    linear cellars, the THUMS case
        ring   P = 2 * 2 pi R_isl             wells on a rim, both faces

    The fan still applies: below the kickoff the cluster spreads, and every
    linear dimension grows with it, so P scales with R_f(z)/R0 whatever the
    packing. Above the kickoff -- which is where the storage is -- the
    packing alone decides.
    """
    if case.n_wells_field is None:
        return None
    N, B0 = case.n_wells_field, case.B_surface_m
    R0 = cluster_R0(case)
    if case.packing == "disc":
        P0 = 2.0 * np.pi * R0
    elif case.packing == "rows":
        n_isl = max(1, int(case.n_islands))
        row_len = (N / n_isl / 2.0) * B0          # double rows
        P0 = n_isl * 2.0 * (row_len + case.row_gap_m)
    elif case.packing == "ring":
        n_isl = max(1, int(case.n_islands))
        circ = (N / n_isl / 2.0) * B0             # rim length occupied
        P0 = n_isl * 2.0 * circ                   # loses from both faces
    else:
        raise ValueError(f"unknown packing {case.packing!r}")
    Rf = field_radius(case, depth)
    return P0 * (Rf / R0)


def field_radius(case, depth):
    """Radius of the well cluster at a given depth [m].

    Vertical above the kickoff point, then fanning at a constant effective
    half-angle. A real directional programme builds angle over a section and
    then holds it, so the fan is not a straight cone -- but the quantity this
    feeds is a logarithm, and the difference between a cone and a build-and-
    hold is well inside the uncertainty in the spacing data itself.
    """
    if case.n_wells_field is None:
        return None
    R0 = case.B_surface_m * np.sqrt(case.n_wells_field / np.pi)
    spread = np.maximum(0.0, np.asarray(depth, float) - case.z_kickoff_m)
    return R0 + spread * np.tan(np.radians(case.fan_angle_deg))


def well_spacing(case, depth):
    """Centre-to-center spacing at depth, scaled with the fan radius [m]."""
    Rf = field_radius(case, depth)
    if Rf is None:
        return None
    R0 = case.B_surface_m * np.sqrt(case.n_wells_field / np.pi)
    return case.B_surface_m * Rf / R0


def formation_resistance(case, depth=None):
    """Total series resistance, PCM bulk to undisturbed rock [m K / W].

    Per meter of BOREHOLE, not per meter of tube. The PCM-to-casing term is
    the crude one: n_ft parallel paths from the tube walls out to the casing,

        R_pcm = ln(R_casing_i / r_e) / (2 pi n_ft k_pcm)

    which ignores that the paths crowd together near the wall. It is an
    approximation and it is flagged as one -- but it is a quarter of a total
    that the ground dominates, so refining it would move the answer less than
    the uncertainty in k_rock does.
    """
    R_in = case.D_well / 2.0
    k_pcm = 0.5 * (case.k_s + case.k_l)
    R_pcm = np.log(R_in / case.r_e) / (2.0 * np.pi * case.n_ft * k_pcm)
    R_steel = np.log(case.r_casing_o / R_in) / (2.0 * np.pi * case.k_wall)
    R_cem = np.log(case.r_bore / case.r_casing_o) / (2.0 * np.pi * case.k_cement)
    R_g = ground_resistance(case)
    if not np.isfinite(R_g):
        return np.inf                      # adiabatic; K_loss becomes 0
    R_near = R_pcm + R_steel + R_cem

    # ---- FIELD SHIELDING -------------------------------------------------
    # A well inside a cluster cannot lose what an isolated one would: its
    # neighbors hold the rock between them warm, and once the cells have
    # merged the only heat leaving is across the cluster PERIMETER, shared
    # among N wells. Writing that as a resistance,
    #
    #     R_perimeter(z) = N sqrt(pi alpha t) / (2 pi k R_field(z))
    #
    # -- the semi-infinite-slab flux at the boundary, over the perimeter the
    # field presents, divided by the wells that share it. It carries no
    # driving temperature, so it is a true resistance and drops straight into
    # the series chain.
    #
    # The two regimes are combined by taking the LARGER resistance. That is
    # not a smooth blend and does not pretend to be: a well cannot leak more
    # than if it stood alone, and a field cannot leak more than its perimeter
    # allows, so whichever constraint binds is the one that governs. The
    # crossover is sharp in reality too -- it is the moment the cells merge.
    if depth is None or case.n_wells_field is None:
        return R_near + R_g
    alpha = case.k_rock / case.rho_cp_rock
    t = case.t_operation_yr * 365.25 * 24.0 * 3600.0
    # Perimeter, not 2*pi*R: the packing decides how much boundary each
    # well has to share. See `field_perimeter`.
    P = field_perimeter(case, depth)
    R_per = (case.n_wells_field * np.sqrt(np.pi * alpha * t)
             / (case.k_rock * P))
    return R_near + np.maximum(R_g, R_per)


def formation_profile(case, n_segments=None, discharge=False):
    """Undisturbed rock temperature [K] at each segment, and its depth [m].

    THE COORDINATE. In BOREAS z is DEPTH, which it could not be in THUMS: the
    hairpin's two legs occupied every depth twice, so one rock temperature
    would have had two PCM segments claiming it. Here the fin tubes only go
    down and the downcomer does not exchange, so the map is one-to-one.

    On CHARGE the water enters the fin tubes at the BOTTOM -- it arrives there
    down the downcomer -- so the solver's z = 0 is the bottom of the well and
    depth decreases with z. On DISCHARGE the flow reverses and the march is
    run mirrored, so the profile is mirrored with it, exactly as T_m_seg is.
    Getting this backwards would put the hot cascade layers against the cold
    shallow rock, which is the THUMS orientation and the thing BOREAS exists
    to avoid -- so it is asserted in `validate_case`, not left to a comment.
    """
    n = n_segments or case.n_segments
    xi = (np.arange(n) + 0.5) / n                 # cell centers, 0 at inlet
    depth = case.L_well * (1.0 - xi)              # charge frame: z=0 is bottom
    T_rock = case.T_surface_C + 273.15 + case.grad_K_per_km / 1000.0 * depth
    if discharge:
        return T_rock[::-1].copy(), depth[::-1].copy()
    return T_rock, depth


def idle_hours(case):
    """Duration of EACH idle period, h. Zero when idling is not simulated."""
    if case.idle_mode == 'none':
        return 0.0
    if case.t_idle is not None:
        return float(case.t_idle)
    return max(0.0, (case.day_h - case.t_ch - case.t_dc) / 2.0)


def idle_h(case, T_m_seg, times, n_segments=None, E0=None,
           T_rock=None, R_form=None, record=False):
    """Relax the store with NO FLOW, exchanging only with the formation.

        dE'_j/dt = -K_loss,j [T_pcm(E'_j) - T_rock,j]

    The daily cycle is t_ch + t_dc + the idle periods between them, and the
    idle hours were simply not simulated: the store jumped from the end of
    charge to the start of discharge with no time elapsed, so their loss was
    missing from Q_loss and every indicator built on it.

    This is NOT a new integrator. It is the same per-segment update
    `march_h` already performs, with the TUBE conductance set to zero: the
    combined sink K_eq = K + K_l collapses to K_l and the equilibrium
    temperature T_eq to T_rock. `advance_segment` stays implicit in T_pcm
    and remains exact branch by branch, so the long idle steps are as safe
    as the long steps at the end of a half-cycle.

    Returns the same shape of dict as `march_h` for the fields that still
    mean something with no fluid: E, Q_cum_J (negative: the store loses),
    q_prime, and the history when `record`.
    """
    n = n_segments or case.n_segments
    dz = case.L_tube / n
    _A, E_lat, C_s, C_l = pcm_capacities(case)
    E = (np.zeros(n) if E0 is None else np.asarray(E0, float).copy())

    if T_rock is None:                      # adiabatic: idling costs nothing
        K_loss = np.zeros(n); T_rock_arr = np.zeros(n)
    else:
        R_tot = (formation_resistance(case) if R_form is None
                 else np.asarray(R_form, float))
        R_arr = np.full(n, R_tot) if np.ndim(R_tot) == 0 else R_tot
        K_loss = np.where(np.isfinite(R_arr), 1.0 / (R_arr * case.n_ft), 0.0)
        K_loss = np.nan_to_num(K_loss, posinf=0.0, neginf=0.0)
        T_rock_arr = np.asarray(T_rock, float)

    times = np.asarray(times, float)
    Q_loss_cum = 0.0
    clausius = 0.0
    hist = {"E": [], "Q_loss_cum": [], "t": []} if record else None
    for s in range(1, len(times)):
        dt = times[s] - times[s - 1]
        if dt <= 0.0:
            continue
        for i in range(n):
            K_l = K_loss[i]
            if K_l <= 0.0:
                continue
            E[i], q_tot = advance_segment(E[i], T_m_seg[i], T_rock_arr[i],
                                          K_l, dt, E_lat, C_s, C_l,
                                          case.sensible_heat)
            # with the rock as the ONLY sink, what the PCM received is
            # exactly minus what it lost; no split to recover.
            q_loss = -q_tot
            T_eff = T_rock_arr[i] - q_tot / K_l
            Q_loss_cum += q_loss * dz * dt
            clausius += q_loss * (T_eff - T_rock_arr[i]) * dz * dt
        if record:
            hist["E"].append(E.copy()); hist["Q_loss_cum"].append(Q_loss_cum)
            hist["t"].append(times[s])
    out = dict(E=E, Q_loss_J=Q_loss_cum, Q_cum_J=-Q_loss_cum,
               clausius=clausius, idle=True)
    if record:
        out["history"] = {k: np.asarray(v) for k, v in hist.items()}
    return out


def march_h(case, T_inlet, T_m_seg, m_dot, k_wall, times,
            n_segments=None, E0=None, record=False, T_rock=None,
            R_form=None):
    """Segment march on the ENTHALPY state. Replaces `march` from v0.4.

    Identical to `march` in every respect except what is integrated: the state
    is E' [J per m of tube] rather than A_melt [m2], so no heat is ever clipped
    and the PCM temperature is free to leave T_m.

    Recording convention. Every history frame is stamped, in `t_hist`, with the
    instant at which its state arrays (`E`, `A_melt`, `T_pcm`, `delta`) are
    exact. The profile arrays (`T_fluid`, `NTU`, `U_i`, `q_prime`) in the same
    frame are the ones evaluated from that state, i.e. the profile that drives
    the step *beginning* there. There are `len(t)+1` frames: one at t = 0 and
    one at the end of each accepted step, so the final state is in the record.

    Earlier builds recorded the pre-step state but stamped it with the
    post-step time, which on a logarithmic grid put the last frame a quarter of
    a charge behind its label, and appended `E` after the update while the
    other state arrays were from before it. Neither affected any marched or
    reported quantity -- `Q_cum`, `closure`, `eps_local` and the returned state
    all come from the march itself -- but every recorded plot was lagged.
    """
    n_segments = n_segments or case.n_segments
    dz = case.L_tube / n_segments
    r_i, r_e = case.r_i, case.r_e
    A_avail, E_lat, C_s, C_l = pcm_capacities(case)

    # ---- the formation sink ---------------------------------------------
    # K_loss is a conductance per unit TUBE length, so the per-borehole
    # conductance is divided among the n_ft tubes that share each depth.
    # Passing T_rock=None recovers the THUMS adiabatic wall exactly, which is
    # how the v0.16 reproduction test is run.
    if T_rock is None:
        K_loss = np.zeros(n_segments)
        T_rock_arr = np.zeros(n_segments)
    else:
        R_tot = (formation_resistance(case) if R_form is None
                 else np.asarray(R_form, float))
        R_arr = np.full(n_segments, R_tot) if np.ndim(R_tot) == 0 else R_tot
        K_loss = np.where(np.isfinite(R_arr), 1.0 / (R_arr * case.n_ft), 0.0)
        K_loss = np.nan_to_num(K_loss, posinf=0.0, neginf=0.0)
        T_rock_arr = np.asarray(T_rock, float)
    Q_loss_cum = 0.0
    # ---- A CLAUSIUS GATE ON THE LOSS TERM --------------------------------
    # The closure CANNOT catch a flipped loss sign, and this was watched:
    # negating q_loss leaves the residual at 5e-15, because q_prime and
    # Q_loss_cum both carry the error and it cancels algebraically. The
    # closure is self-consistent, which is the DN-26 trap -- a check reading
    # only its own inputs.
    #
    # This one reads the STATE instead. Heat must flow from hot to cold, so
    # q_loss and (T_pcm - T_rock) must share a sign at every segment and
    # every step. The product is accumulated and gated on being non-negative.
    # With the correct convention it is K (dT)^2 and cannot be negative; with
    # the sign flipped it is -K (dT)^2 and cannot be positive. That is the
    # independent input the closure lacks.
    clausius = 0.0

    E = np.zeros(n_segments) if E0 is None else np.array(E0, float)
    E_start = E.copy()
    t_prev, Q_cum = 0.0, 0.0
    ts, Qs, T_outs = [], [], []
    hist = {k: [] for k in ("T_fluid", "T_pcm", "A_melt", "delta", "NTU",
                            "U_i", "q_prime", "E")} if record else None
    t_hist = []

    for t in np.asarray(times, float):
        dt = t - t_prev
        if dt <= 0:
            t_prev = t
            continue

        A, T_pcm = pcm_state(E, T_m_seg, case)
        # `delta` stays the MELT thickness: it is what the H3 merge diagnostic
        # is about. The conduction path is a separate question -- which side of
        # the front the heat has to cross -- and depends on the direction.
        delta = delta_from_area(A, r_e, case.num_fins, case.fin_t, case.fin_L)
        d_melt, k_melt = conduction_shell(case, A, A_avail, True, E, E_lat)
        d_froz, k_froz = conduction_shell(case, A, A_avail, False, E, E_lat)
        T0 = T_inlet
        q_prime = np.zeros(n_segments)
        if record:
            E_before = E.copy()          # the state this frame is stamped at
            T_prof = np.empty(n_segments + 1); T_prof[0] = T_inlet
            NTU_prof = np.empty(n_segments); U_prof = np.empty(n_segments)

        for i in range(n_segments):
            # Which shell the heat crosses depends on which way the front is
            # moving; the sign of the driving difference settles it before K is
            # known, and it has the same sign as q'.
            melting = T0 > T_pcm[i]
            d_path = float(d_melt[i] if melting else d_froz[i])
            k_m = float(k_melt[i] if melting else k_froz[i])
            h_i, cp_d, _, _ = h_internal(case.fluid2, T0, case.P, r_i, m_dot)
            U_i = compute_U_i(h_i, r_i, r_e, k_wall, case.Rf_i, k_m,
                              d_path, case.L_tube, case.fin_t,
                              case.fin_L, case.num_fins)
            NTU = float(np.clip((2 * np.pi * r_i * U_i * dz) / (m_dot * cp_d),
                                -50.0, 50.0))
            # conductance of the segment to the fluid, per unit tube length:
            # q' = K (T0 - T_pcm), from the NTU effectiveness
            K = m_dot * cp_d * (1.0 - np.exp(-NTU)) / dz

            # TWO linear sinks on one node, combined EXACTLY rather than
            # split operator. For any T_pcm,
            #     K (T0 - T) + K_l (T_rock - T) = (K + K_l) (T_eq - T)
            # with T_eq the conductance-weighted mean, so `advance_segment`
            # is called unchanged and remains implicit in T_pcm. Adding the
            # loss explicitly after the step would have been a half-order
            # scheme on a term that can reverse sign down the well.
            K_l = K_loss[i]
            K_eq = K + K_l
            T_eq = (K * T0 + K_l * T_rock_arr[i]) / K_eq if K_eq > 0 else T0
            E[i], q_tot = advance_segment(E[i], T_m_seg[i], T_eq, K_eq, dt,
                                          E_lat, C_s, C_l,
                                          case.sensible_heat)
            # Recover the split. q_tot is what the PCM NET received, so the
            # effective PCM temperature over the step follows from it, and
            # the two streams separate without any further assumption.
            if K_l > 0.0:
                T_eff = T_eq - q_tot / K_eq
                q_loss = K_l * (T_eff - T_rock_arr[i])   # +ve = leaving
                Q_loss_cum += q_loss * dz * dt
                clausius += q_loss * (T_eff - T_rock_arr[i]) * dz * dt
            else:
                q_loss = 0.0
            q_prime[i] = q_tot + q_loss                  # what the TUBE gave
            # the fluid gives up exactly what the PCM took FROM IT -- the
            # formation term is not the fluid's to pay
            T0 = T0 - q_prime[i] * dz / (m_dot * cp_d)
            if record:
                T_prof[i + 1] = T0; NTU_prof[i] = NTU; U_prof[i] = U_i

        if record:
            # stamped at t_prev, where E_before / A / T_pcm / delta are exact
            t_hist.append(t_prev)
            hist["T_fluid"].append(T_prof.copy()); hist["T_pcm"].append(T_pcm.copy())
            hist["A_melt"].append(A.copy());       hist["delta"].append(delta.copy())
            hist["NTU"].append(NTU_prof.copy());   hist["U_i"].append(U_prof.copy())
            hist["q_prime"].append(q_prime.copy()); hist["E"].append(E_before)

        Q_cum += float(np.sum(q_prime) * dz) * dt
        ts.append(t); Qs.append(float(np.sum(q_prime) * dz)); T_outs.append(T0)
        t_prev = t

    if record:
        # the end state, which the loop above never gets to record
        end = segment_profile(case, T_inlet, T_m_seg, m_dot, k_wall, E,
                              n_segments=n_segments)
        t_hist.append(t_prev)
        for k in hist:
            hist[k].append(end[k])

    A, T_pcm = pcm_state(E, T_m_seg, case)
    dE = float(np.sum(E - E_start) * dz)
    delta_end = delta_from_area(A, r_e, case.num_fins, case.fin_t, case.fin_L)

    # ---- H3 proximity ----------------------------------------------------
    # delta can never EXCEED delta_merge here (see Case.delta_merge), so the
    # useful diagnostic is not a violation flag but how close to the limit the
    # solution runs, and over how much of the well. The annular conduction
    # resistance is least trustworthy where this approaches 1.
    d_merge = case.delta_merge
    prox = delta_end / d_merge if d_merge > 0 else np.full_like(delta_end, np.nan)

    return {
        "E": E, "A_melt": A, "T_pcm": T_pcm,
        "delta": delta_end,
        "Q_cum_J": Q_cum,
        "Q_loss_J": Q_loss_cum,
        # >= 0 always, by the second law. Negative means the loss term is
        # pushing heat uphill, which is the sign error the closure misses.
        "loss_clausius": clausius,
        # THE CLOSURE NOW HAS THREE TERMS. In THUMS it was
        # |Q_cum - dE| / |Q_cum|: everything the tube gave went into storage,
        # because nothing could leave. Here what the tube gives either stays
        # or goes to the rock, so the identity is Q_tube = dE + Q_loss and the
        # residual is what is left over. This is still a bookkeeping identity
        # rather than a gate -- it cannot fail without an arithmetic bug --
        # but it would have caught the sign error of dropping q_loss from
        # q_prime, which is the mistake this split invites.
        "closure": (abs(Q_cum - dE - Q_loss_cum) / abs(Q_cum)
                    if Q_cum else np.nan),
        "t": np.array(ts), "Q": np.array(Qs), "T_out": np.array(T_outs),
        "V_melt_tube": float(np.sum(A * dz)),
        "eps_local": A / A_avail,
        "delta_merge": d_merge,
        "merge_proximity": prox,
        "merge_proximity_max": float(np.max(prox)),
        "f_near_merge": float(np.mean(prox > 0.90)),
        "E_sensible_J": float(np.sum(np.where(E > E_lat, E - E_lat,
                                              np.where(E < 0, E, 0.0))) * dz),
        "z": np.linspace(0.0, case.L_tube, n_segments),
        "history": {k: np.array(v) for k, v in hist.items()} if record else None,
        "t_hist": np.array(t_hist) if record else None,
    }


# ==========================================================================
# cycles + energy budget
# ==========================================================================

def T_m_bottom(case):
    """Melting temperature of the COLDEST cascade layer [K].

    The cascade is graded so that each layer sits DT_4C_M below the charging
    fluid at its own leading face, under a linear glide. The layers are spaced
    DT_3C_2C / N_lay apart, so the span of the melting temperatures is one
    layer width SHORT of the glide:

        T_m,bottom = T_m,top - DT_3C_2C * (N_lay - 1) / N_lay

    For the default case that is 150 - 55*8/9 = 101.111 C against a 55 K glide.
    The discharge enters the exchanger at this end, which is why this is the
    reference for DT_M_4D and not T_m,top.
    """
    return case.T_m - case.cascade_span


def _orc_cold_composite(rank, fluid, n=300):
    """The ORC-side composite curve of the evaporator: (Q, T), cold end first.

    Two cold streams share the water:

      * the MAIN stream, 1 kg per kg of evaporator flow, from state 10e to
        state 3e at p_evape -- liquid preheat, then boiling at the constant
        temperature T_3e;
      * the REHEAT stream, y_frac of the flow, from 5e to 6e at p_inte,
        superheated vapour throughout.

    Both are built on real enthalpy, so the latent plateau appears as a
    vertical segment rather than being smeared into a straight line in T.
    That matters: the plateau is where this evaporator pinches.
    """
    y = rank["y_frac"]
    p_e, p_i = rank["p_evape"], rank["p_5e"]
    h10, h3 = rank["h_10e"], rank["h_3e"]
    h5, h6 = rank["h_5e"], rank["h_6e"]
    T10, T3 = rank["T_10e"], rank["T_3e"]
    T5, T6 = rank["T_5e"], rank["T_6e"]
    h_bub = CP.PropsSI("H", "P", p_e, "Q", 0, fluid) / 1000.0

    # sample in ENTHALPY, not temperature: a (T, p) call exactly on the
    # saturation line raises, and the preheat ends exactly there
    h_pre = np.linspace(h10, h_bub, n)
    T_pre = np.array([CP.PropsSI("T", "H", x * 1000.0, "P", p_e, fluid)
                      for x in h_pre])
    h_rh = np.linspace(h5, h6, n)
    T_rh = np.array([CP.PropsSI("T", "H", x * 1000.0, "P", p_i, fluid)
                     for x in h_rh])

    def Q_main(t):
        if t <= T10:
            return 0.0
        if t >= T3:
            return h3 - h10                      # preheat AND latent
        return float(np.interp(t, T_pre, h_pre)) - h10

    def Q_reheat(t):
        if t <= T5:
            return 0.0
        if t >= T6:
            return y * (h6 - h5)
        return y * (float(np.interp(t, T_rh, h_rh)) - h5)

    lo, hi = min(T10, T5), max(T3, T6)
    grid = set(np.linspace(lo, hi, 4 * n))
    grid |= {T10, T5, T6, T3, np.nextafter(T3, lo)}   # break points
    Ts = np.array(sorted(t for t in grid if lo <= t <= hi))
    Qs = np.array([Q_main(t) + Q_reheat(t) for t in Ts])
    keep = np.concatenate(([True], np.diff(Qs) > 1e-12))
    return Qs[keep], Ts[keep]


def orc_pinch(rank, T_w_hot, T_w_cold, fluid, P_water, n=600):
    """Smallest water-minus-ORC temperature difference in the evaporator [K].

    Negative when the two composite curves cross, which is a second-law
    violation: the reported ORC efficiency is then unreachable, whatever the
    component efficiencies.

    The water is the hot stream and follows REAL enthalpy at `P_water`, not a
    straight line in temperature, and the search runs over an even grid in Q
    that includes every break point. The pinch is found wherever it lies; it
    is emphatically not assumed to sit at an end. For the case as shipped in
    v0.8 it sat at 23 % of the duty, where boiling begins.
    """
    Qc, Tc = _orc_cold_composite(rank, fluid)
    Q_total = Qc[-1]
    Tw = np.linspace(T_w_cold, T_w_hot, 400)
    hw = np.array([CP.PropsSI("H", "T", t, "P", P_water, "Water") / 1000.0
                   for t in Tw])
    Qh = (hw - hw[0]) * Q_total / (hw[-1] - hw[0])    # water flow scales out
    Qq = np.unique(np.concatenate([np.linspace(0.0, Q_total, n), Qc, Qh]))
    Qq = Qq[(Qq >= 0.0) & (Qq <= Q_total)]
    return float(np.min(np.interp(Qq, Qh, Tw) - np.interp(Qq, Qc, Tc)))


def feasible_rankine(case, T_1e, T_w_hot, T_w_cold):
    """The ORC evaporating temperature, as the MINIMUM of two constraints.

        T_3e = min( T_2d - DT_2D_3E ,  T_3e^pinch )

    The first is a minimum approach at the HOT END, which is what anybody
    writes first and what this model used alone until v0.9. The second is the
    largest boiling temperature for which the composite curves keep
    `case.DT_pinch_ORC` everywhere, found by bisection. Neither subsumes the
    other, so the binding one is whichever is lower, and the returned dict
    says which it was in `T_3e_binding`.

    At the default DT_2D_3E = 12 K the PINCH binds across the whole useful
    range of glides -- the hot-end rule never becomes active -- so that field
    is inert in practice while remaining a real constraint if raised. Writing
    it as a minimum rather than as "start from the old rule and correct it"
    is the same arithmetic and a more honest statement of intent.
    """
    T_3e_hot_end = T_w_hot - case.DT_2D_3E

    def pinch_at(t):
        return orc_pinch(double_stage_rankine(case.fluid, T_1e, t,
                                              case.eta_turb_s,
                                              case.eta_pump_s,
                                              x_int=case.x_int_orc),
                         T_w_hot, T_w_cold, case.fluid, case.P)

    # is the hot-end value already pinch-feasible?
    if pinch_at(T_3e_hot_end) >= case.DT_pinch_ORC:
        rank = double_stage_rankine(case.fluid, T_1e, T_3e_hot_end,
                                    case.eta_turb_s, case.eta_pump_s,
                                    x_int=case.x_int_orc)
        rank["T_3e_uncorrected"] = T_3e_hot_end
        rank["T_3e_hot_end"] = T_3e_hot_end
        rank["T_3e_binding"] = "hot-end approach DT_2D_3E"
        rank["pinch"] = orc_pinch(rank, T_w_hot, T_w_cold, case.fluid, case.P)
        return rank

    lo = T_1e + 10.0                     # floor: no cycle worth the name below
    hi = T_3e_hot_end
    if pinch_at(lo) < case.DT_pinch_ORC:
        raise RuntimeError(
            f"no ORC boiling temperature above {lo - 273.15:.1f} C clears the "
            f"required pinch of {case.DT_pinch_ORC:.1f} K against water at "
            f"{T_w_hot - 273.15:.1f} -> {T_w_cold - 273.15:.1f} C "
            f"(best is {pinch_at(lo):+.2f} K). The water glide is too large "
            f"for a cycle that boils at one temperature: reduce the "
            f"discharging glide, or raise the water temperatures, or relax "
            f"DT_pinch_ORC.")
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        if pinch_at(mid) >= case.DT_pinch_ORC:
            lo = mid
        else:
            hi = mid
    rank = double_stage_rankine(case.fluid, T_1e, lo,
                                case.eta_turb_s, case.eta_pump_s,
                                x_int=case.x_int_orc)
    rank["T_3e_uncorrected"] = T_3e_hot_end      # kept: reported in css_report
    rank["T_3e_hot_end"] = T_3e_hot_end
    rank["T_3e_binding"] = "evaporator pinch DT_pinch_ORC"
    rank["pinch"] = orc_pinch(rank, T_w_hot, T_w_cold, case.fluid, case.P)
    return rank


def _stream_curve(h_cold_end, h_hot_end, p, fluid, n=400):
    """(Q, T) for a single-pressure stream, from its COLD end.

    Q is duty per unit mass of that stream. Sampling is in enthalpy, so a
    latent plateau comes out flat and a (T, p) call never lands exactly on
    the saturation line.
    """
    hs = np.linspace(h_cold_end, h_hot_end, n)
    Ts = np.array([CP.PropsSI("T", "H", x * 1000.0, "P", p, fluid)
                   for x in hs])
    return hs - h_cold_end, Ts


def _water_curve(T_cold, T_hot, P, Q_total, n=400):
    """(Q, T) for the water, scaled so its total duty matches Q_total."""
    Ts = np.linspace(T_cold, T_hot, n)
    hs = np.array([CP.PropsSI("H", "T", t, "P", P, "Water") / 1000.0
                   for t in Ts])
    if abs(hs[-1] - hs[0]) < 1e-12:                 # isothermal reservoir
        return np.array([0.0, Q_total]), np.array([T_cold, T_hot])
    return (hs - hs[0]) * Q_total / (hs[-1] - hs[0]), Ts


def _UA_from_curves(Qh, Th, Qc, Tc, Q_duty_kW, n=2000):
    """UA and the effective mean approach, from two composite curves.

    Both curves are (Q, T) measured from the COLD END of the exchanger, on
    whatever duty basis; only their SHAPE is used. The conductance follows
    from the local driving difference:

        UA = int dQ / dT      ->     UA = Q_duty * <1/dT>

    and the effective mean approach is the number that reproduces it,
    dT_eff = Q_duty / UA. For a counterflow exchanger with both streams
    sensible and constant cp, dT_eff is the log-mean difference exactly;
    with a phase change on one side it is the correct generalisation, which
    the LMTD is not.

    Returns (UA [kW/K], dT_eff [K], pinch [K]). UA is infinite and dT_eff
    zero if the curves touch -- which is the honest answer, and is what the
    heat-pump evaporator returned before v0.9a gave it an approach.
    """
    Q_top = min(Qh[-1], Qc[-1])
    q = np.linspace(0.0, Q_top, n)
    dT = np.interp(q, Qh, Th) - np.interp(q, Qc, Tc)
    pinch = float(np.min(dT))
    if pinch <= 0.0:
        return float("inf"), 0.0, pinch
    # trapezoid on 1/dT, in units of the curves' own duty, then rescaled
    inv = np.trapezoid(1.0 / dT, q) if hasattr(np, "trapezoid") \
        else np.trapz(1.0 / dT, q)
    UA = Q_duty_kW * inv / Q_top
    return float(UA), float(Q_duty_kW / UA), pinch


def exchanger_UA(case, rank, hp, T, Eb):
    """Conductance UA [kW/K] required by each of the four exchangers.

    The model reports efficiencies but has never reported what they cost in
    hardware. That gap makes every approach temperature a free lunch: tighten
    `DT_pinch_ORC` to 1 K and eta_ORC rises with nothing to pay for it. UA is
    the missing axis, and it is the one that makes the approach temperatures
    genuine design variables rather than free parameters.

    Each exchanger is integrated over its real composite curves -- the same
    construction as `orc_pinch`, so a latent plateau is a flat segment and
    the water follows real enthalpy. Duties are plant-level, from the energy
    budget, so the UA values are for the whole field and add up.

    Returns a dict of dicts, one per exchanger, each with:
        duty_kW, UA_kW_per_K, dT_eff_K, pinch_K
    """
    F, R = case.fluid, case.refrig
    out = {}

    # ---- ORC evaporator: water (hot) against the two ORC streams ----------
    Q_orc = Eb["Q_dot_in_ORC"]
    Qc, Tc = _orc_cold_composite(rank, F)
    Qh, Th = _water_curve(T["T_3d"], T["T_2d"], case.P, Qc[-1])
    out["ORC evaporator"] = dict(zip(
        ("UA_kW_per_K", "dT_eff_K", "pinch_K"),
        _UA_from_curves(Qh, Th, Qc, Tc, Q_orc))) | {"duty_kW": Q_orc}

    # ---- ORC condenser: ORC fluid (hot) against the sink ------------------
    Q_cond_orc = Q_orc * (1.0 - rank["rank_eff"])
    Qh, Th = _stream_curve(rank["h_1e"], rank["h_4e"], rank["p_conde"], F)
    Qc, Tc = _water_curve(case.T_sink_C + 273.15,
                          case.T_sink_C + case.DT_sink_glide + 273.15,
                          case.P, Qh[-1])
    out["ORC condenser"] = dict(zip(
        ("UA_kW_per_K", "dT_eff_K", "pinch_K"),
        _UA_from_curves(Qh, Th, Qc, Tc, Q_cond_orc))) | {"duty_kW": Q_cond_orc}

    # ---- HTHP condenser: refrigerant (hot) against the charging water -----
    Q_cond_hp = Eb["Q_dot_out_HP"]
    Qh, Th = _stream_curve(hp["h_3h"], hp["h_2h"], hp["p_condh"], R)
    Qc, Tc = _water_curve(T["T_2c"], T["T_3c"], case.P, Qh[-1])
    out["HTHP condenser"] = dict(zip(
        ("UA_kW_per_K", "dT_eff_K", "pinch_K"),
        _UA_from_curves(Qh, Th, Qc, Tc, Q_cond_hp))) | {"duty_kW": Q_cond_hp}

    # ---- HTHP evaporator: source water (hot) against the refrigerant ------
    # Duty is the condenser heat less the compressor work: what the source
    # must actually give up.
    Q_evap_hp = Q_cond_hp * (1.0 - 1.0 / hp["hp_cop"])
    Qc, Tc = _stream_curve(hp["h_4h"], hp["h_13h"], hp["p_evaph"], R)
    Qh, Th = _water_curve(T["T_4a"], case.T_source_C + 273.15,
                          case.P, Qc[-1])
    out["HTHP evaporator"] = dict(zip(
        ("UA_kW_per_K", "dT_eff_K", "pinch_K"),
        _UA_from_curves(Qh, Th, Qc, Tc, Q_evap_hp))) | {"duty_kW": Q_evap_hp}

    finite = [v["UA_kW_per_K"] for v in out.values()
              if np.isfinite(v["UA_kW_per_K"])]
    out["TOTAL"] = dict(duty_kW=sum(v["duty_kW"] for v in out.values()),
                        UA_kW_per_K=sum(finite) if len(finite) == 4
                        else float("inf"),
                        dT_eff_K=float("nan"), pinch_K=min(
                            v["pinch_K"] for v in out.values()))
    return out


def ua_report(case, r):
    """Print the conductance each exchanger needs, and what it buys."""
    ua = exchanger_UA(case, r["rank"], r["hp"], r["T"], r["budget"])
    print("=" * 74)
    print("EXCHANGER CONDUCTANCE  (plant totals, all wells)")
    print("=" * 74)
    print(f"  {'exchanger':<18}{'duty':>10}{'UA':>12}"
          f"{'dT_eff':>10}{'pinch':>9}")
    print(f"  {'':<18}{'kW':>10}{'kW/K':>12}{'K':>10}{'K':>9}")
    print("  " + "-" * 60)
    for name in ("HTHP evaporator", "HTHP condenser",
                 "ORC evaporator", "ORC condenser"):
        v = ua[name]
        print(f"  {name:<18}{v['duty_kW']:>10.1f}{v['UA_kW_per_K']:>12.2f}"
              f"{v['dT_eff_K']:>10.2f}{v['pinch_K']:>9.2f}")
    print("  " + "-" * 60)
    t = ua["TOTAL"]
    print(f"  {'TOTAL':<18}{t['duty_kW']:>10.1f}{t['UA_kW_per_K']:>12.2f}"
          f"{'':>10}{t['pinch_K']:>9.2f}")
    print()
    print("  dT_eff is the single approach that would need the same UA. For a")
    print("  counterflow exchanger with both streams sensible it IS the")
    print("  log-mean difference; with a phase change on one side it is the")
    print("  correct generalisation and the LMTD is not.")
    print()
    print("  Read this beside eta_RTE. An approach temperature bought cheaply")
    print("  in efficiency is paid for here, and until v0.10 nothing said so.")
    return ua


def cycle_state_points(case, T_2d=None):
    """ORC and heat-pump state points, and the two cycle efficiencies.

    Note the expansions and compressions carry no isentropic efficiency, so
    eta_ORC and COP are idealized -- only the electrical and mechanical
    efficiencies appear.

    THE DISCHARGE CLOSURE (changed in v0.6). Only the two exchanger INLETS are
    boundary conditions on the march; both outlets are results of the heat
    transfer and cannot be prescribed. Until v0.5a the discharge was closed the
    other way round -- the outlet was pinned at T_2d = T_m,top - DT_m_2D with
    DT_m_2D = 0, and the inlet derived by subtracting the glide. That is not
    merely arbitrary, it is contradicted by the model's own solution: at the
    start of a discharge at CSS the water leaves at 157.93 C, nearly 8 K above
    the top layer's melting point, because the PCM is
    superheated. It is also silently tied to N_lay -- the implied inlet
    approach was DT_3C_2C / N_lay, one layer width, chosen by nobody.

    The inlets are now prescribed symmetrically:

        charge:    T_4c = T_m,top    + DT_4C_M     (superheat, drives melting)
        discharge: T_3d = T_m,bottom - DT_M_4D     (subcooling, drives freezing)

    and T_2d is DEMOTED to a provisional estimate, the value it would take if
    the water achieved the full glide. It is provisional because the ORC needs
    an evaporating temperature before the store has been marched: T_3e is tied
    to T_2d. Pass the realized outlet back in as `T_2d` for the correction pass
    (see `simulate_css`); the estimate is good enough that one pass suffices.

    Setting DT_M_4D = DT_3C_2C / N_lay recovers the retired closure exactly,
    which is how the change is verified to be a no-op.
    """
    T_3d_C = T_m_bottom(case) - 273.15 - case.DT_M_4D
    # WATER-LOOP NUMBERING (round 25, Fig. 1): 1 is the FIELD OUTLET and 4
    # the FIELD INLET, in both half-cycles.
    #   charge     1c -> pump -> 2c -> condenser -> 3c -> line -> 4c
    #   discharge  1d -> line -> 2d -> evaporator -> 3d -> pump -> 4d
    # The argument named T_2d is the FIELD OUTLET, 1d: it is what the store
    # hands over, provisional until the march returns the realized value.
    # The evaporator inlet 2d is that less the line loss.
    T_1d_C = (T_3d_C + case.glide_dc_spec) if T_2d is None else (T_2d - 273.15)
    T_2d_C = T_1d_C - case.line_dc_dT
    T = dict(
        T_1e=case.T_sink_C + case.DT_sink_glide + case.DT_E_sink + 273.15,
        T_3e=T_2d_C - case.DT_2D_3E + 273.15,
        T_1d=T_1d_C + 273.15,                     # FIELD OUTLET on discharge
        T_2d=T_2d_C + 273.15,                     # evaporator inlet
        T_3d=T_3d_C + 273.15,                     # evaporator outlet
        T_4c=case.T_m_C + case.DT_4C_M + 273.15,  # FIELD INLET on charge
        # The source stream, and the evaporating temperature DERIVED from its
        # outlet. Until v0.9 these were two independent fields and the
        # approach between them was their difference, unchecked: the default
        # pair made it exactly zero, and reversing them made it negative with
        # no symptom anywhere, because T_4a was written here and never read
        # again. See DN-17.
        T_4a=case.T_source_C - case.DT_3A_4A + 273.15,
        T_13h=case.T_source_C - case.DT_3A_4A - case.DT_pinch_HPE + 273.15,
    )
    # 3c is what the CONDENSER must deliver so that the field receives 4c:
    # the line loses line_ch_dT on the way, so the condenser works that much
    # harder. The causality runs 4c -> 3c because the field inlet is pinned
    # by the top of the cascade, not by the plant.
    T["T_3c"] = T["T_4c"] + case.line_ch_dT
    T["T_2c"] = T["T_3c"] - case.DT_3C_2C
    T["T_2h"] = T["T_3c"] + case.DT_2H_3C
    # PUMP HEATING IS NEGLECTED, written out rather than left implicit so
    # that it is one line to replace when it is not.
    T["T_1c"] = T["T_2c"]        # field outlet on charge == pump inlet
    T["T_4d"] = T["T_3d"]        # evaporator outlet == field inlet on dc

    # T["T_3e"] above is only the OLD hot-end rule, kept so the uncorrected
    # value stays visible. What the water can actually deliver is decided by
    # the whole evaporator, so the boiling temperature comes back from the
    # pinch check and is written into T -- otherwise the reported state points
    # and the efficiency would describe different cycles. This runs on the
    # correction pass too, with the realized T_2d.
    rank = feasible_rankine(case, T["T_1e"], T["T_2d"], T["T_3d"])
    T["T_3e_hot_end_rule"] = T["T_3e"]
    T["T_3e"] = rank["T_3e"]
    hp = two_stage_htheatpump_2regs(case.refrig, T["T_13h"], T["T_2h"],
                                    case.DT_sub, case.eta_comp_s,
                                    x_int=case.x_int_hp)
    for name, v in (("rank_eff", rank.get("rank_eff")),
                    ("hp_cop", hp.get("hp_cop"))):
        if v is None or not np.isfinite(v):
            raise RuntimeError(f"{name} is not finite ({v})")
    return rank, hp, T


def water_h(case, T_K):
    """Specific enthalpy of the heat-transfer water, kJ/kg."""
    return CP.PropsSI("H", "T", T_K, "P", case.P, case.fluid2) / 1000.0


def water_s(case, T_K):
    """Specific entropy of the heat-transfer water, kJ/kg-K."""
    return CP.PropsSI("S", "T", T_K, "P", case.P, case.fluid2) / 1000.0


def stream_exergy_rate(case, m_dot, T_hot, T_cold):
    """Exergy given up by a water stream cooling from T_hot to T_cold, kW.

        m_dot [ (h_hot - h_cold) - T_0 (s_hot - s_cold) ],   T_0 = T_sink

    Positive when T_hot > T_cold >= T_0. Used for the process water and for
    the geothermal stream, which are the same fluid at the same pressure.
    Passing T_cold = T_0 gives the stream's exergy relative to the dead
    state, which is what the 'resource' convention charges for.
    """
    T_0 = case.T_sink_C + 273.15
    return m_dot * ((water_h(case, T_hot) - water_h(case, T_cold))
                    - T_0 * (water_s(case, T_hot) - water_s(case, T_cold)))


def energy_budget(case, rank_eff, hp_cop, T):
    """Work backwards from the specified electrical output to the charging duty."""
    # rank_eff and hp_cop now ALREADY carry the machine losses, so only
    # the electrical efficiencies remain here. Dividing by Turb_eff as
    # well would double-count them.
    W_dot_T = case.W_dot_el_out / case.ElG_eff
    Q_dot_in_ORC = W_dot_T / rank_eff
    D_E_in_ORC = Q_dot_in_ORC * case.t_dc * 3600.0                 # kJ
    D_E_out_HP = D_E_in_ORC * (1.0 + case.loss_surplus)
    Q_dot_out_HP = D_E_out_HP / case.t_ch / 3600.0
    W_dot_C_HP = Q_dot_out_HP / hp_cop
    W_dot_el_in = W_dot_C_HP / case.ElH_eff

    # v0.13: the ENTHALPY difference, not c_p at the mean temperature times
    # the glide. Over the 105 -> 160 C charging rise c_p varies about 3 %, so
    # the linearised form gives a flow rate 0.096 % inconsistent with the
    # model's own CoolProp data. That is invisible in the energy balance --
    # both sides use the same wrong flow -- but it fabricates 7.17 kW of
    # exergy and stopped `exergy_audit` closing. The identical formula was
    # written a SECOND time in `run_cycle`, where it is live; see the note
    # there. DN-24.
    m_dot_w_ch = Q_dot_out_HP / (water_h(case, T["T_3c"])
                                 - water_h(case, T["T_2c"]))
    return dict(Q_dot_in_ORC=Q_dot_in_ORC, D_E_in_ORC=D_E_in_ORC,
                D_E_out_HP=D_E_out_HP, Q_dot_out_HP=Q_dot_out_HP,
                W_dot_el_in=W_dot_el_in, m_dot_w_ch=m_dot_w_ch)


def melting_temperatures(case):
    """Layer melting temperatures for charging and for discharging."""
    dTm = case.cascade_param / case.N_lay
    i = np.arange(case.N_lay, dtype=float)
    return ((case.T_m - i * dTm).tolist(),
            (case.T_m - case.cascade_param + (i + 1.0) * dTm).tolist())


# ==========================================================================
# sizing
# ==========================================================================


# ==========================================================================
# run   (charge, size, discharge, KPIs)
# ==========================================================================


# ==========================================================================
# cyclic steady state   (v0.5)
# ==========================================================================
# run_cycle() above answers "how many wells does one charge from a COLD START
# need?". That is a first-cycle question, and the cycle does not return to the
# state it started from, so the answer is not what a plant repeating a 24 h
# schedule would see.
#
# WHY SIZING CANNOT SIMPLY BE MOVED TO CSS. At cyclic steady state the state
# returns to itself, so the enthalpy change over a cycle is zero. This model has
# no loss path -- H2 no axial conduction, H6 adiabatic wall, H9 no azimuthal
# exchange -- so charge energy EQUALS discharge energy identically:
#
#       eta_storage(CSS) == 1     exactly, for every N and every flow.
#
# Two consequences follow, and both are structural rather than numerical.
#
#   1. The assumed loss surplus lambda is UNATTAINABLE at CSS. The budget
#      requires D_E_out_HP = (1+lambda) D_E_in_ORC, so the ratio of CSS charge
#      to requirement pins at 1/(1+lambda) for EVERY N. There is no root.
#      On the first cycle the surplus is not lost but PARKED in the store as
#      residual melt; at CSS there is nowhere left to park it.
#
#   2. Even with lambda = 0 the rate criterion goes DEGENERATE. Charge and
#      discharge are then the same number, so one equation is left for two
#      unknowns: it fixes the FLOW RATIO and says nothing about N.
#
# THE REFORMULATION. Stop asking the energy balance to determine N. Specify the
# hardware, march to CSS, and report what comes out:
#
#       N        from latent heat alone, D_E_out_HP / (rho V_well h_m)
#       m_ch     pinned by the charging glide
#       m_dc     pinned by the discharging glide
#
# Nothing is solved -- there is no root-find anywhere in this routine -- so the
# question of convergence does not arise. What was an unsatisfiable constraint
# becomes a reported output: the deviation of the delivered energy, and hence of
# the net electrical output, from its target.
#
# This also dissolves an older objection. size_well_field's docstring argues
# that the discharge flow cannot be pinned to its glide because the delivered
# energy then saturates below the requirement for any well count, leaving no
# root. Under this framing that saturation is not a failure; it is the answer.


def simulate_css(case, N=None, n_cycles=80, tol=1e-9, record=False, T_2d=None):
    """March a specified well field to cyclic steady state.

    One pass. `T_2d` overrides the provisional exchanger outlet used to set the
    ORC evaporating temperature; `simulate_css_corrected` is the wrapper that
    supplies the realized value. Called directly, this is the uncorrected
    result.

    No root-finding. `N` defaults to the latent-heat-only well count and both
    flows are pinned by their glides, so the routine is a pure simulation.

    `lambda` is forced to zero regardless of `case.loss_surplus`, because a
    non-zero surplus cannot be absorbed at CSS in a model with no loss path
    (see the note above). The override is reported in the returned dict as
    `lambda_overridden` so it can never happen silently.

    Returns the converged half-cycle states, the convergence history, and the
    deviation of the delivered energy from the target.
    """
    lambda_overridden = case.loss_surplus != 0.0
    case = case.with_(loss_surplus=0.0) if lambda_overridden else case

    rank, hp, T = cycle_state_points(case, T_2d=T_2d)
    Eb = energy_budget(case, rank["rank_eff"], hp["hp_cop"], T)
    n = case.n_segments
    T_m_lay, T_m_lay_dc = melting_temperatures(case)
    T_m_seg = layer_map(T_m_lay, n, case.N_lay)
    T_m_seg_dc = T_m_seg[::-1].copy()     # flow reverses; see run_cycle
    # The rock profile is mirrored with exactly the same rule as T_m_seg, for
    # exactly the same reason: on discharge the flow enters at the top.
    T_rock_ch, depth_ch = formation_profile(case, n, discharge=False)
    T_rock_dc, depth_dc = formation_profile(case, n, discharge=True)
    # The resistance is now a PROFILE, because the spacing is. Built on the
    # same two frames as T_rock so a mirroring slip cannot put the shielded
    # shallow resistance against the deep rock.
    R_form_ch = formation_resistance(case, depth_ch)
    R_form_dc = formation_resistance(case, depth_dc)
    times_ch = np.logspace(0.0, np.log10(case.t_ch * 3600.0), case.n_times)
    times_dc = np.logspace(0.0, np.log10(case.t_dc * 3600.0), case.n_times)
    t_idle_h = idle_hours(case)
    times_idle = (np.logspace(0.0, np.log10(t_idle_h * 3600.0), case.n_times)
                  if t_idle_h > 0.0 else None)

    # ---- hardware and flows: all SPECIFIED, none solved -------------------
    if N is None:
        N = Eb["D_E_out_HP"] * 1000.0 / (case.rho_latent * case.V_well * case.h_m)
    # v0.13: enthalpy differences, not c_p at the mean temperature times the
    # glide. This is the LIVE copy of the formula fixed in `energy_budget`;
    # the two were written independently and both were wrong the same way,
    # which is why patching only one of them produced an encouraging no-op
    # and left the defect in place. See DN-24.
    m1_ch = (Eb["Q_dot_out_HP"] / (water_h(case, T["T_3c"])
                                   - water_h(case, T["T_2c"]))
             / (N * case.n_ft))
    # NOTE the discharge flow stays pinned by the SPECIFIED glide, as it has
    # been since v0.6, and not by the realized T_2d that the correction pass
    # knows about. Using the realized outlet here is tempting -- it cuts the
    # reported deviation from +4.46 % to +1.45 % and lifts eta_RTE by 0.28 %
    # -- but that is partly circular: feeding last pass's outlet into this
    # pass's flow definition drives the delivered energy toward the target by
    # construction, without iterating to a fixed point. It would also change
    # the model's SPECIFICATION ("the flow is pinned by the glide"), which is
    # a different decision from fixing a linearisation. Measured and left
    # alone. See DN-25.
    m1_dc = (Eb["Q_dot_in_ORC"]
             / (water_h(case, T["T_3d"] + case.glide_dc_spec)
                - water_h(case, T["T_3d"]))
             / (N * case.n_ft))

    # ---- march until the state repeats -----------------------------------
    def cycle(E0, rec=False):
        ch = march_h(case, T["T_4c"], T_m_seg, m1_ch, case.k_wall, times_ch,
                     n_segments=n, E0=E0, record=rec, T_rock=T_rock_ch,
                     R_form=R_form_ch)
        # IDLE 1: after charge, in CHARGE indexing. The store is at its
        # hottest here, so this is the costlier of the two periods.
        if times_idle is None:
            id1 = None
            E_after_ch = ch["E"]
        else:
            id1 = idle_h(case, T_m_seg, times_idle, n_segments=n,
                         E0=ch["E"], T_rock=T_rock_ch, R_form=R_form_ch,
                         record=rec)
            E_after_ch = id1["E"]
        dc = march_h(case, T["T_3d"], T_m_seg_dc, m1_dc, case.k_wall, times_dc,
                     n_segments=n, E0=E_after_ch[::-1], record=rec,
                     T_rock=T_rock_dc, R_form=R_form_dc)
        # IDLE 2: after discharge, in DISCHARGE indexing, so the mirrored
        # cascade and rock profile are the right ones to hand it.
        if times_idle is None:
            id2 = None
        else:
            id2 = idle_h(case, T_m_seg_dc, times_idle, n_segments=n,
                         E0=dc["E"], T_rock=T_rock_dc, R_form=R_form_dc,
                         record=rec)
            dc = dict(dc); dc["E"] = id2["E"]
        return ch, dc, id1, id2

    E = np.zeros(n)
    history = []
    for k in range(n_cycles):
        ch, dc, id1, id2 = cycle(E)
        Q_ch = ch["Q_cum_J"] * case.n_ft * N / 1000.0          # kJ, field
        # SIGNED, not abs(). The march returns heat INTO the store, so a
        # delivering discharge is negative and the sign flip is all that is
        # wanted. abs() instead reports a store that ABSORBS on discharge --
        # which happens whenever T_2d < T_3d, i.e. isolated wells shorter
        # than ~700 m -- as though it had delivered, and every downstream
        # quantity inherits the lie (e', RTE, and eta_storage = 1 - loss
        # with loss > 1). Verified: an isolated 300 m well integrates to
        # +8.7e7 J/m with T_2d = 85.5 C against T_3d = 91.1 C.
        Q_dc = -dc["Q_cum_J"] * case.n_ft * N / 1000.0
        drift = float(np.max(np.abs(dc["E"][::-1] - E)))
        history.append(dict(cycle=k + 1, Q_charge_kJ=Q_ch, Q_discharge_kJ=Q_dc,
                            drift=drift,
                            eps_end_charge=float(ch["eps_local"].mean()),
                            eps_end_discharge=float(dc["eps_local"].mean())))
        E = dc["E"][::-1]                 # back to charge-march indexing
        if drift < tol:
            break

    if record:
        # One extra cycle from the converged state, this time recording. The
        # state repeats to `tol`, so this reproduces the last cycle; doing it
        # here rather than inside the loop means the profile sweep that closes
        # each recorded march is paid once instead of `cycles` times.
        ch, dc, id1, id2 = cycle(E, rec=True)

    # The discharge marches from the far end, so its segment index runs
    # backwards. Everything returned to the caller is in CHARGE (depth)
    # indexing, which is why there is one cascade, `T_m_seg`, and not two.
    dc = unmirror_march(dc)

    _Q_idle = ((id1["Q_loss_J"] if id1 else 0.0)
               + (id2["Q_loss_J"] if id2 else 0.0))

    req = Eb["D_E_in_ORC"]
    deviation = Q_dc / req - 1.0

    # ---- the outlet temperatures, which are results and not inputs ---------
    # THE MARCH RETURNS THE FIELD OUTLET, which is state 1d. What the
    # evaporator sees is that less the discharge line loss, state 2d. The
    # two are equal while line_dc_dT is zero, which is the shipped
    # default, so nothing downstream moves.
    T_1d_realized = mixed_mean_outlet(dc)      # FIELD OUTLET, state 1d
    T_2d_realized = T_1d_realized - case.line_dc_dT   # evaporator inlet
    T_2c_realized = mixed_mean_outlet(ch)      # what returns to the HP condenser
    glide_dc = T_1d_realized - T["T_3d"]       # the STORE's glide
    glide_ch = T["T_4c"] - T_2c_realized
    return dict(
        N_wells=N, m1_ch=m1_ch, m1_dc=m1_dc, flow_ratio=m1_dc / m1_ch,
        cycles=len(history), converged=history[-1]["drift"] < tol, tol=tol,
        history=history, charge=ch, discharge=dc, E_css=E,
        Q_charge_kJ=Q_ch, Q_discharge_kJ=Q_dc, required_kJ=req,
        # Does the store actually give heat back? Both conditions, because
        # either alone can be satisfied by a store that does not.
        delivers=bool(T_1d_realized > T["T_3d"] and Q_dc > 0.0),
        deviation=deviation,
        W_el_out_implied=case.W_dot_el_out * (1.0 + deviation),
        # ---- THE NUMBER THAT WAS 1.0000 BY CONSTRUCTION ------------------
        # In THUMS eta_storage was Q_dc/Q_ch with an adiabatic wall, so it
        # was identically 1 and carried no information whatever: it could not
        # move, it could not fail, and reporting it was theatre. Here it is a
        # measurement. Q_loss is signed and SUMMED, not accumulated in
        # magnitude -- the deep segments can sit in rock hotter than they are
        # and run the other way, and that is a result, not an error to be
        # absorbed by an abs().
        Q_loss_ch_kJ=ch["Q_loss_J"] * case.n_ft * N / 1000.0,
        Q_loss_dc_kJ=dc["Q_loss_J"] * case.n_ft * N / 1000.0,
        # The idle loss has to be ADDED here explicitly, but it needs no
        # help reaching eta_storage: idling leaves the store cooler, so the
        # discharge that follows delivers less and Q_dc/Q_ch falls on its
        # own. The two routes are independent, which is the point -- if the
        # idle loss showed up in Q_loss_kJ and NOT in eta_storage, the
        # bookkeeping and the physics would have come apart.
        Q_loss_idle_kJ=_Q_idle * case.n_ft * N / 1000.0,
        t_idle_h=t_idle_h,
        Q_loss_kJ=(ch["Q_loss_J"] + dc["Q_loss_J"] + _Q_idle)
                  * case.n_ft * N / 1000.0,
        R_formation=float(np.mean(np.atleast_1d(
            formation_resistance(case, depth_ch)))),
        B_top=(None if case.n_wells_field is None
               else float(np.atleast_1d(well_spacing(case, 0.0))[0])),
        B_bottom=(None if case.n_wells_field is None
                  else float(np.atleast_1d(
                      well_spacing(case, case.L_well))[0])),
        R_ground=ground_resistance(case),
        eta_storage=Q_dc / Q_ch, lambda_overridden=lambda_overridden,
        T_2d_assumed=T["T_2d"], T_2d_realized=T_2d_realized,
        T_1d_assumed=T["T_1d"], T_1d_realized=T_1d_realized,
        T_2c_assumed=T["T_2c"], T_2c_realized=T_2c_realized,
        glide_dc=glide_dc, glide_ch=glide_ch,
        glide_ratio_dc=glide_dc / case.glide_dc_spec,
        T_2d_was_provisional=T_2d is None,
        T=T, budget=Eb, T_m_seg=T_m_seg, rank=rank, hp=hp)
        # no `T_m_seg_dc`: the discharge is returned in depth indexing, where
        # the cascade is the same physical object as on charge. The mirrored
        # copy is an internal detail of the march and does not leave here.


def geothermal_resource(case, r):
    """The 60 C source as a FINITE, priced resource.

    Until v0.13 the source was effectively unlimited: `DT_3A_4A` said how far
    the stream was cooled, the flow followed from the duty, and no output ever
    recorded how much flow that was. Nothing in the model objected to a design
    that quietly demanded five times the geothermal field.

    The flow is DERIVED, not prescribed -- the same choice already made for
    `N_wells`. Prescribing a plant-level flow would put the uncertainty in a
    quantity nobody has measured; deriving it puts the uncertainty in the
    per-well yield, which is field data. See DN-22.

    Note that the exergy of the source is homogeneous of degree one in the
    flow, and m_geo = Q_src / (c_p dT), so BOTH conventions below reduce to
    the duty, the glide and the two temperatures. Deriving the flow rather
    than prescribing it costs the second-law analysis nothing.
    """
    hp, T = r["hp"], r["T"]
    T_0 = case.T_sink_C + 273.15
    T_src = case.T_source_C + 273.15
    T_rei = T["T_4a"]                          # T_source - DT_3A_4A

    x = hp["x_8h"]
    m_ref = r["budget"]["Q_dot_out_HP"] / (hp["h_2h"] - hp["h_3h"])
    Q_src = m_ref * (1.0 - x) * (hp["h_13h"] - hp["h_4h"])          # kW
    m_geo = Q_src / (water_h(case, T_src) - water_h(case, T_rei))   # kg/s

    ex_full = stream_exergy_rate(case, m_geo, T_src, T_0)
    ex_rei = stream_exergy_rate(case, m_geo, T_rei, T_0)
    ex_stripped = stream_exergy_rate(case, m_geo, T_src, T_rei)

    N_geo = m_geo / case.m_dot_geo_well
    return dict(
        Q_src_kW=Q_src, m_geo=m_geo, m_ref=m_ref,
        T_reinject_C=T_rei - 273.15,
        Ex_full_kW=ex_full, Ex_reinject_kW=ex_rei, Ex_stripped_kW=ex_stripped,
        utilization=ex_stripped / ex_full if ex_full else float("nan"),
        N_geo=N_geo,
        # TARGET basis, kept for traceability only. The reported
        # N_geo_per_MWe is in performance_indices, on W_net.
        N_geo_per_MWe_target=N_geo / (case.W_dot_el_out / 1000.0),
        # which of the two the audit will charge for
        Ex_charged_kW=(ex_full if case.exergy_convention == "resource"
                       else ex_stripped),
        Ex_loss_kW=(ex_rei if case.exergy_convention == "resource" else 0.0),
        convention=case.exergy_convention)


def source_pumping(case, r):
    """Source-loop pumping power during CHARGE, kW.

    The heat-pump source loop runs only while the store is charging, so
    this is a charge-side parasitic and enters eta_RTE through the same
    denominator as the field charge pumping.

        W = m_src / rho * dp / eta_pump

    `m_src` is the source mass flow the HTHP evaporator needs, which
    `geothermal_resource` already computes from the evaporator duty and
    DT_3A_4A. A scarce source needs a big DT and little flow; seawater
    needs a small DT and a lot of flow, and this is where it pays for it.
    Set `source_loop_dp_bar = 0` to recover the v2.8 behaviour, in which
    the source arrived free.
    """
    if not case.source_loop_dp_bar:
        return 0.0
    m_src = geothermal_resource(case, r)["m_geo"]
    dp = case.source_loop_dp_bar * 1e5                       # Pa
    return m_src / case.rho_source * dp / case.source_pump_eff / 1000.0


def sink_pumping(case, r):
    """ORC condenser seawater pumping during DISCHARGE, kW.

    The condenser runs only while the store discharges, so unlike the
    source loop this is a DISCHARGE-side parasitic: it comes off the
    numerator of eta_RTE, not the denominator, and it is therefore the
    more expensive of the two per kilowatt.
    """
    if not case.sink_loop_dp_bar or case.DT_sink_glide <= 0.0:
        return 0.0
    try:
        ua = exchanger_UA(case, r["rank"], r["hp"], r["T"], r["budget"])
        duty = ua["ORC condenser"]["duty_kW"]
    except Exception:
        return 0.0
    cp = 4.0                                    # kJ/kg/K, seawater ~ 17 C
    m = duty / (cp * case.DT_sink_glide)        # kg/s
    dp = case.sink_loop_dp_bar * 1e5
    return m / case.rho_sink * dp / case.sink_pump_eff / 1000.0


def borehole_pumping(case, r):
    """Field pumping power for both half-cycles, kW, at the realized flows.

    ONE copy, called by `performance_indices` and by `exergy_audit`. It was
    briefly written twice -- which is the DN-24 trap exactly, a formula
    duplicated so that a later fix can land in one copy and miss the other.
    """
    gv = case.geom_vector()
    N, T = r["N_wells"], r["T"]
    _, pp_dc = calculate_pressure_drop(gv, r["m1_dc"], case.fluid2,
                                       r["T_2d_realized"], T["T_3d"], case.P)
    _, pp_ch = calculate_pressure_drop(gv, r["m1_ch"], case.fluid2,
                                       T["T_4c"], r["T_2c_realized"], case.P)
    f = case.n_ft * N / 1000.0

    # ---- THE DOWNCOMER IS ALSO A PIPE -----------------------------------
    # Easy to forget, because it is the passage that does no heat transfer --
    # and forgetting it would flatter BOREAS against THUMS on precisely the
    # quantity Study 2 showed decides efficiency below ground. The fin tubes
    # now run L_well where the hairpin ran 2*L_well, so leaving the downcomer
    # out would halve the pumping rather than keep it equal.
    #
    # It carries the WHOLE well's flow (n_ft * m1) through a bore sized by
    # r_id_bore = r_i * sqrt(n_ft), i.e. n_ft times the area. Same velocity,
    # same length, so to the extent friction factors match it contributes the
    # same drop as one fin tube's worth -- which is the sense in which this
    # geometry has the hairpin's pressure drop, not half of it.
    gv_id = list(gv)
    gv_id[3] = case.r_id_bore                    # r_i
    gv_id[5] = 2.0 * case.r_id_bore              # D_i_tube, what the drop uses
    gv_id[8], gv_id[9] = 0.0, 0                  # no fins inside the downcomer
    # n_bends = 0: the 180 deg turn at the bottom is ONE fitting shared by
    # the fin tube and the downcomer, and it is already counted on the
    # fin-tube leg above. Counting it on both legs charged the well four
    # bends where it has two.
    _, pid_ch = calculate_pressure_drop(gv_id, r["m1_ch"] * case.n_ft,
                                        case.fluid2, T["T_4c"],
                                        T["T_4c"], case.P, n_bends=0)
    _, pid_dc = calculate_pressure_drop(gv_id, r["m1_dc"] * case.n_ft,
                                        case.fluid2, T["T_3d"],
                                        T["T_3d"], case.P, n_bends=0)
    f_id = N / 1000.0                            # ONE downcomer per well

    # ---- the surface lines between the plant and the field -------------
    # Fig. 1's coils. Zero by default, so this adds nothing today; when a
    # line pressure drop is set it is a field-level duty, carried by the
    # whole discharge or charge flow rather than per tube.
    m_ch = r["m1_ch"] * case.n_ft * N
    m_dc = r["m1_dc"] * case.n_ft * N
    rho_w = 1000.0
    p_line_ch = case.line_ch_dp * m_ch / rho_w / 1000.0     # kW
    p_line_dc = case.line_dc_dp * m_dc / rho_w / 1000.0     # kW
    return (pp_ch * f + pid_ch * f_id + p_line_ch,
            pp_dc * f + pid_dc * f_id + p_line_dc)


def downcomer_leak(case, r):
    """Heat crossing the downcomer insulation, per half-cycle [kJ, field].

    THE SIGN REVERSES BETWEEN HALF-CYCLES, and the two are not equal.

    On CHARGE the downcomer carries the 160 C supply down past PCM grading
    101 C at the top to 150 C at the bottom, so heat leaks OUT of the pipe and
    INTO the store. The energy is not lost -- it is deposited in the wrong
    layer, at a large temperature mismatch. An exergy penalty, not an energy
    one. This is the advantage the geometry has over a nested coaxial, where
    the insulated passage carries the COLD return and its leak un-stores heat
    and hands it back to the heat pump.

    On DISCHARGE the downcomer carries the hot outlet back up past the same
    PCM, and the leak re-deposits heat that was just extracted. That one is a
    true capacity loss, and it is the larger of the two because the driving
    difference is larger.

    Quasi-steady and evaluated against the CASCADE temperature rather than the
    realized PCM temperature: the downcomer is a boundary condition on the
    store, not a participant in its march, and resolving it properly would
    mean coupling it segment by segment into `march_h`. Reported separately so
    it is never silently folded into the formation loss, which is a different
    mechanism with a different sign convention.
    """
    T, n = r["T"], case.n_segments
    T_m_seg = r["T_m_seg"]
    R_ins = (np.log(case.r_id_outer / case.r_id_bore)
             / (2.0 * np.pi * case.k_ins))
    dz = case.L_well / n

    # charge: supply descends at T_4c; discharge: return ascends at T_2d
    q_ch = float(np.sum((T["T_4c"] - T_m_seg) / R_ins) * dz)
    q_dc = float(np.sum((r["T_2d_realized"] - T_m_seg) / R_ins) * dz)
    N = r["N_wells"]
    return (q_ch * N * case.t_ch * 3600.0 / 1e3,      # kJ, into the store
            q_dc * N * case.t_dc * 3600.0 / 1e3)


def exergy_audit(case, r):
    """Exergy destruction by component over one cycle at CSS, in kWh.

    THE DEAD STATE is T_0 = T_sink. It is the lowest temperature in the
    system, so every exergy is positive, and the heat dumped at the ORC
    condenser is genuinely unrecoverable. It is NOT the heat-pump evaporator
    source: that is the 60 C geothermal stream, a second and quite separate
    reservoir, which is why `geothermal_resource` has to price it.

    THE CLOSURE, and why this function is a gate rather than a report:

        W_el_in + Ex_geo  =  W_el_out + Ex_reinject + sum(I_j)

    The left and right sides are computed from DIFFERENT things. The boundary
    terms come from four stream states. Every I_j comes from the entropy
    generated inside one component, from its own state points and flow
    fractions. If the cycle state points, the flow splits and the duties are
    mutually consistent, the two agree to machine precision; if any one of
    them is wrong, they do not. That is what caught the c_p linearisation in
    the water flow rates (DN-24), which was invisible to every energy balance
    in the model because both sides of those balances used the same wrong
    flow.

    WHAT IS A DESTRUCTION AND WHAT IS A LOSS. The reinjected geothermal
    stream leaves the boundary intact -- its exergy is not destroyed, it is
    discarded. It is reported on its own line and attributed to NO component,
    because no component is responsible for it; the decision to size the
    field this way is. Under the 'stripped' convention it is not charged at
    all and the line is zero.

    THE BOREHOLE is one bucket here, obtained from the difference of the two
    water-stream exergies over a full cycle. That is exact at CSS -- the PCM
    returns to its own initial state, so its net exergy change is zero, and no
    PCM entropy function is needed. It does not say WHERE in the well or WHEN
    the destruction happens; that needs s(E') and is the next step.
    """
    T_0 = case.T_sink_C + 273.15
    hp, rank, T, Eb = r["hp"], r["rank"], r["T"], r["budget"]
    t_ch, t_dc = case.t_ch, case.t_dc                       # hours
    geo = geothermal_resource(case, r)
    m_ref = geo["m_ref"]
    x = hp["x_8h"]

    # ---- heat pump, per kg of condenser flow ------------------------------
    s = lambda k: hp["s_" + k] / 1000.0                     # kJ/kg-K
    h = lambda k: hp["h_" + k]
    gen = {}
    gen["HTHP compressor HP"] = 1.0 * (s("2h") - s("6h"))
    gen["HTHP compressor LP"] = (1.0 - x) * (s("5h") - s("1h"))
    gen["HTHP throttle 7h-8h"] = 1.0 * (s("8h") - s("7h"))
    gen["HTHP throttle 9h-4h"] = (1.0 - x) * (s("4h") - s("9h"))
    gen["HTHP flash separator"] = (x * s("10h") + (1.0 - x) * s("9h")
                                   - s("8h"))
    gen["HTHP IHX-1"] = 1.0 * (s("12h") - s("3h")) + x * (s("11h") - s("10h"))
    gen["HTHP IHX-2"] = (1.0 * (s("7h") - s("12h"))
                         + (1.0 - x) * (s("1h") - s("13h")))
    m_w_ch = Eb["m_dot_w_ch"]
    gen["HTHP condenser"] = (1.0 * (s("3h") - s("2h"))
                             + m_w_ch * (water_s(case, T["T_3c"])
                                         - water_s(case, T["T_2c"])) / m_ref)
    gen["HTHP evaporator"] = ((1.0 - x) * (s("13h") - s("4h"))
                              + geo["m_geo"]
                              * (water_s(case, T["T_4a"])
                                 - water_s(case, case.T_source_C + 273.15))
                              / m_ref)
    I = {k: T_0 * m_ref * v * t_ch for k, v in gen.items()}     # kWh/cycle

    W_comp = m_ref * ((h("2h") - h("6h")) + (1.0 - x) * (h("5h") - h("1h")))
    W_el_in = Eb["W_dot_el_in"]
    I["HTHP motor"] = (W_el_in - W_comp) * t_ch

    # ---- ORC, per kg through the first turbine stage ----------------------
    y = rank["y_frac"]
    se = lambda k: rank["s_" + k] / 1000.0
    he = lambda k: rank["h_" + k]
    m_orc = Eb["Q_dot_in_ORC"] / ((he("3e") - he("10e"))
                                  + y * (he("6e") - he("5e")))
    ge = {}
    ge["ORC turbine 1"] = 1.0 * (se("5e") - se("3e"))
    ge["ORC turbine 2"] = y * (se("4e") - se("6e"))
    ge["ORC pump 1"] = y * (se("2e") - se("1e"))
    ge["ORC pump 2"] = (1.0 - y) * (se("8e") - se("7e"))
    ge["ORC regenerator"] = ((1.0 - y) * (se("7e") - se("5e"))
                             + y * (se("9e") - se("2e")))
    ge["ORC mixer"] = se("10e") - (y * se("9e") + (1.0 - y) * se("8e"))
    m_w_dc = Eb["Q_dot_in_ORC"] / (water_h(case, T["T_2d"])
                                   - water_h(case, T["T_3d"]))
    ge["ORC evaporator"] = ((1.0 * (se("3e") - se("10e"))
                             + y * (se("6e") - se("5e")))
                            + m_w_dc * (water_s(case, T["T_3d"])
                                        - water_s(case, T["T_2d"])) / m_orc)
    # the sink is AT the dead state, so the rejected heat carries no exergy
    # and the whole of it is destroyed in the condenser
    Q_rej = m_orc * y * (he("4e") - he("1e"))
    ge["ORC condenser"] = y * (se("1e") - se("4e")) + Q_rej / T_0 / m_orc
    for k, v in ge.items():
        I[k] = T_0 * m_orc * v * t_dc

    W_turb = m_orc * ((he("3e") - he("5e")) + y * (he("6e") - he("4e"))
                      - y * (he("2e") - he("1e"))
                      - (1.0 - y) * (he("8e") - he("7e")))
    I["ORC generator"] = (W_turb - case.W_dot_el_out) * t_dc

    # ---- THE TWO GATES THAT CAN ACTUALLY FAIL -----------------------------
    # Each machine bucket is summed from entropy generated INSIDE its
    # components and checked against a boundary balance built from stream
    # states only. These are independent computations and they are what
    # caught DN-24.
    I_hp = sum(v for k, v in I.items() if k.startswith("HTHP"))
    I_orc = sum(v for k, v in I.items() if k.startswith("ORC"))
    dEx_ch = stream_exergy_rate(case, m_w_ch, T["T_3c"], T["T_2c"]) * t_ch
    dEx_dc = stream_exergy_rate(case, m_w_dc, T["T_2d"], T["T_3d"]) * t_dc
    bnd_hp = (W_comp * t_ch + geo["Ex_stripped_kW"] * t_ch - dEx_ch
              + (W_el_in - W_comp) * t_ch)
    bnd_orc = dEx_dc - W_turb * t_dc + (W_turb - case.W_dot_el_out) * t_dc
    gates = {"HTHP": (I_hp, bnd_hp), "ORC": (I_orc, bnd_orc)}

    # ---- borehole: BY DIFFERENCE, and therefore not a gate ----------------
    # T_3c = T_4c and T_2d/T_3d are shared with the surface exchangers, so
    # this term is algebraically the residual of the water loop: it cancels
    # the two exchanger terms exactly and the GLOBAL sum below is an
    # identity, not a test. An earlier draft printed that identity as a
    # closure check reading 0.00e+00 and called it a pass. It cannot fail,
    # so it cannot pass. The independent value comes from the resolved
    # integral over z and t once s(E') exists; until then this number is
    # correct but unverified.
    I["borehole"] = dEx_ch - dEx_dc

    # The water loop does NOT in fact close at CSS: the well returns water at
    # T_2c_realized, not at the T_2c the condenser assumes, and likewise on
    # discharge. Reported rather than absorbed silently.
    mism = float("nan")
    if r.get("T_2c_realized") is not None:
        mism = (stream_exergy_rate(case, m_w_ch, T["T_3c"],
                                   r["T_2c_realized"]) * t_ch - dEx_ch
                - (stream_exergy_rate(case, m_w_dc, r["T_2d_realized"],
                                      T["T_3d"]) * t_dc - dEx_dc))

    # ---- REALISED BASIS, and the same boundary as eta_RTE ----------------
    # Until v0.15 this function evaluated the whole audit on the TARGET
    # duties out of `energy_budget`, and charged no pumping. `eta_RTE` in
    # `performance_indices` does neither: it uses the energy the field
    # actually moved, and it charges the borehole parasitics. So psi and
    # eta_RTE were computed on different bases and different boundaries, and
    # quoting "0.317 becomes 0.246" compared two things that were not
    # like for like. The manuscript caught this before the code did.
    #
    # At CSS the storage efficiency is identically 1 and lambda is forced to
    # zero, so Q_charge = Q_discharge and D_E_out_HP = D_E_in_ORC: both half
    # cycles scale by the SAME factor, and the rebase is one multiplication.
    scale = r["Q_discharge_kJ"] / Eb["D_E_in_ORC"]
    for key in I:
        I[key] *= scale
    W_in = W_el_in * t_ch * scale
    W_out = case.W_dot_el_out * t_dc * scale
    Ex_geo = geo["Ex_charged_kW"] * t_ch * scale
    Ex_loss = geo["Ex_loss_kW"] * t_ch * scale

    # The borehole parasitics are work in, dissipated as friction in the
    # well, so they are an input AND a destruction. Charged with eta_RTE's
    # sign convention: the charge pump adds to the input, the discharge pump
    # subtracts from the output. They are computed at the realized flows
    # already, so they are not scaled.
    pump_ch, pump_dc = borehole_pumping(case, r)
    P_ch, P_dc = pump_ch * t_ch, pump_dc * t_dc
    I["borehole pumping"] = P_ch + P_dc
    W_in += P_ch
    W_out -= P_dc

    boundary = W_in + Ex_geo - W_out - Ex_loss
    total = sum(I.values())

    # A THIRD GATE, and the only one that spans two functions. eta_RTE is
    # computed here from the scaled budget plus the parasitics, and in
    # `performance_indices` from the energy the field actually moved. Two
    # routes, one answer. It is what would have caught the basis mismatch
    # that the manuscript found before the code did, and it can only hold
    # while psi and eta_RTE share a boundary -- which is the point.
    # Recomputed the way `performance_indices` does it -- from the energy the
    # field moved and the same parasitics -- rather than from the budget.
    Q_in_ORC = r["Q_discharge_kJ"] / (t_dc * 3600.0)
    Q_out_HP = r["Q_charge_kJ"] / (t_ch * 3600.0)
    rk_eff = case.W_dot_el_out / case.ElG_eff / Eb["Q_dot_in_ORC"]
    cop_r = Eb["Q_dot_out_HP"] / (Eb["W_dot_el_in"] * case.ElH_eff)
    W_out_pi = (Q_in_ORC * rk_eff * case.ElG_eff - pump_dc) * t_dc
    W_in_pi = (Q_out_HP / cop_r / case.ElH_eff + pump_ch) * t_ch
    gates["eta_RTE"] = (W_out / W_in * W_in_pi, W_out_pi)

    return dict(
        T_0=T_0, components_kWh=I, total_kWh=total, boundary_kWh=boundary,
        scale=scale, pumping_ch_kWh=P_ch, pumping_dc_kWh=P_dc,
        gates={k: dict(sum_kWh=a, boundary_kWh=b, residual_kWh=a - b,
                       residual_rel=abs(a - b) / abs(b) if b else float("nan"))
               for k, (a, b) in gates.items()},
        gates_pass=all(abs(a - b) <= 1e-9 * abs(b) for a, b in gates.values()),
        global_is_identity=True,
        loop_mismatch_kWh=mism,
        W_el_in_kWh=W_in, W_el_out_kWh=W_out,
        Ex_geo_kWh=Ex_geo, Ex_reinject_kWh=Ex_loss,
        eta_RTE=W_out / W_in,
        psi=W_out / (W_in + Ex_geo),
        psi_stripped=W_out / (W_in + geo["Ex_stripped_kW"] * t_ch * scale),
        shares={k: v / total for k, v in I.items()},
        geo=geo)


def exergy_report(case, r, a=None):
    """Print the destruction table, largest first, and the closure check."""
    a = exergy_audit(case, r) if a is None else a
    g = a["geo"]
    print(f"EXERGY AUDIT   dead state T_0 = {a['T_0'] - 273.15:.1f} C "
          f"(the sink), convention '{g['convention']}'")
    print(f"  IN   electrical           {a['W_el_in_kWh']:10.1f} kWh")
    print(f"  IN   geothermal           {a['Ex_geo_kWh']:10.1f} kWh   "
          f"({g['m_geo']:.1f} kg/s, {g['N_geo']:.2f} producers)")
    print(f"  OUT  electrical           {a['W_el_out_kWh']:10.1f} kWh")
    print(f"  OUT  reinjected (LOSS)    {a['Ex_reinject_kWh']:10.1f} kWh   "
          f"at {g['T_reinject_C']:.1f} C, "
          f"{100 * (1 - g['utilization']):.1f} % of the resource discarded")
    print(f"  {'-' * 56}")
    for k, v in sorted(a["components_kWh"].items(), key=lambda kv: -kv[1]):
        print(f"    {k:<26}{v:10.1f} kWh{100 * a['shares'][k]:7.1f} %")
    print(f"  {'-' * 56}")
    print(f"    {'SUM of components':<26}{a['total_kWh']:10.1f} kWh")
    print(f"    {'boundary balance':<26}{a['boundary_kWh']:10.1f} kWh"
          "   (identity -- see below)")
    print("\n  GATES. Only these can fail. Each sums the entropy generated "
          "inside\n  one machine's components and checks it against a "
          "balance built from\n  stream states alone.")
    for k, g in a["gates"].items():
        ok = g["residual_rel"] <= 1e-9
        print(f"    {k:<8}{g['sum_kWh']:11.2f} vs {g['boundary_kWh']:11.2f} "
              f"kWh   rel {g['residual_rel']:.2e}   "
              f"{'OK' if ok else '*** FAIL ***'}")
    print("    borehole  by difference -- the water-loop residual. It cancels"
          "\n              the two exchanger terms exactly, so the global sum"
          "\n              above is an IDENTITY and cannot fail. Verified only"
          "\n              when the resolved s(E') integral lands.")
    if a["loop_mismatch_kWh"] == a["loop_mismatch_kWh"]:
        print(f"    loop      {a['loop_mismatch_kWh']:+.1f} kWh between the "
              "outlets the surface assumes\n              and the outlets the "
              "well actually delivers at CSS")
    print(f"\n  eta_RTE {a['eta_RTE']:.4f}    psi (resource) {a['psi']:.4f}"
          f"    psi (stripped) {a['psi_stripped']:.4f}")
    return a


def performance_indices(case, r):
    """The performance indicators of the factorial study, evaluated at CSS.

    These were defined for the first-cycle framing, where the field delivered
    the target energy by construction. At cyclic steady state it delivers
    whatever it delivers, so every one of them is computed from the ENERGY THE
    FIELD ACTUALLY MOVES rather than from the target. That matters: charging
    with the target input while crediting the realized output would flatter the
    round-trip efficiency by exactly the deviation.

        eta_T       thermal -> net electric conversion on the discharge side,
                    W_el_out / Q_in_ORC. Identically eta_ORC * eta_turb * eta_gen,
                    so it is a property of the power block alone.
        eta_T_eff   the same with the discharge pumping parasitic charged
                    against it. This is the "effective discharge efficiency".
        eta_RTE     round-trip: net electricity out over electricity in, both
                    including their pumping parasitics.
        eps_RTE     eta_RTE weighted by the fraction of the store that actually
                    CYCLES, eps(end of charge) - eps(end of discharge).
        dE_therm    thermal energy delivered per well per cycle   [kWh]
        dE_elec     net electric energy delivered per well,
                    dE_therm * eta_T_eff                          [kWh]

    A note on eps_RTE, because it changed meaning and the change matters when
    reading a factorial table. The first-cycle version weighted by eps_m, the
    melted fraction at end of charge. Under the enthalpy formulation that
    SATURATES -- once a segment has melted all its PCM the further energy goes
    into superheat, which eps_m cannot see -- and at this design point it is
    1.0000 exactly, so eps_RTE was identical to eta_RTE and carried no
    information at all. The cycled fraction does not saturate: a store that
    fills completely but only half empties returns 0.5, which is the useful
    statement.

    Note also that eta_RTE is INDEPENDENT of the deviation, exactly. A field
    that delivers 1 % more than target also drew 1 % more in, because the charge
    flow is pinned by its glide in the same way. The deviation is a statement
    about plant size, not about efficiency.
    """
    T, Eb = r["T"], r["budget"]
    N = r["N_wells"]
    t_ch_s, t_dc_s = case.t_ch * 3600.0, case.t_dc * 3600.0

    # ---- what the field actually moved, per cycle ------------------------
    Q_dot_in_ORC = r["Q_discharge_kJ"] / t_dc_s          # kW, field
    Q_dot_out_HP = r["Q_charge_kJ"] / t_ch_s             # kW, field
    rank_eff = Eb["Q_dot_in_ORC"] and (case.W_dot_el_out / case.ElG_eff
                                       / Eb["Q_dot_in_ORC"])
    W_el_out = Q_dot_in_ORC * rank_eff * case.ElG_eff
    cop = Eb["Q_dot_out_HP"] / (Eb["W_dot_el_in"] * case.ElH_eff)
    W_el_in = Q_dot_out_HP / cop / case.ElH_eff

    # ---- parasitics, at the realized temperatures ------------------------
    pumping_ch, pumping_dc = borehole_pumping(case, r)
    # The source loop runs only while charging, so it loads the SAME
    # denominator as the field charge pumping. With a scarce source this
    # was negligible and was left out; with seawater it is not.
    pumping_src = source_pumping(case, r)
    pumping_ch = pumping_ch + pumping_src
    # The sink loop runs on DISCHARGE, so it loads the numerator.
    pumping_sink = sink_pumping(case, r)
    pumping_dc = pumping_dc + pumping_sink

    # ---- ONE NORMALIZATION BASIS ----------------------------------------
    # Every per-MWe quantity below is divided by W_net, the REALISED NET
    # output of the field, and by nothing else.
    #
    #     W_net = W*_el,out - W_f,dc
    #
    # which is the numerator of eta_RTE. v1.4 used three different bases --
    # UA from the TARGET budget duties over the realized GROSS output,
    # wells over the realized gross, N_geo over the TARGET -- so no two
    # per-MWe numbers in the same table were comparable. At the reference
    # case this matters: R4 loses 52 % of the charge, so a store sized for
    # 1 MWe of latent inventory delivers roughly half that, and anything
    # per TARGET MWe understates the real plant by about a factor of two.
    W_net = W_el_out - pumping_dc

    def per_MWe(x):
        """x per MWe of realized net output. The only divisor in use."""
        return x / (W_net / 1000.0) if W_net else float('nan')

    # ---- the indicators --------------------------------------------------
    eta_T = W_el_out / Q_dot_in_ORC
    eta_T_eff = eta_T - pumping_dc / Q_dot_in_ORC
    eta_RTE = ((W_el_out - pumping_dc) * case.t_dc
               / ((W_el_in + pumping_ch) * case.t_ch))
    eps_cycled = float(r["charge"]["eps_local"].mean()
                       - r["discharge"]["eps_local"].mean())

    # ---- what BOREAS measures that THUMS could not ----------------------
    # eta_storage was 1.0000 by construction under H6. It is now a number
    # that can move, and the sign of the loss is NOT fixed: deep segments can
    # sit in rock hotter than they are and gain heat, so a negative loss is a
    # result rather than a bug and nothing here takes an absolute value.
    Q_loss = r.get("Q_loss_kJ", 0.0)
    dc_ch, dc_dc = downcomer_leak(case, r)
    f_loss = Q_loss / r["Q_charge_kJ"] if r["Q_charge_kJ"] else float("nan")
    f_bypass = ((dc_ch + dc_dc) / r["Q_charge_kJ"]
                if r["Q_charge_kJ"] else float("nan"))
    dE_therm = r["Q_discharge_kJ"] / 3600.0 / N          # kWh per well
    out = dict(
        eta_T=eta_T, eta_T_eff=eta_T_eff,
        eta_RTE=(eta_RTE if r.get("delivers", True) else 0.0),
        eps_RTE=((eta_RTE * eps_cycled) if r.get("delivers", True) else 0.0),
        eps_cycled=eps_cycled,
        # ---- THE SCALE-FREE NUMBER --------------------------------------
        # Deliverable electricity per meter of well per cycle. This is the
        # only quantity the model produces that is independent of plant size,
        # well count and field geometry, and it is therefore the one to
        # report and the one that transfers to another field.
        #
        # It is also nearly a constant: 0.336 to 0.340 kWh/m across a 4x
        # range of well depth at island spacing, a spread of 1.3 %. Depth,
        # within that range, buys capacity strictly in proportion to the
        # meters drilled and changes almost nothing else.
        #
        # Everything else is then arithmetic. For a field of n(L) wells
        # vertical to depth L,  E_total = n(L) * L * e'  -- see
        # `field_capacity`. Prescribing W_dot_el_out is scaffolding: the
        # model is exactly linear in plant size (a 4x plant gives 4.0000x
        # the wells and bit-identical per-well numbers), so the plant size
        # cancels out of every per-well and per-meter quantity.
        # Zeroed when the store does not deliver (section 1.4 of the v0.2
        # handoff): a non-delivering store has no electricity per meter and
        # no round trip, and reporting the signed numbers instead invites
        # them to be averaged into a sweep as though they were small
        # positives. The loss is still reported as computed.
        kWh_per_m_well=(dE_therm * eta_T_eff / case.L_well
                        if r.get("delivers", True) else 0.0),
        delivers=bool(r.get("delivers", True)),
        eta_storage=r["eta_storage"],
        Q_loss_kJ=Q_loss, f_formation_loss=f_loss,
        Q_bypass_ch_kJ=dc_ch, Q_bypass_dc_kJ=dc_dc, f_bypass=f_bypass,
        R_formation=r.get("R_formation", float("nan")),
        R_ground=r.get("R_ground", float("nan")),
        dE_therm_kWh=dE_therm, dE_elec_kWh=dE_therm * eta_T_eff,
        pumping_ch_kW=pumping_ch, pumping_dc_kW=pumping_dc,
        pumping_src_kW=pumping_src,
        pumping_sink_kW=pumping_sink,
        f_pump=(pumping_ch + pumping_dc) / W_el_out,
        W_el_out_kW=W_el_out, W_el_in_kW=W_el_in,
        rho_E_kWh_m3=dE_therm / case.V_well,
        # ---- POWER density, which is not energy density -----------------
        # rho_E says how much energy a cubic meter of store holds; it says
        # nothing about how fast it can be delivered, and the two are traded
        # against each other by the discharge window. A store with twice the
        # energy density and half the discharge rate is a different machine.
        # Per unit DEPTH as well, because drilling is priced by the meter.
        rho_P_kW_m3=(W_el_out / N) / case.V_well,
        rho_P_kW_per_m=(W_el_out / N) / case.L_well,
        t_discharge_h=case.t_dc)

    # ---- what the efficiencies above cost in hardware --------------------
    # Without these, every approach temperature is a free lunch: tighten
    # DT_pinch_ORC from 10 K to 1 K and eta_RTE rises 11 % with nothing to
    # pay, while the evaporator conductance more than doubles. Reported per
    # unit of delivered electricity as well, which is the form that lets two
    # designs of different size be compared.
    if "rank" in r and "hp" in r:
        # UA at the duties the field ACTUALLY moves, not at the target
        # budget's. UA is linear in duty at fixed temperatures, so this is
        # the same integration over the same composite curves with the two
        # duties replaced; the temperatures are already realized (T_2d).
        Eb_real = dict(Eb, Q_dot_in_ORC=Q_dot_in_ORC,
                       Q_dot_out_HP=Q_dot_out_HP)
        ua = exchanger_UA(case, r["rank"], r["hp"], r["T"], Eb_real)
        ua_target = exchanger_UA(case, r["rank"], r["hp"], r["T"], Eb)
        out["UA_total_kW_K"] = ua["TOTAL"]["UA_kW_per_K"]
        out["UA_per_MWe"] = per_MWe(ua["TOTAL"]["UA_kW_per_K"])
        # kept only so a v1.4 number can be traced; the paper does not use it
        out["UA_per_MWe_target"] = (ua_target["TOTAL"]["UA_kW_per_K"]
                                    / (case.W_dot_el_out / 1000.0))
        out["pinch_min_K"] = ua["TOTAL"]["pinch_K"]
        for key, tag in (("HTHP evaporator", "hpe"), ("HTHP condenser", "hpc"),
                         ("ORC evaporator", "orce"), ("ORC condenser", "orcc")):
            out[f"UA_{tag}_kW_K"] = ua[key]["UA_kW_per_K"]
            out[f"UA_{tag}_per_MWe"] = per_MWe(ua[key]["UA_kW_per_K"])
            # duty made dimensionless by the same basis: Q / W_net
            out[f"Q_{tag}_over_Wnet"] = ua[key]["duty_kW"] / W_net
            out[f"duty_{tag}_kW"] = ua[key]["duty_kW"]
            out[f"dTeff_{tag}_K"] = ua[key]["dT_eff_K"]
            # The four pinches individually, not just their minimum. Which
            # exchanger is tightest is design information; `pinch_min_K`
            # throws it away, and a design can move the binding one without
            # the minimum changing at all.
            out[f"pinch_{tag}_K"] = ua[key]["pinch_K"]

        # ---- second law, and the resource the first law books as free ----
        # eta_RTE treats the 60 C geothermal stream as costless. It is not:
        # a dedicated producer is drilled and paid for whether or not its
        # exergy is used. psi prices it, and comes out well BELOW eta_RTE.
        # The well count is the number the operator actually cares about,
        # and until v0.13 the model reported only half of it.
        ex = exergy_audit(case, r)
        out["psi"] = ex["psi"]
        out["psi_stripped"] = ex["psi_stripped"]
        out["I_total_kWh"] = ex["total_kWh"]
        out["Ex_reinject_kWh"] = ex["Ex_reinject_kWh"]
        out["resource_utilization"] = ex["geo"]["utilization"]
        out["m_geo_kg_s"] = ex["geo"]["m_geo"]
        out["N_geo"] = ex["geo"]["N_geo"]
        out["W_net_kW"] = W_net
        out["W_el_in_per_MWe"] = per_MWe(W_el_in)
        out["m_w_ch_per_MWe"] = per_MWe(Q_dot_out_HP / max(
            water_h(case, T["T_3c"]) - water_h(case, T["T_2c"]), 1e-9))
        out["m_w_dc_per_MWe"] = per_MWe(Q_dot_in_ORC / max(
            water_h(case, T["T_2d"]) - water_h(case, T["T_3d"]), 1e-9))
        out["T_reinject_C"] = ex["geo"]["T_reinject_C"]
        N_st = r.get("N_wells")
        if N_st is not None:
            out["N_wells_total"] = N_st + ex["geo"]["N_geo"]
            # STORAGE wells per MWe. v1.4 added the geothermal producers
            # into this number and divided by the gross output, so it was
            # neither the storage count nor on the reporting basis.
            out["wells_per_MWe"] = per_MWe(N_st)
            out["N_geo_per_MWe"] = per_MWe(ex["geo"]["N_geo"])
            out["wells_total_per_MWe"] = per_MWe(N_st + ex["geo"]["N_geo"])

            # V12: wells per MWe must equal t_dc / (e' L), paper Eq.
            # wells-per-MW, with e' net of pumping. This ties the plant
            # count to the scale-free number; if they disagree, one of the
            # two is not what it claims to be.
            e_prime = out.get("kWh_per_m_well", 0.0)
            if e_prime > 0:
                v12 = 1000.0 * case.t_dc / (e_prime * case.L_well)
                out["wells_per_MWe_from_eprime"] = v12
                out["V12_rel_error"] = abs(v12 / out["wells_per_MWe"] - 1.0)
        for name, v in ex["components_kWh"].items():
            out["I_" + name.replace(" ", "_").replace("-", "")] = v
        out["exergy_gates_pass"] = ex["gates_pass"]

        # ---- per-well productivity, and one scalar for the whole audit ----
        # rho_P says power per cubic meter of store; this says power per WELL,
        # which is the number an operator compares against a drilling cost.
        if N_st is not None:
            out["kW_per_well"] = W_el_out / N_st
        out["I_per_MWh"] = ex["total_kWh"] / (ex["W_el_out_kWh"] / 1000.0)

    # ---- REGIME LABELS, not performance -------------------------------
    # These do not say whether a design is good. They say whether two runs
    # are the same plant, and a screen that averages across them will report
    # main effects taken over a regime change. A run whose ORC switched from
    # pinch-bound to hot-end-bound is a different question, not a worse
    # answer. Carried in the KPI dict so a factorial or a Morris screen
    # cannot lose them.
    out["deviation"] = r["deviation"]
    out["merge_proximity_max"] = float(max(r["charge"]["merge_proximity_max"],
                                           r["discharge"]["merge_proximity_max"]))
    tb = r["rank"].get("T_3e_binding", "")
    out["T_3e_binding"] = tb
    out["regime_pinch_bound"] = 1.0 if "pinch" in tb else 0.0
    msgs = validate_case(case, verbose=False)
    out["feasible"] = 0.0 if any(k == "error" for k, _ in msgs) else 1.0
    out["n_warnings"] = float(sum(1 for k, _ in msgs if k == "warn"))
    return out


def field_capacity(case, r, n_wells_total, survival=None, k=None):
    """Deliverable electricity per cycle for a WHOLE FIELD [MWh].

    THE QUESTION, FLIPPED. Everywhere else the plant size is prescribed and
    the well count follows. That is backwards for a repurposing study: the
    wells are the asset and they are already in the ground. Here the count is
    the input and the energy is the result.

        E_total(L) = n(L) * L * e'(L)

    with e' the per-meter figure above and n(L) the number of wells still
    USABLE to depth L.

    `survival` is a callable s(depth_m) -> fraction in [0, 1], the share of
    the field still near-vertical at that depth. It is deliberately an
    argument rather than a built-in correlation, because it is a property of
    the drilling programme and must come from directional survey data. A
    geometric estimate from a maximum inclination and a reach is possible --
    with an instantaneous kick at 80 deg and 8200 ft of reach over 4175 ft of
    TVD the knee sits near 2729 ft -- but that estimate is extremely
    sensitive to the build rate, moving from 327 ft at 2 deg/100 ft to
    2681 ft at an instantaneous kick. An eightfold range on the headline
    number is not something to bury inside a default, so there is no default:
    pass None and every well is assumed usable, which is an upper bound and
    is reported as such.

    WHY THERE IS AN OPTIMUM once `survival` is supplied. Per-well energy
    grows with L while n(L) falls, so the product has a maximum -- at the
    shallowest kickoff, i.e. drill to the knee and stop. Without a survival
    function capacity rises monotonically with depth, which is an artifact of
    assuming every well reaches every depth.
    """
    k = k or performance_indices(case, r)
    e_prime = k["kWh_per_m_well"]                      # kWh per m per cycle
    frac = 1.0 if survival is None else float(survival(case.L_well))
    n = n_wells_total * frac
    E = n * case.L_well * e_prime / 1000.0             # MWh
    return dict(kWh_per_m_well=e_prime, survival_fraction=frac,
                n_wells_usable=n, L_well_m=case.L_well,
                E_total_MWh=E, P_MW=E / case.t_dc,
                bounded=survival is None)


def kpi_report(case, r, k=None):
    """Print the factorial-study indicators."""
    k = k or performance_indices(case, r)
    print("PERFORMANCE INDICATORS   (at cyclic steady state, per cycle)")
    print("=" * 66)
    print(f"  eta_T       discharge thermal -> electric   {k['eta_T']:9.4f}")
    print(f"  eta_T,eff   the same, less discharge pumping{k['eta_T_eff']:9.4f}"
          f"   <- effective")
    print(f"  eta_RTE     round trip, both parasitics     {k['eta_RTE']:9.4f}")
    print(f"  eps_RTE     eta_RTE x cycled fraction       {k['eps_RTE']:9.4f}")
    print()
    print(f"  dE_therm    thermal energy per well      {k['dE_therm_kWh']:12.1f} kWh")
    print(f"  dE_elec     net electric energy per well {k['dE_elec_kWh']:12.1f} kWh")
    print(f"  rho_E       thermal energy density       {k['rho_E_kWh_m3']:12.2f} kWh/m3")
    print(f"  rho_P       electric POWER density      {k['rho_P_kW_m3']:12.4f} kW/m3"
          f"   ({k['t_discharge_h']:.0f} h window)")
    print(f"              per meter of well           {k['rho_P_kW_per_m']:12.4f} kW/m")
    print("      rho_E and rho_P are traded against each other by the")
    print("      discharge window; a store with twice the energy density and")
    print("      half the rate is a different machine, not a better one.")
    print()
    print(f"  cycled fraction of the store   {k['eps_cycled']:8.4f}"
          f"   (melted {r['charge']['eps_local'].mean():.4f},"
          f" residual {r['discharge']['eps_local'].mean():.4f})")
    print(f"  pumping, charge / discharge    {k['pumping_ch_kW']:8.2f}"
          f" / {k['pumping_dc_kW']:.2f} kW"
          f"   = {100*k['f_pump']:.2f} % of gross output")
    if "UA_total_kW_K" in k:
        print()
        print(f"  UA total, all four exchangers  "
              f"{k['UA_total_kW_K']:8.1f} kW/K"
              f"   = {k['UA_per_MWe']:.1f} per MWe")
        print(f"      HTHP evap {k['UA_hpe_kW_K']:7.1f}  "
              f"HTHP cond {k['UA_hpc_kW_K']:7.1f}  "
              f"ORC evap {k['UA_orce_kW_K']:7.1f}  "
              f"ORC cond {k['UA_orcc_kW_K']:7.1f}")
        print(f"  the four pinches, individually (K)")
        print(f"      HTHP evap {k['pinch_hpe_K']:7.2f}  "
              f"HTHP cond {k['pinch_hpc_K']:7.2f}  "
              f"ORC evap {k['pinch_orce_K']:7.2f}  "
              f"ORC cond {k['pinch_orcc_K']:7.2f}")
        print(f"      minimum   {k['pinch_min_K']:7.2f}  <- which exchanger is"
              f" tightest is design information")
        print( "                         that the minimum alone throws away.")
        print("      Read UA beside eta_RTE. Every approach temperature is")
        print("      bought here; `ua_report` breaks it down.")
    print()
    # This used to say eta_RTE was independent of the deviation, because a
    # field delivering 1 % above target also drew 1 % more in. That is true
    # of an ADIABATIC store only. With a lossy one the two are tied:
    # eta_RTE ~= eta_RTE,adiabatic * eta_storage, and at the reference case
    # (R4) the deviation is dominated by the formation loss rather than by
    # any sizing residual.
    print(f"  deviation {100 * r.get('deviation', float('nan')):+7.2f} %   "
          f"eta_storage {r['eta_storage']:8.4f}")
    print("  These are not independent. The deviation is a statement about")
    print("  plant SIZE only when the store is adiabatic; with a lossy store")
    print("  eta_RTE carries the loss, eta_RTE ~ eta_RTE,ad * eta_storage,")
    print("  and at R4 the loss is most of what the deviation is reporting.")


def simulate_css_corrected(case, N=None, record=False, **kw):
    """CSS with a single ORC correction pass. The v0.6 default path.

    The ORC needs an evaporating temperature before the store has been marched,
    because T_3e is tied to T_2d. Since v0.6 T_2d is a result, so the first pass
    uses the provisional estimate -- the outlet the water would reach if it
    achieved the full glide -- and the second re-evaluates the ORC, the energy
    budget, the well count and both flows at the outlet the first pass actually
    produced.

    ONE pass is enough, and that is a property of the problem rather than a
    hopeful choice: the outlet is pinned by the store, not by the plant. Over a
    sweep of the discharge subcooling from 6 to 15 K -- which moves the inlet by
    9 K and the deviation by twelve percentage points -- the realized outlet
    moves by less than 0.3 K. `dT_2d_pass` is returned so the assumption is
    visible rather than assumed; if it is ever large, iterate.

    This keeps the plant level decoupled from the store in the sense that
    matters for Figure "procedure": there is no outer loop, only a second
    evaluation.
    """
    first = simulate_css(case, N=N, record=False, **kw)
    second = simulate_css(case, N=N, record=record,
                          T_2d=first["T_1d_realized"], **kw)
    second["pass1"] = first
    second["dT_2d_pass"] = second["T_2d_realized"] - first["T_2d_realized"]
    second["corrected"] = True
    return second


def css_report(case, r):
    """Print the cyclic-steady-state result."""
    print("CYCLIC STEADY STATE")
    print("=" * 66)
    if r["lambda_overridden"]:
        print(f"  NOTE  lambda forced to 0, from case.loss_surplus = "
              f"{case.loss_surplus:.3f}. A non-zero loss surplus cannot be")
        print("        absorbed at CSS: the state returns to itself and this")
        print("        model has no loss path, so charge == discharge exactly.")
        print("        The ZERO is what drove this run; the 0.050 in the Case")
        print("        is inert here and affects only energy_budget called on")
        print("        its own.")
        print()
    if r["T"].get("T_3e_hot_end_rule") is not None:
        old = r["T"]["T_3e_hot_end_rule"] - 273.15
        new = r["T"]["T_3e"] - 273.15
        if new < old - 1e-6:
            print(f"  NOTE  ORC boiling temperature set by the EVAPORATOR "
                  f"PINCH, not by DT_2D_3E:")
            print(f"        hot-end rule would give {old:7.2f} C; the pinch "
                  f"allows {new:7.2f} C.")
            print()
    print(f"  N_wells          {r['N_wells']:10.4f}   from latent heat alone, not solved")
    print(f"  m_dot charge     {r['m1_ch']:10.4f} kg/s per leg-pair, pinned by the glide")
    print(f"  m_dot discharge  {r['m1_dc']:10.4f} kg/s per leg-pair, pinned by the glide")
    print(f"  flow ratio       {r['flow_ratio']:10.4f}   a consequence, not a solve")
    print()
    # Two convergence statements, because they can disagree and the one
    # that matters is the second. The state drift stalls on round-off at
    # about 2e-8 J/m for a lossless or well-shielded store, so those runs
    # hit the cycle cap while their DISCHARGE INTEGRAL -- the quantity
    # every result depends on -- has been constant for many cycles.
    _h = r.get("history") or []
    _q = [x["Q_discharge_kJ"] for x in _h[-5:]]
    _ec = ((max(_q) - min(_q)) / abs(_q[-1])) if len(_q) > 1 and _q[-1] \
        else float("nan")
    if r["converged"]:
        print(f"  converged in {r['cycles']} cycles"
              f"   (drift {r['history'][-1]['drift']:.1e})")
    else:
        # Loud, because the result is still printed and looks perfectly normal.
        # Slow convergence is not rare: it scales with the well count, and a
        # case with low latent heat or many wells can need three times the
        # default. Re-run with a larger n_cycles before believing anything
        # below this line.
        print(f"  *** NOT CONVERGED *** stopped at the n_cycles limit of"
              f" {r['cycles']} with drift {r['history'][-1]['drift']:.2e},"
              f" tolerance {r['tol']:.0e}")
        print("      Re-run with a larger n_cycles. Everything below assumes")
        print("      the state has returned to itself, and it has not.")
    # NOT "must be 1". That was true under THUMS hypothesis H6, which this
    # model exists to drop: eta_storage is a result now, and at the
    # reference case it is about 0.48.
    print(f"  dE_dc spread, last 5 cycles  {_ec:.2e}"
          f"   <- the convergence that matters")
    print(f"  eta_storage      {r['eta_storage']:10.6f}   "
          f"a result, not a constraint (1 only as k_rock -> 0)")
    print()
    print(f"  required   D_E_in_ORC  {r['required_kJ']:12.5e} kJ")
    print(f"  delivered  at CSS      {r['Q_discharge_kJ']:12.5e} kJ")
    print(f"  DEVIATION              {100*r['deviation']:+12.3f} %")
    print()
    print(f"  implied net output     {r['W_el_out_implied']/1000:10.4f} MWe"
          f"   (target {case.W_dot_el_out/1000:.4f})")
    print()
    print("  the deviation IS the glide shortfall")
    print(f"    realized discharge glide / assumed   {r['glide_ratio_dc']:10.6f}")
    print(f"    delivered energy / required          "
          f"{r['Q_discharge_kJ']/r['required_kJ']:10.6f}")
    print("    The flow is pinned by the ASSUMED glide, so the energy")
    print("    delivered is just the ratio of realized to assumed glide.")
    print()
    print("  exchanger outlets are RESULTS, not inputs (v0.6)")
    print(f"    discharge, to the ORC       assumed {r['T_2d_assumed']-273.15:8.3f} C"
          f"   realized {r['T_2d_realized']-273.15:8.3f} C")
    print(f"    charge, to the HP condenser assumed {r['T_2c_assumed']-273.15:8.3f} C"
          f"   realized {r['T_2c_realized']-273.15:8.3f} C")
    if r.get("corrected"):
        print(f"    ORC corrected at the realized outlet; second pass moved it"
              f" {r['dT_2d_pass']:+.4f} K")
    elif r["T_2d_was_provisional"]:
        print("    UNCORRECTED: the ORC still runs on the provisional outlet.")
        print("    Use simulate_css_corrected for the reported result.")
    ch, dc = r["charge"], r["discharge"]
    print()
    print("  melted fraction at CSS")
    print(f"    end of charge     mean {ch['eps_local'].mean():.4f}"
          f"   min {ch['eps_local'].min():.4f}   max {ch['eps_local'].max():.4f}")
    print(f"    end of discharge  mean {dc['eps_local'].mean():.4f}"
          f"   min {dc['eps_local'].min():.4f}   max {dc['eps_local'].max():.4f}")
    if dc["eps_local"].mean() > 0.05 and ch["eps_local"].mean() > 0.99:
        print()
        print("    The store melts completely but does not refreeze completely,")
        print("    so the shortfall is a DISCHARGE RATE limit, not an inventory")
        print("    limit. More PCM per well is not the lever.")


def validate_case(case, verbose=True):
    """Check a Case for the inconsistencies that are easy to create.

    This is not a check that the model is right -- that lives in the
    verification notebook. It is a check that YOUR PARAMETERS are consistent,
    and it fires on the traps that have actually caught people:

      * `n_segments` not a multiple of `N_lay`, which makes `layer_map` place
        a layer boundary mid-segment;
      * a discharge inlet that is not below the coldest layer, or a charge
        inlet not above the hottest, so the half-cycle cannot run;
      * a glide so large that the coldest layer falls below the sink, or so
        small that the cascade collapses;
      * a time grid too coarse for the fastest segment time constant.

    Returns a list of (level, message). `level` is 'error' for a case that
    cannot be run and 'warn' for one that can but probably should not be.
    """
    msgs = []
    def err(m): msgs.append(('error', m))
    def warn(m): msgs.append(('warn', m))

    # --- the cascade -----------------------------------------------------
    if case.N_lay < 1:
        err(f'N_lay = {case.N_lay}; must be at least 1')
    if case.n_segments % case.N_lay:
        warn(f'n_segments = {case.n_segments} is not a multiple of '
             f'N_lay = {case.N_lay}: layer_map puts a layer boundary inside a '
             f'segment, so one segment carries the wrong melting temperature '
             f'by up to {case.DT_3C_2C / case.N_lay:.3f} K. '
             f'Nearest clean values: '
             f'{case.N_lay * (case.n_segments // case.N_lay)} or '
             f'{case.N_lay * (case.n_segments // case.N_lay + 1)}')

    T_bot = T_m_bottom(case) - 273.15
    spacing = case.DT_3C_2C / case.N_lay

    # --- the two inlets --------------------------------------------------
    T_in_ch = case.T_m_C + case.DT_4C_M
    T_in_dc = T_bot - case.DT_M_4D
    if case.DT_4C_M <= 0:
        err(f'DT_4C_M = {case.DT_4C_M} K; the charge inlet must be ABOVE the '
            f'hottest layer or nothing melts')
    if case.DT_M_4D <= 0:
        err(f'DT_M_4D = {case.DT_M_4D} K; the discharge inlet must be BELOW '
            f'the coldest layer or nothing freezes')
    if 0 < case.DT_4C_M < spacing:
        warn(f'DT_4C_M = {case.DT_4C_M:.3f} K is less than one layer width '
             f'({spacing:.3f} K), so the charging fluid drops below a layer\'s '
             f'melting point before leaving it')
    if 0 < case.DT_M_4D < spacing:
        warn(f'DT_M_4D = {case.DT_M_4D:.3f} K is less than one layer width '
             f'({spacing:.3f} K); the discharge pinches at each layer exit')

    # --- glide and the sink ----------------------------------------------
    if case.DT_3C_2C <= 0:
        err(f'DT_3C_2C = {case.DT_3C_2C} K; the glide sets both mass flows')
    if T_in_dc <= case.T_sink_C:
        err(f'discharge inlet {T_in_dc:.1f} C is at or below the sink '
            f'{case.T_sink_C:.1f} C')
    if T_in_ch <= T_bot:
        err(f'charge inlet {T_in_ch:.1f} C is below the coldest layer '
            f'{T_bot:.1f} C')

    # --- properties ------------------------------------------------------
    for f in ('h_m', 'cp_s', 'cp_l', 'k_s', 'k_l', 'rho_s', 'rho_l'):
        v = getattr(case, f, None)
        if v is not None and v <= 0:
            err(f'{f} = {v}; must be positive')

    # --- the time grid ---------------------------------------------------
    if case.n_times < 8:
        warn(f'n_times = {case.n_times}; the logarithmic grid needs enough '
             f'levels that the last step does not span most of the window')

    # --- the working fluid is liquid water, and must stay that way --------
    T_sat_w = CP.PropsSI('T', 'P', case.P, 'Q', 0, 'Water') - 273.15
    if T_in_ch >= T_sat_w:
        err(f'the charging inlet T_4c = {T_in_ch:.2f} C is at or above the '
            f'saturation temperature of water at P = {case.P/1e5:.2f} bar, '
            f'which is {T_sat_w:.2f} C. The model assumes single-phase '
            f'liquid throughout (H7) and carries no boiling correlation, so '
            f'it would report nonsense rather than fail. Raise case.P or '
            f'lower T_m_C + DT_4C_M.')
    elif T_sat_w - T_in_ch < 10.0:
        warn(f'the charging inlet is {T_sat_w - T_in_ch:.2f} K below the '
             f'saturation temperature of water at {case.P/1e5:.2f} bar; '
             f'there is very little margin against boiling')

    # --- approaches that are zero by construction -------------------------
    # None of these is an error. Each is a place where the model quietly
    # assumes an infinite exchanger, and a parametric study that leans on it
    # will report an efficiency no hardware can reach.
    for old, new in (('Turb_eff', 'eta_turb_s'), ('Comp_eff', 'eta_comp_s')):
        if getattr(case, old, None) is not None:
            err(f'{old} = {getattr(case, old)} was RETIRED at v0.12. It was '
                f'the isentropic efficiency applied as a multiplier on the '
                f'work AFTER an isentropic cycle; it is now applied inside '
                f'the cycle, to the state points, as {new}. Setting both '
                f'would double-count. Use {new} and leave {old} at None.')
    for f in ('eta_turb_s', 'eta_pump_s', 'eta_comp_s', 'ElG_eff', 'ElH_eff'):
        v = getattr(case, f)
        if not (0.0 < v <= 1.0):
            err(f'{f} = {v}; must be in (0, 1]')
    if getattr(case, 'DT_3A_13H', None) is not None:
        err(f'DT_3A_13H = {case.DT_3A_13H} was RETIRED at v0.9 and is now '
            f'ignored. It used to set the evaporating temperature '
            f'independently of DT_3A_4A, so the approach in the heat-pump '
            f'evaporator was the difference of two free fields and nothing '
            f'checked its sign: the defaults made it exactly zero, and '
            f'reversing them made it negative with no symptom. The '
            f'evaporating temperature is now derived, '
            f'T_13h = T_source_C - DT_3A_4A - DT_pinch_HPE. Set '
            f'DT_pinch_HPE instead, and leave DT_3A_13H at None.')
    # ---- the geothermal resource, new at v0.13 --------------------------
    if case.exergy_convention not in ('resource', 'stripped'):
        err(f"exergy_convention = {case.exergy_convention!r}; must be "
            f"'resource' (charge everything the well lifts, book the "
            f"reinjected stream as a loss) or 'stripped' (charge only what "
            f"the evaporator removes). They give OPPOSITE guidance on "
            f"DT_3A_4A, so the choice is not cosmetic. See DN-23.")
    if case.m_dot_geo_well <= 0:
        err(f'm_dot_geo_well = {case.m_dot_geo_well} kg/s; N_geo is directly '
            f'proportional to it and it must be positive.')
    T_rei = case.T_source_C - case.DT_3A_4A
    if T_rei < case.T_reinject_min_C:
        warn(f'reinjection at {T_rei:.1f} C is below T_reinject_min_C = '
             f'{case.T_reinject_min_C:.1f} C. DT_3A_4A = {case.DT_3A_4A} K '
             f'draws the source down further than the formation constraint '
             f'allows. This is a WARNING and not an error only because the '
             f'default limit is a placeholder: the project has no measured '
             f'value for it yet. The second law pushes DT_3A_4A UP -- it '
             f'cuts m_geo far faster than it cuts COP -- so this bound is '
             f'the one that will bind in an optimization.')
    if T_rei <= case.T_sink_C:
        err(f'reinjection at {T_rei:.1f} C is at or below the sink '
            f'({case.T_sink_C} C), which is the dead state. The source would '
            f'carry no exergy worth taking and the audit would go negative.')
    if case.DT_pinch_HPE <= 0:
        err(f'DT_pinch_HPE = {case.DT_pinch_HPE} K; the refrigerant must '
            f'evaporate BELOW the source outlet or the evaporator runs heat '
            f'uphill. Zero means infinite area.')
    elif case.DT_pinch_HPE < 2.0:
        warn(f'DT_pinch_HPE = {case.DT_pinch_HPE:.2f} K is a very close '
             f'approach in the heat-pump evaporator; the area needed grows '
             f'roughly as 1/DT as this goes to zero')
    T_evap = case.T_source_C - case.DT_3A_4A - case.DT_pinch_HPE
    if T_evap <= case.T_sink_C:
        err(f'the derived evaporating temperature {T_evap:.2f} C is at or '
            f'below the sink at {case.T_sink_C:.2f} C; there is no lift left '
            f'to speak of. Reduce DT_3A_4A or DT_pinch_HPE.')

    # --- how hard the far end of the well is driven -----------------------
    # NOT an error, and an earlier version of this check wrongly made it one.
    # The margin below is computed on a LINEAR GLIDE against melting
    # temperatures, i.e. it assumes the PCM sits at T_m. Under Formulation C
    # it does not: the store superheats and subcools, so water arriving on
    # the "wrong" side of T_m still exchanges heat, and `conduction_shell`
    # takes the direction from sign(T_j - T_pcm) segment by segment. The
    # model handles a negative margin correctly and reports the consequence
    # through eps and the deviation. DT_M_4D = DT_3C_2C/N_lay -- the retired
    # v0.5 closure, margin exactly zero -- runs and is one of the frozen
    # verification fixtures. See DN-20.
    span = case.cascade_span
    g_ch, g_dc = case.DT_3C_2C, case.glide_dc_spec
    for gl, appr, name, half, far in (
            (g_ch, case.DT_4C_M, 'DT_4C_M', 'charging', 'coldest'),
            (g_dc, case.DT_M_4D, 'DT_M_4D', 'discharging', 'hottest')):
        margin = span - (gl - appr)
        if margin <= 0.0:
            warn(f'under a linear glide the {half} water would reach the '
                 f'{far} layer {-margin:.3f} K on the wrong side of its '
                 f'melting point (span {span:.3f} K, glide {gl:.3f} K, '
                 f'{name} {appr:.3f} K). This is NOT infeasible -- the PCM '
                 f'leaves T_m under Formulation C and the march takes the '
                 f'heat-flow direction per segment -- but that end of the '
                 f'store is being driven by the sensible branch, or briefly '
                 f'in reverse, rather than melting or freezing as intended. '
                 f'Expect a lower cycled fraction. Raise DT_cascade or '
                 f'{name}, or narrow the glide, to restore the margin.')
        elif margin < 2.0:
            warn(f'the {half} margin at the {far} layer is only '
                 f'{margin:.3f} K (span {span:.3f} K against a glide of '
                 f'{gl:.3f} K less {name} = {appr:.3f} K); that end of the '
                 f'store is driven very softly')

    if case.DT_cascade is not None:
        msgs.append(('info',
                     f'cascade prescribed independently: DT_cascade = '
                     f'{case.DT_cascade:.3f} K, span {span:.3f} K, against a '
                     f'charging glide of {g_ch:.3f} K. The leading-face '
                     f'approach is no longer uniform along the well.'))
    if case.DT_3D_2D is not None:
        msgs.append(('info',
                     f'discharging glide prescribed independently: '
                     f'DT_3D_2D = {case.DT_3D_2D:.3f} K against a charging '
                     f'glide of {g_ch:.3f} K'))
    if case.DT_sink_glide > 0.0:
        warn(f'DT_sink_glide = {case.DT_sink_glide:.2f} K. The ORC condenser '
             f'then has an INTERIOR pinch at the desuperheat corner, around '
             f'93 % of its duty, and the model does not check it. It crosses '
             f'near {case.DT_E_sink + 0.4:.1f} K of sink glide.')
    warn('the ORC regenerator has a zero approach by construction '
         '(T_9e = T_7e in double_stage_rankine), so its duty is an upper '
         'bound rather than a design value. This is structural, not a '
         'setting you can change from Case.')

    # --- the ORC evaporator has to obey the second law --------------------
    try:
        rank, _hp, Tst = cycle_state_points(case)
        got = rank.get('pinch')
        old = rank.get('T_3e_uncorrected')
        if old is not None and rank['T_3e'] < old - 1e-6:
            rank_old = double_stage_rankine(case.fluid, Tst['T_1e'], old,
                                            x_int=case.x_int_orc)
            warn(f'the ORC pinch forced the boiling temperature DOWN, from '
                 f'{old - 273.15:.2f} C to {rank["T_3e"] - 273.15:.2f} C, and '
                 f'with it eta_ORC from {rank_old["rank_eff"]:.4f} to '
                 f'{rank["rank_eff"]:.4f}. The hot-end rule DT_2D_3E = '
                 f'{case.DT_2D_3E:.1f} K is not what sets the evaporating '
                 f'temperature here; the pinch at DT_pinch_ORC = '
                 f'{case.DT_pinch_ORC:.1f} K is. This is the honest cost of a '
                 f'{case.DT_3C_2C:.0f} K water glide against a fluid that '
                 f'boils at one temperature.')
        elif got is not None:
            msgs.append(('info', f'ORC evaporator pinch {got:+.2f} K, '
                                 f'requirement {case.DT_pinch_ORC:.1f} K'))
    except Exception as e:                       # never block on a diagnostic
        warn(f'could not evaluate the ORC pinch: {e}')

    if verbose:
        if not msgs:
            print('validate_case: OK')
            print(f'  cascade   {case.T_m_C:.3f} C down to {T_bot:.3f} C '
                  f'in {case.N_lay} layers of {spacing:.3f} K')
            print(f'  inlets    charge {T_in_ch:.3f} C, '
                  f'discharge {T_in_dc:.3f} C')
        for level, m in msgs:
            print(f'{level.upper():5s} {m}')
    # ======================================================================
    # BOREAS v1.0 additions
    # ======================================================================

    # --- the gradient and the source are INDEPENDENT, and checked ---------
    # An earlier draft derived the gradient from T_source_C by setting the
    # formation temperature at producing depth EQUAL to the wellhead
    # temperature. That is a lower bound written as an equality: the water
    # cools on the way up, so the rock is at least that hot and generally
    # hotter, and the producing wells may be deeper than the retrofits. The
    # deficit is real physics and deriving one from the other throws it away.
    # So both are specified, and only the INEQUALITY is enforced.
    T_at_prod = (case.T_surface_C
                 + case.grad_K_per_km / 1000.0 * case.L_producing_ft * 0.3048)
    if case.T_source_C > T_at_prod + 1e-9:
        err(f'T_source_C = {case.T_source_C:.1f} C but the gradient only '
            f'reaches {T_at_prod:.1f} C at {case.L_producing_ft:.0f} ft; '
            f'the source cannot be hotter than the rock it comes from. '
            f'Deepen L_producing_ft or raise grad_K_per_km.')
    elif case.T_source_C > T_at_prod - 5.0:
        warn(f'T_source_C = {case.T_source_C:.1f} C is within 5 K of the '
             f'formation temperature at producing depth ({T_at_prod:.1f} C), '
             f'which leaves no allowance for cooling up the production string')

    # --- the cascade must run HOT AT THE BOTTOM --------------------------
    # This is the whole point of the downcomer and it is the one thing a
    # careless edit to melting_temperatures() would silently reverse. Hot at
    # the top is the THUMS orientation: it puts the hottest PCM against the
    # coldest shallow rock and the coldest PCM against rock that, at 8000 ft
    # and a Wilmington gradient, is hotter than it is -- so the deep layers
    # never freeze and that capacity is dead. Watched, not commented.
    try:
        T_rock_arr, depth = formation_profile(case, case.n_segments)
        T_m_lay, _ = melting_temperatures(case)
        T_m_seg = layer_map(T_m_lay, case.n_segments, case.N_lay)
        if depth[0] < depth[-1]:
            err('formation_profile is not in charge-frame indexing: '
                'z = 0 must be the BOTTOM of the well')
        if T_m_seg[0] < T_m_seg[-1]:
            err('the cascade runs hot-at-top. BOREAS requires hot-at-bottom '
                'so the cascade aligns with the geothermal gradient; '
                'check melting_temperatures()')
        stuck = float(np.mean(T_m_seg < T_rock_arr))
        if stuck > 0.02:
            warn(f'{100 * stuck:.0f} % of the well sits in rock HOTTER than '
                 f'its melting point and cannot freeze on discharge; '
                 f'that capacity is dead. Shorten the well, raise T_m_C, '
                 f'or lower grad_K_per_km.')
    except Exception as exc:              # pragma: no cover
        warn(f'could not check the cascade orientation: {exc}')

    # --- the geometry has to close ---------------------------------------
    R = case.D_well / 2.0
    a = case.r_e + case.fin_L                     # finned tube outer radius
    b = case.r_id_outer
    # Area is NECESSARY but not sufficient; 0.85 is a generous packing bound
    # for a handful of unequal circles and is used only to catch the gross
    # case. The arrangement is the one in the schematic -- the downcomer
    # OFFSET, not centered -- which packs better than a concentric annulus.
    if np.pi * (b ** 2 + case.n_ft * a ** 2) > 0.85 * np.pi * R ** 2:
        err(f'the downcomer ({2000 * b:.0f} mm OD) and {case.n_ft} finned '
            f'tubes ({2000 * a:.0f} mm over fins) cannot pack into a '
            f'{1000 * case.D_well:.0f} mm hole')
    elif b + 2.0 * a > R + 1e-6:
        warn(f'the downcomer must be run OFF-CENTRE: {2000 * b:.0f} mm OD '
             f'plus two finned tubes is {2000 * (b / 2 + a):.0f} mm against a '
             f'{1000 * R:.0f} mm radius, so a concentric arrangement does not '
             f'close. The schematic arrangement does; centralisers must hold '
             f'the offset.')
    if case.V_well <= 0:
        err('no PCM volume left: the tubes and downcomer fill the hole')
    f_pcm = case.V_well / case.V_borehole
    if f_pcm < 0.15:
        warn(f'PCM is only {100 * f_pcm:.0f} % of the borehole; the hardware '
             f'is crowding out the storage')

    # --- the formation model ---------------------------------------------
    if not np.isfinite(ground_resistance(case)):
        warn(f'the line-source ground resistance is outside its validity at '
             f'k_rock = {case.k_rock} W/m/K, t = {case.t_operation_yr} yr; '
             f'the formation has been treated as ADIABATIC for this run, so '
             f'eta_storage will come back as exactly 1')
    if case.t_operation_yr > 50.0:
        warn('t_operation_yr beyond 50 yr: the line source is being '
             'extrapolated well past any calibration')

    return msgs


In [ ]:
%%writefile boreas_ground.py
"""boreas_field.py -- well-field geometry for BOREAS, without editing boreas.py.

Builds the ground resistance seen by a representative storage well from the
actual positions of all N storage wells, at every measured depth (MD), and
runs BOREAS against it.

Geometry
    disc_positions      N wells filling a circle at one well per B0^2
    rows_positions      one straight double row, B0 along a row, w between rows
    grissom_positions   Active+Idle Grissom slots on the ring (WellSTAR)
    trajectories        vertical, or literature S-curves radiating from the
                        island center (Clark & Barth 1983; Truex & Hunter 1969;
                        Robertson et al. 1987)

Ground models (all evaluated at the age of the field, t_op)
    'per'   perimeter rule, Eq. (Rper) of the paper: max(R_g, N sqrt(pi a t)/(k P))
    'flux'  line sources of EQUAL STRENGTH over the actual positions; the
            field-mean wall temperature rise per unit heat rate
    'temp'  line sources of EQUAL TEMPERATURE: every well at a given MD has the
            same PCM-to-rock difference, the strengths are solved for (edge
            wells lose more), and the field-mean conductance is returned.
            The near-well series resistance (PCM, casing, cement) is inside
            the solve, so the result is consistent with BOREAS' series chain.

Line-source interaction is local and two-dimensional: a target point on well i
interacts with well k through the infinite line source E1(d^2/4at)/(4 pi k),
where d is the SHORTEST distance from the target to well k's trajectory. For
parallel vertical wells this is the classical superposition; for inclined
wells it is the perpendicular distance to the locally straight neighbor.
The ground surface and the well ends are not represented (as in Eq. (Rg)).
"""
import numpy as np
import pandas as pd
from scipy.special import exp1
from scipy.spatial import cKDTree
import scipy.sparse as sp
import scipy.sparse.linalg as spla

FT = 0.3048
# Nominal wellhead spacing, 6 ft and 12 ft, from the Oxy Long Beach
# drilling deck. B0_MEASURED is what the WellSTAR slot coordinates actually
# show for Grissom: median nearest-neighbor 1.65 m over 363 slots, 10 %
# tighter than nominal. The idealized rungs (disc, double row) are built at
# the MEASURED value so that the step from them to the real slot positions
# isolates the LAYOUT and is not contaminated by a spacing change.
# Above this many unknowns the direct sparse solve is replaced by CG; see
# Ground.profiles. 6000 is below the smallest size at which the direct
# factorisation has been seen to exhaust memory, and the two agree to 1e-12
# wherever both run.
_DIRECT_MAX = 6000

# THE station spacing, in meters of measured depth. One number, used by
# every run of the ladder, the L sweep and the sensitivities. See
# Ground.stations_for for why this is a spacing and not a count.
DS_STATION = 50.0

B0_NOMINAL, W_DEF = 1.83, 3.66
B0_MEASURED = 1.65
B0_DEF = B0_MEASURED


# ----------------------------------------------------------------- slots --
def disc_positions(N, B0=B0_DEF):
    i = np.arange(N) + 0.5
    R0 = B0 * np.sqrt(N / np.pi)
    r = np.sqrt(i / N) * R0
    th = np.pi * (3 - np.sqrt(5)) * i
    return np.c_[r * np.cos(th), r * np.sin(th)]


def rows_positions(N, B0=B0_DEF, w=W_DEF):
    n1 = int(np.ceil(N / 2)); n2 = N - n1
    x1 = np.arange(n1) * B0; x2 = np.arange(n2) * B0
    P = np.r_[np.c_[x1, np.zeros(n1)], np.c_[x2, np.full(n2, w)]]
    return P - P.mean(0)


# Which Grissom slots are available as STORAGE wells.
#
# R4 converts every live slot, which is the upper bound on mutual
# shielding but assumes the island stops producing. If the island keeps
# producing the co-produced water that feeds the heat pump, its producers
# and injectors stay in service and only the IDLE slots can be converted.
# That is R4i, and it is a much smaller field: 65 wells against 363.
STORAGE_SETS = {
    'all_slots': ("Active", "Idle"),
    'idle_only': ("Idle",),
    'idle_plus': ("Idle", "Canceled", "Plugged"),
}


def statuses_for(storage_set):
    try:
        return STORAGE_SETS[storage_set]
    except KeyError:
        raise ValueError(
            f'storage_set must be one of {sorted(STORAGE_SETS)}, '
            f'not {storage_set!r}') from None


def grissom_positions(csv, statuses=("Active", "Idle"), dedupe=0.6, stray=5.0):
    df = pd.read_csv(csv, dtype={"API": str})
    g = df[(df.Island == "Grissom") & df.WellStatus.isin(statuses)]
    lat0 = g.Latitude.mean(); R = 6371000.0
    P = np.c_[np.radians(g.Longitude) * R * np.cos(np.radians(lat0)),
              np.radians(g.Latitude) * R]
    t = cKDTree(P); keep = np.ones(len(P), bool)
    for i, j in sorted(t.query_pairs(dedupe)):
        if keep[i] and keep[j]:
            keep[j] = False
    P = P[keep]
    d, _ = cKDTree(P).query(P, 2)
    P = P[d[:, 1] < stray]                       # drop isolated stray slots
    return P - P.mean(0)


def perimeter_buffer(P, r):
    from shapely.geometry import Point
    from shapely.ops import unary_union
    return unary_union([Point(p).buffer(r, 24) for p in P]).length


# ---------------------------------------------------------- trajectories --
def s_curve(md, kop, build, theta_max, hold_to=None, drop=2.5, theta_end=45.0):
    """Inclination [deg] along MD [m]: vertical to kop, build [deg/100 ft] to
    theta_max, hold; optional drop [deg/100 ft] from hold_to [m] to theta_end."""
    b = build / (100 * FT); dr = drop / (100 * FT)
    inc = np.clip((md - kop) * b, 0.0, theta_max)
    if hold_to is not None and theta_end < theta_max:
        inc = np.where(md > hold_to,
                       np.maximum(theta_max - (md - hold_to) * dr, theta_end), inc)
    return inc


def trajectories(P, L, kind="vertical", ds=5.0, kop=None, build=4.0,
                 theta_max=55.0, hold_to=3000 * FT, center=None):
    """Polylines for all wells: array (N, n, 3) of (x, y, TVD), MD grid md."""
    md = np.arange(0.0, L + 1e-9, ds)
    N = len(P)
    if kind == "vertical":
        X = np.repeat(P[:, None, :], len(md), 1)
        return md, np.concatenate([X, np.broadcast_to(md, (N, len(md)))[..., None]], 2)
    c = P.mean(0) if center is None else center
    az = np.arctan2(P[:, 1] - c[1], P[:, 0] - c[0])
    kop = np.broadcast_to(np.asarray(kop, float), (N,))
    out = np.empty((N, len(md), 3))
    for i in range(N):
        inc = np.radians(s_curve(md, kop[i], build, theta_max, hold_to))
        incm = 0.5 * (inc[1:] + inc[:-1])
        dh = np.r_[0, np.cumsum(np.sin(incm) * ds)]
        dz = np.r_[0, np.cumsum(np.cos(incm) * ds)]
        out[i, :, 0] = P[i, 0] + dh * np.cos(az[i])
        out[i, :, 1] = P[i, 1] + dh * np.sin(az[i])
        out[i, :, 2] = dz
    return md, out


def staggered_kop(P, center=None, outer=(100, 175, 250), inner=(300, 400, 500)):
    """Outer-row slots kick off shallow (long drift goes 'over the top'),
    inner-row slots deep; values cycle along the ring (Clark & Barth 1983)."""
    c = P.mean(0) if center is None else center
    r = np.hypot(*(P - c).T); ang = np.arctan2(*(P - c).T[::-1])
    t = cKDTree(P); _, nb = t.query(P, 12)
    is_out = r >= np.median(r[nb], 1)
    order = np.argsort(ang); kop = np.empty(len(P))
    io = ii = 0
    for j in order:
        if is_out[j]:
            kop[j] = outer[io % len(outer)]; io += 1
        else:
            kop[j] = inner[ii % len(inner)]; ii += 1
    return kop * FT, is_out


# ------------------------------------------------------ ground resistance --
class Ground:
    def __init__(self, k=2.5, rhocp=2.34e6, t_yr=10.0, rb=0.110, cutoff=80.0):
        self.k, self.a = k, k / rhocp
        self.t = t_yr * 365.25 * 86400.0
        self.rb, self.cutoff = rb, cutoff
        self.Rg = np.log(2 * np.exp(-np.euler_gamma / 2) *
                         np.sqrt(self.a * self.t) / rb) / (2 * np.pi * k)

    def kern(self, d):
        return exp1(d ** 2 / (4 * self.a * self.t)) / (4 * np.pi * self.k)

    def pairs(self, md, X, nst=50):
        """For targets at nst MD stations on every well: shortest distance to
        every other well within the cutoff, and the neighbor's station."""
        N, n, _ = X.shape
        st = (np.arange(nst) + 0.5) * md[-1] / nst
        ist = np.clip(np.searchsorted(md, st), 1, n - 1)
        T = X[:, ist, :].reshape(-1, 3)                      # targets
        owner_t = np.repeat(np.arange(N), nst)
        pts = X.reshape(-1, 3)
        owner = np.repeat(np.arange(N), n); idx = np.tile(np.arange(n), N)
        tree = cKDTree(pts)
        rows, cols, dist = [], [], []
        # Chunked, and the chunk is small on purpose. query_ball_point
        # returns a Python list per target, and near the surface an S-curve
        # puts every one of the 363 wells' ~250 samples inside the cutoff --
        # tens of thousands of indices per target. At a chunk of 400 that
        # intermediate is hundreds of megabytes, and four of these running
        # in parallel is what exhausts the machine. Smaller chunks cost a
        # few per cent in Python overhead and bound the peak.
        for t0 in range(0, len(T), 50):
            lists = tree.query_ball_point(T[t0:t0 + 50], self.cutoff + 6.0)
            for tt, L in enumerate(lists):
                t_id = t0 + tt
                L = np.asarray(L)
                L = L[owner[L] != owner_t[t_id]]
                if len(L) == 0:
                    continue
                o = owner[L]; j = idx[L]
                p = T[t_id]
                # exact distance to the two polyline segments around each sample
                best = np.full(len(L), np.inf)
                for dj in (-1, 0):
                    a_ = np.clip(j + dj, 0, n - 2)
                    A = X[o, a_]; Bq = X[o, a_ + 1]
                    v = Bq - A; wv = p - A
                    tpar = np.clip((wv * v).sum(1) / np.maximum((v * v).sum(1), 1e-12), 0, 1)
                    dd = np.linalg.norm(wv - tpar[:, None] * v, axis=1)
                    best = np.minimum(best, dd)
                srt = np.lexsort((best, o))
                o_s, b_s, j_s = o[srt], best[srt], j[srt]
                first = np.r_[True, o_s[1:] != o_s[:-1]]
                o_u, d_u, j_u = o_s[first], b_s[first], j_s[first]
                keep = d_u < self.cutoff
                o_u, d_u, j_u = o_u[keep], d_u[keep], j_u[keep]
                s_nb = np.clip(np.searchsorted(st, md[j_u]), 0, nst - 1)
                rows.append(np.full(len(o_u), t_id, dtype=np.int32))
                cols.append((o_u * nst + s_nb).astype(np.int32))
                dist.append(np.maximum(d_u, self.rb))
        if not rows:
            # A single well, or a field so spread that nothing is inside
            # the cutoff: no interactions at all. The caller must still get
            # arrays, and the solve then returns the isolated R_g, which is
            # what V3 checks.
            z32 = np.zeros(0, np.int32)
            return st, z32, z32, np.zeros(0), N
        return st, np.concatenate(rows), np.concatenate(cols), np.concatenate(dist), N

    @staticmethod
    def stations_for(L, ds=None):
        """How many MD stations to assemble the superposition on.

        ONE FIXED STATION SPACING, ds, for every run. Not a count, and not
        a per-run search.

        The method treats each station as an infinite line source in the
        plane normal to the trajectory, so a station must represent a
        segment long compared with the distance to the neighboring wells.
        Fix the COUNT instead and that condition changes with L: 50
        stations is 25 m apart in a 1250 m hole and 2 m apart in a 100 m
        one, against a well spacing of 1.65 m. The short case then returns
        negative strengths -- wells absorbing heat from rock they are
        hotter than -- and R_g,eff goes negative with them.

        Fix the SPACING and the condition is the same at every length, the
        stations sit at the same measured depths whatever L is (which is
        what makes V10 meaningful), and every run of a sweep is discretized
        identically.

        ds = 50 m is the shipped value. Measured over the whole L sweep,
        100 to 2000 m: zero negative strengths at every length for R1, R3
        and R4, with the off-station kernel weight rising smoothly from
        0 to 6.4 %. Refining to 25 m introduces negatives at L = 1250 m
        while moving the loss by 0.04 points, so the fine grid is worse
        posed and not more accurate. See station_convergence.csv.
        """
        return int(max(3, round(L / float(ds if ds else DS_STATION))))

    @staticmethod
    def _is_vertical(X, tol=1e-9):
        """True when every trajectory keeps its surface (x, y)."""
        return bool(np.abs(X[:, :, :2] - X[:, :1, :2]).max() <= tol)

    def profiles(self, md, X, R_near, nst=50, perimeter=None):
        """Field-mean R_g,eff(MD) for 'flux' and 'temp' (and 'per' if given).

        VERTICAL FIELDS ARE SOLVED ONCE, NOT nst TIMES. For parallel
        vertical wells the nearest point on a neighbor is at the same
        depth, so the interaction matrix is exactly block diagonal in
        station (measured: off-station kernel weight 0.00 %) and every
        block is the same matrix -- the horizontal distances do not change
        with depth. R_g,eff is then a single number, independent of both
        nst and L, which is what the measured sweep shows (R1: 15.783 m K/W
        at every length and every count; R3: 3.230).

        This is not an approximation, and it is what makes the disc rung
        tractable: at 1.65 m spacing all 363 wells sit inside 18 m, so
        every target has ~90 000 candidate samples inside the 80 m cutoff
        and doing that 25 times over was not finishing.
        """
        if self._is_vertical(X) and nst > 1:
            # One station, on a two-sample stub of the same trajectories.
            # The kernel sees only the horizontal distance, which is the
            # same at every depth for vertical wells, so the stub gives the
            # identical matrix at a fraction of the cost. Its own tvd is
            # meaningless and is discarded; the real one is taken below.
            one = self.profiles(md[:2], X[:, :2, :], R_near, nst=1)
            st = (np.arange(nst) + 0.5) * md[-1] / nst
            rep = lambda v: np.repeat(np.asarray(v).reshape(-1)[:1], nst)
            out = dict(md=st, flux=rep(one['flux']), temp=rep(one['temp']),
                       flux_raw=rep(one['flux_raw']),
                       temp_raw=rep(one['temp_raw']),
                       q_spread=rep(one['q_spread']),
                       tvd=X[:, np.clip(np.searchsorted(md, st), 0,
                                        len(md) - 1), 2].mean(0))
            self.n_negative = one and self.n_negative
            self.off_station_weight = 0.0
            if perimeter is not None:
                out['per'] = np.maximum(
                    self.Rg, N_ := X.shape[0] * np.sqrt(np.pi * self.a * self.t)
                    / (self.k * perimeter(st)))
            return out
        st, r, c, d, N = self.pairs(md, X, nst)
        g = self.kern(d)
        M = N * nst
        self_t = self.kern(np.array([self.rb]))[0]
        # equal flux: mean wall rise per unit q. Row sums only, so this
        # needs no matrix at all.
        # Structural diagnostic, recorded while the entries are in hand:
        # how much kernel weight couples a target to a neighbor at a
        # DIFFERENT station. Zero means the system is exactly block
        # diagonal in station and the answer cannot depend on nst; a large
        # value means the stations have been refined past the point where
        # a station still represents a line source, which is the same
        # failure that produces negative strengths.
        self.off_station_weight = float(
            g[(r % nst) != (c % nst)].sum() / g.sum()) if len(g) else 0.0
        rowsum = np.bincount(r, weights=g, minlength=M)
        flux = (self_t + rowsum).reshape(N, nst).mean(0)
        del rowsum
        # equal temperature: (R_near + G_ii) q + G q = 1.
        #
        # ASSEMBLED AS IT STANDS. Earlier versions solved
        # (G + G^T)/2 instead, and that was wrong. G is not symmetric for
        # inclined wells and has no reason to be: row (i, s) says "the
        # temperature rise at MY location, on well i at station s, due to
        # well k", and the nearest point on k from there is at some other
        # station s'. The reverse entry is about a different target point
        # and a different distance. Averaging the two mixes rows that are
        # not about the same place.
        #
        # Measured here: ||G - G^T|| / ||G|| is 0.25 at 50 m station
        # spacing and 0.90 at 25 m. The symmetrized system returns 130
        # negative strengths at 25 m and 569 at 8 m; the system as
        # assembled returns NONE at any spacing tested, and its mean
        # R_g,eff moves under 0.8 % across a fourfold refinement. The
        # negative strengths were an artifact of the symmetrization, not
        # of the grid.
        #
        # For parallel vertical wells the nearest point IS at the same
        # depth and the distances are mutual, so G is symmetric there and
        # the symmetrization was a no-op -- which is why this went unseen.
        A = sp.coo_matrix(
            (np.concatenate([g, np.full(M, R_near + self_t)]),
             (np.concatenate([r, np.arange(M, dtype=r.dtype)]),
              np.concatenate([c, np.arange(M, dtype=r.dtype)]))),
            shape=(M, M))
        del g, r, c, d
        b = np.ones(M)
        if M <= _DIRECT_MAX:
            q = spla.spsolve(A.tocsc(), b)
        else:
            # NOT CG. CG needs a symmetric positive-definite matrix, and
            # this one is neither symmetric nor diagonally dominant: the
            # diagonal is R_near + G_ii = 0.567 m K/W against a largest
            # off-diagonal row sum of 3.73, because near the surface all
            # 363 wells are inside the 80 m cutoff. BiCGSTAB with a
            # Jacobi preconditioner, checked against the direct solve
            # wherever both run.
            Ac = A.tocsr()
            del A
            Mp = spla.LinearOperator(
                (M, M), matvec=lambda x, dg=Ac.diagonal(): x / dg)
            q, info = spla.bicgstab(Ac, b, rtol=1e-12, atol=0.0,
                                    maxiter=5000, M=Mp)
            if info != 0:
                raise RuntimeError(
                    f'BiCGSTAB did not converge (info={info}) at M={M}')
        q = q.reshape(N, nst)
        # V11: every solved strength must be positive. All wells sit at the
        # same temperature and all are hotter than the far field, so every
        # one of them rejects heat; a negative q is the discretization
        # failing, not a result. Recorded so the caller can assert on it.
        self.n_negative = int((q < 0).sum())
        temp = 1.0 / q.mean(0) - R_near
        out = dict(md=st, flux=np.maximum(flux, self.Rg), temp=np.maximum(temp, self.Rg),
                   flux_raw=flux, temp_raw=temp, q_spread=q.std(0) / q.mean(0),
                   tvd=X[:, np.clip(np.searchsorted(md, st), 0, len(md) - 1), 2].mean(0))
        if perimeter is not None:
            out["per"] = np.maximum(self.Rg, N * np.sqrt(np.pi * self.a * self.t)
                                    / (self.k * perimeter(st)))
        return out


# ------------------------------------------------------------ BOREAS hook --
def run_boreas(B, L, md_prof, Rg_prof, tvd_prof=None, n_segments=99, **case_kw):
    """Run simulate_css_corrected with R_g,eff(MD) and T_f(TVD(MD)).

    Patches B.formation_profile / B.formation_resistance for the duration of
    the call only. The 'depth' passed between them is MD from the wellhead.
    """
    case = B.Case(n_segments=n_segments, L_ft=L / FT, **case_kw)
    fp0, fr0 = B.formation_profile, B.formation_resistance
    R_near = fr0(case) - B.ground_resistance(case)
    md_p = np.asarray(md_prof, float); Rg_p = np.asarray(Rg_prof, float)
    tvd_p = md_p if tvd_prof is None else np.asarray(tvd_prof, float)

    def tvd_of(mdv):
        if tvd_prof is None:
            return np.asarray(mdv, float)
        x = np.r_[0.0, md_p]; y = np.r_[0.0, tvd_p]
        slope = (y[-1] - y[-2]) / (x[-1] - x[-2])
        mdv = np.asarray(mdv, float)
        return np.where(mdv <= x[-1], np.interp(mdv, x, y), y[-1] + slope * (mdv - x[-1]))

    def fprof(c, n_segments=None, discharge=False):
        n = n_segments or c.n_segments
        xi = (np.arange(n) + 0.5) / n
        mdv = c.L_well * (1.0 - xi)                 # charge frame: 0 = bottom
        T = c.T_surface_C + 273.15 + c.grad_K_per_km / 1000.0 * tvd_of(mdv)
        if discharge:
            return T[::-1].copy(), mdv[::-1].copy()
        return T, mdv

    def fres(c, depth=None):
        if depth is None:
            return fr0(c)
        return R_near + np.interp(np.asarray(depth, float), md_p, Rg_p)

    B.formation_profile, B.formation_resistance = fprof, fres
    try:
        r = B.simulate_css_corrected(case)
        k = B.performance_indices(case, r)
    finally:
        B.formation_profile, B.formation_resistance = fp0, fr0
    Nw = r["N_wells"]
    E_out = (k["W_el_out_kW"] - k["pumping_dc_kW"]) * case.t_dc / Nw   # net, per well
    E_in = (k["W_el_in_kW"] + k["pumping_ch_kW"]) * case.t_ch / Nw
    return dict(L=L, loss=k["f_formation_loss"], RTE=k["eta_RTE"],
                # v1.4 onwards the package reports whether the store
                # actually gives heat back; surface it rather than let a
                # non-delivering rung look like a small positive.
                delivers=bool(k.get("delivers", r.get("delivers", True))),
                eta_storage=k["eta_storage"], e_prime=k["kWh_per_m_well"],
                E_out_kWh=E_out, E_in_kWh=E_in, T2d_C=r["T_2d_realized"] - 273.15,
                N_wells_1MW=Nw, R_near=R_near)


# ======================================================================
#  Making the measured field the DEFAULT, not an add-on.
#
#  A notebook declares the field once and every later cell -- profiles,
#  animation, parametric studies -- inherits it. Without this the
#  idealized R_field(z) leaks back in wherever someone calls
#  simulate_css_corrected directly, which is most places.
# ======================================================================
from contextlib import contextmanager as _ctx


def reference_field(B, case, csv, island='Grissom', trajectory='s_curve',
                    kop_rule='staggered', theta_max=55.0, kop_ft=300.0,
                    L_md=None, ds=None, verbose=True,
                    storage_set='all_slots', positions=None):
    """Positions, trajectories and R_g,eff(MD) for the design point.

    Returns (md, Rg, tvd, info). Hand the first three to `field_ground`.

    The station count follows from ONE fixed spacing, `ds` (default
    DS_STATION = 50 m). There is no search: V11 is asserted instead.
    """
    import numpy as _np
    L = float(L_md if L_md is not None else case.L_well)
    if island != 'Grissom':
        raise ValueError(f'no slot data for island {island!r}')
    # `positions` lets a caller hand in an explicit slot subset -- used by
    # the storage/producer allocation, where the storage count is the
    # unknown and the slots are chosen from the ABLE set rather than by
    # well status. It overrides storage_set entirely.
    if positions is not None:
        P = _np.asarray(positions, float)
    else:
        P = grissom_positions(csv, statuses=statuses_for(storage_set))
    kop = None
    if trajectory == 's_curve':
        kop = (staggered_kop(P)[0] if kop_rule == 'staggered'
               else _np.full(len(P), kop_ft * FT))
    md, X = trajectories(P, L, kind=trajectory, kop=kop, theta_max=theta_max)
    R_near = float(_np.atleast_1d(
        B.formation_resistance(case, _np.array([0.0])))[0]
        - B.ground_resistance(case))
    gr = Ground(k=case.k_rock, rhocp=case.rho_cp_rock,
                t_yr=case.t_operation_yr, rb=case.r_bore)
    nst = Ground.stations_for(L, ds)
    pr = gr.profiles(md, X, R_near, nst=nst)
    # V11, as an assertion and not a search. Every well sits at the same
    # temperature and every one of them is hotter than the far field, so
    # every solved strength must be positive. A negative one is the
    # discretization failing, and the response is to report it, not to
    # quietly try a different station count -- that is what made two runs
    # of the same sweep incomparable in v2.0.
    if gr.n_negative:
        raise RuntimeError(
            f'V11 FAILED: {gr.n_negative} of {len(P) * nst} strengths are '
            f'negative at L = {L:.0f} m with ds = {ds or DS_STATION:.0f} m '
            f'({nst} stations). This is a geometry or kernel problem; do '
            f'not work around it by changing the station count.')
    # tvd_bottom is the TVD at the BOTTOM OF THE STORE, md = L, taken
    # from the trajectory. It used to be pr['tvd'][-1], the TVD at the
    # last STATION CENTRE -- at ds = 50 m that is md = 1225 m, so the
    # reference case was reported 18 m too shallow, 891 against 909.
    info = dict(N=len(P), L_md=L, n_stations=nst,
                ds_station=float(ds or DS_STATION),
                tvd_last_station=float(pr['tvd'][-1]),
                off_station_weight=gr.off_station_weight, R_near=R_near,
                Rg_mean=float(_np.mean(pr['temp'])),
                Rg_top=float(pr['temp'][0]), Rg_bottom=float(pr['temp'][-1]),
                tvd_bottom=float(X[:, -1, 2].mean()), island=island,
                trajectory=trajectory, kop_rule=kop_rule,
                theta_max=theta_max, positions=P, kop=kop)
    if verbose:
        print(f"field: {info['N']} {island} slots, {trajectory}, "
              f"L = {L:.0f} m MD -> TVD {info['tvd_bottom']:.0f} m")
        print(f"  R_g,eff {info['Rg_top']:.2f} (top) to "
              f"{info['Rg_bottom']:.2f} (bottom), mean {info['Rg_mean']:.2f} "
              f"m K/W over {nst} stations")
    return pr['md'], pr['temp'], pr['tvd'], info


# The superposition solve is the expensive step in the whole model: 363
# wells x 50 stations is an 18150-unknown sparse system, about 85 s, and
# reference_field tries several station counts before one solves cleanly.
# That is fine in a batch runner and absurd in the first cell of a
# notebook, so the result is cached by the SPECIFICATION it came from.
# WHICH COMMIT the cached field is fetched from. A branch tracks the live
# line; a TAG freezes it. A notebook pinned to a tag sets GITREF here too,
# otherwise it would download a field solve from a moving branch while
# believing itself frozen.
GITREF = 'notebook-model'


def _ref_url():
    return ('https://raw.githubusercontent.com/JaderBarbosaLSU/'
            f'wellbore-thermal-storage/{GITREF}/field_ladder/')


def _ref_key(case, island, trajectory, kop_rule, theta_max, kop_ft, L,
             ds=None, storage_set='all_slots'):
    """Everything the profile depends on, and nothing else.

    The case enters only through r_bore (the kernel's inner radius), the
    rock properties and the field age; the PCM, the plant and the fin
    geometry do not touch the ground solve, so changing them must not
    invalidate the cache.
    """
    bits = [island, trajectory, kop_rule, f'{theta_max:.4g}',
            f'{kop_ft:.4g}', f'{L:.6g}', f'{case.r_bore:.6g}',
            f'{case.k_rock:.6g}', f'{case.rho_cp_rock:.6g}',
            f'{case.t_operation_yr:.6g}', f'ds{float(ds or DS_STATION):.4g}']
    if storage_set != 'all_slots':
        bits.append(storage_set)   # 'all_slots' keeps the v2.8 key
    return 'fieldref_' + '_'.join(bits).replace('.', 'p').replace('+', '')


def cached_reference_field(B, case, csv, cache_dir='.', fetch=True,
                           verbose=True, **kw):
    """`reference_field`, read from disk or from the repo when possible.

    Returns the same (md, Rg, tvd, info). `info` carries 'cached', which is
    the provenance: True means nothing was solved here. A cache miss is not
    an error -- it recomputes and writes the file -- but it costs minutes,
    so it says so first.
    """
    import os
    import urllib.request
    import numpy as _np
    L = float(kw.get('L_md') or case.L_well)
    key = _ref_key(case, kw.get('island', 'Grissom'),
                   kw.get('trajectory', 's_curve'),
                   kw.get('kop_rule', 'staggered'),
                   float(kw.get('theta_max', 55.0)),
                   float(kw.get('kop_ft', 300.0)), L,
                   kw.get('ds'),
                   kw.get('storage_set', 'all_slots'))
    path = os.path.join(cache_dir, key + '.npz')
    if not os.path.exists(path) and fetch:
        try:
            urllib.request.urlretrieve(_ref_url() + key + '.npz', path)
        except Exception:
            pass
    if os.path.exists(path):
        try:
            d = _np.load(path, allow_pickle=True)
            md, Rg, tvd = d['md'], d['Rg'], d['tvd']
            info = dict(d['info'].item())
            info['cached'] = True
            if verbose:
                print(f"field: {info['N']} {info['island']} slots, "
                      f"{info['trajectory']}, L = {L:.0f} m MD -> TVD "
                      f"{info['tvd_bottom']:.0f} m   [cached]")
                print(f"  R_g,eff {info['Rg_top']:.2f} (top) to "
                      f"{info['Rg_bottom']:.2f} (bottom), mean "
                      f"{info['Rg_mean']:.2f} m K/W over "
                      f"{info['n_stations']} stations")
            return md, Rg, tvd, info
        except Exception:
            pass
    if verbose:
        print('no cached field for this specification -- solving the '
              f'superposition.\n  Written to {path} and reused from then '
              'on.', flush=True)
    md, Rg, tvd, info = reference_field(B, case, csv, verbose=verbose, **kw)
    info['cached'] = False
    try:
        store = {k: v for k, v in info.items() if k not in ('positions', 'kop')}
        _np.savez(path, md=md, Rg=Rg, tvd=tvd, info=_np.array(store, dtype=object))
    except Exception:
        pass
    return md, Rg, tvd, info


@_ctx
def field_ground(B, md, Rg, tvd):
    """Run the model against a measured field, not the idealized R_field(z).

    Patches formation_profile and formation_resistance for the duration, so
    ANY call inside -- including simulate_css_corrected(record=True), which
    is what the profile and animation figures need -- sees the
    superposition resistance and the true vertical depth. Restored on exit
    even if the body raises.
    """
    import numpy as _np
    fp0, fr0 = B.formation_profile, B.formation_resistance
    md_a = _np.asarray(md, float)
    Rg_a = _np.asarray(Rg, float)
    tvd_a = _np.asarray(tvd, float)
    x = _np.r_[0.0, md_a]
    y = _np.r_[0.0, tvd_a]
    slope = (y[-1] - y[-2]) / (x[-1] - x[-2])

    def _tvd(mdv):
        mdv = _np.asarray(mdv, float)
        return _np.where(mdv <= x[-1], _np.interp(mdv, x, y),
                         y[-1] + slope * (mdv - x[-1]))

    def fprof(c, n_segments=None, discharge=False):
        n = n_segments or c.n_segments
        xi = (_np.arange(n) + 0.5) / n
        mdv = c.L_well * (1.0 - xi)          # charge frame: z = 0 is bottom
        T = c.T_surface_C + 273.15 + c.grad_K_per_km / 1000.0 * _tvd(mdv)
        if discharge:
            return T[::-1].copy(), mdv[::-1].copy()
        return T, mdv

    # A TVD mapping for the PLOTTING layer. formation_profile's second
    # return value has to stay MEASURED depth, because fres interpolates
    # R_g,eff against it -- but a figure drawn on that axis shows a
    # 1250 m well where the S-curve reaches 891 m. depth_axis picks this
    # up when present, so every figure and the animation get TVD without
    # the solver's own lookup changing at all.
    B._boreas_tvd_of = _tvd

    def fres(c, depth=None):
        if depth is None:
            return fr0(c)
        rn = float(_np.atleast_1d(fr0(c, _np.array([0.0])))[0]
                   - B.ground_resistance(c))
        return rn + _np.interp(_np.asarray(depth, float), md_a, Rg_a)

    B.formation_profile, B.formation_resistance = fprof, fres
    try:
        yield
    finally:
        B.formation_profile, B.formation_resistance = fp0, fr0
        if hasattr(B, '_boreas_tvd_of'):
            del B._boreas_tvd_of


### 1.1 The inputs, handed to every script

`spec.py` and `field.py` are how a value typed in section 2 reaches the
twenty-odd scripts that draw the figures and build the macro pack. They
read `ref_case.json` and `boreas_spec.json`, which section 2 writes.

Before v2.19 six figure generators carried their own inline copy of the
reference case and nine repeated the field geometry as literal arguments,
so a change in section 2 reached some of the output and not the rest, with
nothing to say which.

In [ ]:
%%writefile spec.py
"""Every list of values the paper's figures and macros sweep over.

One place. The notebook's input section writes `boreas_spec.json` and this
reads it, so a value changed there changes every driver, every figure and
every macro that depends on it. If the file is absent these defaults are
the published ones, so the module is usable outside the notebook too.

Nothing here is a result. Results live in the run artefacts.
"""
import json, os

DEFAULTS = dict(
    # ---- the sweeps every figure and parametric macro draws from -------
    T_m_C=[120.0, 125.0, 130.0, 135.0, 140.0, 150.0, 160.0],
    t_ch_h=[9, 10, 11, 12, 13, 14],
    h_m_kJ_kg=[200.0, 260.0, 380.0, 500.0],
    sink_glide_K=[2.0, 3.0, 4.0, 5.0, 7.0, 10.0],
    grad_K_per_km=[35.0, 40.0, 45.0, 50.0, 56.5],
    storage_length_m_MD=[100.0, 200.0, 300.0, 400.0, 500.0, 600.0,
                         800.0, 1000.0, 1250.0],
    # the reduced length set used for the gradient-by-length cutoff map
    cutoff_length_m_MD=[100.0, 500.0, 600.0, 800.0, 1000.0, 1250.0],
    # ---- the field ladder ---------------------------------------------
    ladder_rungs=['adiabatic', 'R0v', 'R0i', 'R1', 'R2', 'R3', 'R4', 'R4i'],
    # ---- one-at-a-time sensitivities ----------------------------------
    sens=dict(weight36=dict(D_in=7.825),
              ann06=dict(k_cement=0.6),
              ann15=dict(k_cement=1.5),
              topone=dict(t_operation_yr=1.0),
              topthirty=dict(t_operation_yr=30.0),
              isosink=dict(DT_sink_glide=0.0),
              seawater=dict(source='seawater', T_source_C=17.0, DT_3A_4A=5.0),
              kop300=dict()),
    # ---- discretisation and sink-accounting runs ----------------------
    extra=dict(convbase=dict(n_segments=99, n_times=40),
               convns=dict(n_segments=198, n_times=40),
               convnt=dict(n_segments=198, n_times=80),
               ds12=dict(_ds=12.5),
               sinkfree=dict(T_sink_C=17.0, DT_sink_glide=0.0),
               glideonly=dict(DT_sink_glide=4.0, sink_loop_dp_bar=0.0)),
    # ---- the field geometry the reference field is built on ------------
    field=dict(trajectory='s_curve', kop_rule='staggered', theta_max=55.0,
               kop_ft=300.0, L_md=1250.0, n_slots=363),
    # ---- the sliding-pressure bracket ---------------------------------
    sliding_cases=['R4', 'R3', 'adiabatic'],
    sliding_n_grid=13,
    # ---- the convergence test used by \\IntegratorCheck ----------------
    integrator=dict(E_over_E_lat=[-0.2, 0.0, 0.5, 1.2], dT_K=[25.0, -25.0],
                    K=[1e-3, 1e-1], dt_s=[600.0, 8491.0]),
    css_tol=1e-6,
)

_PATH = os.environ.get('BOREAS_SPEC', 'boreas_spec.json')
SPEC = dict(DEFAULTS)
if os.path.exists(_PATH):
    SPEC.update(json.load(open(_PATH)))


def get(k):
    return SPEC[k]


In [ ]:
%%writefile field.py
"""The reference case and the well field, in one place, for every generator.

Six figure generators carried their own inline copy of the reference case
(`mkgeom216`, `mkladder216`, `mkR3prof`, `mkts`, `mkprofiles`, `mkS1`),
and nine files repeated the field geometry as literal arguments --
`theta_max=55.0, kop_ft=300.0, L_md=1250.0`. They were made standalone in
v2.16 so the notebook could run them, and the cost only became visible
when the input section became the thing the user edits: changing T_m or the
storage length in one place left six figures drawn at the old value, with
nothing to say so.

Everything now comes from here, and here reads two files:

  ref_case.json    every field of the reference Case -- written by the
                   notebook's input section, so the figures follow it
  boreas_spec.json the sweep lists and the field geometry (spec.py)

Both have defaults, so a generator still runs outside the notebook.
"""
import json, os
import boreas as B
import boreas_ground as G
import spec

BUILD = os.environ.get('BOREAS_BUILD', '.')
CSV = os.path.join(BUILD, os.environ.get('BOREAS_CSV',
                                         'THUMS_wells_WellSTAR_clean.csv'))
if not os.path.exists(CSV):
    CSV = 'THUMS_wells_WellSTAR_clean.csv'

_RC = os.path.join(BUILD, 'ref_case.json')
_RAW = json.load(open(_RC)) if os.path.exists(_RC) else None

# the published reference, used only when ref_case.json is absent
_FALLBACK = dict(T_m_C=130.0, cascade_param=48.9, N_lay=12, h_m=380e3,
                 source='produced_water', T_source_C=60.0, DT_3A_4A=15.0,
                 T_sink_C=17.0, DT_sink_glide=4.0, sink_loop_dp_bar=0.5,
                 t_ch=12.0, t_dc=12.0, x_int_orc=0.31, x_int_hp=0.30,
                 grad_K_per_km=56.5, n_segments=99, t_operation_yr=10.0,
                 D_in=201.2 / 25.4, r_casing_o=0.21910 / 2,
                 r_bore=0.31120 / 2, k_cement=0.7)

FIELD = dict(spec.SPEC['field'])
FIELD.setdefault('ds', G.DS_STATION)
L_MD = float(FIELD['L_md'])
N_SLOTS = int(FIELD['n_slots'])


def case(**kw):
    """The reference Case. kw overrides, exactly as ref.v216 did."""
    if _RAW is None:
        d = dict(_FALLBACK); d['L_ft'] = L_MD / 0.3048
    else:
        base = B.Case()
        d = {}
        for k, v in _RAW.items():
            try:
                dv = getattr(base, k)
            except AttributeError:
                continue
            if isinstance(dv, bool):
                d[k] = v in (True, 'True', 'true', 1)
            elif isinstance(dv, int):
                d[k] = int(float(v))
            elif isinstance(dv, float):
                d[k] = float(v)
            elif isinstance(dv, str):
                d[k] = str(v)
            elif isinstance(dv, tuple):
                d[k] = tuple(v)      # JSON has no tuple; brackets come back
            else:                    # as lists and compare unequal
                d[k] = v
    d.update(kw)
    return B.Case(**d)


def positions():
    return G.grissom_positions(CSV)


def build(mod_B, c, trajectory=None, L_md=None, ds=None, P=None, **kw):
    """The reference field, with the geometry taken from the spec.

    One call site for the nine that repeated theta_max / kop_ft / L_md as
    literals. Pass trajectory='vertical' for the R3 rungs.
    """
    a = dict(trajectory=trajectory or FIELD['trajectory'],
             kop_rule=FIELD['kop_rule'], theta_max=float(FIELD['theta_max']),
             kop_ft=float(FIELD['kop_ft']), L_md=float(L_md or L_MD),
             ds=ds or FIELD['ds'], verbose=False)
    a.update(kw)
    return G.reference_field(mod_B, c, CSV,
                             positions=positions() if P is None else P, **a)


In [ ]:
%%writefile ref.py
"""Compatibility shim: the generators' historical entry point.

They were written against `ref.v216()`. The reference now comes from
field.py, which reads cell 2.1. This keeps the old name working so the
scripts in the pack are the scripts that ran, unedited.
"""
import field

v216 = field.case
v218 = field.case
v219 = field.case


The slot coordinates come from the CalGEM WellSTAR export. They are fetched
rather than embedded: the file is the field, and a copy pasted into a
notebook is a copy that goes stale.

In [ ]:
import os, urllib.request

# WHICH COMMIT THIS NOTEBOOK DOWNLOADS FROM.
#   a BRANCH name tracks the live line and changes under you;
#   a TAG is frozen for ever.
# Your copy was built pinned to the value below. Changing it repoints
# every download in this notebook at once.
GITREF = 'notebook-model'
RAW = ('https://raw.githubusercontent.com/JaderBarbosaLSU/'
       'wellbore-thermal-storage/' + GITREF + '/')

CSV = 'THUMS_wells_WellSTAR_clean.csv'
if not os.path.exists(CSV):
    urllib.request.urlretrieve(RAW + 'field_ladder/' + CSV, CSV)
import boreas_ground as G
G.GITREF = GITREF          # the cached field solve comes from the same ref
print(f'slot data ready  (fetched from {GITREF})')

In [ ]:
import importlib, sys
for _m in ('boreas',):
    sys.modules.pop(_m, None)     # %%writefile changed the file on disk
import boreas as B
import numpy as np, matplotlib.pyplot as plt, pandas as pd
print(B.__doc__.strip().splitlines()[0])
c = B.Case()
print(f'\ndefault case: {c.n_ft} fin tubes + 1 downcomer, '
      f'{c.L_ft:.0f} ft, T_m = {c.T_m_C:.0f} C')

---
## 2. The reference case (R4) — **edit here, nowhere else**

Every parameter you are likely to change lives in the next cell. Nothing
below it hard-codes a number: each later section starts from `CASE` and
says, in one visible `.with_(...)`, what it is changing and why.

The split is deliberate:

* **`CASE`** is the well, the material and the plant — what you are
  building. It says nothing about how many of them there are or how they
  are arranged on the surface.
* **`FIELD`** is the arrangement — which island, which trajectory, how
  deep the wells kick off. It is kept separate because the *same* well
  gives losses from 3 % to 69 % depending only on this, and because the
  model's own defaults describe an **isolated** well, which is a bound and
  not a design.

`Case` has 83 fields. The ones written out below are those worth touching;
the rest keep their defaults, and the audit cell that follows prints
everything you have changed so nothing hides.

In [ ]:
# =============================================================================
#  THE DESIGN POINT.  Edit the numbers HERE -- not in the model cell, and not
#  in the sections below.  Units are in front of every value.
# =============================================================================
CASE = B.Case(
    # --- PCM and cascade -----------------------------------------------------
    T_m_C      = 130.0,      # melting temperature of the HOTTEST layer     [C]
                             #   v2.9: WAS 150. Lowering the cascade top cuts
                             #   the heat-pump lift AND the store's dT to the
                             #   formation at once, and costs only on the ORC
                             #   side. 130 is the measured optimum once the
                             #   idle periods are simulated (135 without).
                             #   the cascade is graded with the geotherm, so
                             #   this layer sits at the BOTTOM of the well
    h_m        = 380_000.0,  # latent heat of fusion                     [J/kg]
    k_s        = 0.60,       # solid conductivity                       [W/m/K]
    k_l        = 0.45,       # liquid conductivity                      [W/m/K]
    N_lay      = 9,          # cascade layers along the well                [-]
    DT_3C_2C   = 55.0,       # cascade grading; SPAN = this*(N_lay-1)/N_lay [K]
                             #   -> 48.889 K.  DT_cascade overrides it if set;
                             #   setting DT_cascade EQUAL to this is a no-op.

    # --- the well ------------------------------------------------------------
    #   NOTE  the storage length is NOT here. It is L_md in FIELD below,
    #   in meters of MEASURED depth, because that is what the trajectory
    #   and the casing shoe are quoted in. Setting L_ft here as well is an
    #   error and the guard below says so rather than silently winning.
    n_ft       = 3,          # fin tubes per well, all downward             [-]
    num_fins   = 24,         # fins per tube                                [-]
    fin_L      = 0.0035,     # fin length                                   [m]
    fin_t      = 0.0015,     # fin thickness                                [m]
    t_ins      = 0.008,      # DOWNCOMER insulation (the casing is bare)    [m]
    k_ins      = 0.007,      # downcomer insulation, VIT                [W/m/K]

    # --- casing: THUMS upper zone, 8-5/8 in 32# (Truex & Hunter, SPE-2562)
    #   D_in is the casing INSIDE diameter and is what bounds the PCM. The
    #   201.2 mm ID carries 46 % more material per tube than the 177.8 mm
    #   legacy string, which is worth +36 % on e' -- a larger effect than
    #   the choice of ground model. Set D_in = 7.0, r_casing_o = 0.098 and
    #   r_bore = 0.110 to recover the legacy reference.
    D_in       = 201.2 / 25.4,  # 8-5/8 in 32# inside diameter             [in]
    r_casing_o = 0.21910 / 2,   # 219.1 mm outside diameter                 [m]
    r_bore     = 0.31120 / 2,   # 12-1/4 in hole                            [m]
    k_cement   = 0.7,           # annulus fill: mud above the TOC       [W/m/K]
                                #   cement (1.0) instead moves the loss by
                                #   0.09 pts and e' by 0.1 %, so the top of
                                #   cement is not worth chasing

    # --- formation -----------------------------------------------------------
    T_surface_C   = 18.0,    # surface temperature                          [C]
    grad_K_per_km = 56.5,    # geothermal gradient                       [K/km]
                             #   v2.16: 56.5 for central Wilmington (CDMG
                             #   1983, DOE/SF/10855-3) is the only sourced
                             #   value; 45 had no source. Worth +10 % on
                             #   eta_RTE, and it removes the cutoff: the
                             #   marginal return never falls below the
                             #   threshold above about 50 K/km.
    k_rock        = 2.5,     # rock conductivity                        [W/m/K]
    rho_cp_rock   = 2.34e6,  # rock volumetric heat capacity          [J/m3/K]
    t_operation_yr = 10.0,   # field age at which R_g is evaluated          [yr]

    # --- plant ---------------------------------------------------------------
    W_dot_el_out = 1000.0,   # net electrical output                      [kWe]
    t_ch         = 12.0,     # charge duration                               [h]
    t_dc         = 12.0,     # discharge duration                            [h]
                             #   v2.9: 12 + 12 FILLS the 24 h day, so nothing
                             #   is left idle. Idling is never optimal: every
                             #   filled-day split beats 10/10 with idle. See
                             #   the split study for the other five.

    # --- the heat-pump source ------------------------------------------------
    #   v2.9: SEAWATER, not the 60 C geothermal producer and not the island's
    #   co-produced water. Both of those are SCARCE, and scarcity forces a
    #   large source glide, which drags the evaporator down about as far as a
    #   cold source would: 60 C water across 15 K evaporates at 40 C, 17 C
    #   seawater across 5 K evaporates at 7 C. The produced water also costs
    #   266 of the island's 382 able slots to supply, which the store cannot
    #   then use. Set source='produced_water', T_source_C=60, DT_3A_4A=15 to
    #   reproduce that comparison.
    #   v2.10: the source is 60 C CO-PRODUCED WATER. Its PROVISION -- which
    #   slots must keep producing to supply it, and the surface buffer -- is
    #   deferred to the companion study. At DT 15 K and with a buffer the
    #   allocation is 116 storage wells against 266 producers; the paper
    #   states that as an assumption rather than assuming it silently.
    #   Set source='seawater', T_source_C=17, DT_3A_4A=5 for the all-seawater
    #   variant, which needs no producers but costs 28 % of eta_RTE.
    source       = 'produced_water',
    T_source_C   = 60.0,     # co-produced water                             [C]
    DT_3A_4A     = 15.0,     # source glide: scarce source, so a wide glide  [K]
    source_loop_dp_bar = 1.0,# intake, piping, evaporator                  [bar]
    source_pump_eff    = 0.75,

    # --- THE ORC SINK: the same seawater, and NOT free ----------------------
    #   DT_sink_glide = 0 made the condenser an infinite reservoir: the water
    #   carried heat away WITHOUT warming, so T_1e sat at T_sink + 5 and the
    #   pump cost nothing. Both favours are now withdrawn. The temperature one
    #   was the larger: -10.7 % against -6.1 % for the pump.
    #   The glide is a design variable -- small keeps the condenser cold and
    #   costs pumping, large saves pumping and raises T_1e kelvin for kelvin.
    #   4 K is the optimum at 0.5 bar (3.5 K is 0.2 % better; 4 K is rounder).
    T_sink_C     = 17.0,     # seawater, same source as the heat pump        [C]
    DT_sink_glide = 4.0,     # condenser water temperature rise              [K]
    sink_loop_dp_bar = 0.5,  # larger, lower-velocity intake               [bar]
    sink_pump_eff    = 0.75,

    # --- intermediate pressures: p_int = p_low + x (p_high - p_low) ----------
    #   None keeps each cycle's own rule (ORC arithmetic, HTHP geometric).
    #   Both are wrong: the ORC's optimum IS the geometric mean (x = 0.31,
    #   not the 0.5 it used), and the HTHP's is NOT (x = 0.21 against a
    #   geometric 0.11), because the flash drum gives the two stages
    #   different flows. Together worth +4.1 % on eta_RTE.
    #   v2.10, re-measured at THIS reference: the GEOMETRIC mean is within
    #   0.5 % of the optimum for BOTH cycles (0.18 % ORC, 0.40 % HTHP), so one
    #   rule covers both. The arithmetic mean the ORC used costs 1.9 %.
    x_int_orc    = 0.31,     # geometric mean sits at 0.351
    x_int_hp     = 0.30,     # geometric mean sits at 0.20

    # --- idling --------------------------------------------------------------
    #   'simulate' integrates the hours when neither half-cycle runs, with
    #   zero flow and the formation as the only sink. At 10/10 those 2x2 h
    #   cost 10.4 % of eta_RTE and are 14 % of the total loss -- they were
    #   simply not simulated before v2.9. At 12/12 there are none, and
    #   idle_hours(CASE) returns 0 on its own.
    idle_mode    = 'simulate',
    t_idle       = None,     # h EACH period; None -> split what the day leaves

    # --- the secondary loop --------------------------------------------------
    #   v2.19: a second `T_sink_C = 20.0` sat here, left over from the
    #   pre-seawater default. Python rejects a repeated keyword, so THIS CELL
    #   DID NOT COMPILE in the v2.16 and v2.18 notebooks -- the first cell
    #   anyone runs after the model. It was never caught because both rounds
    #   verified the figure and pack cells against a case built in the task
    #   sandbox, not the one this cell defines. The sink is 17 C, set in the
    #   ORC sink block above, and every cell below is now checked by running
    #   this notebook top to bottom.
    P            = 10.0e5,   # secondary-loop pressure                      [Pa]
                             #   water saturates at 179.9 C here, which CAPS
                             #   T_m near 165 C; see section 10.5
    DT_pinch_ORC = 5.0,      # ORC evaporator pinch                          [K]
    DT_2H_3C     = 10.0,     # HTHP condenser approach                       [K]

    # --- line losses between the plant and the field -------------------------
    #   Figure 1 draws a coil on the charge line 3c -> 4c and on the
    #   discharge line 1d -> 2d. Each stands for the heat and pumping loss
    #   of the run between the plant and the wellhead.
    #   PLACEHOLDERS, NOT YET MODELED: at zero, T_4c = T_3c and
    #   T_2d = T_1d exactly, and every number matches results pack v2.7.
    line_ch_dT = 0.0,        # 3c -> 4c temperature drop                    [K]
    line_ch_dp = 0.0,        # 3c -> 4c pressure drop                      [Pa]
    line_dc_dT = 0.0,        # 1d -> 2d temperature drop                    [K]
    line_dc_dp = 0.0,        # 1d -> 2d pressure drop                      [Pa]

    # --- numerics ------------------------------------------------------------
    n_segments = 99,         # a MULTIPLE of N_lay, or a layer boundary falls
                             #   inside a segment and validate_case warns
)

# =============================================================================
#  THE FIELD.  Measured slot positions and real trajectories, not an
#  idealized arrangement. The old packing / n_islands / fan_angle_deg
#  description is retired: it could not represent where the wells actually
#  are, and at this spacing the difference is not small.
#
#    island      : slot coordinates from the WellSTAR export
#    trajectory  : 'vertical' or 's_curve' (build-and-hold, SPE-2562)
#    kop_rule    : 'staggered' outer rows kick off shallow, inner deep
#                  'fixed'     every well at kop_ft
#    L_md        : storage length in MEASURED depth, bounded by the shoe
# =============================================================================
FIELD = dict(
    island     = 'Grissom',   # 363 slots after dedup and stray removal   [-]
    trajectory = 's_curve',   # 'vertical' | 's_curve'
    kop_rule   = 'staggered', # 'staggered' | 'fixed'
    theta_max  = 55.0,        # maximum inclination                     [deg]
    kop_ft     = 300.0,       # used only when kop_rule = 'fixed'        [ft]
    L_md       = 1250.0,      # storage length in measured depth          [m]
    ds_station = 50.0,        # superposition station spacing             [m]
                              #   ONE spacing for every run of every sweep,
                              #   not a count and not a per-run search.
                              #   50 m is the shipped value, chosen on
                              #   CONVERGENCE: the loss moves 0.001 points
                              #   and e' 0.05 % to the next finer grid.
                              #   See section 5.2 and
                              #   station_convergence.csv.
)

# The storage length reaches Case through L_ft, and ONLY from here.
REF = CASE.with_(L_ft=FIELD['L_md'] / 0.3048)
G.DS_STATION = FIELD['ds_station']

# ---- no silent overrides ----------------------------------------------
# If L_ft was also set in CASE above and disagrees with L_md, that is an
# error, not a warning: the student would otherwise edit L_ft, see nothing
# change, and have no way to find out why.
_L_ft_default = B.Case().L_ft
if abs(CASE.L_ft - _L_ft_default) > 1e-9 and \
        abs(CASE.L_ft * 0.3048 - FIELD['L_md']) > 1e-6:
    raise ValueError(
        f"two storage lengths: CASE.L_ft = {CASE.L_ft:.1f} ft "
        f"({CASE.L_ft * 0.3048:.1f} m) against FIELD['L_md'] = "
        f"{FIELD['L_md']:.1f} m. Set the length in FIELD['L_md'] only.")

# The superposition profile for that field. Built once here and reused by
# every later cell through `field_ground`, so no section can quietly fall
# back to the idealized geometry.
#
# Cached by specification, including the station spacing. The file for the
# default FIELD is fetched from the repository; editing FIELD above forces
# a fresh solve, which the cell says it is doing.
MD, RG, TVD, FINFO = G.cached_reference_field(
    B, REF, CSV, island=FIELD['island'], trajectory=FIELD['trajectory'],
    kop_rule=FIELD['kop_rule'], theta_max=FIELD['theta_max'],
    kop_ft=FIELD['kop_ft'], L_md=FIELD['L_md'], ds=FIELD['ds_station'])
SLOTS = G.grissom_positions(CSV)        # plan view, for the figures below
print('reference case (R4) set.  REF = CASE at the field length; '
      'MD/RG/TVD describe the field.')

# ---- hand these inputs to every generator ------------------------------
# The sweeps, the figures and the macro pack all run as separate scripts.
# They read the case from ref_case.json and the field and sweep lists from
# boreas_spec.json, both written HERE. That is what makes this the only
# place an input is written down: change a value above and every script
# below follows it. Nothing reads a number from anywhere else.
import dataclasses, json as _json
_json.dump({f.name: getattr(REF, f.name) for f in dataclasses.fields(REF)},
           open('ref_case.json', 'w'), indent=1, default=str)
print('ref_case.json written -- %d fields' % len(dataclasses.fields(REF)))


### 2.2 The sweeps — the other half of the inputs

Cell 2.1 is the design point. These are the **lists of values the paper
sweeps over**: the cascade top, the division of the day, the latent heat,
the sink glide, the geothermal gradient, the storage length, the field
rungs and the one-at-a-time sensitivities.

They were buried in six driver scripts, one list each, which meant a
figure could be redrawn over a range nobody had looked at in months. They
are here now, and `boreas_spec.json` carries them to every script.

**Changing a list here changes the figures, the tables and the macros that
read it** — but only in `recompute` mode (2.3), because the published
results were marched at the values below. If you change a list and stay in
`published` mode, the cell that unpacks the results says so and stops.

In [ ]:
# =============================================================================
#  THE SWEEPS.  Edit the lists HERE.  Every driver, figure and macro that
#  sweeps a value reads it from this cell through boreas_spec.json.
# =============================================================================
SWEEPS = dict(
    # --- the cascade top -----------------------------------------------------
    #   eta_RTE peaks at 130 C, the delivered energy at 150 C, and 170 C does
    #   not deliver at all (the store loses more than it is charged), so the
    #   sweep stops at 160. Both optima are interior.
    T_m_C = [120.0, 125.0, 130.0, 135.0, 140.0, 150.0, 160.0],

    # --- the division of the 24 h day ----------------------------------------
    #   t_ch; t_dc is 24 - t_ch, so no window is left idle. The rating varies
    #   by more than half across this range while eta_RTE moves a few percent.
    t_ch_h = [9, 10, 11, 12, 13, 14],

    # --- the latent heat of the PCM ------------------------------------------
    #   380 kJ/kg is an ASSUMPTION, not a measurement: 260 needs 86 wells per
    #   MWe against 51. This range is the paper's honest bracket.
    h_m_kJ_kg = [200.0, 260.0, 380.0, 500.0],

    # --- the condenser water temperature rise --------------------------------
    sink_glide_K = [2.0, 3.0, 4.0, 5.0, 7.0, 10.0],

    # --- the geothermal gradient ---------------------------------------------
    #   56.5 K/km is the reference (CDMG 1983, central Wilmington). The lower
    #   values are the sensitivity, and they are what locate the cutoff: it
    #   exists only below about 50 K/km.
    grad_K_per_km = [35.0, 40.0, 45.0, 50.0, 56.5],
    grad_ref = 56.5,

    # --- the storage length, measured depth ----------------------------------
    #   The full nine-point sweep runs at the reference gradient; the reduced
    #   set runs at each of the others, which is enough to place the cutoff.
    storage_length_m_MD = [100.0, 200.0, 300.0, 400.0, 500.0, 600.0,
                           800.0, 1000.0, 1250.0],
    cutoff_length_m_MD = [100.0, 500.0, 600.0, 800.0, 1000.0, 1250.0],

    # --- the field ladder ----------------------------------------------------
    #   Each rung changes ONE thing: R0v a single vertical well, R0i a single
    #   S-curve, R1 a compact disc, R2 a double row, R3 the real slots
    #   vertical, R4 the real slots on S-curves (the reference), R4i the idle
    #   slots only. 'adiabatic' removes the formation altogether.
    ladder_rungs = ['adiabatic', 'R0v', 'R0i', 'R1', 'R2', 'R3', 'R4', 'R4i'],

    # --- one-at-a-time sensitivities -----------------------------------------
    sens = dict(weight36  = dict(D_in=7.825),              # 36 lb/ft string
                ann06     = dict(k_cement=0.6),            # annulus range
                ann15     = dict(k_cement=1.5),
                topone    = dict(t_operation_yr=1.0),      # field age
                topthirty = dict(t_operation_yr=30.0),
                isosink   = dict(DT_sink_glide=0.0),       # the idealisation
                seawater  = dict(source='seawater',        # the all-seawater
                                 T_source_C=17.0,          #   variant
                                 DT_3A_4A=5.0),
                kop300    = dict()),                       # fixed kickoff

    # --- discretisation, and the sink accounting -----------------------------
    extra = dict(convbase  = dict(n_segments=99,  n_times=40),
                 convns    = dict(n_segments=198, n_times=40),
                 convnt    = dict(n_segments=198, n_times=80),
                 ds12      = dict(_ds=12.5),               # station spacing
                 sinkfree  = dict(T_sink_C=17.0, DT_sink_glide=0.0),
                 glideonly = dict(DT_sink_glide=4.0, sink_loop_dp_bar=0.0)),

    # --- the sliding-pressure bracket ----------------------------------------
    sliding_cases = ['R4', 'R3', 'adiabatic'],
    sliding_n_grid = 13,

    # --- the integrator check, and the CSS criterion -------------------------
    integrator = dict(E_over_E_lat=[-0.2, 0.0, 0.5, 1.2], dT_K=[25.0, -25.0],
                      K=[1e-3, 1e-1], dt_s=[600.0, 8491.0]),
    css_tol = 1e-6,
)

# the field geometry lives in FIELD in cell 2.1; it travels with the spec
SWEEPS['field'] = dict(trajectory=FIELD['trajectory'], kop_rule=FIELD['kop_rule'],
                       theta_max=FIELD['theta_max'], kop_ft=FIELD['kop_ft'],
                       L_md=FIELD['L_md'], n_slots=len(SLOTS),
                       ds=FIELD['ds_station'])

import json as _json
_json.dump(SWEEPS, open('boreas_spec.json', 'w'), indent=1)
print('boreas_spec.json written.  %d marches if nothing is cached:'
      % (len(SWEEPS['T_m_C']) + len(SWEEPS['t_ch_h']) + len(SWEEPS['h_m_kJ_kg'])
         + len(SWEEPS['sink_glide_K']) + len(SWEEPS['sens'])
         + len(SWEEPS['ladder_rungs']) + len(SWEEPS['storage_length_m_MD'])
         + (len(SWEEPS['grad_K_per_km']) - 1) * len(SWEEPS['cutoff_length_m_MD'])
         + len(SWEEPS['extra']) + len(SWEEPS['sliding_cases']) + 3))
for _k in ('T_m_C', 't_ch_h', 'h_m_kJ_kg', 'grad_K_per_km'):
    print(f'   {_k:16s} {SWEEPS[_k]}')


### 2.3 Published results, or recompute everything

Two ways to run this notebook.

| | `published` | `recompute` |
|---|---|---|
| where the numbers come from | the marches that produced the paper | your own run |
| follows cells 2.1 and 2.2 | no — fixed at the published values | **yes** |
| time | seconds | **hours** |

`published` is the default and reproduces the paper's figures, tables and
macros exactly. Use it to check the notebook works, to read the code, or
to redraw a figure.

`recompute` re-marches everything from cells 2.1 and 2.2. It is the mode to
use after changing an input. It takes a few hours on one core — the paper
is about eighty marches — so it is built to be **resumable**: every result
is written as soon as it finishes, and re-running continues where it
stopped. Colab can cut a session off at any time; just run the cell again.

Set it below. In `recompute` mode the driver prints what it is about to do
and how long it expects to take before it starts.

In [ ]:
# =============================================================================
#  RUN_MODE: 'published'  -- the paper's numbers, seconds
#            'recompute'  -- re-march everything from cells 2.1 and 2.2, hours
# =============================================================================
NOTEBOOK_VERSION = '2.25'   # checked against the pack in section 15
RUN_MODE = 'published'

assert RUN_MODE in ('published', 'recompute'), RUN_MODE
print(f'RUN_MODE = {RUN_MODE!r}')
if RUN_MODE == 'published':
    print('  the published results will be unpacked; cells 2.1 and 2.2 do NOT\n'
          '  change them. Set RUN_MODE = "recompute" to follow your own inputs.')
else:
    print('  everything will be re-marched from cells 2.1 and 2.2. Hours, and\n'
          '  resumable: if the session dies, run the driver cell again.')

# Written to disk, not just left in the kernel: section 15 and anything
# else downstream reads it from here. A check that depends on a variable
# defined eleven sections earlier fails the moment the kernel restarts,
# which is the same class of bug as the one it is checking for.
import json as _json
_json.dump(dict(notebook_version=NOTEBOOK_VERSION, run_mode=RUN_MODE),
           open('notebook_meta.json', 'w'), indent=1)


### 2.1 What you actually changed

`Case` carries 83 fields and the cell above writes out 30. This prints
every field of `REF` that differs from the model default, so nothing you
set — or forgot to set — is invisible.

In [ ]:
import dataclasses
base = B.Case()
diff = [(f.name, getattr(base, f.name), getattr(REF, f.name))
        for f in dataclasses.fields(base)
        if getattr(base, f.name) != getattr(REF, f.name)]
print(f'{len(diff)} field(s) differ from the model defaults:\n')
print(f"  {'field':18s}{'default':>14}  ->  {'yours':<14}")
for n, d, v in diff:
    print(f'  {n:18s}{str(d):>14}  ->  {str(v):<14}')

# No later cell may correct an input. Every .with_(...) below section 2
# is a VISIBLE sensitivity, never a fix; this prints what section 2 itself
# changed between CASE and REF so an override cannot hide.
_ref_vs_case = [(f.name, getattr(CASE, f.name), getattr(REF, f.name))
                for f in dataclasses.fields(CASE)
                if getattr(CASE, f.name) != getattr(REF, f.name)]
print(f'\nREF differs from CASE in {len(_ref_vs_case)} field(s) '
      f'(should be L_ft alone, set from FIELD["L_md"]):')
for n, a, b in _ref_vs_case:
    print(f'  {n:18s}{str(a):>14}  ->  {str(b):<14}')
if [n for n, _, _ in _ref_vs_case] not in ([], ['L_ft']):
    raise ValueError('section 2 changed something other than the storage '
                     'length between CASE and REF')

print(f'\nstation spacing {FIELD["ds_station"]:.0f} m -> '
      f'{G.Ground.stations_for(FIELD["L_md"], FIELD["ds_station"])} stations '
      f'at L = {FIELD["L_md"]:.0f} m')

print('\nvalidate_case on REF:')
msgs = B.validate_case(REF, verbose=False)
if not msgs:
    print('  (clean)')
for lvl, m in msgs:
    print(f'  [{lvl}] {m[:100]}')

---
## 3. The hole, in cross-section

`n_ft` finned tubes and one insulated downcomer, sized so its velocity matches
theirs. That sizing is the whole reason this geometry costs nothing in
pumping: the downcomer carries the flow of all `n_ft` tubes through `n_ft`
times the area, so the velocity is the same everywhere and the total path is
still $2L_{well}$ — exactly what the hairpin had.

The downcomer runs **off-center**, as in the schematic. A concentric
arrangement does not close for three 1¼″ tubes in a 7″ hole; an offset one
packs comfortably. `validate_case` says so rather than leaving it to a comment.

In [ ]:
def pack(case):
    # Place the downcomer tangent to the wall, then fit the fin tubes round it.
    # SOLVED, not eyeballed: a hand-placed sketch had a fin tube overlapping
    # the downcomer by 30 mm, which is the kind of thing a figure will happily
    # show you forever. The constraint is only that no two circles intersect
    # and all sit inside the hole -- and it is checked below the plot, because
    # a schematic nobody verifies is a drawing, not a result.
    R = case.D_well / 2
    a, b = case.r_e + case.fin_L, case.r_id_outer
    d_id, d_ft = R - b, R - a                     # both tangent to the wall
    # smallest angle off the downcomer that clears it
    cosmin = (d_id**2 + d_ft**2 - (a + b)**2) / (2 * d_id * d_ft)
    th_min = np.arccos(np.clip(cosmin, -1, 1))
    span = 2 * np.pi - 2 * th_min
    n = case.n_ft
    if n == 1:
        th = np.array([np.pi])
    else:
        th = th_min + np.linspace(0, span, n)
    return (d_id, 0.0), [(d_ft * np.cos(t), d_ft * np.sin(t)) for t in th], a, b, R


(xid, yid), fts, a, b, R = pack(c)
fig, ax = plt.subplots(figsize=(6, 6))
ax.add_patch(plt.Circle((0, 0), R * 1000, fc='#2ecc71', ec='k', lw=2, zorder=1))
ax.add_patch(plt.Circle((xid * 1000, yid * 1000), b * 1000, fc='w',
                        ec='#1a5276', lw=6, zorder=2))
ax.text(xid * 1000, yid * 1000, 'ID', ha='center', va='center', fontsize=12, zorder=3)
for (x, y) in fts:
    ax.add_patch(plt.Circle((x * 1000, y * 1000), a * 1000, fc='w',
                            ec='#1a5276', lw=5, zorder=2))
    ax.text(x * 1000, y * 1000, 'FT', ha='center', va='center', fontsize=10, zorder=3)
L = R * 1100
ax.set_xlim(-L, L); ax.set_ylim(-L, L); ax.set_aspect('equal'); ax.axis('off')
ax.set_title(f'{c.n_ft} fin tubes + 1 insulated downcomer\n'
             f'{2000*R:.0f} mm hole, PCM = {100*c.V_well/c.V_borehole:.0f} % of it',
             fontsize=12)
plt.show()

# the figure is only worth looking at if it is a real packing
gaps = [np.hypot(x - xid, y - yid) - a - b for (x, y) in fts]
print(f'  smallest downcomer-to-tube gap  {1000*min(gaps):+6.1f} mm')
if len(fts) > 1:
    pp = [np.hypot(fts[i][0]-fts[j][0], fts[i][1]-fts[j][1]) - 2*a
          for i in range(len(fts)) for j in range(i+1, len(fts))]
    print(f'  smallest tube-to-tube gap       {1000*min(pp):+6.1f} mm')
print(f'  fin tube      {2000*c.r_e:.1f} mm OD, {1000*c.fin_L:.1f} mm fins, '
      f'{c.num_fins} of them')
print(f'  downcomer     {2000*c.r_id_bore:.1f} mm bore, {2000*c.r_id_outer:.0f} mm over '
      f'{1000*c.t_ins:.0f} mm insulation (k = {c.k_ins} W/m/K)')
print(f'  r_cell        {1000*c.r_cell:.1f} mm   (melt fronts meet here)')
print(f'  L_tube        {c.L_tube:.0f} m = L_well  <-- not 2x, and that is the point')

---
## 4. The formation

The thermal circuit from the PCM outward is a series chain, and the terms are
wildly unequal:

$$R_{tot} = \underbrace{R_{PCM}}_{\sim 25\%} + \underbrace{R_{casing}}_{0.1\%}
+ \underbrace{R_{cement}}_{5\%} + \underbrace{R_{ground}(t)}_{\sim 70\%}$$

The ground resistance uses the **exact** large-time Carslaw--Jaeger prefactor
$2e^{-\gamma/2} = 1.49861$, not the $2.5$ carried by v1.0 on an attribution
that could not be substantiated. The difference was a *constant* additive
offset of \SI{0.0326}{} m·K/W that never decayed with time; under the
`max()` rule it moved only the isolated-well bound and the sparse fan.

The casing is thermally invisible and the cement is a rounding error. **The
ground resistance is the model**, and unlike the others it *grows with time*
as the thermal front spreads — which is why `t_operation_yr` is a parameter
and why a quoted loss figure is meaningless without the age beside it.

### The gradient is independent, and only checked

`T_source_C` is a **lower bound** on the formation temperature — the water
cools coming up the production string, and the producing wells may be deeper
than the retrofits. An earlier draft derived the gradient from the source
temperature, which writes a bound as an equality and throws away exactly the
physics the deficit represents. Here both are specified and `validate_case`
enforces only the inequality.

In [ ]:
print(f'  R_PCM->casing   {np.log((c.D_well/2)/c.r_e)/(2*np.pi*c.n_ft*0.5*(c.k_s+c.k_l)):8.4f} m K/W')
print(f'  R_cement        {np.log(c.r_bore/c.r_casing_o)/(2*np.pi*c.k_cement):8.4f}')
print(f'  R_ground        {B.ground_resistance(c):8.4f}   <-- at t = {c.t_operation_yr:.0f} yr')
print(f'  R_total         {B.formation_resistance(c):8.4f}')

T_rock, depth = B.formation_profile(c)
T_m_lay, _ = B.melting_temperatures(c)
T_m_seg = B.layer_map(T_m_lay, c.n_segments, c.N_lay)
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot(T_m_seg - 273.15, depth, lw=2.5, label='PCM cascade')
ax.plot(T_rock - 273.15, depth, lw=2.5, ls='--', label='undisturbed rock')
ax.fill_betweenx(depth, T_rock - 273.15, T_m_seg - 273.15,
                 where=(T_m_seg > T_rock), alpha=.18, color='#c0392b', label='loses')
ax.fill_betweenx(depth, T_rock - 273.15, T_m_seg - 273.15,
                 where=(T_m_seg <= T_rock), alpha=.3, color='#2471a3', label='GAINS')
ax.invert_yaxis(); ax.set_xlabel('temperature, C'); ax.set_ylabel('depth, m')
ax.set_title(f'cascade hot at the BOTTOM, aligned with the gradient\n'
             f'{c.grad_K_per_km:.0f} K/km from {c.T_surface_C:.0f} C', fontsize=11)
ax.legend(); ax.grid(alpha=.3); plt.show()

### Why hot-at-the-bottom matters

The rock is coldest at the surface and hottest at depth. THUMS ran the cascade
**hot at the top**, so the hottest PCM faced the coldest rock and the coldest
PCM faced rock that — at 8000 ft and a Wilmington gradient — is *hotter than
it is*. Those deep layers never freeze, and that capacity is dead.

The net loss is identical either way (the mean of $T_{PCM}$ is the same), but
the **distribution** is not:

| at 8000 ft, 56.5 K/km | ΔT at top | ΔT at bottom | cannot freeze |
|---|---|---|---|
| hot-top (THUMS) | +132 K | −55 K | **29 %** |
| hot-bottom (BOREAS) | +83 K | −6 K | **7 %** |

This is not a free choice: the downcomer *makes* it hot-at-bottom, because
charge water reaches the bottom before it has given up any heat.

---
## 5. The field — where the wells actually are

Until v1.2 the field was an abstraction: a well count, a wellhead spacing
and a cone half-angle, from which a perimeter and a shielding factor
followed. That description cannot represent **where the wells are**, and at
1.65 m spacing the difference is not a refinement — it is most of the
answer.

So the geometry is now read from the **CalGEM WellSTAR export**: 363 Grissom
slots, deduplicated and stripped of strays. Three facts about them, all
from the data rather than from a description:

* the measured median nearest-neighbor spacing is **1.65 m**, 10 % tighter
  than the nominal 6 ft. The idealized rungs below are built at the measured
  value, so that the step from them to the real slots isolates the *layout*;
* Grissom is a **thick closed ring**, not a filled disc and not a straight
  double row: no slot lies inside $0.3\,R_\mathrm{max}$ and 72 % lie beyond
  $0.7\,R_\mathrm{max}$, against 0.09 and 0.51 for a disc;
* the wells are **directionally drilled**, so measured depth is not true
  vertical depth. At $L = 1250$ m MD the S-curve reaches only about 890 m
  TVD, where the rock is 58 °C rather than the 73 °C a vertical well would
  see. The store is colder *and* better shielded than the vertical
  idealisation says.

### The ground model

Every well in the field is a line source. At a given MD, well $i$ sees its
own wall rise plus the rise its neighbors impose,

$$\Delta T_i = R_\mathrm{near}\,q_i + \sum_k G_{ik}\,q_k, \qquad
G_{ik} = \frac{1}{4\pi k}\,E_1\!\left(\frac{d_{ik}^{2}}{4\alpha t}\right)$$

with $d_{ik}$ the shortest distance from the target to well $k$'s
trajectory. Two ways to close this:

| model | assumption | what it gets wrong |
|---|---|---|
| `flux` | every well emits the **same** $q$ | edge wells are in fact hotter; it averages a temperature that is not uniform |
| `temp` | every well sits at the **same** $\Delta T$ | nothing; the strengths are solved for and edge wells come out stronger |

`temp` is the one used throughout, because the PCM in every well is at the
same melting temperature — equal temperature is the physical boundary
condition, not equal flux. Solving $(R_\mathrm{near}\mathbf{I} +
\mathbf{G})\,\mathbf{q} = \mathbf{1}$ gives
$R_{g,\mathrm{eff}} = 1/\overline{q} - R_\mathrm{near}$.

**The specification degenerates correctly.** With a single well, $G_{ii}$
must reduce to the isolated-well $R_g$ already in the model. It does, to
$8\times10^{-7}$ — which is the check worth having, because it ties the new
ground model to the old one rather than to a reference value.

In [ ]:
# the degeneracy check: one well, no neighbors
_c = REF
_g = G.Ground(k=_c.k_rock, rhocp=_c.rho_cp_rock, t_yr=_c.t_operation_yr,
              rb=_c.r_bore)
print(f'  G_ii (new superposition kernel)  {_g.kern(_c.r_bore):.9f} m K/W')
print(f'  R_g  (shipped isolated-well)     {B.ground_resistance(_c):.9f} m K/W')
print(f'  difference                       '
      f'{abs(_g.kern(_c.r_bore) - B.ground_resistance(_c)):.2e}')

### 5.2 One station spacing, fixed, for every run

Stations are the measured depths at which the interaction matrix is
assembled. The input is a **spacing**, `FIELD['ds_station'] = 50 m`, and
the count follows as $\max(3, \mathrm{round}(L/\Delta s))$ — one value for
the ladder, the $L$ sweep and every sensitivity. Two things follow:

* the stations sit at the same measured depths whatever $L$ is, so
  $R_{g,\mathrm{eff}}$ at a given MD can be *compared* across lengths —
  that is check V10, and it agrees to 0.00 % between 200 m and 2000 m;
* every run of a sweep is discretized identically, so a difference between
  two runs is the thing being varied and not the grid.

### What the matrix is, and what it is not

Each row of the system says *the temperature rise at my location, on well
$i$ at station $s$, due to well $k$* — and the nearest point on $k$ seen
from there sits at some other station $s'$. The reverse entry is about a
**different target point** at a **different distance**. So

$$G_{(i,s),(k,s')} \ne G_{(k,s'),(i,s)}$$

and $\mathbf{G}$ has no reason to be symmetric. Measured here:
$\lVert \mathbf{G}-\mathbf{G}^\mathsf{T}\rVert / \lVert\mathbf{G}\rVert$
is 0.25 at 50 m spacing and 0.90 at 25 m.

**An earlier version solved $(\mathbf{G}+\mathbf{G}^\mathsf{T})/2$
instead, and that was wrong.** Averaging mixes rows that are not about the
same place. It also produced negative well strengths — 130 at 25 m
spacing, 569 at 8 m — and those were then read as evidence that refining
the grid broke the line-source assumption. They were not. Solved as
assembled, **there are no negative strengths at any spacing tested**, and
the claim that "the coarse grid is the well-posed one" is withdrawn.

Two consequences for the solver, both in `Ground.profiles`:

* the system is solved **as assembled**, $\mathbf{A} = \mathbf{G} +
  (R_\mathrm{near}+G_{ii})\mathbf{I}$;
* above `_DIRECT_MAX` it uses **BiCGSTAB** with a Jacobi preconditioner,
  not CG. CG needs a symmetric positive-definite matrix and this one is
  neither symmetric nor diagonally dominant — the diagonal is 0.567 m K/W
  against a largest off-diagonal row sum of 3.73, because near the surface
  all 363 wells are inside the 80 m cutoff.

### The grid study, redone

| $\Delta s$ | stations | negatives | off-station weight | loss | $e'$ |
|---|---|---|---|---|---|
| 104 m | 12 | 0 | 25.9 % | 51.75 % | 0.2177 |
| **50 m** | **25** | **0** | **5.2 %** | **52.42 %** | **0.2151** |
| 25 m | 50 | 0 | 25.8 % | 52.42 % | 0.2150 |
| 12.5 m | 100 | 0 | 27.6 % | 52.50 % | 0.2147 |
| 6.25 m | 200 | 0 | 32.7 % | 52.53 % | 0.2146 |

$\Delta s = 50$ m is adopted because it **converges**: the loss moves
0.001 points and $e'$ 0.05 % to the next finer grid, and 0.12 points across
the whole eightfold refinement. 104 m is too coarse (0.67 points). At
$L = 400$ m the same refinement leaves about 0.4 points of residual drift,
so the honest statement is that the discretization is worth **up to a few
tenths of a point of loss**, not that it is exact.

### One structural result that makes most of this moot

**Vertical fields are exactly block diagonal in station.** For parallel
vertical wells the nearest point on a neighbor is at the same depth, so
every coupling stays within its own station — the off-station kernel weight
is 0.00 % — and every block is the *same* matrix, because horizontal
distances do not change with depth. $R_{g,\mathrm{eff}}$ is therefore
independent of both the station count and $L$: R1 gives 15.783 m K/W and R3
gives 3.230, at every length and every spacing tested. $\mathbf{G}$ is also
symmetric there, which is precisely why the symmetrization went unnoticed
for so long.

The solver exploits this directly, which is what makes the disc rung
computable at all — at 1.65 m spacing all 363 wells sit inside 18 m, so
every target has about 90 000 candidate samples inside the cutoff.

The S-curve is the only case where the count does anything: there the
wells converge and diverge, a neighbor's nearest point lands at most one
station away, and 5.2 % of the kernel weight sits off the diagonal block.

In [ ]:
# the structure claim, measured rather than asserted
import numpy as np
_c, _P = REF, SLOTS
_kopm, _ = G.staggered_kop(_P)
_gr = G.Ground(k=_c.k_rock, rhocp=_c.rho_cp_rock, t_yr=_c.t_operation_yr,
               rb=_c.r_bore)
_Rn = float(np.atleast_1d(B.formation_resistance(_c, np.array([0.0])))[0]
            - B.ground_resistance(_c))
_nst = G.Ground.stations_for(FIELD['L_md'], FIELD['ds_station'])
print(f"ds = {FIELD['ds_station']:.0f} m -> {_nst} stations "
      f"at L = {FIELD['L_md']:.0f} m MD\n")
for _t, _k in (('vertical', None), ('s_curve', _kopm)):
    _md, _X = G.trajectories(_P, FIELD['L_md'], kind=_t, kop=_k,
                             theta_max=FIELD['theta_max'])
    _pr = _gr.profiles(_md, _X, _Rn, nst=_nst)
    print(f"  {_t:9s} off-station kernel weight "
          f"{100*_gr.off_station_weight:5.2f} %   "
          f"negative strengths {_gr.n_negative:5d}   "
          f"R_g,eff mean {_pr['temp'].mean():7.3f} m K/W")
print('\n0.00 % is the block-diagonal case: the count cannot matter there.')

In [ ]:
import numpy as np, matplotlib.pyplot as plt
P = SLOTS
kop_m, is_out = G.staggered_kop(P)

fig, ax = plt.subplots(1, 3, figsize=(13.2, 4.0))
for grp, col, lab in ((is_out, '#b03028', 'outer row'),
                      (~is_out, '#23356b', 'inner row')):
    ax[0].plot(P[grp, 0], P[grp, 1], 'o', ms=2.2, color=col, label=lab)
ax[0].set_aspect('equal'); ax[0].legend(frameon=False, markerscale=3)
ax[0].set_xlabel('east / m'); ax[0].set_ylabel('north / m')
ax[0].set_title(f'(a) {len(P)} Grissom slots')

_md2, _X2 = G.trajectories(P, 2000.0, kind='s_curve', kop=kop_m,
                           theta_max=FIELD['theta_max'])
_rad = np.hypot(_X2[:, :, 0], _X2[:, :, 1])
for i in range(0, len(P), 7):
    ax[1].plot(_rad[i], _X2[i, :, 2], lw=0.4, color='#6a6a6a', alpha=.7)
ax[1].axhline(FINFO['tvd_bottom'], color='#b03028', lw=1.2, ls='--')
ax[1].annotate(f"L = {FIELD['L_md']:.0f} m MD", (0, FINFO['tvd_bottom']),
               color='#b03028', fontsize=9, va='bottom')
ax[1].invert_yaxis()
ax[1].set_xlabel('radius from centroid / m'); ax[1].set_ylabel('TVD / m')
ax[1].set_title('(b) S-curve trajectories')

from scipy.spatial import cKDTree
for kind, col in (('vertical', '#23356b'), ('s_curve', '#b03028')):
    m2, X2 = G.trajectories(P, 2000.0, kind=kind, kop=kop_m,
                            theta_max=FIELD['theta_max'])
    nn = [np.median(cKDTree(X2[:, j, :2]).query(X2[:, j, :2], k=2)[0][:, 1])
          for j in range(0, len(m2), 8)]
    ax[2].plot(m2[::8], nn, color=col, lw=1.8, label=kind)
ax[2].set_yscale('log'); ax[2].legend(frameon=False)
ax[2].set_xlabel('MD / m'); ax[2].set_ylabel('median spacing / m')
ax[2].set_title('(c) nearest-neighbor spacing')
plt.tight_layout(); plt.show()

print(f"measured median spacing at the wellhead: "
      f"{np.median(cKDTree(P).query(P, k=2)[0][:, 1]):.3f} m "
      f"(nominal 6 ft = {G.B0_NOMINAL:.2f} m)")

In [ ]:
# R_g,eff along the well: vertical against the real trajectory
_MDv, _RGv, _TVDv, _Iv = G.cached_reference_field(
    B, REF, CSV, trajectory='vertical', L_md=FIELD['L_md'], verbose=False)

fig, a = plt.subplots(figsize=(5.6, 4.4))
a.plot(_RGv, _TVDv, color='#23356b', lw=2.0, label='vertical')
a.plot(RG, TVD, color='#b03028', lw=2.0, label=f"S-curve, "
       rf"$\theta_\mathrm{{max}}$ = {FIELD['theta_max']:.0f}$^\circ$")
a.axvline(B.ground_resistance(REF), color='#6a6a6a', lw=1.2, ls='--')
a.annotate('isolated well', (B.ground_resistance(REF), 0), color='#6a6a6a',
           fontsize=9, rotation=90, ha='right', va='top')
a.invert_yaxis(); a.set_xscale('log')
a.set_xlabel(r'$R_{g,\mathrm{eff}}$ / m K W$^{-1}$'); a.set_ylabel('TVD / m')
a.legend(frameon=False)
a.set_title('Shielding along the well, equal-temperature superposition',
            fontsize=10)
plt.tight_layout(); plt.show()

print(f"vertical : mean {_Iv['Rg_mean']:6.2f} m K/W, "
      f"TVD at the shoe {_Iv['tvd_bottom']:.0f} m")
print(f"S-curve  : mean {FINFO['Rg_mean']:6.2f} m K/W, "
      f"TVD at the shoe {FINFO['tvd_bottom']:.0f} m")
print(f"isolated : {B.ground_resistance(REF):6.2f} m K/W")
print(f"the field is {_Iv['Rg_mean']/B.ground_resistance(REF):.0f}x more "
      f"resistive than a lone well when the wells are vertical, and only "
      f"{FINFO['Rg_mean']/B.ground_resistance(REF):.0f}x once they deviate: "
      f"the trajectory gives back most of the shielding.")

---
## 6. Run it — the field ladder

One run does not tell you what a field is worth; the *difference* between
two runs does. So the result is reported as a **ladder**, in which each rung
changes exactly one thing and nothing else moves:

| rung | geometry | what the step from the rung above isolates |
|---|---|---|
| **R0v** | isolated well, vertical | the bound. No field at all |
| **R0i** | isolated well, S-curve | the **trajectory alone**: same MD, colder rock, no neighbors |
| **R1** | disc, vertical, 1.65 m | the **field**: 363 neighbors in the most favorable packing |
| **R2** | straight double row, vertical | the **layout**: same count and spacing, strung out instead of compact |
| **R3** | Grissom slots, vertical | the **real positions**: a ring, not an idealized row |
| **R4** | Grissom slots, S-curve | the **real trajectories**: the **reference case** |

Two rules make the comparison honest, and both were violated in an earlier
draft:

1. **Every rung is at the measured 1.65 m.** Building the idealized rungs at
   the nominal 6 ft mixes a layout change with a spacing change, and that
   understated the layout penalty by 40 % — R2 → R3 is $+4.2$ points, not
   $+3.0$.
2. **Every rung carries the same casing**, the THUMS upper-zone 8-5/8 in
   string of section 2. Its 201.2 mm ID holds 46 % more PCM per tube than the
   legacy 177.8 mm, worth **+36 % on $e'$** — a larger effect than the choice
   of ground model, and a reason not to compare across casings by accident.

The 111 runs behind this section take about two hours, so they are read
from a cached JSON. Set `RECOMPUTE = True` to rebuild any of them locally;
the cache is keyed by the full specification, so nothing is recomputed twice.

In [ ]:
import os, json, urllib.request
RECOMPUTE = False            # True -> run the ladder here instead of reading it

PACK = 'pack_runs.json'
if not os.path.exists(PACK):
    urllib.request.urlretrieve(RAW + 'field_ladder/' + PACK, PACK)
RUNS = [r for r in json.load(open(PACK)).values() if 'error' not in r]
print(f'{len(RUNS)} runs in the cache')

def pick(step, L=None, ground='superposition', **kw):
    c = [r for r in RUNS if r['step'] == step and r['ground'] == ground
         and all(abs(r[k] - v) < 1e-9 if isinstance(v, float) else r[k] == v
                 for k, v in kw.items())]
    if L is not None:
        c = [r for r in c if abs(r['L_md'] - L) < 1e-6]
    return sorted(c, key=lambda r: r['L_md'])

In [ ]:
import pandas as pd
LADDER = ['R0v', 'R0i', 'R1', 'R2', 'R3', 'R4']
WHAT = {'R0v': 'isolated, vertical', 'R0i': 'isolated, S-curve',
        'R1': 'disc, vertical', 'R2': 'double row, vertical',
        'R3': 'Grissom slots, vertical (R3)', 'R4': 'Grissom slots, S-curve'}

tab, prev = {}, None
for s in LADDER:
    r = pick(s, FIELD['L_md'])
    if not r:
        continue
    r = r[0]
    tab[s] = {'geometry': WHAT[s], 'loss %': 100 * r['loss'],
              'step': '' if prev is None else f"{100*(r['loss']-prev):+.1f}",
              'eta_storage': r['eta_storage'], 'eta_RTE': r['RTE'],
              "e' [kWh/m]": r['e_prime'],
              'T_2d [C]': r['T2d_C'], 'TVD [m]': r['tvd_bottom'],
              'delivers': r['delivers']}
    prev = r['loss']
display(pd.DataFrame(tab).T)
print(f"field at the reference case (R4): "
      f"{pick('R4', FIELD['L_md'])[0]['field_MWh_per_cycle']:.1f} MWh "
      f"per cycle from {pick('R4', FIELD['L_md'])[0]['N_field']:.0f} wells")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12.0, 4.0))
steps = [s for s in LADDER if pick(s, FIELD['L_md'])]
loss = [100 * pick(s, FIELD['L_md'])[0]['loss'] for s in steps]
ep = [pick(s, FIELD['L_md'])[0]['e_prime'] for s in steps]
x = np.arange(len(steps))
lab = [WHAT[s].replace(', ', ',\n') for s in steps]
ax[0].bar(x, loss, color=['#b03028' if v > 50 else '#23356b' for v in loss])
ax[1].bar(x, ep, color='#2f6f4f')
for a, v, f in ((ax[0], loss, '{:.1f}'), (ax[1], ep, '{:.3f}')):
    a.set_xticks(x); a.set_xticklabels(lab, fontsize=8)
    for i, vi in enumerate(v):
        a.text(i, vi + 0.02 * max(v), f.format(vi), ha='center', fontsize=8)
ax[0].set_ylabel('formation loss / %')
ax[1].set_ylabel(r"$e'$ / kWh per m of well")
fig.suptitle(f"The field ladder at L = {FIELD['L_md']:.0f} m MD, "
             f"Grissom, N = 363", y=1.0, fontsize=11)
plt.tight_layout(); plt.show()

### Reading the ladder

**The field is worth an order of magnitude, and the trajectory takes a
third of it back.** R0 → R1 is the big step: 363 neighbors cut the loss
from 64 % to 3 %, because each well only has to push heat out through its
share of the cluster boundary. R3 → R4 then puts 39 points back on,
entirely through the trajectory: below the kickoff the wells separate, the
shielding weakens, and the store spends most of its length in rock it does
not share with anybody.

**Layout matters more than compactness suggests.** R1 → R2 is $+7$ points
for the same 363 wells at the same spacing — stringing them into a line
roughly triples the perimeter per well. R2 → R3 is a further $+3$: the real
ring is *better* than a straight double row of the same length, because a
closed loop has no end-caps, but worse than the idealized row's uniformity
allows for.

**Nothing here fails to deliver.** Earlier versions reported short isolated
wells that absorbed heat on discharge, and v1.3 and earlier hid that behind
an `abs()`. With the signed fix and the THUMS casing the regime is gone: an
isolated 100 m well goes from absorbing (81.9 °C, loss 115 %) to delivering
(99.2 °C, loss 86 %). Both changes were needed — the sign fix reports the
legacy case correctly, the casing removes the case.

---
## 7. The gates, watched to fail

A check that has never been seen to fail is not evidence. Both of these were
corrupted deliberately, and the second exists *because* the first was silent.

**The closure cannot see a flipped loss sign.** Negating `q_loss` leaves the
residual at 5e-15 — `q_prime` and `Q_loss_cum` both carry the error and it
cancels algebraically. The closure reads only its own inputs.

So the loss term carries a **Clausius gate**: accumulate
$q_{loss}\,(T_{PCM}-T_{rock})$, which is $K(\Delta T)^2$ under the correct
convention and $-K(\Delta T)^2$ under a flipped one.

In [ ]:
import dataclasses, gc
src = open('boreas.py').read()
r_ok = B.simulate_css_corrected(REF)
print(f'{"(uncorrupted)":<28} closure {r_ok["charge"]["closure"]:.2e}   '
      f'clausius {r_ok["charge"]["loss_clausius"]:+.3e}')
for lab, bad in [
        ('drop q_loss from q_prime',
         src.replace('q_prime[i] = q_tot + q_loss', 'q_prime[i] = q_tot')),
        ('flip the loss sign',
         src.replace('q_loss = K_l * (T_eff - T_rock_arr[i])',
                     'q_loss = K_l * (T_rock_arr[i] - T_eff)'))]:
    assert bad != src, lab
    ns = {}
    exec(compile(bad, 'corrupt', 'exec'), ns)
    rb = ns['simulate_css_corrected'](ns['Case'](**dataclasses.asdict(REF)))
    cl, cz = rb['charge']['closure'], rb['charge']['loss_clausius']
    print(f'{lab:<28} closure {cl:.2e} {"FIRES" if cl > 1e-6 else "silent":>6}   '
          f'clausius {cz:+.3e} {"FIRES" if cz < 0 else "silent"}')
    # each corrupted copy is a second full module plus its recorded march.
    # Keeping both alive alongside the real one is about 1 GB, enough to be
    # killed on a small runtime, and nothing below needs them.
    del ns, rb
    gc.collect()

### The adiabatic limit

Driving `k_rock` to zero must recover THUMS hypothesis H6 exactly: no loss,
$\eta_{storage} = 1$.

This gate found a real bug on its first run. `ground_resistance` returned
`0.0` when the line source fell outside its validity, meaning *"the front has
not cleared the hole"*. But zero **resistance** is infinite **conductance** —
it claims the rock is a perfect heat sink. The test returned
$\eta_{storage}=0.20$ and a loss **larger** than the physical case.

In [ ]:
r0 = B.simulate_css_corrected(REF.with_(k_rock=1e-9))
print(f'  k_rock -> 0 : Q_loss {r0["Q_loss_kJ"]:.3e} kJ, '
      f'eta_storage {r0["eta_storage"]:.9f}')
print(f'  {"PASS" if r0["Q_loss_kJ"] == 0.0 else "FAIL"}'
      f'   (residual is the CSS drift tolerance, not a loss)')

---
## 8. How deep should the storage go?

With a real trajectory the question changes shape. A vertical well gains
from every extra meter: the rock is hotter and the shielding does not
weaken, so $e'$ rises monotonically and there is no optimum — depth is a
pure capital lever. **A deviated well does not.** Past the kickoff each
extra meter of measured depth buys less true vertical depth and sits in
rock the neighbors no longer share, so the marginal return falls away.

The cutoff is defined on the **marginal** round-trip efficiency, not the
average: $\Delta E_\mathrm{out}/\Delta E_\mathrm{in}$ across each interval
in $L$. The storage stops paying where that marginal value first drops
below half its value **over the first 100 m**.

Which interval supplies that reference is not a detail. Because $E(0) = 0$,
the marginal over $0 \to 100$ m *is* $\eta_\mathrm{RTE}(100)$, and it is
larger than the marginal over $100 \to 200$ m. Using the $100 \to 200$
interval instead lowers the threshold from 0.131 to 0.122 and moves the
cutoff a full step, from 400--500 m to 500--600 m. Both are defensible
definitions; they are not the same definition, and the paper reports the
first.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13.8, 4.0))
COL = dict(zip(LADDER, ('#6a6a6a', '#9c27a0', '#23356b', '#0a9396',
                        '#2f6f4f', '#b03028')))
cut = {}
for s in LADDER:
    rr = pick(s)
    if len(rr) < 2:
        continue
    L = np.array([r['L_md'] for r in rr])
    ax[0].plot(L, [r['e_prime'] for r in rr], 'o-', ms=3.5, lw=1.5,
               color=COL[s], label=f'{s}: {WHAT[s]}')
    ax[1].plot(L, [r['RTE'] for r in rr], 'o-', ms=3.5, lw=1.5, color=COL[s])
    Eo = np.array([r['E_out_kWh'] for r in rr])
    Ei = np.array([r['E_in_kWh'] for r in rr])
    # Marginals on the interior intervals, plus the first one, 0 -> L[0].
    # E(0) = 0, so the marginal over that first interval is just RTE(L[0]) --
    # and it is the reference the criterion is defined against. Taking
    # marg[0] (the 100 -> 200 m interval) instead is a different and smaller
    # number, and it moves the cutoff a full step, to 500-600 m.
    marg = np.diff(Eo) / np.where(np.diff(Ei) == 0, np.nan, np.diff(Ei))
    Lm = 0.5 * (L[1:] + L[:-1])
    ax[2].plot(Lm, marg, 'o-', ms=3.5, lw=1.5, color=COL[s])
    first = Eo[0] / Ei[0] if abs(L[0] - 100.0) < 1e-6 else marg[0]
    ax[2].plot([0.5 * L[0]], [first], 'o', ms=3.5, color=COL[s])
    thr = 0.5 * first
    below = np.where(marg < thr)[0]
    if len(below):
        cut[s] = (L[below[0]], L[below[0] + 1], first, thr)
        ax[2].plot(Lm[below[0]], marg[below[0]], '*', ms=15, color=COL[s])

for a, lab in zip(ax, (r"$e'$ / kWh m$^{-1}$", r'$\eta_\mathrm{RTE}$',
                       r'marginal $\Delta E_\mathrm{out}/\Delta E_\mathrm{in}$')):
    a.set_xlabel('storage length $L$ / m MD'); a.set_ylabel(lab)
    a.axvspan(FIELD['L_md'], 2000.0, color='#cccccc', alpha=.35, lw=0)
ax[0].legend(frameon=False, fontsize=7.5, ncol=1)
ax[2].set_ylim(bottom=0)
ax[1].annotate('beyond the provisional shoe',
               (FIELD['L_md'], ax[1].get_ylim()[1]), fontsize=8, color='#6a6a6a',
               rotation=90, ha='right', va='top')
plt.tight_layout(); plt.show()

print('cutoff -- marginal RTE first below half its value over the first')
print('100 m, which is RTE(100) because E(0) = 0:')
for s in LADDER:
    if s in cut:
        a_, b_, m0, thr = cut[s]
        print(f'  {s:4s} {WHAT[s]:26s} {a_:5.0f}-{b_:5.0f} m MD   '
              f'(first 100 m {m0:.3f}, threshold {thr:.3f})')
    elif len(pick(s)) > 1:
        print(f'  {s:4s} {WHAT[s]:26s} none within the sweep')

**Vertical wells have no cutoff; deviated ones stop paying between 400 and
600 m MD**, and the reference case's own crossing falls in 400--500. That is
not a statement about heat transfer in a tube — it is a statement about the
kickoff. The S-curve reaches about 890 m TVD at 1250 m
MD, and most of the shortfall is accumulated in the build section, where
the wells are simultaneously diverging from each other and falling behind
in depth.

### How sharp is that number?

The marginal curve is flat where it crosses the threshold — 0.123 against
0.131 over 400--500 m — so the crossing is resolved to one interval of the
$L$ grid and no finer. Within that limit it is now **stable**: the
45$^\circ$ and 70$^\circ$ variants put it in the same 400--500 m interval
as the reference case.

They did not, in v2.0. There the station count was chosen per run, as the
first count that solved with no negative strengths, so two runs of the same
sweep were discretized differently — at $L = 500$ m the reference case
landed on 25 stations and the 45$^\circ$ variant on 6 — and that moved $e'$
by 2.8 % against the 0.4 % the inclination was worth. The cutoff came out
one interval later for both inclination variants, in an order that was not
monotonic in angle. **That was the grid, not physics**, and section 5.2
describes the fixed spacing that removed it.

Two things are solid underneath. Below the kickoff the three inclinations
are now *identical to four decimals*, as they must be — there is nothing to
deviate yet, and in v2.0 they were not. And a deep kickoff removes the
cutoff altogether: `R4_kop300` has none within the sweep.

The grey band marks measured depths beyond the provisional casing shoe.
**The shoe depth is the one genuinely blocking unknown in the geometry**:
the top of cement moves the loss by 0.09 points across its whole plausible
range, the mud conductivity by 0.17, the casing grade by 2.2 % of $e'$ —
all sub-2 %, because the near-well resistance they move is itself only
about 7 % of the total. The shoe is different in kind, because it is a
constraint on $L$ rather than a resistance.

---
## 8b. What the answer is sensitive to

Four one-at-a-time variations on the design rung R4, each re-solving the
superposition from scratch rather than rescaling anything.

In [ ]:
import pandas as pd
SENS = [('R4', 'reference case: staggered KOP, 55 deg, 10 yr'),
        ('R4_kop300', 'every well kicks off at 300 ft'),
        ('R4_th45', 'theta_max = 45 deg'),
        ('R4_th70', 'theta_max = 70 deg'),
        ('R4_top1', 'field age 1 yr'),
        ('R4_top30', 'field age 30 yr')]
base = pick('R4', FIELD['L_md'])[0]
rows, missing = {}, []
for step, what in SENS:
    r = pick(step, FIELD['L_md'])
    if not r:
        missing.append(f'{step} ({what})')
        continue
    r = r[0]
    rows[step] = {'what': what, 'loss %': 100 * r['loss'],
                  'd(loss) [pts]': 100 * (r['loss'] - base['loss']),
                  "e' [kWh/m]": r['e_prime'],
                  "d(e') %": 100 * (r['e_prime'] / base['e_prime'] - 1),
                  'eta_RTE': r['RTE'], 'TVD [m]': r['tvd_bottom']}
display(pd.DataFrame(rows).T.round(3))
if missing:
    print('not in this cache yet, so not shown rather than shown as blank:')
    for m in missing:
        print('  ' + m)

**The kickoff is the largest single lever, and the field age is the widest
spread.** Moving every well to a 300 ft kickoff takes the loss from 52 % to
35 % — more than either inclination change — which is the mechanism behind
the cutoff: a deep kickoff keeps the store inside the tightly spaced
section. Inclination at 45° and 70° gives 50 % and 55 %, a narrow band
either side.

Field age runs 72 % at one year to 39 % at thirty, the widest of anything
tested. That is not a design variable, and it cuts the right way: the
thermal field matures around the store and the loss falls with it. It also
means **any loss quoted without an age is meaningless** — everything here
is at ten years.

### Two ground models, one geometry

The `flux` rows in the cache are the equal-strength alternative at the same
geometry. They are kept because the difference between them is the size of
the modeling choice, and that belongs in the limitations rather than in a
footnote.

In [ ]:
rows = {}
for s in ('R1', 'R2', 'R3', 'R4'):
    t = pick(s, FIELD['L_md'], ground='superposition')
    f = pick(s, FIELD['L_md'], ground='flux')
    if t and f:
        rows[s] = {'geometry': WHAT[s],
                   'temp: loss %': 100 * t[0]['loss'],
                   'flux: loss %': 100 * f[0]['loss'],
                   'difference [pts]': 100 * (f[0]['loss'] - t[0]['loss']),
                   "temp: e'": t[0]['e_prime'], "flux: e'": f[0]['e_prime']}
display(pd.DataFrame(rows).T.round(3))
print('equal flux is the optimistic reading: it averages a wall temperature')
print('that is not uniform, and the edge wells it under-weights are the ones')
print('doing the losing.')

# V1/V7: the two legacy-casing rows exist only to regress against the
# prototype, which was computed at the 177.8 mm string. They are NOT a
# result -- they are the check that this implementation reproduces the one
# it replaced once the casing is set back.
print()
for s_, want in (('R3_legacy', 17.4), ('R4_legacy', 70.7)):
    r = pick(s_, FIELD['L_md'])
    if r:
        print(f'{s_:10s} {100*r[0]["loss"]:5.1f} % against the prototype\'s '
              f'{want:4.1f} %')

---
## 9. Change something

Every parameter is a field of `Case`. The ones BOREAS adds:

| | |
|---|---|
| `n_ft` | fin tubes per hole (the downcomer is always one more) |
| `t_ins`, `k_ins` | downcomer insulation |
| `T_surface_C`, `grad_K_per_km` | the geothermal gradient |
| `k_rock`, `rho_cp_rock`, `k_cement` | formation and annulus |
| `t_operation_yr` | how old the field is when you evaluate it |
| `D_in`, `r_casing_o`, `r_bore` | the casing string; `D_in` is what bounds the PCM |
| `line_ch_dT`, `line_dc_dT`, and the two `_dp` | the line-loss placeholders |

### The water loop is numbered 1 = field outlet, 4 = field inlet

Figure 1 numbers both half-cycles the same way: **1 is the outlet of the
borehole field and 4 is its inlet.**

| half-cycle | 1 | 2 | 3 | 4 |
|---|---|---|---|---|
| charge | field outlet | condenser inlet | condenser outlet | **field inlet** |
| discharge | **field outlet** | evaporator inlet | evaporator outlet | field inlet |

The coils between 3c and 4c and between 1d and 2d stand for heat and
pumping losses in the lines between plant and field. They are not
computed yet, so $T_\mathrm{4c} = T_\mathrm{3c}$ and
$T_\mathrm{2d} = T_\mathrm{1d}$. Pump heating is neglected, so
$T_\mathrm{1c} = T_\mathrm{2c}$ and $T_\mathrm{4d} = T_\mathrm{3d}$.

The discharge inlet approach was `DT_M_1D` until round 25 and is now
**`DT_M_4D`**; the old name still works and warns.

**Nothing below section 2 may correct an input.** Every `.with_(...)` in
the sections above is a *sensitivity* --- a visible, commented variation
on the reference case --- and never a fix for something set in section 2.
The audit cell enforces the one case that used to break this rule: the
storage length, which section 2 set as `L_ft = 5000` and then silently
replaced with 1250 m. There is now one length input, `FIELD['L_md']`, and
setting `L_ft` as well raises.

The field is **not** a `Case` field any more. `n_wells_field`,
`B_surface_m`, `fan_angle_deg` and `packing` still exist for backward
compatibility with v1.2 results, but nothing in this notebook uses them:
the arrangement lives in `FIELD` and reaches the model through
`boreas_ground.field_ground`, which patches `formation_profile` and
`formation_resistance` for the duration of a run. To change the field,
edit `FIELD` in section 2 and re-run — not a `Case` field.

Always run `validate_case` first — it is fast, and it catches the gradient
contradicting the source temperature, a cascade that has been flipped
hot-to-top, a geometry that will not pack, and a well whose deep layers cannot
freeze.

---
## 10. The surface cycles: $T$--$s$ diagrams and state points

The borehole model and the plant models meet at the water loop, and the only
quantity the store sets is $T_\mathrm{2d}$, the temperature of the water
arriving at the ORC evaporator. Everything else in the two cycles below is
frozen from the hairpin study. So these diagrams are *context*, not a result:
they say what machine the store is feeding, and they make visible how much of
the store's temperature the ORC actually uses.

Three things in these figures are deliberate and worth stating, because each
is a place where a $T$--$s$ plot is usually drawn wrongly:

1. **Isobars are sampled in enthalpy, not temperature.** Inside the dome $T$
   and $p$ are not independent, so a state cannot be recovered from $(T, p)$
   there and a temperature-sampled isobar breaks silently at the saturation
   line --- which is exactly where every heat-exchanger leg lives. Enthalpy is
   monotonic along an isobar through subcooled liquid, dome and superheat
   alike.
2. **Throttles are drawn as isenthalps**, sampled in pressure, not as the
   straight lines a sketch usually shows. The $7\mathrm{h} \to 8\mathrm{h}$
   expansion is strongly curved in $T$--$s$ and the curvature *is* the
   irreversibility.
3. **The connectivity is read from the model's enthalpies, not from the
   numbering.** The ORC is a *reheat* cycle ($3\mathrm{e} \to 5\mathrm{e}$,
   reheat at $p_\mathrm{int}$ to $6\mathrm{e}$, then $\to 4\mathrm{e}$); the
   HTHP is a two-stage *flash-economizer* cycle. Neither is apparent from the
   state list alone.

### A note on fonts

Times New Roman is proprietary and is **not** installed on Colab. Asking
`matplotlib` for it by name there does not fail --- it silently falls back to
DejaVu Sans and the figure comes out in the wrong face. The style helper asks
for Times first and then for its metric-compatible clones (Nimbus Roman,
Liberation Serif), which are present on virtually every Linux image and are
indistinguishable from Times at figure scale. `active_serif()` reports which
one actually resolved, so the fallback is visible rather than silent. On a Mac
with Times installed, you get the real thing.

In [ ]:
%%writefile boreas_plots.py
"""Publication-quality cycle diagrams and the reference state-point table.

FONTS. Times New Roman is proprietary and is not installed on Colab, on most
Linux images, or in this sandbox. Asking matplotlib for it by name therefore
produces a silent fallback to DejaVu Sans -- the request appears to succeed
and the figure comes out in the wrong face. The chain below asks for Times
first and then for its metric-compatible clones, Nimbus Roman and Liberation
Serif, which ARE present almost everywhere and are visually indistinguishable
from Times at figure scale. `active_serif()` reports which one resolved, so
the fallback is visible rather than silent.

ISOBARS ARE SAMPLED IN ENTHALPY, NOT TEMPERATURE. Inside the dome T and p are
not independent, so CoolProp cannot return a state from (T, p) there and a
temperature-sampled isobar silently breaks at the saturation line -- which is
exactly where every heat-exchanger leg in both cycles lives. Enthalpy
increases monotonically along an isobar through subcooled liquid, the dome and
superheat alike, so sampling in h gives one unbroken path with no branching on
phase. Throttles are drawn the same way, sampled in pressure at constant h,
rather than as the straight lines a T--s sketch usually shows.

CONNECTIVITY IS READ FROM THE MODEL, NOT GUESSED. The ORC is a REHEAT cycle:
3e expands to 5e, is reheated at p_int to 6e, then expands to 4e; 7e--10e are
the regenerative sub-circuit. The HTHP is a two-stage FLASH-ECONOMIZER cycle:
7h throttles to 8h into a flash drum at p_int, the saturated liquid 9h
throttles on to 4h, and the flash vapour 10h joins the second stage at 11h.
That ordering comes from the enthalpies in the state dictionaries, not from
the numbering.
"""
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import CoolProp.CoolProp as CP

SERIF = ["Times New Roman", "Nimbus Roman", "Liberation Serif",
         "STIXGeneral", "DejaVu Serif"]

RED, BLUE, NAVY, PURPLE, GREY = "#b03028", "#1b6ca8", "#23356b", "#5b3a8e", "#6a6a6a"

# Annotations in a busy T-s plot land on curves. A white, edgeless patch
# behind the text masks whatever it crosses; without it the approach-
# temperature symbols and the isobar labels are unreadable where they sit
# on a process line.
MASK = dict(facecolor="white", edgecolor="none", alpha=0.82, pad=1.3)


def use_paper_style(scale=1.0):
    """Serif, hairline axes, no top/right spine. Call once per session.

    v2.22: THE SIZES HERE ARE THE SIZES THE READER SEES. Every figure is now
    drawn at the width the paper prints it at (see PRINTED_IN below), so
    LaTeX does not scale it and a 9 pt label prints as 9 pt. Before this
    round the panels were drawn about twice their printed width, so the same
    9.5 pt label reached the page at about 5 pt, against 12 pt body text --
    which is what JRB was looking at when he said the figures were too small.

    Tick labels 8 pt, axis labels and legends 9 pt, in-plot annotations no
    smaller than ANNOT_MIN. Curves 1.0 pt, axes 0.6 pt.

    `scale` multiplies every type size, for a figure deliberately reproduced
    at another size. Leave it at 1.0 to get the printed sizes.
    """
    s = float(scale)
    mpl.rcParams.update({
        "font.family": "serif", "font.serif": SERIF,
        "mathtext.fontset": "stix",
        "font.size": 9 * s, "axes.labelsize": 9 * s,
        "axes.titlesize": 9 * s,
        "xtick.labelsize": 8 * s, "ytick.labelsize": 8 * s,
        "legend.fontsize": 8 * s,
        "axes.linewidth": 0.6,
        "axes.spines.top": False, "axes.spines.right": False,
        "xtick.direction": "in", "ytick.direction": "in",
        "xtick.major.width": 0.6, "ytick.major.width": 0.6,
        "xtick.minor.width": 0.5, "ytick.minor.width": 0.5,
        "xtick.major.size": 2.6, "ytick.major.size": 2.6,
        "lines.linewidth": 1.0, "lines.markersize": 3.6,
        "legend.frameon": False, "legend.handlelength": 1.8,
        "legend.borderaxespad": 0.3, "legend.labelspacing": 0.3,
        "figure.dpi": 140, "savefig.dpi": 300, "savefig.bbox": "tight",
    })


# --------------------------------------------------- the printed geometry
# elsarticle preprint at 12 pt: \linewidth = 390 pt. A figure drawn at any
# other width is scaled by LaTeX, and its type is scaled with it.
LINEWIDTH_IN = 390.0 / 72.27          # 5.40 in
PRINTED_IN = dict(half=0.48 * LINEWIDTH_IN,    # a half-width subfigure
                  profile=0.49 * LINEWIDTH_IN, # the twelve profile panels
                  full=LINEWIDTH_IN,           # \linewidth
                  loss=0.55 * LINEWIDTH_IN)    # prof_loss_cmp
ANNOT_MIN = 7.5                       # smallest in-plot annotation, pt

# v2.23: the well-profile panels only. well_integrals() keeps its weights,
# because Figs. 16 and S4 are approved and must not move.
PROFILE_LW = 0.7                      # the profile curves
PROFILE_LW_OVERLAY = 0.8              # dotted T_pcm, dashed melting staircase


def figsize(kind, aspect=0.80, width_in=None):
    """Width as printed, height from the aspect ratio.

    aspect is height/width, so the shapes the figures already had are kept:
    pass the old (w, h) as aspect = h / w.
    """
    w = float(width_in) if width_in else PRINTED_IN[kind]
    return (w, w * float(aspect))


def active_serif():
    """Which font actually resolved. Never assume the first choice won."""
    from matplotlib.font_manager import findfont, FontProperties
    got = findfont(FontProperties(family="serif"))
    norm = lambda x: x.lower().replace(" ", "").replace("-", "")
    for name in SERIF:
        if norm(name) in norm(got):
            return name, got
    return "(fallback)", got


# ----------------------------------------------------------------- helpers
def _dome(ax, fluid, n=320):
    Tc = CP.PropsSI("Tcrit", fluid)
    T = np.linspace(max(CP.PropsSI("Ttriple", fluid) + 1.0, 180.0), Tc - 0.15, n)
    sl = np.array([CP.PropsSI("S", "T", t, "Q", 0, fluid) for t in T]) / 1e3
    sv = np.array([CP.PropsSI("S", "T", t, "Q", 1, fluid) for t in T]) / 1e3
    ax.plot(np.r_[sl, sv[::-1]], np.r_[T, T[::-1]] - 273.15,
            color="0.30", lw=1.3, zorder=1)


def _iso_p(ax, fluid, p, h1, h2, n=160, **kw):
    """Constant-pressure path, sampled in enthalpy so it crosses the dome."""
    H = np.linspace(h1, h2, n) * 1e3
    T = np.array([CP.PropsSI("T", "P", p, "H", h, fluid) for h in H]) - 273.15
    s = np.array([CP.PropsSI("S", "P", p, "H", h, fluid) for h in H]) / 1e3
    ax.plot(s, T, zorder=4, **kw)


def _iso_h(ax, fluid, h, p1, p2, n=120, **kw):
    """Constant-enthalpy path (a throttle), sampled in pressure."""
    P = np.geomspace(p1, p2, n)
    T = np.array([CP.PropsSI("T", "P", p, "H", h * 1e3, fluid) for p in P]) - 273.15
    s = np.array([CP.PropsSI("S", "P", p, "H", h * 1e3, fluid) for p in P]) / 1e3
    ax.plot(s, T, zorder=3, **kw)


def _frame(ax, pts, padx=0.16, pady=0.18):
    """Zoom to the cycle. The full dome dwarfs both cycles otherwise."""
    s = np.array([p[0] for p in pts]); T = np.array([p[1] for p in pts])
    ds = (s.max() - s.min()) or 1.0
    dT = (T.max() - T.min()) or 1.0
    ax.set_xlim(s.min() - padx * ds, s.max() + padx * ds)
    ax.set_ylim(T.min() - pady * dT, T.max() + pady * dT)


def _label(ax, s, T, lab, off, marker="o", fc="white", ec="#1a1a1a"):
    ax.plot(s, T, marker, ms=4.4, mfc=fc, mec=ec, mew=1.2, zorder=6)
    ax.annotate(lab, (s, T), textcoords="offset points", xytext=off,
                fontsize=8.4, color="#1a1a1a", zorder=7, bbox=MASK)


def _axes(ax, title, loc="lower right"):
    ax.set_xlabel(r"specific entropy, $s$  /  kJ kg$^{-1}$ K$^{-1}$")
    ax.set_ylabel(r"temperature, $T$  /  $^{\circ}$C")
    ax.set_title(title, pad=8)
    ax.legend(loc=loc, handlelength=1.7, borderpad=0.2, labelspacing=0.35)


# ------------------------------------------------------------------- ORC
_ORC_OFF = {"1e": (8, -11), "2e": (8, 2), "3e": (-18, 7), "4e": (7, 0),
            "5e": (-20, -11), "6e": (6, 5), "7e": (-24, -13), "8e": (7, 4),
            "9e": (-25, 2), "10e": (7, -11)}


def ts_orc(case, rank, ax=None, zoom=True):
    """T-s diagram of the double-stage reheat ORC at the reference case."""
    f = case.fluid
    if ax is None:
        _, ax = plt.subplots(figsize=(5.7, 4.5))
    _dome(ax, f)
    S = lambda k: rank["s_" + k] / 1e3
    T = lambda k: rank["T_" + k] - 273.15
    h = lambda k: rank["h_" + k]

    _iso_p(ax, f, rank["p_evape"], h("2e"), h("3e"), color=RED, lw=1.8,
           label=r"heat addition, $p_\mathrm{evap}$")
    ax.plot([S("3e"), S("5e")], [T("3e"), T("5e")], color=NAVY, lw=1.8,
            zorder=4, label="turbine / pump")
    _iso_p(ax, f, rank["p_inte"], h("5e"), h("6e"), color=RED, lw=1.8,
           ls=(0, (5, 2)), label=r"reheat, $p_\mathrm{int}$")
    ax.plot([S("6e"), S("4e")], [T("6e"), T("4e")], color=NAVY, lw=1.8, zorder=4)
    _iso_p(ax, f, rank["p_conde"], h("4e"), h("1e"), color=BLUE, lw=1.8,
           label=r"condensation, $p_\mathrm{cond}$")
    ax.plot([S("1e"), S("2e")], [T("1e"), T("2e")], color=NAVY, lw=1.8, zorder=4)

    pts = []
    for k in ("1e", "2e", "3e", "4e", "5e", "6e"):
        _label(ax, S(k), T(k), k, _ORC_OFF[k]); pts.append((S(k), T(k)))
    # 7e--10e differ by hundredths of a degree and would be four overlapping
    # labels on one dot; they are drawn as four markers but annotated once.
    for k in ("7e", "8e", "9e", "10e"):
        ax.plot(S(k), T(k), "s", ms=4.4, mfc="#b9b9c8", mec="#1a1a1a",
                mew=1.2, zorder=6)
        pts.append((S(k), T(k)))
    ax.annotate("7e--10e", (S("8e"), T("8e")), textcoords="offset points",
                xytext=(8, 3), fontsize=8.4, zorder=7)
    ax.plot([], [], "s", ms=4.4, mfc="#b9b9c8", mec="#1a1a1a", mew=1.2,
            label="regenerator, 7e--10e")

    if zoom:
        _frame(ax, pts)
    # v2.22: on two lines. At the printed width each panel is 2.7 in, and
    # this title on one line is wider than that -- the (a) and (b) titles
    # ran into each other.
    _axes(ax, rf"(a) ORC, {f}" + "\n"
              rf"$\eta_\mathrm{{ORC}} = {rank['rank_eff']:.4f}$,"
              rf"  $y = {rank['y_frac']:.4f}$", loc="upper left")
    return ax


# ------------------------------------------------------------------ HTHP
_HP_OFF = {"1h": (7, -10), "2h": (7, 1), "3h": (7, 2), "4h": (-19, -11),
           "5h": (-20, 4), "6h": (-20, -11), "7h": (-22, -11), "8h": (-8, -12),
           "9h": (-6, 8), "10h": (-24, -5), "11h": (7, 1), "12h": (7, -10),
           # v2.22: 9h was (-23, 1), which at the printed panel width put the
           # label on the "100" tick of the temperature axis
           "13h": (-25, -2)}   # v2.20: was (7, 2), which collided with 1h


def ts_hthp(case, hp, ax=None, zoom=True):
    """T-s diagram of the two-stage flash-economizer HTHP."""
    f = case.refrig
    if ax is None:
        _, ax = plt.subplots(figsize=(5.7, 4.5))
    _dome(ax, f)
    S = lambda k: hp["s_" + k] / 1e3
    T = lambda k: hp["T_" + k] - 273.15
    h = lambda k: hp["h_" + k]
    pe, pi, pc = hp["p_evaph"], hp["p_inth"], hp["p_condh"]

    _iso_p(ax, f, pe, h("4h"), h("1h"), color=BLUE, lw=1.8,
           label=r"evaporation $+$ superheat, $p_\mathrm{evap}$")
    ax.plot([S("1h"), S("5h")], [T("1h"), T("5h")], color=PURPLE, lw=1.8,
            zorder=4, label="compression")
    ax.plot([S("11h"), S("2h")], [T("11h"), T("2h")], color=PURPLE, lw=1.8, zorder=4)
    _iso_p(ax, f, pc, h("2h"), h("7h"), color=RED, lw=1.8,
           label=r"condensation $+$ subcool, $p_\mathrm{cond}$")
    _iso_h(ax, f, h("7h"), pc, pi, color=GREY, lw=1.3, ls=(0, (2, 2)),
           label=r"throttle, $h =$ const.")
    _iso_h(ax, f, h("9h"), pi, pe, color=GREY, lw=1.3, ls=(0, (2, 2)))
    ax.plot([S("10h"), S("11h")], [T("10h"), T("11h")], color=PURPLE, lw=1.0,
            ls=(0, (1, 2)), zorder=3, label="flash vapour to stage 2")

    # 5h, 6h and 11h are the SAME state in the model -- three labels on one
    # dot. Annotated once rather than stacked.
    pts = []
    for k in [f"{i}h" for i in range(1, 14)]:
        sq = k in ("5h", "6h", "9h", "10h", "11h", "12h")
        if k in ("5h", "6h"):
            ax.plot(S(k), T(k), "s", ms=4.4, mfc="#c3b9d8", mec="#1a1a1a",
                    mew=1.2, zorder=6)
        else:
            _label(ax, S(k), T(k),
                   "5h, 6h, 11h" if k == "11h" else k, _HP_OFF[k],
                   marker="s" if sq else "o", fc="#c3b9d8" if sq else "white")
        pts.append((S(k), T(k)))

    if zoom:
        _frame(ax, pts)
    # v2.22: the legend moves off the upper left, where it sat across the
    # 3h / 12h / 7h cluster once the panel was drawn at its printed width.
    # The middle left of this panel is empty at every reference setting.
    _axes(ax, rf"(b) HTHP, {f}" + "\n"
              rf"COP $= {hp['hp_cop']:.4f}$,"
              rf"  $x_\mathrm{{8h}} = {hp['x_8h']:.4f}$", loc="center left")
    return ax


def figsize_full_pair(aspect=0.66):
    # v2.22: the old 0.40 left each panel 2.7 x 2.15 in at the
    # printed width, and the legend sat on the curves. A T-s
    # diagram takes portrait panels well: the temperature range
    # is what the reader is reading.
    return figsize('full', aspect)


def ts_both(case, rank, hp, figsize=None):
    # v2.22: two panels across \linewidth. Drawn at 11.8 in and
    # printed at 5.4 it was scaled 0.46 -- the worst in the set, with
    # text reaching the page at 2.8 pt.
    figsize = figsize or figsize_full_pair()
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    ts_orc(case, rank, ax=axes[0])
    ts_hthp(case, hp, ax=axes[1])
    # v2.22: the legends go BELOW the panels. At the printed width each
    # panel is 2.7 in and the longest entry is nearly that wide, so inside
    # the axes it lands on the curves and the node labels wherever it is
    # put -- upper left crossed 3h/12h/7h, centre left crossed 8h and 10h.
    for ax in axes:
        leg = ax.get_legend()
        if leg is None:
            continue
        handles, labels = ax.get_legend_handles_labels()
        leg.remove()
        ax.legend(handles, labels, loc="upper center",
                  bbox_to_anchor=(0.5, -0.20), ncol=1, fontsize=7.6,
                  frameon=False, handlelength=1.6, borderaxespad=0.0,
                  labelspacing=0.25)
    fig.tight_layout()
    return fig, axes


# --------------------------------------------------- both on one axes
_CMB_ORC = {"1e": (-22, -11), "2e": (7, 3), "3e": (-16, 8), "4e": (7, -2),
            "5e": (-24, -5), "6e": (-20, 7)}
# v2.22: 5e was (-19, -13). At the printed width that put it on 13h, which
# is 17 mK of entropy away and 11 K below.
_CMB_HP = {"1h": (7, -11), "2h": (-20, 5), "3h": (-17, 7), "4h": (-20, -13),
           "7h": (-21, -12), "8h": (-6, -13), "9h": (-24, 5),
           "10h": (-26, -10), "11h": (7, 0), "12h": (7, -11), "13h": (-21, 3)}


def ts_combined(case, rank, hp, Tdict, ax=None, show_water=True,
                water="level", model=None, isobars=True, show_values=False,
                title=None, dimension_style=True):
    """Both cycles on one T-s axes with the external water levels.

    This is the coupling diagram: the two cycles never exchange working
    fluid, they exchange water, so the horizontal lines are the only thing
    that makes the pair a single machine rather than two unrelated plots.
    The four levels are the sink, the HTHP evaporator (geothermal source),
    the HTHP condenser inlet (charge loop) and the ORC evaporator inlet
    (the store outlet, T_2d) -- the last being the one quantity the
    borehole model actually sets.
    """
    f = case.fluid
    if ax is None:
        _, ax = plt.subplots(figsize=(7.6, 5.6))
    _dome(ax, f)
    Se = lambda k: rank["s_" + k] / 1e3
    Te = lambda k: rank["T_" + k] - 273.15
    he = lambda k: rank["h_" + k]
    Sh = lambda k: hp["s_" + k] / 1e3
    Th = lambda k: hp["T_" + k] - 273.15
    hh = lambda k: hp["h_" + k]

    pts = []

    # ---- water as REAL GLIDES, duty-mapped onto the working fluid's
    # entropy, with every pinch marked. `water="level"` falls back to the
    # four inlet temperatures as horizontal lines, which is the conventional
    # presentation but hides where each exchanger actually binds.
    if show_water and water == "glide":
        if model is None:
            raise ValueError("water='glide' needs the model module, "
                             "e.g. ts_combined(..., water='glide', model=B)")
        ov = _water_overlay(model, case, rank, hp, Tdict)
        # symbol, not value: the approach temperatures are specified
        # quantities and belong in the nomenclature, not on the drawing
        style = {
            "ORC evaporator": ("#1a1a1a",
                               r"ORC evaporator water, $T_\mathrm{2d}\to T_\mathrm{3d}$",
                               r"$\Delta T_\mathrm{pinch,ORC}$", (9, 7)),
            "ORC condenser": ("#2a6fb0", "sink water, ORC condenser",
                              r"$\Delta T_\mathrm{E,sink}$", (10, -14)),
            "HTHP condenser": ("#0a9396",
                               r"charge water, $T_\mathrm{2c}\to T_\mathrm{3c}$",
                               r"$\Delta T_\mathrm{2H,3C}$", (8, -2)),
            "HTHP evaporator": ("#9c27a0", "source water, HTHP evaporator",
                                r"$\Delta T_\mathrm{pinch,HPE}$", (9, -15)),
        }
        # v2.20: where each approach is DIMENSIONED, from JRB's marked-up
        # page. Every one is led out of the cluttered region and labelled in
        # clear space; the geometry is computed from the record, so the
        # arrows mark gaps the solver found rather than drawn guesses.
        DIM = {
            # led LEFT from the pinch, arrows outside, label up and left
            "ORC evaporator": dict(side="left", ext=0.17, gap=0.015, arm=5.0,
                                   lab_xy=(-0.050, 83.0), lab_ha="left",
                                   lab_va="bottom"),
            # the two thin grey guide lines already run left from 4h and act
            # as the extension lines, so only the label moves
            "HTHP evaporator": dict(side="left", ext=0.17, gap=0.015,
                                    draw_ext=(False, False), arrows=False,
                                    lab_xy=(0.300, 42.5), lab_ha="right",
                                    lab_va="center"),
            # a double arrow left of the saturation curve, between the 2h
            # level and a new extension line led left from the 3C end; the
            # 10 K gap is wide enough to take the arrow between the lines
            "HTHP condenser": dict(side="left", ext=0.085, gap=0.012,
                                   double=True,
                                   lab_xy=(1.313, 144.0), lab_ha="left",
                                   lab_va="center"),
            # led RIGHT at the condensing temperature, label lower right
            "ORC condenser": dict(side="right", ext=0.090, gap=0.010, arm=4.5,
                                  lab_xy=(1.300, 16.0), lab_ha="left",
                                  lab_va="center"),
        }
        for nm, (col, lab, sym, off) in style.items():
            d = ov[nm]
            txt = f"{lab}  ({d['pinch_K']:.1f} K)" if show_values else lab
            _draw_water(ax, d, col, label=txt,
                        symbol=None if show_values else sym,
                        dx=off[0], dy=off[1],
                        dim=DIM.get(nm) if dimension_style else None)
            pts += list(zip(d["s"], d["T_w"]))

    # ---- water-side levels, drawn first so the cycles sit on top
    if show_water and water == "level":
        levels = [
            (case.T_sink_C, "#2a6fb0", r"sink, ORC condenser"),
            (case.T_source_C, "#9c27a0", r"source, HTHP evaporator"),
            (Tdict["T_2c"] - 273.15, "#0a9396", r"HTHP condenser inlet, $T_\mathrm{2c}$"),
            (Tdict["T_2d"] - 273.15, "#1a1a1a", r"ORC evaporator inlet, $T_\mathrm{2d}$"),
        ]
        for Tl, col, lab in levels:
            ax.axhline(Tl, color=col, lw=1.0, ls=(0, (1, 2.5)), zorder=2,
                       label=f"{lab}  ({Tl:.0f} $^\\circ$C)")
            pts.append((0.0, Tl))

    # ---- ORC, blue
    _iso_p(ax, f, rank["p_evape"], he("2e"), he("3e"), color=NAVY, lw=1.5)
    ax.plot([Se("3e"), Se("5e")], [Te("3e"), Te("5e")], color=NAVY, lw=1.5, zorder=4)
    _iso_p(ax, f, rank["p_inte"], he("5e"), he("6e"), color=NAVY, lw=1.5)
    ax.plot([Se("6e"), Se("4e")], [Te("6e"), Te("4e")], color=NAVY, lw=1.5, zorder=4)
    _iso_p(ax, f, rank["p_conde"], he("4e"), he("1e"), color=NAVY, lw=1.5)
    ax.plot([Se("1e"), Se("2e")], [Te("1e"), Te("2e")], color=NAVY, lw=1.5, zorder=4)

    # ---- HTHP, red
    g = case.refrig
    _iso_p(ax, g, hp["p_evaph"], hh("4h"), hh("1h"), color=RED, lw=1.5)
    ax.plot([Sh("1h"), Sh("5h")], [Th("1h"), Th("5h")], color=RED, lw=1.5, zorder=4)
    ax.plot([Sh("11h"), Sh("2h")], [Th("11h"), Th("2h")], color=RED, lw=1.5, zorder=4)
    _iso_p(ax, g, hp["p_condh"], hh("2h"), hh("7h"), color=RED, lw=1.5)
    _iso_h(ax, g, hh("7h"), hp["p_condh"], hp["p_inth"], color=RED, lw=1.1,
           ls=(0, (3, 2)))
    _iso_h(ax, g, hh("9h"), hp["p_inth"], hp["p_evaph"], color=RED, lw=1.1,
           ls=(0, (3, 2)))
    ax.plot([Sh("10h"), Sh("11h")], [Th("10h"), Th("11h")], color=RED, lw=1.0,
            ls=(0, (1, 2)), zorder=3)

    for k, off in _CMB_ORC.items():
        _label(ax, Se(k), Te(k), k, off, fc=NAVY, ec=NAVY); pts.append((Se(k), Te(k)))
    for k in ("7e", "8e", "9e", "10e"):
        ax.plot(Se(k), Te(k), "o", ms=4.0, mfc=NAVY, mec=NAVY, zorder=6)
        pts.append((Se(k), Te(k)))
    ax.annotate("7e--10e", (Se("8e"), Te("8e")), textcoords="offset points",
                xytext=(-26, 5), fontsize=8.4, color=NAVY, zorder=7)
    for k, off in _CMB_HP.items():
        lab = "5h, 6h, 11h" if k == "11h" else k
        _label(ax, Sh(k), Th(k), lab, off, fc=RED, ec=RED); pts.append((Sh(k), Th(k)))
    for k in ("5h", "6h"):
        ax.plot(Sh(k), Th(k), "o", ms=4.0, mfc=RED, mec=RED, zorder=6)

    ax.plot([], [], "o-", color=NAVY, ms=4.4, lw=1.5, label="ORC (Rankine)")
    ax.plot([], [], "o-", color=RED, ms=4.4, lw=1.5, label="HTHP (heat pump)")

    _frame(ax, pts, padx=0.10, pady=0.12)

    # ---- constant-pressure reference curves, drawn last so the frame is
    # known and they can be clipped to it. The two superheated legs that
    # look like chords -- 13h->1h and 4e to the saturated vapour -- ARE on
    # their isobars; those isobars are straight there to 0.009 K and
    # 0.057 K respectively. Drawing the isobars makes that visible rather
    # than something the reader has to take on trust.
    if isobars:
        y0, y1 = ax.get_ylim()
        levels = sorted({round(p, 3) for p in
                         (rank["p_conde"], rank["p_inte"], rank["p_evape"],
                          hp["p_evaph"], hp["p_inth"], hp["p_condh"])})
        # Group pressures that are within a few per cent: their isobars are
        # one line at this scale, so two labels would sit on top of each
        # other. Here p_int,HP = 3.82 bar and p_evap,ORC = 3.70 bar.
        groups = []
        for p in levels:
            if groups and abs(p - groups[-1][-1]) / p < 0.06:
                groups[-1].append(p)
            else:
                groups.append([p])
        fracs = [0.97, 0.52, 0.86, 0.97, 0.68, 0.40]
        for q, grp in enumerate(groups):
            # only the NUMBER goes in math mode; "bar" set upright
            lab = "$" + r"\,/\,".join(f"{pp / 1e5:.2f}" for pp in grp) + "$ bar"
            for j, pp in enumerate(grp):
                _isobar_bg(ax, f, pp, max(y0, -40.0), min(y1 + 40.0, 230.0),
                           label=(lab if j == 0 else None),
                           label_frac=fracs[q % len(fracs)])
        ax.set_ylim(y0, y1)

    ax.set_xlabel(r"specific entropy, $s$  /  kJ kg$^{-1}$ K$^{-1}$")
    ax.set_ylabel(r"temperature, $T$  /  $^{\circ}$C")
    ax.set_title(title or "Cyclopentane in the Heat Pump and Organic "
                          "Rankine Cycles", pad=9)
    ax.legend(loc="upper left", handlelength=1.9, borderpad=0.25,
              labelspacing=0.35)
    return ax


# ------------------------------------------- profiles along the well
#
# THE COORDINATE. The solver's z is NOT depth. Quoting formation_profile:
# "On CHARGE the water enters the fin tubes at the BOTTOM -- it arrives
# there down the downcomer -- so the solver's z = 0 is the bottom of the
# well and depth decreases with z." So depth = L_well - z, and plotting
# against the raw z and calling it depth puts the well upside down. Use
# `depth_seg`, taken from formation_profile so there is one definition.
#
# WHICH FRAME THE HISTORY IS IN. Both half-cycles come back in the CHARGE
# frame -- index 0 is the BOTTOM for charge and discharge alike -- even
# though the discharge march is run mirrored. Verified, not assumed: the
# test is that the PCM can never sit above its own melting point while it
# is freezing, and at the end of discharge max(T_pcm - T_m) is +0.00 K read
# forward against +26.9 K read reversed. The fluid boundary alone does not
# settle it, because the inlet is at opposite ends in the two half-cycles.
#
# PHYSICALLY. The hottest layer (150 C) sits at the BOTTOM, against the
# hottest rock, which is what keeps its loss down. On charge the water
# descends the insulated downcomer, enters the fin tubes at the bottom and
# cools as it RISES; on discharge it enters at the top, warms as it
# DESCENDS, and returns up the downcomer. Counter-current.

def _time_subset(t_hist, n=9):
    """Roughly log-spaced time levels, always including first and last."""
    t = np.asarray(t_hist, float)
    want = np.unique(np.concatenate(
        ([t[0]], t[0] + (t[-1] - t[0]) * np.logspace(-3, 0, n - 1))))
    return [int(np.argmin(np.abs(t - w))) for w in want]


def _staircase_xy(z, T_m_seg):
    """Explicit staircase polyline, so it can be drawn either way round.

    `ax.step` only ever steps in y as a function of x, which is wrong when
    depth is the ORDINATE. Building the vertices by hand keeps one
    definition of the staircase for both orientations.
    """
    z = np.asarray(z, float)
    T = np.asarray(T_m_seg, float) - 273.15
    dz = z[1] - z[0] if len(z) > 1 else 1.0
    edges = np.concatenate(([z[0] - dz / 2], z + dz / 2))
    return np.repeat(T, 2), np.repeat(edges, 2)[1:-1]


def _staircase(ax, z, T_m_seg, vertical=False, **kw):
    kw = dict(color="#1a1a1a", lw=1.1, ls=(0, (4, 2)), zorder=6) | kw
    T, zz = _staircase_xy(z, T_m_seg)
    ax.plot(*((T, zz) if vertical else (zz, T)), **kw)


def depth_axis(model, case, n=None):
    """(depth of segment centers, depth of fluid nodes) in meters.

    Taken from the model's own formation_profile so the plotting layer
    cannot drift from the solver's idea of which end is the bottom.
    """
    n = n or case.n_segments
    depth_seg = np.asarray(model.formation_profile(case, n,
                                                   discharge=False)[1], float)
    L = case.L_well
    depth_node = L - np.linspace(0.0, L, n + 1)
    # Inside boreas_ground.field_ground the model's depth is MEASURED
    # depth, because that is what the resistance lookup is indexed on.
    # For a FIGURE it has to be true vertical depth: at the reference
    # case 1250 m of hole reaches 891 m. The context manager leaves the
    # mapping on the model for exactly this.
    tvd_of = getattr(model, "_boreas_tvd_of", None)
    if tvd_of is not None:
        depth_seg = np.asarray(tvd_of(depth_seg), float)
        depth_node = np.asarray(tvd_of(depth_node), float)
    return depth_seg, depth_node


# --------------------------------------------------------- profile style
def profile_kwargs(case, field):
    """The arguments the PAPER's well-profile panels are drawn with.

    v2.21. The R4 set (Figs. 13-16) and the R3 set (SM S1-S4) were drawn by
    two call sites with different arguments, so the R3 panels came out in
    kelvin, without the melting staircase, the PCM overlay or the time
    colour bar, with melted AREA instead of melted fraction, and -- worst --
    against the raw march coordinate, which runs from the BOTTOM of the
    well. The R3 panels were therefore upside down against an axis labelled
    TVD. One function now owns the style; the two sets differ only in the
    record and in the depth axis, which is what the SM claims.
    """
    common = dict(vertical=True, colorbar=True)
    if field == "T_fluid":
        return dict(common, overlay="T_pcm",
                    overlay_label=r"$T_\mathrm{pcm}$", offset=-273.15,
                    staircase=True, ylabel=r"temperature / $^\circ$C")
    if field == "A_melt":
        return dict(common, scale=1.0 / pcm_capacities_A(case),
                    ylabel="melted fraction of the PCM")
    if field == "U_i":
        # U_i is the OVERALL coefficient on the inner (tube-side) area, not
        # the convective film h_i: it carries the tube-side film, the wall
        # and the PCM-side path in series. Eq. (Ui) of the SM.
        return dict(common, ylabel=r"$U_i$ / W m$^{-2}$ K$^{-1}$")
    raise KeyError(field)


def depth_axes(case, md, tvd):
    """TVD of each segment centre and node, from the trajectory.

    The solver's z runs from the BOTTOM (the charge inlet), so a figure
    that plots against z and labels it TVD is upside down. This maps the
    march coordinate onto true vertical depth, which is also what makes the
    R3 and R4 panels comparable: 1250 m of hole reaches 1250 m TVD when the
    well is vertical and 909 m when it is an S-curve.
    """
    import numpy as _np
    n = case.n_segments
    xi = (_np.arange(n) + 0.5) / n
    mdv = case.L_well * (1.0 - xi)
    dseg = _np.interp(mdv, _np.r_[0.0, md], _np.r_[0.0, tvd])
    dnode = _np.interp(case.L_well * (1.0 - _np.linspace(0, 1, n + 1)),
                       _np.r_[0.0, md], _np.r_[0.0, tvd])
    return dseg, dnode


def check_orientation(rec, case, dseg, name=""):
    """The charge inlet must be at the DEEPEST segment, the discharge at the
    shallowest.

    Asked for in handoff v2.21 after the R3 panels shipped flipped. This is
    the invariant that was violated, stated once and checked wherever a
    profile is drawn, so a figure cannot silently come out upside down
    again.
    """
    import numpy as _np
    i_deep = int(_np.argmax(dseg))
    i_shal = int(_np.argmin(dseg))
    Tc = _np.asarray(rec["charge"]["history"]["T_fluid"], float)
    Td = _np.asarray(rec["discharge"]["history"]["T_fluid"], float)
    ok_c = Tc[0][i_deep] > Tc[0][i_shal]      # hot water enters at depth
    ok_d = Td[0][i_shal] < Td[0][i_deep]      # cold water enters at the top
    if not (ok_c and ok_d):
        raise AssertionError(
            f"{name}: profile orientation is wrong. charge inlet "
            f"{Tc[0][i_deep]-273.15:.1f} C at the deep end against "
            f"{Tc[0][i_shal]-273.15:.1f} C at the top; discharge inlet "
            f"{Td[0][i_shal]-273.15:.1f} C at the top against "
            f"{Td[0][i_deep]-273.15:.1f} C at the deep end")
    return dict(charge_deep_C=float(Tc[0][i_deep] - 273.15),
                charge_top_C=float(Tc[0][i_shal] - 273.15),
                discharge_top_C=float(Td[0][i_shal] - 273.15),
                discharge_deep_C=float(Td[0][i_deep] - 273.15))


def well_profiles(r, case, field="T_fluid", axes=None, n_times=9,
                  figsize=None, cmap="viridis", ylabel=None,
                  title=None, staircase=False, scale=1.0, offset=0.0,
                  logy=False, vertical=True, depth_seg=None, depth_node=None,
                  overlay=None, overlay_label=None, overlay_ls=(0, (2, 1.6)),
                  colorbar=False, depth_label=r"TVD / m",
                  lw=None, lw_overlay=None):
    """One field along the well at several times, charge and discharge.

    `r` must come from simulate_css_corrected(..., record=True).
    `depth_seg` / `depth_node` come from `depth_axis`; without them the raw
    solver coordinate is used, which runs from the BOTTOM of the well and is
    not depth.
    `overlay` names a second field drawn dashed in the same color per time
    (used for T_pcm beside T_fluid).
    """
    if figsize is None:
        figsize = (8.8, 5.8) if vertical else (11.6, 4.4)
    if axes is None:
        _, axes = plt.subplots(1, 2, figsize=figsize,
                              sharex=vertical, sharey=not vertical)
    cm = plt.get_cmap(cmap)
    qlabel = ylabel or field
    dlabel = depth_label
    for ax, half, lab in ((axes[0], "charge", "(a) charging"),
                          (axes[1], "discharge", "(b) discharging")):
        H, th = r[half]["history"], r[half]["t_hist"]
        zs = depth_seg if depth_seg is not None else r[half]["z"]
        zn = depth_node if depth_node is not None else None
        idx = _time_subset(th, n_times)
        Y = np.asarray(H[field], float)
        O = np.asarray(H[overlay], float) if overlay else None
        for j, i in enumerate(idx):
            col = cm(j / max(1, len(idx) - 1))
            for arr, ls in ((Y, "-"), (O, overlay_ls)):
                if arr is None:
                    continue
                y = arr[i] * scale + offset
                if len(y) == len(zs):
                    zz = zs
                elif zn is not None and len(y) == len(zn):
                    zz = zn
                else:
                    zz = np.linspace(zs[0], zs[-1], len(y))
                # v2.23, JRB: "the lines are a tad too thick". The curves
                # go to 0.7 pt and the dotted overlay and dashed staircase
                # to 0.8, at the printed size.
                _lw = (lw if lw is not None else PROFILE_LW) if ls == "-" \
                    else (lw_overlay if lw_overlay is not None
                          else PROFILE_LW_OVERLAY)
                ax.plot(*((y, zz) if vertical else (zz, y)), color=col,
                        lw=_lw, ls=ls,
                        label=(f"{th[i] / 60:.1f}" if ls == "-" else None))
        if staircase:
            _staircase(ax, zs, r["T_m_seg"], vertical=vertical,
                       lw=lw_overlay if lw_overlay is not None
                       else PROFILE_LW_OVERLAY)
        ax.set_title(lab, pad=7)
        if vertical:
            ax.set_xlabel(qlabel)
            if logy:
                ax.set_xscale("log")
        else:
            ax.set_xlabel(dlabel)
            if logy:
                ax.set_yscale("log")
    if vertical:
        # Label BOTH panels when they are going to separate files: in
        # per-panel mode there is no shared left-hand axis to carry it,
        # and an unlabelled depth axis invites the reader to take TVD
        # for the storage length.
        for ax in (axes if colorbar else axes[:1]):
            ax.set_ylabel(dlabel)
        zs = depth_seg if depth_seg is not None else r["charge"]["z"]
        for ax in axes:
            ax.set_ylim(max(zs) * 1.02, -0.02 * max(zs))   # 0 at the top
    else:
        axes[0].set_ylabel(qlabel)
    if overlay:
        axes[1].plot([], [], color="0.35", lw=1.3, ls=overlay_ls,
                     label=overlay_label or overlay)
    if staircase:
        axes[1].plot([], [], color="#1a1a1a", lw=1.1, ls=(0, (4, 2)),
                     label=r"$T_{m,k}$")
    if colorbar:
        # One key for both panels, on the right of (b): a nine-entry
        # legend inside the axes covered the curves it was labeling.
        import matplotlib as _mpl
        th_all = r["charge"]["t_hist"]
        sm = _mpl.cm.ScalarMappable(
            cmap=cm, norm=_mpl.colors.Normalize(0.0, max(th_all) / 3600.0))
        cb = axes[1].figure.colorbar(sm, ax=axes[1], pad=0.02)
        cb.set_label("time from the start of the half-cycle / h",
                     fontsize=8)
        cb.ax.tick_params(labelsize=7.5)
        if overlay or staircase:
            axes[1].legend(fontsize=7.5, loc="best", handlelength=1.6,
                           frameon=False)
    else:
        leg = axes[1].legend(title="time / min", ncol=2, fontsize=7.5,
                             title_fontsize=7.8, loc="best",
                             handlelength=1.6, labelspacing=0.25,
                             columnspacing=1.0)
        leg._legend_box.align = "left"
    if title:
        axes[0].figure.suptitle(title, y=0.99)
    return axes


def well_integrals(r, case, axes=None, figsize=(11.6, 4.2),
                   per_tube=False, titles=True):
    """Heat rate integrated along the well, and the melted fraction, vs time.

    The melted fraction is the volume-weighted mean of the local melted
    cross-section, A_melt / A_avail, which is the same quantity the model
    reports as the cycled fraction at the end of each half-cycle.
    """
    if axes is None:
        _, axes = plt.subplots(1, 2, figsize=figsize)
    A_avail = pcm_capacities_A(case)
    for ax, half, lab in ((axes[0], "charge", "(a) charging"),
                          (axes[1], "discharge", "(b) discharging")):
        H, th, z = r[half]["history"], r[half]["t_hist"], r[half]["z"]
        dz = z[1] - z[0]
        # q_prime is ALREADY per unit length of FIN TUBE -- Eq. (ode) is
        # written per tube and K_j is per tube length -- so the per-tube
        # rate is the bare sum and the PER WELL rate is n_ft times it.
        # v2.3 divided by n_ft here, which made the plotted rates a
        # factor of three low. Checked against the budget: n_ft times
        # Q_cum_J reproduces dE_ch and dE_dc per well exactly.
        q = np.asarray(H["q_prime"], float).sum(axis=1) * dz / 1e3  # kW/tube
        if not per_tube:
            q = q * case.n_ft
        eps = np.asarray(H["A_melt"], float).mean(axis=1) / A_avail
        ax.plot(th / 60.0, q, color=RED, lw=1.8)
        ax.set_xlabel("time / min")
        ax.set_ylabel(r"$\dot{Q}$ per fin tube / kW" if per_tube
                      else r"$\dot{Q}$ per well / kW", color=RED)
        ax.tick_params(axis="y", colors=RED)
        axb = ax.twinx()
        axb.plot(th / 60.0, eps, color="#2f6f4f", lw=1.8, ls=(0, (5, 2)))
        axb.set_ylabel("melted fraction of the PCM", color="#2f6f4f")
        axb.tick_params(axis="y", colors="#2f6f4f")
        axb.spines["right"].set_visible(True)
        if titles:
            ax.set_title(lab, pad=7)
    return axes


def _resample(Y, t_src, t_dst):
    """Linear interpolation of a (time, segment) field onto a new time grid.

    The march records on a roughly LOGARITHMIC time grid, which is right for
    static profiles but makes an animation played at constant frame rate
    appear to accelerate. Resampling onto a uniform grid puts the playback
    clock in proportion to physical time; the sub-minute transient is then
    genuinely brief on screen, which is the honest depiction.
    """
    Y = np.asarray(Y, float)
    return np.column_stack([np.interp(t_dst, t_src, Y[:, j])
                            for j in range(Y.shape[1])])


def animate_well(r, case, path, fps=12, frames_per_half=60, dpi=110,
                 writer=None):
    """Animate the well through one full cycle at cyclic steady state.

    Three panels with depth on the ordinate: fluid temperature against the
    melting staircase, local melted fraction, and the overall coefficient.
    Axis limits are fixed over the whole cycle so that nothing jumps between
    the half-cycles and the eye can compare them directly.

    `path` ending .gif uses pillow, .mp4 uses ffmpeg.
    """
    import matplotlib.animation as manim
    import boreas as _b
    A = pcm_capacities_A(case)
    zs, zn = depth_axis(_b, case, len(r["T_m_seg"]))
    seq = []                              # (phase, t_s, Tf, Tp, eps, Ui)
    for half, lab in (("charge", "charging"), ("discharge", "discharging")):
        H, th = r[half]["history"], r[half]["t_hist"]
        tu = np.linspace(th[0], th[-1], frames_per_half)
        Tf = _resample(H["T_fluid"], th, tu) - 273.15
        Tp = _resample(H["T_pcm"], th, tu) - 273.15
        eps = _resample(H["A_melt"], th, tu) / A
        Ui = _resample(H["U_i"], th, tu)
        for k in range(frames_per_half):
            seq.append((lab, tu[k], Tf[k], Tp[k], eps[k], Ui[k]))

    allT = np.concatenate([np.concatenate([s[2], s[3]]) for s in seq])
    allU = np.concatenate([s[5] for s in seq])
    lims = [(allT.min() - 3, allT.max() + 3), (-0.04, 1.04),
            (0.0, allU.max() * 1.05)]
    labs = [r"temperature / $^\circ$C", r"$A_\mathrm{melt}/A_\mathrm{av}$",
            r"$U_i$ / W m$^{-2}$ K$^{-1}$"]
    cols = [NAVY, "#2f6f4f", PURPLE]

    fig, axes = plt.subplots(1, 3, figsize=(11.0, 5.4), sharey=True)
    lines = []
    for ax, lim, lab, col in zip(axes, lims, labs, cols):
        ax.set_xlim(*lim)
        ax.set_xlabel(lab)
        ln, = ax.plot([], [], color=col, lw=2.0)
        lines.append(ln)
    lpcm, = axes[0].plot([], [], color=RED, lw=1.5, ls=(0, (2, 1.6)))
    lines.append(lpcm)
    _staircase(axes[0], zs, r["T_m_seg"], vertical=True)
    axes[0].plot([], [], color=NAVY, lw=2.0, label=r"$T_f$")
    axes[0].plot([], [], color=RED, lw=1.5, ls=(0, (2, 1.6)),
                 label=r"$T_\mathrm{pcm}$")
    axes[0].plot([], [], color="#1a1a1a", lw=1.1, ls=(0, (4, 2)),
                 label=r"$T_{m,k}$")
    axes[0].legend(loc="lower left", fontsize=8.2, handlelength=1.8)
    # TVD, like the still figures: depth_axis returns true vertical
    # depth whenever the field patch is active, and on the S-curve
    # that is 909 m for 1250 m of hole.
    axes[0].set_ylabel(r"TVD / m")
    # Depth increases downward: setting the limits in descending order IS
    # the inversion -- invert_yaxis() on top of this would undo it.
    top = float(np.max(zs))
    axes[0].set_ylim(top * 1.02, -0.02 * top)
    head = fig.suptitle("", y=0.975, fontsize=11)
    fig.subplots_adjust(left=0.075, right=0.985, top=0.90, bottom=0.11,
                        wspace=0.10)

    def draw(i):
        lab, t, Tf, Tp, eps, Ui = seq[i]
        lines[0].set_data(Tf, zn if len(Tf) == len(zn) else zs)
        lines[1].set_data(eps, zs)
        lines[2].set_data(Ui, zs)
        lines[3].set_data(Tp, zs)
        head.set_text(f"{lab}   $t$ = {t / 60:6.1f} min   "
                      f"mean melted fraction {eps.mean():.3f}")
        return lines + [head]

    anim = manim.FuncAnimation(fig, draw, frames=len(seq), interval=1000 / fps,
                               blit=False)
    if writer is None:
        writer = "pillow" if str(path).endswith(".gif") else "ffmpeg"
    anim.save(str(path), writer=writer, fps=fps, dpi=dpi)
    plt.close(fig)
    return len(seq)


def pcm_capacities_A(case):
    """PCM cross-section available per fin tube [m^2]."""
    import boreas as _b
    return _b.pcm_capacities(case)[0]


# ----------------------------------------------------------- heat maps
def heatmap(rows, xkey, ykey, zkey, xs, ys, ax=None, cmap="viridis",
            title="", cbar_label="", fmt="{:.3f}", mark=None, annotate=True,
            vmin=None, vmax=None, cbar=True, fontsize=7.6,
            best="max", box_ref=True):
    """One metric over a two-parameter grid, with the cells labeled.

    Cells are ANNOTATED with their value. A heat map whose color scale is
    the only channel invites the reader to misjudge small differences, and
    several of these maps are deliberately flat -- the flatness is the
    result, and it is only legible if the numbers are on the cells.

    Infeasible points (validate_case blocking, or an exception) are hatched
    rather than dropped, so a hole in the map reads as a constraint and not
    as a bug in the plot.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(6.2, 4.8))
    m = {(r[xkey], r[ykey]): r for r in rows}
    Z = np.full((len(ys), len(xs)), np.nan)
    bad = np.zeros_like(Z, dtype=bool)
    for i, y in enumerate(ys):
        for j, x in enumerate(xs):
            r = m.get((float(x), float(y))) or m.get((x, y))
            if r is None or not r.get("feasible", True) or zkey not in r:
                bad[i, j] = True
            else:
                Z[i, j] = r[zkey]
    im = ax.imshow(Z, origin="lower", aspect="auto", cmap=cmap,
                   vmin=vmin, vmax=vmax, interpolation="nearest",
                   extent=(-0.5, len(xs) - 0.5, -0.5, len(ys) - 0.5))
    # Which cells get called out. The reference case and the best cell are
    # marked by STYLING THE VALUE, not by drawing a symbol on top of it: an
    # overlaid star or ring hides the number it is pointing at, which is the
    # one thing the reader wants from an annotated map.
    ref_ij = None
    if mark is not None and mark[0] in list(xs) and mark[1] in list(ys):
        ref_ij = (list(ys).index(mark[1]), list(xs).index(mark[0]))
    best_ij = None
    if best and np.isfinite(Z).any():
        pick = np.nanargmax(Z) if best == "max" else np.nanargmin(Z)
        best_ij = np.unravel_index(pick, Z.shape)

    for i in range(len(ys)):
        for j in range(len(xs)):
            if bad[i, j]:
                ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                                           facecolor="0.92", edgecolor="0.6",
                                           hatch="///", lw=0.5))
                continue
            if not annotate:
                continue
            lo, hi = np.nanmin(Z), np.nanmax(Z)
            t = 0.5 if hi == lo else (Z[i, j] - lo) / (hi - lo)
            col = "white" if t < 0.55 else "#111111"
            fs, weight = fontsize, "normal"
            if best_ij is not None and (i, j) == tuple(best_ij):
                fs, weight = fontsize * 1.45, "bold"
            elif ref_ij is not None and (i, j) == ref_ij:
                weight = "bold"
            ax.text(j, i, fmt.format(Z[i, j]), ha="center", va="center",
                    fontsize=fs, fontweight=weight, color=col, zorder=5)
    if ref_ij is not None and box_ref:
        ax.add_patch(plt.Rectangle((ref_ij[1] - 0.5, ref_ij[0] - 0.5), 1, 1,
                                   fill=False, edgecolor="#d62828", lw=1.8,
                                   zorder=6))
    ax.set_xticks(range(len(xs)))
    ax.set_xticklabels([f"{x:g}" for x in xs])
    ax.set_yticks(range(len(ys)))
    ax.set_yticklabels([f"{y:g}" for y in ys])
    ax.set_title(title, pad=8)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(True)
    if cbar:
        cb = ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
        cb.set_label(cbar_label)
        cb.outline.set_linewidth(0.6)
    return ax, Z


def spread(Z):
    """Peak-to-peak as a percentage of the mean -- is this map flat?"""
    f = Z[np.isfinite(Z)]
    return 100.0 * (f.max() - f.min()) / abs(f.mean()) if f.size else np.nan


# --------------------------------------- surface cycles vs T_m
def cycle_vs_Tm(rows_autoP, rows_fixedP, case, axes=None, figsize=(13.2, 4.3)):
    """How the two surface machines trade against storage temperature.

    `rows_autoP` has the water-loop pressure raised to keep the loop liquid
    at each T_m; `rows_fixedP` holds it at the shipped value and goes
    infeasible once the charging inlet reaches saturation. Plotting both is
    the point: the first gives the thermodynamic optimum, the second says
    what it costs to reach it.
    """
    if axes is None:
        _, axes = plt.subplots(1, 3, figsize=figsize)
    ok = [r for r in rows_autoP if r.get("feasible")]
    T = np.array([r["T_m_C"] for r in ok])
    eta = np.array([r["eta_ORC"] for r in ok])
    cop = np.array([r["COP"] for r in ok])
    prod = np.array([r["product"] for r in ok])
    Pb = np.array([r["P_bar"] for r in ok])
    T2d = np.array([r["T_2d_C"] for r in ok])
    T3e = np.array([r["T_3e_C"] for r in ok])
    okf = [r["T_m_C"] for r in rows_fixedP if r.get("feasible")]
    T_lim = max(okf) if okf else None
    i_opt = int(np.argmax(prod))
    T_ref = case.T_m - 273.15

    def shade(ax):
        if T_lim is not None and T_lim < T.max():
            ax.axvspan(T_lim, T.max(), color="0.88", zorder=0)
            # y is in AXES fraction here, so it must be 1.0 -- not
            # ax.get_ylim()[1], which is a data value and would place the
            # text at several hundred percent of the axes height. With
            # savefig.bbox='tight' that silently inflates the saved figure
            # to tens of thousands of pixels instead of raising.
            # bottom of the band, not the top: the top is where the
            # optimum callout lives
            ax.annotate(f"needs $P>{case.P / 1e5:.0f}$ bar",
                        (0.5 * (T_lim + T.max()), 0.02),
                        ha="center", va="bottom", fontsize=8.0, color="0.35",
                        xycoords=("data", "axes fraction"), xytext=(0, 2),
                        textcoords="offset points", bbox=MASK)
        ax.axvline(T_ref, color="0.45", lw=0.9, ls=(0, (1, 2.5)), zorder=1)

    ax = axes[0]
    ax.plot(T, eta, color=NAVY, lw=1.9, label=r"$\eta_\mathrm{ORC}$")
    ax.set_ylabel(r"$\eta_\mathrm{ORC}$", color=NAVY)
    ax.tick_params(axis="y", colors=NAVY)
    axb = ax.twinx()
    axb.plot(T, cop, color=RED, lw=1.9, label="COP")
    axb.set_ylabel("COP", color=RED)
    axb.tick_params(axis="y", colors=RED)
    axb.spines["right"].set_visible(True)
    shade(ax)
    ax.set_title("(a) the two machines pull opposite ways", pad=8)

    ax = axes[1]
    ax.plot(T, prod, color="#2f6f4f", lw=2.0)
    ax.plot([T[i_opt]], [prod[i_opt]], "o", ms=6, mfc="white",
            mec="#2f6f4f", mew=1.6, zorder=5)
    ax.annotate(rf"optimum {T[i_opt]:.0f} $^\circ$C" "\n"
                rf"$\eta_\mathrm{{ORC}}\,$COP$\,={prod[i_opt]:.4f}$",
                (T[i_opt], prod[i_opt]), textcoords="offset points",
                xytext=(-4, -30), fontsize=8.4, bbox=MASK)
    j = int(np.argmin(np.abs(T - T_ref)))
    ax.plot([T[j]], [prod[j]], "s", ms=5, mfc="white", mec="0.35", mew=1.3,
            zorder=5)
    ax.annotate(rf"reference {T_ref:.0f} $^\circ$C, {prod[j]:.4f}"
                "\n" rf"({100 * (prod[i_opt] / prod[j] - 1):+.1f} % to optimum)",
                (T[j], prod[j]), textcoords="offset points", xytext=(-58, -52),
                fontsize=8.4, ha="left", bbox=MASK)
    shade(ax)
    axb = ax.twinx()
    axb.plot(T, Pb, color="#9c27a0", lw=1.3, ls=(0, (4, 2)))
    axb.set_ylabel("required loop pressure  /  bar", color="#9c27a0")
    axb.tick_params(axis="y", colors="#9c27a0")
    axb.spines["right"].set_visible(True)
    ax.set_ylabel(r"$\eta_\mathrm{ORC}\times$ COP")
    ax.set_title("(b) a shallow optimum, bought with pressure", pad=8)

    ax = axes[2]
    ax.plot(T, T2d - T3e, color="#b06a28", lw=1.9)
    shade(ax)
    ax.set_ylabel(r"$T_\mathrm{2d} - T_\mathrm{3e}$  /  K")
    ax.set_title("(c) the pinch shortfall is not a quirk\nof the reference case",
                 pad=8)
    ax.set_ylim(0, max(60.0, float((T2d - T3e).max()) * 1.15))

    for a in axes:
        a.set_xlabel(r"storage melting temperature, $T_\mathrm{m}$  /  $^{\circ}$C")
    return axes, dict(T_opt_C=float(T[i_opt]), product_opt=float(prod[i_opt]),
                      T_ref_C=float(T_ref), product_ref=float(prod[j]),
                      gain_pct=float(100 * (prod[i_opt] / prod[j] - 1)),
                      P_feasible_limit_C=T_lim)


# ------------------------- water streams drawn ON the T-s diagram
#
# A water stream has its own entropy, on a scale that has nothing to do with
# cyclopentane's, so it cannot simply be plotted on these axes. What CAN be
# plotted, and is what matters for a pinch, is the water temperature at each
# point OF THE EXCHANGER, placed at the entropy the working fluid has at that
# same point. The two streams are tied together by cumulative duty: at a given
# Q the working fluid is at (s, T_wf) and the water is at T_w. Plotting
# (s, T_w) puts the water line directly above (evaporator) or below
# (condenser) the working-fluid path, and the VERTICAL GAP at every abscissa
# is then the real local approach, with the pinch as its narrowest point.
#
# This is a duty mapping, not a thermodynamic path: the water line is not a
# process in cyclopentane's coordinates and its abscissa carries no meaning
# for the water itself. Said plainly in the legend, it is an honest and very
# readable construction; left unsaid it would be a category error.

def _water_overlay(model, case, rank, hp, Tdict, n=400):
    """name -> dict(s, T_wf, T_w, pinch_K, i_pinch, hot) for each exchanger.

    The stream definitions, the water endpoints and the duty bases are taken
    from `exchanger_UA`, so the pinch recovered here is the same number that
    function reports rather than a parallel construction of it.
    """
    F, R, P = case.fluid, case.refrig, case.P
    out = {}

    def pack(name, s, T_wf, T_w, hot):
        dT = (T_wf - T_w) if hot else (T_w - T_wf)
        i = int(np.argmin(dT))
        out[name] = dict(s=s, T_wf=T_wf - 273.15, T_w=T_w - 273.15,
                         pinch_K=float(dT[i]), i_pinch=i, hot=hot)

    def s_of(h, p, fluid):
        return np.array([CP.PropsSI("S", "P", p, "H", x * 1e3, fluid)
                         for x in h]) / 1e3

    def simple(name, h_cold, h_hot, p, fluid, Tw_cold, Tw_hot, hot):
        """Single-pressure working-fluid stream against a water glide."""
        Q, T = model._stream_curve(h_cold, h_hot, p, fluid, n=n)
        Qw, Tw = model._water_curve(Tw_cold, Tw_hot, P, Q[-1])
        h = h_cold + Q
        pack(name, s_of(h, p, fluid), T, np.interp(Q, Qw, Tw), hot)

    # ---- ORC evaporator. The cold side is TWO streams (main + reheat), so
    # the duty at a given point of the main stream includes whatever the
    # reheat stream has already given up at that temperature. Mapping the
    # main stream alone would shift the water line along Q and move the
    # apparent pinch off the bubble point.
    y, p_e, p_i = rank["y_frac"], rank["p_evape"], rank["p_5e"]
    Qc, _ = model._orc_cold_composite(rank, F)
    # The bubble point must be a NODE. The pinch sits exactly on it, so an
    # evenly spaced grid that steps over it reports a slightly loose value
    # (5.06 K instead of 5.00) and the arrow lands next to the corner rather
    # than on it.
    h_bub = CP.PropsSI("H", "P", p_e, "Q", 0, F) / 1e3
    h = np.unique(np.append(np.linspace(rank["h_10e"], rank["h_3e"], n),
                            h_bub))
    T = np.array([CP.PropsSI("T", "P", p_e, "H", x * 1e3, F) for x in h])
    h_rh = np.linspace(rank["h_5e"], rank["h_6e"], n)
    T_rh = np.array([CP.PropsSI("T", "P", p_i, "H", x * 1e3, F) for x in h_rh])
    Q = (h - rank["h_10e"]) + y * (np.interp(T, T_rh, h_rh) - rank["h_5e"])
    Qw, Tw = model._water_curve(Tdict["T_3d"], Tdict["T_2d"], P, Qc[-1])
    pack("ORC evaporator", s_of(h, p_e, F), T, np.interp(Q, Qw, Tw), False)

    simple("ORC condenser", rank["h_1e"], rank["h_4e"], rank["p_conde"], F,
           case.T_sink_C + 273.15,
           case.T_sink_C + case.DT_sink_glide + 273.15, True)
    simple("HTHP condenser", hp["h_3h"], hp["h_2h"], hp["p_condh"], R,
           Tdict["T_2c"], Tdict["T_3c"], True)
    simple("HTHP evaporator", hp["h_4h"], hp["h_13h"], hp["p_evaph"], R,
           Tdict["T_4a"], case.T_source_C + 273.15, False)
    return out


def _draw_water(ax, d, color, label=None, mark_pinch=True, lw=1.6,
                symbol=None, fontsize=8.8, dx=6, dy=-3, dim=None):
    """Water glide plus its pinch. `symbol` replaces the numeric value."""
    ax.plot(d["s"], d["T_w"], color=color, lw=lw, ls=(0, (6, 2)),
            zorder=5, label=label)
    if mark_pinch:
        i = d["i_pinch"]
        if dim is None:
            # the old presentation: a double arrow across the gap, which at
            # 5 K on a 180 K axis collapses into the blob the markup calls an
            # x, with the label pushed onto whatever curve is nearest
            ax.annotate("", xy=(d["s"][i], d["T_w"][i]),
                        xytext=(d["s"][i], d["T_wf"][i]),
                        arrowprops=dict(arrowstyle="<->", color="#1a1a1a",
                                        lw=1.0))
            txt = symbol if symbol is not None else rf"${d['pinch_K']:.1f}$ K"
            ax.annotate(txt, (d["s"][i], 0.5 * (d["T_w"][i] + d["T_wf"][i])),
                        textcoords="offset points", xytext=(dx, dy),
                        fontsize=fontsize, zorder=9, bbox=MASK)
        else:
            # v2.21, JRB: no marker at the pinch. The x was the old double
            # arrow collapsing, and once the gap is dimensioned outside the
            # clutter the two extension lines already say where the pinch is
            # and what it measures. One annotation, not two.
            lo, hi = sorted((d["T_w"][i], d["T_wf"][i]))
            _dim(ax, d["s"][i], lo, hi,
                 label=symbol if symbol is not None
                 else rf"${d['pinch_K']:.1f}$ K", **dim)


def _dim(ax, s0, T_lo, T_hi, side="left", ext=0.16, gap=0.02, arm=6.0,
         color="#1a1a1a", lw=0.9, ext_lw=0.6, label=None, lab_xy=None,
         lab_ha="left", lab_va="bottom", fontsize=8.8, draw_ext=(True, True),
         double=False, arrows=True, lab_color=None, zorder=9):
    """An engineering dimension between two temperatures at one entropy.

    The pinch gaps here are 5-10 K on a 180 K axis, so a double arrow drawn
    BETWEEN them collapses into a blob -- which is what the figure used to
    show, and why the labels ended up on top of the curves. Drawn as a
    dimension instead: two extension lines led out of the clutter, and the
    arrowheads placed OUTSIDE the gap pointing in, which is how the same
    problem is solved on a drawing.

      side    which way the extension lines are led, 'left' or 'right'
      ext     their length, in data units of entropy
      gap     distance from the pinch to the start of the extension lines
      arm     length of each arrow, in temperature units
      double  one double-headed arrow between the lines instead, when the
              gap is wide enough to take it
      draw_ext  skip one or both extension lines when the figure already
              has a line at that temperature doing the job
    """
    sgn = -1.0 if side == "left" else 1.0
    s1 = s0 + sgn * gap
    s2 = s0 + sgn * (gap + ext)
    for T, want in ((T_hi, draw_ext[0]), (T_lo, draw_ext[1])):
        if want:
            ax.plot([s1, s2], [T, T], color=color, lw=ext_lw, zorder=zorder,
                    solid_capstyle="butt")
    s_arr = s0 + sgn * (gap + 0.62 * ext)
    if not arrows:
        pass
    elif double:
        ax.annotate("", xy=(s_arr, T_hi), xytext=(s_arr, T_lo),
                    arrowprops=dict(arrowstyle="<->", color=color, lw=lw,
                                    shrinkA=0, shrinkB=0), zorder=zorder)
    else:
        ax.annotate("", xy=(s_arr, T_hi), xytext=(s_arr, T_hi + arm),
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=lw,
                                    shrinkA=0, shrinkB=0), zorder=zorder)
        ax.annotate("", xy=(s_arr, T_lo), xytext=(s_arr, T_lo - arm),
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=lw,
                                    shrinkA=0, shrinkB=0), zorder=zorder)
    if label is not None:
        x, y = lab_xy if lab_xy is not None else (s2, T_hi + arm)
        ax.annotate(label, (x, y), fontsize=fontsize,
                    color=lab_color if lab_color else color,
                    ha=lab_ha, va=lab_va, zorder=zorder + 1, bbox=MASK)


def _isobar_bg(ax, fluid, p, T_lo, T_hi, n=240, label=None, lw=0.7,
               color="0.78", fontsize=7.6, zorder=0, label_frac=0.97):
    """A constant-pressure reference curve spanning the frame.

    Sampled in ENTHALPY between two SINGLE-PHASE anchors, so the path runs
    liquid -> dome -> vapour without a (T, p) call ever landing inside the
    two-phase region, where temperature and pressure are not independent.
    """
    h_lo = CP.PropsSI("H", "T", T_lo + 273.15, "P", p, fluid) / 1e3
    h_hi = CP.PropsSI("H", "T", T_hi + 273.15, "P", p, fluid) / 1e3
    H = np.linspace(h_lo, h_hi, n)
    T = np.array([CP.PropsSI("T", "P", p, "H", h * 1e3, fluid)
                  for h in H]) - 273.15
    s = np.array([CP.PropsSI("S", "P", p, "H", h * 1e3, fluid)
                  for h in H]) / 1e3
    ax.plot(s, T, color=color, lw=lw, zorder=zorder)
    if label:
        # Put the label on the last point that is actually INSIDE the frame.
        # Anchoring it at a fixed fraction of the curve puts it off-plot,
        # because the curve is deliberately drawn wider than the axes.
        x0, x1 = ax.get_xlim()
        y0, y1 = sorted(ax.get_ylim())
        inside = np.where((s > x0) & (s < x1) & (T > y0) & (T < y1))[0]
        if len(inside):
            k = inside[int(np.clip(label_frac, 0.0, 1.0) * (len(inside) - 1))]
            ax.annotate(label, (s[k], T[k]), fontsize=fontsize, color="0.45",
                        zorder=8, ha="right", va="bottom",
                        textcoords="offset points", xytext=(-3, 2),
                        bbox=MASK)
    return s, T


def ts_water(model, case, rank, hp, Tdict, axes=None, figsize=(11.8, 8.8)):
    """One panel per exchanger: working fluid and water on the same T-s axes.

    Four panels rather than one plot, because the four exchangers live in
    four different corners of the diagram and overlaying them all produces a
    figure in which no pinch can actually be read.
    """
    if axes is None:
        _, axes = plt.subplots(2, 2, figsize=figsize)
    ov = _water_overlay(model, case, rank, hp, Tdict)
    order = ["ORC evaporator", "ORC condenser",
             "HTHP condenser", "HTHP evaporator"]
    fluid_of = {"ORC evaporator": case.fluid, "ORC condenser": case.fluid,
                "HTHP condenser": case.refrig, "HTHP evaporator": case.refrig}
    tag = dict(zip(order, "abcd"))
    for name, ax in zip(order, np.ravel(axes)):
        d = ov[name]
        _dome(ax, fluid_of[name])
        wf_col = RED if d["hot"] else NAVY
        ax.plot(d["s"], d["T_wf"], color=wf_col, lw=1.9, zorder=4,
                label="working fluid")
        _draw_water(ax, d, "#0a9396", label="water (duty-mapped)")
        pts = list(zip(d["s"], d["T_wf"])) + list(zip(d["s"], d["T_w"]))
        _frame(ax, pts, padx=0.14, pady=0.22)
        ax.set_xlabel(r"specific entropy of the working fluid,"
                      "\n" r"$s$  /  kJ kg$^{-1}$ K$^{-1}$")
        ax.set_ylabel(r"temperature, $T$  /  $^{\circ}$C")
        ax.set_title(rf"({tag[name]}) {name}:  pinch $= "
                     rf"{d['pinch_K']:.2f}$ K", pad=8)
        ax.legend(loc="best", handlelength=1.9, labelspacing=0.3)
    return axes, ov


# ------------------------------------- ORC evaporator composite curves
def _composites(model, case, rank, Tdict):
    """(Qc, Tc, Qh, Th, Q_total) from the MODEL's own constructions.

    `_orc_cold_composite` and `_water_curve` are reused rather than
    reimplemented, so what is plotted is by construction the same pair of
    curves that `orc_pinch` minimizes over. A composite curve redrawn by the
    plotting layer is a second implementation of the physics, and the two
    drift apart exactly when it matters.
    """
    Qc, Tc = model._orc_cold_composite(rank, case.fluid)
    Qh, Th = model._water_curve(Tdict["T_3d"], Tdict["T_2d"], case.P, Qc[-1])
    return Qc, Tc, Qh, Th, Qc[-1]


def orc_composite(model, case, rank, Tdict, axes=None, show_hot_end=True,
                  figsize=(11.8, 4.7)):
    """ORC evaporator composite curves and the approach profile.

    The point of the figure is that the pinch is INTERIOR. It sits at the
    onset of boiling, where the working fluid's curve goes flat against a
    water stream that keeps gliding down, and neither end of the exchanger
    shows it. A hot-end approach rule -- the thing everyone writes first --
    is blind to it by construction.

    With `show_hot_end`, the cold composite that the hot-end rule alone would
    permit (T_3e = T_2d - DT_2D_3E) is overlaid dashed. It crosses the water
    curve, so it is not a conservative approximation of the right answer; it
    is a second-law violation, and the efficiency computed from it is
    unreachable at any size of exchanger.
    """
    if axes is None:
        _, axes = plt.subplots(1, 2, figsize=figsize)
    ax, ax2 = axes
    Qc, Tc, Qh, Th, Qtot = _composites(model, case, rank, Tdict)
    pc = 100.0 * np.asarray(Qc) / Qtot
    ph = 100.0 * np.asarray(Qh) / Qtot

    # --- approach profile and the pinch location, on a common Q grid
    q = np.linspace(0.0, 100.0, 4000)
    Tw = np.interp(q, ph, np.asarray(Th) - 273.15)
    Tf = np.interp(q, pc, np.asarray(Tc) - 273.15)
    dT = Tw - Tf
    i = int(np.argmin(dT))
    q_p, dT_p = q[i], dT[i]

    # --- onset of boiling, read off the composite itself.
    #
    # NOT (h_bub - h_10e)/Qtot. That is the main stream's preheat alone, and
    # it lands 5 points of duty too low, because the composite also carries
    # the REHEAT stream (y of the flow, 5e->6e) whose whole duty lies BELOW
    # the bubble temperature and is therefore already banked by the time the
    # main stream starts boiling. Locating the break on the model's own
    # curve avoids re-deriving the assembly and getting it wrong.
    i_b = int(np.searchsorted(np.asarray(Tc), rank["T_3e"] - 1e-9))
    i_b = min(i_b, len(Qc) - 1)
    q_bub = 100.0 * Qc[i_b] / Qtot

    ax.plot(ph, np.asarray(Th) - 273.15, color=RED, lw=1.9,
            label=r"water (hot), $T_\mathrm{2d}\to T_\mathrm{3d}$")
    ax.plot(pc, np.asarray(Tc) - 273.15, color=NAVY, lw=1.9,
            label="ORC working fluid (cold)")

    cross_pct = float("nan")
    if show_hot_end:
        T_he = rank["T_3e_hot_end"]
        rk = model.double_stage_rankine(case.fluid, rank["T_1e"], T_he,
                                        case.eta_turb_s, case.eta_pump_s)
        Qc2, Tc2 = model._orc_cold_composite(rk, case.fluid)
        p2 = 100.0 * np.asarray(Qc2) / Qc2[-1]
        ax.plot(p2, np.asarray(Tc2) - 273.15,
                color="#8a8a8a", lw=1.3, ls=(0, (5, 2)), zorder=3,
                label=r"hot-end rule alone (infeasible)")
        # how much of the exchanger would actually run BACKWARDS
        cross_pct = 100.0 * float(np.mean(
            np.interp(q, p2, np.asarray(Tc2) - 273.15) > Tw))

    ax.axvline(q_bub, color="0.75", lw=0.9, ls=(0, (1, 2.5)), zorder=1)
    ax.annotate("onset of boiling", (q_bub, ax.get_ylim()[0]),
                textcoords="offset points", xytext=(4, 14), rotation=90,
                fontsize=8.2, color="0.35")
    ax.annotate("", xy=(q_p, Tw[i]), xytext=(q_p, Tf[i]),
                arrowprops=dict(arrowstyle="<->", color="#1a1a1a", lw=1.1))
    ax.annotate(rf"pinch $= {dT_p:.2f}$ K", (q_p, 0.5 * (Tw[i] + Tf[i])),
                textcoords="offset points", xytext=(8, -3), fontsize=8.8)
    ax.set_xlabel("cumulative duty  /  % of evaporator load")
    ax.set_ylabel(r"temperature, $T$  /  $^{\circ}$C")
    ax.set_title("(a) ORC evaporator composite curves", pad=8)
    ax.legend(loc="lower right", handlelength=1.9, labelspacing=0.35)

    ax2.plot(q, dT, color="#2f6f4f", lw=1.9)
    ax2.axhline(case.DT_pinch_ORC, color=RED, lw=1.0, ls=(0, (4, 2)),
                label=rf"$\Delta T_\mathrm{{pinch,ORC}} = "
                      rf"{case.DT_pinch_ORC:.0f}$ K")
    ax2.axvline(q_bub, color="0.75", lw=0.9, ls=(0, (1, 2.5)), zorder=1)
    ax2.plot([q_p], [dT_p], "o", ms=5.0, mfc="white", mec="#1a1a1a", mew=1.4,
             zorder=5)
    ax2.annotate(rf"${dT_p:.2f}$ K at ${q_p:.1f}\,\%$", (q_p, dT_p),
                 textcoords="offset points", xytext=(9, 4), fontsize=8.8)
    ax2.set_xlabel("cumulative duty  /  % of evaporator load")
    ax2.set_ylabel(r"approach, $T_\mathrm{water} - T_\mathrm{ORC}$  /  K")
    ax2.set_title("(b) the approach is worst in the interior", pad=8)
    ax2.set_ylim(0.0, max(dT.max() * 1.12, case.DT_pinch_ORC * 1.6))
    ax2.legend(loc="upper right", handlelength=1.9)
    return axes, dict(pinch_K=dT_p, pinch_at_pct=q_p, boiling_at_pct=q_bub,
                      Q_total_kJ_per_kg=Qtot,
                      hot_end_reversed_pct=cross_pct,
                      T_3e_C=rank["T_3e"] - 273.15,
                      T_3e_hot_end_C=rank["T_3e_hot_end"] - 273.15,
                      T_2d_C=Tdict["T_2d"] - 273.15)


# ------------------------------------------------------------- the table
def state_table(rank, hp):
    """State points of both cycles as one DataFrame.

    NOTE h is already kJ/kg in the state dictionaries while s is J/kg/K.
    The mixed units are the model's, not a bug here -- do not rescale h.
    """
    import pandas as pd
    rows = []
    for d, cyc, keys in ((rank, "ORC", [f"{i}e" for i in range(1, 11)]),
                         (hp, "HTHP", [f"{i}h" for i in range(1, 14)])):
        for k in keys:
            rows.append(dict(cycle=cyc, point=k,
                             T_C=d["T_" + k] - 273.15,
                             p_bar=d["p_" + k] / 1e5,
                             h_kJkg=d["h_" + k],
                             s_kJkgK=d["s_" + k] / 1e3))
    return pd.DataFrame(rows).set_index(["cycle", "point"]).round(
        {"T_C": 2, "p_bar": 4, "h_kJkg": 2, "s_kJkgK": 4})


def cycle_summary(rank, hp, res=None):
    """The scalar figures that go in the caption."""
    import pandas as pd
    d = {
        "eta_ORC": rank["rank_eff"],
        "COP": hp["hp_cop"],
        "y, turbine bleed fraction": rank["y_frac"],
        "x_8h, flash quality": hp["x_8h"],
        "eps_R1, IHX-1 effectiveness": hp["epsilon_IHX_1"],
        "eps_R2, IHX-2 effectiveness": hp["epsilon_IHX_2"],
        "ORC evaporator pinch, K": rank["pinch"],
        "ORC T_3e, C": rank["T_3e"] - 273.15,
        "p_evap ORC, bar": rank["p_evape"] / 1e5,
        "p_int  ORC, bar": rank["p_inte"] / 1e5,
        "p_cond ORC, bar": rank["p_conde"] / 1e5,
        "p_evap HTHP, bar": hp["p_evaph"] / 1e5,
        "p_int  HTHP, bar": hp["p_inth"] / 1e5,
        "p_cond HTHP, bar": hp["p_condh"] / 1e5,
    }
    if res is not None:
        for key, lab, scale in (
            ("T_2d_realized", "T_2d realized, C", None),
            ("N_wells", "N wells", 1.0),
            ("m1_ch", "m_dot per tube, charge, kg/s", 1.0),
            ("m1_dc", "m_dot per tube, discharge, kg/s", 1.0),
            ("Q_charge_kJ", "Q charge, GJ", 1e-6),
            ("Q_discharge_kJ", "Q discharge, GJ", 1e-6),
            ("W_in_kJ", "W in, GJ", 1e-6),
            ("W_out_kJ", "W out, GJ", 1e-6),
            ("eta_RTE", "eta_RTE", 1.0),
        ):
            if key in res:
                v = res[key]
                d[lab] = (v - 273.15) if scale is None else v * scale
    return pd.DataFrame({"value": d}).round(5)


In [ ]:
import importlib, boreas_plots as BP
importlib.reload(BP)
BP.use_paper_style()
print('serif resolved to:', BP.active_serif()[0])

ref = REF

# THE CYCLES ARE DRAWN AT THE OUTLET THE GRISSOM FIELD ACTUALLY DELIVERS.
# cycle_state_points' own T_2d is provisional -- the value the outlet WOULD
# take if the top layer stayed fully molten -- and at the reference case the
# field misses it by nearly 30 K. Drawing the T-s diagram at the provisional
# value while the tables report the realized one is how an earlier version
# ended up with a figure and a state-point table that were not the same case.
import pickle, os
if os.path.exists('rec_field.pkl'):
    REC = pickle.load(open('rec_field.pkl', 'rb'))['rec']
else:
    with G.field_ground(B, MD, RG, TVD):          # ~1 min
        REC = B.simulate_css_corrected(REF, record=True)
    pickle.dump(dict(rec=REC), open('rec_field.pkl', 'wb'))

_prov = B.cycle_state_points(ref)[2]['T_2d'] - 273.15
rank, hp, Tdict = B.cycle_state_points(ref, T_2d=REC['T_2d_realized'])
# ALL EIGHT WATER STATES, in the Fig. 1 numbering: 1 is the FIELD OUTLET
# and 4 the FIELD INLET in both half-cycles. The coils 3c -> 4c and
# 1d -> 2d are placeholders at zero, so T_4c = T_3c and T_2d = T_1d; they
# are printed anyway so the placeholders are visible in the bookkeeping
# rather than taken on trust.
print('water loop at the reference case, 1 = field outlet, '
      '4 = field inlet:')
for _half, _sts in (('charge   ', ('1c', '2c', '3c', '4c')),
                    ('discharge', ('1d', '2d', '3d', '4d'))):
    print('  ' + _half + '  ' + '  '.join(
        f'{_s} {Tdict["T_" + _s] - 273.15:7.2f} C' for _s in _sts))
print(f'  line losses: charge {REF.line_ch_dT:.1f} K, '
      f'discharge {REF.line_dc_dT:.1f} K   (placeholders)\n')

print(f'T_2d provisional (top layer fully molten)   {_prov:7.2f} C')
print(f"T_2d realized on the Grissom field          "
      f"{REC['T_2d_realized'] - 273.15:7.2f} C"
      f"   ({REC['T_2d_realized'] - 273.15 - _prov:+.1f} K)")
print('every diagram and state-point table below is at the realized value.')

### 10.1 The two cycles side by side

In [ ]:
fig, _ = BP.ts_both(ref, rank, hp)
fig.savefig('BOREAS_Ts_cycles.pdf')
fig.show()

### 10.2 Both on one axes, with the water levels

This is the coupling diagram. The two cycles never exchange working fluid ---
they exchange *water* --- so the horizontal dotted lines are the only thing
that makes the pair a single machine rather than two unrelated plots. Four
levels are shown: the sink, the geothermal source feeding the HTHP evaporator,
the HTHP condenser inlet $T_\mathrm{2c}$ (the charge loop), and
$T_\mathrm{2d}$, the store outlet --- the one line the borehole model sets.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8.0, 5.8))
BP.ts_combined(ref, rank, hp, Tdict, ax=ax)
fig.tight_layout()
fig.savefig('BOREAS_Ts_combined.pdf')
fig.show()

### 10.2b The water on the $T$--$s$ diagram, with every pinch marked

The horizontal lines above are *inlet temperatures*. They say where the water
starts, not what it does, and a pinch cannot be read off them. The water
actually glides --- $T_\mathrm{2d} \to T_\mathrm{3d}$ through the ORC
evaporator, $T_\mathrm{2c} \to T_\mathrm{3c}$ through the HTHP condenser ---
and it is the glide against a *flat* boiling or condensing plateau that
creates the binding approach.

Water cannot be plotted on these axes directly: its entropy is on a scale with
nothing to do with cyclopentane's. What can be plotted is the water
**temperature at each point of the exchanger**, placed at the entropy the
working fluid has at that same point. The two streams are tied together by
cumulative duty, so at a given $Q$ the working fluid sits at $(s, T_\mathrm{wf})$
and the water at $T_\mathrm{w}$; plotting $(s, T_\mathrm{w})$ puts the water
line directly above (evaporator) or below (condenser) the working-fluid path,
and the **vertical gap at every abscissa is the real local approach**.

This is a duty mapping, not a thermodynamic path. The water line is not a
process in cyclopentane's coordinates and its abscissa means nothing for the
water itself. Stated plainly it is an honest and very readable construction;
left unstated it would be a category error, so the axis label says so.

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 6.2))
BP.ts_combined(ref, rank, hp, Tdict, ax=ax, water='glide', model=B)
fig.tight_layout()
fig.savefig('BOREAS_Ts_combined_water.pdf')
fig.show()

Two things are now visible that the level lines hid.

The ORC-evaporator water (black) descends across the diagram and **touches the
working fluid at the bubble-point corner** --- the pinch is there, not at
either end. And the charge water (teal, $T_\mathrm{2c} \to T_\mathrm{3c}$) runs
nearly parallel to it, offset by the storage approach: that offset is the
round trip through the PCM, and it is the only part of this figure the
borehole model can move.

### 10.2c One panel per exchanger

Four exchangers in four corners of the diagram; overlaying them all makes a
figure in which no individual pinch can be read. Each panel below carries the
dome, the working-fluid path, the duty-mapped water line and the pinch arrow.
The recovered pinches should reproduce the specification
($\Delta T_\mathrm{pinch,ORC}$, $\Delta T_\mathrm{E,sink}$,
$\Delta T_\mathrm{2H,3C}$, $\Delta T_\mathrm{pinch,HPE}$) --- if one of them
does not, the overlay and the model disagree and the overlay is wrong.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.8, 8.8))
_, ov = BP.ts_water(B, ref, rank, hp, Tdict, axes=axes)
fig.tight_layout()
fig.savefig('BOREAS_Ts_water.pdf')

print(f"{'exchanger':20s}{'pinch, K':>10}{'specified':>12}")
spec = {'ORC evaporator': ref.DT_pinch_ORC,
        'ORC condenser': ref.DT_E_sink,
        'HTHP condenser': ref.DT_2H_3C,
        'HTHP evaporator': ref.DT_pinch_HPE}
for k, v in ov.items():
    print(f"{k:20s}{v['pinch_K']:>10.3f}{spec[k]:>12.3f}")
fig.show()

### 10.3 ORC evaporator composite curves: the pinch is *interior*

This is the figure that explains the gap. The water glides down from
$T_\mathrm{2d}$ to $T_\mathrm{3d}$ as a nearly straight sensible line; the
working fluid preheats, then goes **flat** at $T_\mathrm{3e}$ for the rest of
the exchanger. The smallest approach therefore occurs neither at the hot end
nor at the cold end but at the corner where boiling starts, and panel (b)
makes the V unmistakable.

The grey dashed curve is the cold composite that the hot-end rule *alone*
would permit, $T_\mathrm{3e} = T_\mathrm{2d} - \Delta T_\mathrm{2D,3E}$. It
does not merely violate the 5 K pinch --- it sits **above** the water over
most of the duty, so heat would have to flow from the cold stream to the hot
one. That is not a slightly optimistic design, it is a second-law violation,
and the efficiency computed from it is unreachable at any exchanger size. The
printed `hot_end_reversed_pct` is the fraction of the load that runs
backwards.

The curves here are the model's own: `orc_composite` calls
`_orc_cold_composite` and `_water_curve` rather than rebuilding them, so what
is plotted is by construction what `orc_pinch` minimizes over. The returned
`pinch_K` should agree with `rank['pinch']` to rounding --- that agreement is
the figure's self-check, and it is worth reading rather than trusting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.7))
_, info = BP.orc_composite(B, ref, rank, Tdict, axes=axes)
fig.tight_layout()
fig.savefig('BOREAS_ORC_composite.pdf')

for k, v in info.items():
    print(f'  {k:24s} {v:10.4f}')
print(f"\n  model's own pinch        {rank['pinch']:10.4f}   <- must agree")

Note where the onset of boiling lands on the duty axis. It is **not**
$(h_\mathrm{bub} - h_\mathrm{10e})/Q_\mathrm{tot}$: that is the main stream's
preheat alone and lands about five points of duty too low, because the
composite also carries the reheat stream ($y$ of the flow, $5\mathrm{e} \to
6\mathrm{e}$), whose entire duty lies *below* the bubble temperature and is
therefore already banked by the time the main stream starts to boil. The break
is read off the assembled composite instead of re-derived --- re-deriving it
is how the marker ends up a few points away from the pinch it is supposed to
coincide with.

### 10.3b Look at the gap between $T_\mathrm{2d}$ and $3\mathrm{e}$

In the reference case the store delivers water at $T_\mathrm{2d}$ but the ORC
evaporates well below it. Which constraint is binding is reported by the model
itself, and it is not always the one you would assume: if the hot-end rule
$T_\mathrm{3e} = T_\mathrm{2d} - \Delta T_\mathrm{2D,3E}$ bound, the approach
would be 12 K. Run the cell and read `T_3e_binding`.

In [ ]:
T_2d = Tdict['T_2d'] - 273.15
print(f"water from the store, T_2d        {T_2d:8.2f} C")
print(f"hot-end rule would give T_3e      {rank['T_3e_hot_end'] - 273.15:8.2f} C"
      f"   (approach {T_2d - (rank['T_3e_hot_end'] - 273.15):.1f} K)")
print(f"realized T_3e                     {rank['T_3e'] - 273.15:8.2f} C"
      f"   (approach {T_2d - (rank['T_3e'] - 273.15):.1f} K)")
print(f"binding constraint                {rank['T_3e_binding']}")
print()
T_carnot = 1.0 - (rank['T_1e'] / rank['T_3e'])
print(f"Carnot between T_1e and T_3e      {T_carnot:8.4f}")
print(f"eta_ORC                           {rank['rank_eff']:8.4f}"
      f"   ({rank['rank_eff'] / T_carnot * 100:.0f} % of Carnot)")

### 10.4 State points and the scalar summary

In [ ]:
st = BP.state_table(rank, hp)
display(st)
st.to_csv('BOREAS_state_points.csv')

# h is kJ/kg in the model's dictionaries while s is J/kg/K. The mixed units
# are the model's; state_table converts s and leaves h alone. Do not "fix"
# this by dividing h -- that was a real error earlier in this work.
# to_latex routes through the Styler in pandas 2.x and wants
# jinja2 >= 3.1.2. Degrade instead of crashing the notebook on a stack that
# has an older one.
try:
    print(st.to_latex(float_format='%.3f'))
except ImportError as e:
    print(f'(to_latex unavailable: {e})\n')
    print(st.to_string())

In [ ]:
display(BP.cycle_summary(rank, hp))

### 10.5 What actually moves the cycles

Worth running before assuming otherwise. The geothermal source temperature
moves the **COP** --- it sets the heat-pump lift --- and *nothing on the ORC
side*. $T_\mathrm{2d}$ is fixed by the cascade specification
($\Delta T_\mathrm{3C,2C}$ and the melting temperature), not by the source, so
$T_\mathrm{3e}$ and $\eta_\mathrm{ORC}$ do not budge. The two cycles are
coupled to the store but decoupled from each other, which is why the DOE found
$\eta_\mathrm{RTE}$ below ground to be almost entirely pumping.

In [ ]:
alt = REF.with_(grad_K_per_km=56.5, T_source_C=70.0)
rank_a, hp_a, T_a = B.cycle_state_points(alt)
print(f"T_2d   {Tdict['T_2d'] - 273.15:7.2f} ->{T_a['T_2d'] - 273.15:7.2f} C")
print(f"T_3e   {rank['T_3e'] - 273.15:7.2f} ->{rank_a['T_3e'] - 273.15:7.2f} C")
print(f"etaORC {rank['rank_eff']:7.4f} ->{rank_a['rank_eff']:7.4f}")
print(f"COP    {hp['hp_cop']:7.4f} ->{hp_a['hp_cop']:7.4f}")

---
## 11. Thermal behavior along the well

The figures above are about the surface machines. These are about the store
itself, and they need a RECORDED march:
`simulate_css_corrected(..., record=True)` keeps the fluid and PCM
temperature, the melted area, $U_i$ and $q'$ at every time level, at the
cost of one extra cycle.

### The coordinate, which is the thing to get right

**The solver's `z` is not depth.** Quoting `formation_profile`: on charge
the water enters the fin tubes at the *bottom* --- it arrives there down the
insulated downcomer --- so `z = 0` is the bottom of the well and depth
decreases with `z`. Plotting against the raw `z` and calling it depth draws
the well upside down, which is exactly what an earlier version of these
figures did. `depth_axis()` takes the depth array from the model itself so
there is one definition.

Both half-cycle histories come back in the **charge** frame, index 0 at the
bottom, even though the discharge march is run mirrored. Reading the fluid
boundary condition does not settle that, because the inlet sits at opposite
ends in the two half-cycles. The test that does: PCM cannot be above its own
melting point while it is freezing, so at the end of discharge
`max(T_pcm - T_m)` must be $\le 0$ when the history is read against the
cascade **forward**, and the cell prints both readings rather than asserting
one.

Read the two numbers for what they are. The forward reading is the one that
has to pass, and it does, at `+0.00` K. The reversed reading is *evidence*
only when it fails — on the idealized vertical geometry it came out at
`+26.9` K, a flagrant violation. On the Grissom S-curve it comes out
negative too, because the cascade and the realized PCM temperatures both
fall towards the top and the mirrored comparison happens not to break. **A
reversed reading that passes is not a second confirmation**; it means this
particular test has stopped discriminating for this geometry, and the
forward reading is carrying the check alone.

In [ ]:
# THE SAME RECORDED MARCH as section 10, run inside field_ground: every
# profile below is on the measured Grissom field, with the superposition
# resistance R_g,eff(MD) and the rock temperature on TRUE VERTICAL DEPTH.
# Run it outside and the rock at the bottom of the store comes out 15 K too
# hot, because the S-curve reaches only 909 m TVD at 1250 m MD.
rec = REC
ref_well = REF
A_av = BP.pcm_capacities_A(ref_well)
# Depth from the TRAJECTORY, not from the solver coordinate: on an
# S-curve 1250 m of hole reaches about 909 m of depth. ONE definition --
# `depth_axis` returns true vertical depth whenever the field patch is
# active, so the figures, the animation and this cell cannot disagree.
# An earlier version interpolated it here by hand and clamped at the
# last station, reporting 891 m.
import numpy as _np
with G.field_ground(B, MD, RG, TVD):
    dseg, dnode = BP.depth_axis(B, REF)
print(f'depth runs {dseg[0]:.0f} m (index 0, bottom) -> '
      f'{dseg[-1]:.0f} m (top), against {REF.L_well:.0f} m of hole')

# the orientation check, printed rather than trusted
Tm = rec['T_m_seg'] - 273.15
Tp = rec['discharge']['history']['T_pcm'][-1] - 273.15
print(f'end of discharge, max(T_pcm - T_m): forward {(Tp - Tm).max():+.2f} K, '
      f'reversed {(Tp - Tm[::-1]).max():+.2f} K  <- forward must be <= 0')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8.8, 5.8), sharex=True)
BP.well_profiles(rec, ref_well, axes=ax, vertical=True, staircase=True,
                 depth_seg=dseg, depth_node=dnode, offset=-273.15, n_times=6,
                 field='T_fluid', overlay='T_pcm',
                 overlay_label=r'$T_\mathrm{pcm}$',
                 ylabel=r'temperature / $^\circ$C')
fig.tight_layout(); fig.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8.8, 5.8), sharex=True)
BP.well_profiles(rec, ref_well, axes=ax, vertical=True, field='A_melt',
                 depth_seg=dseg, depth_node=dnode, scale=1.0 / A_av,
                 ylabel=r'melted fraction, $A_\mathrm{melt}/A_\mathrm{av}$')
fig.tight_layout(); fig.show()

fig, ax = plt.subplots(1, 2, figsize=(8.8, 5.8), sharex=True)
BP.well_profiles(rec, ref_well, axes=ax, vertical=True, field='U_i',
                 depth_seg=dseg, depth_node=dnode,
                 ylabel=r'$U_i$ / W m$^{-2}$ K$^{-1}$')
fig.tight_layout(); fig.show()

fig, ax = plt.subplots(1, 2, figsize=(11.6, 4.2))
BP.well_integrals(rec, ref_well, axes=ax)
fig.tight_layout(); fig.show()

### 11.1 Animation of the full cycle

Charge then discharge, three panels, with the frames resampled onto a
**uniform** time grid --- the march records on a roughly logarithmic one,
which would make constant-rate playback appear to accelerate.

In [ ]:
from IPython.display import Video
BP.animate_well(rec, ref_well, 'BOREAS_cycle.mp4', fps=12, frames_per_half=60)
Video('BOREAS_cycle.mp4', embed=True, width=760)

In [ ]:
mine = REF.with_(
    n_ft=5,                    # more, smaller tubes -> more PCM, more pumping
    grad_K_per_km=56.5,        # central Wilmington, 1949 survey
)
for lvl, m in B.validate_case(mine, verbose=False):
    print(f'  [{lvl}] {m}')

r = B.simulate_css_corrected(mine)
B.css_report(mine, r)
B.kpi_report(mine, r)

---
### What this model will not tell you

- **Anything about the surface machines.** They are frozen here. Round-trip
  efficiency is set by the heat-pump *lift* — $(T_h - T_0)/(T_h - T_{src})$ —
  and so by the geothermal source temperature, not by anything below ground.
- **A multi-year loss curve.** `t_operation_yr` evaluates the ground
  resistance at one age. The decay from year 1 to year 30 comes from running
  it several times, not from an outer loop.
- **An optimum.** This is a model, not a search.

## 12. The paper figures

Every figure of the ECM paper, drawn by the generators the pack ships and
in the paper's own style: serif (Times), hairline axes, no top or right
spine, one panel per file. With sections 12.2 and 12.3 this covers all 39
result files — figures, state tables and Video S1.

Where the numbers come from depends on `RUN_MODE` in section 2.3.

- **`published`** — the results of about eighty marches are unpacked from
  this notebook (57 kB, gzipped) and the figures are redrawn from them.
  Seconds, and they match the paper exactly. Changing an input in section 2
  does **not** move them: they are the published figures.
- **`recompute`** — section 14 re-marches everything from sections 2.1 and
  2.2 first, and then these figures *are* yours. Hours, and resumable.

Two of the figures below need their own marches either way, because the
records they draw from are also what the macro pack reads: section 12.2
marches R3 and R4 and keeps the records beside the notebook, so a second
run is instant.


### 12.0 The run results

Everything below — the figures, the tables and the macro pack — is drawn
from the results of about eighty marches. In `published` mode they are
unpacked here from the notebook itself (57 kB, gzipped), so the published
figures and numbers come out with no network and no waiting. In
`recompute` mode this cell unpacks only the contract and the stated
reasons, and section 14 produces the results from cells 2.1 and 2.2.

In [ ]:
import base64, gzip, json, pathlib

# The PUBLISHED run results: the ladder, every sweep, the sliding bracket,
# the gradient x length map, the isolated well and the convergence runs --
# about eighty marches, which is what `recompute` mode redoes. The contract,
# the paper's macro list, the stated reasons and the documented type changes
# travel with them.
#
# In 'recompute' mode these are NOT unpacked: section 14 writes its own.
_ART = (
    "..."
)

_REQ = """AdiabaticDeviation
AdiabaticEtaORC
AdiabaticEtaRTE
AdiabaticEtaStorage
AdiabaticPumping
AdiabaticTThreee
AdiabaticTTwod
AdiabaticUAPerMWe
AdiabaticWellsPerMWe
AllocYieldKgs
ConvEpSpread
ConvLossSpread
ConvNsBaseEp
ConvNsBaseLoss
ConvNsFineEp
ConvNsFineLoss
ConvNtFineEp
ConvNtFineLoss
CutoffFirstHundred
CutoffGradThreshold
CutoffGradThresholdFit
CutoffMarginalMin
CutoffMarginalSlopePerK
CutoffThreshold
DsStation
DtLast
GradCutoffA
GradLossA
GradLossB
GradLowA
GradLowB
GradRTEA
GradRTEB
GradRecrossA
GradRef
GrissomExtentLong
GrissomExtentShort
GrissomRminOverRmax
HPSuctionBar
HPSuctionVolIsland
HPSuctionVolPerMWe
IntegratorCheck
IsoLossKWhDay
IsoLossWm
IsoMeanDT
LadderAdiabaticEp
LadderAdiabaticLoss
LadderAdiabaticN
LadderAdiabaticRTE
LadderAdiabaticWellsPerMWe
LadderRFourEp
LadderRFourIEp
LadderRFourIIslandMWe
LadderRFourILoss
LadderRFourIN
LadderRFourIRTE
LadderRFourIWellsPerMWe
LadderRFourLoss
LadderRFourLossPct
LadderRFourN
LadderRFourRTE
LadderRFourWellsPerMWe
LadderROneEp
LadderROneLoss
LadderROneLossPct
LadderROneN
LadderROneRTE
LadderROneWellsPerMWe
LadderRThreeEp
LadderRThreeLoss
LadderRThreeLossPct
LadderRThreeN
LadderRThreeRTE
LadderRThreeWellsPerMWe
LadderRTwoEp
LadderRTwoLoss
LadderRTwoLossPct
LadderRTwoN
LadderRTwoRTE
LadderRTwoWellsPerMWe
LadderRZeroIEp
LadderRZeroILoss
LadderRZeroILossPct
LadderRZeroIN
LadderRZeroIRTE
LadderRZeroIWellsPerMWe
LadderRZeroVEp
LadderRZeroVLoss
LadderRZeroVLossPct
LadderRZeroVN
LadderRZeroVRTE
LadderRZeroVWellsPerMWe
LadderSlotStep
LadderTrajectoryStep
ORCExhaustBar
ORCExhaustVolPerMWe
ParHmEprimeFactor
ParHmFiveHundredEprime
ParHmFiveHundredLoss
ParHmFiveHundredWells
ParHmRangeFactor
ParHmThreeEightyWells
ParHmTtwodHigh
ParHmTtwodLow
ParHmTwoHundredEprime
ParHmTwoHundredLoss
ParHmTwoHundredWells
ParHmTwoSixtyWells
ParRefAtCutoffEprime
ParRefAtCutoffIslandMWe
ParRefAtCutoffL
ParRefAtCutoffRTE
ParRefAtCutoffWells
ParRefAtRefEprime
ParRefAtRefIslandMWe
ParRefAtRefRTE
ParRefAtRefWells
PenetrationDepth
ProfEpsCh
ProfEpsChRThree
ProfEpsCycled
ProfEpsCycledRThree
ProfEpsDc
ProfQlossMeanRFour
ProfQlossMeanRThree
ProfQlossRatio
ProfRateChEnd
ProfRateChMean
ProfRateChStart
ProfRateDcMean
ProfRateDcStart
ProfSubcoolMax
ProfSubcoolMean
ProfTlimORC
ProfToutDcEnd
ProfTtwocHi
ProfTtwocLo
ProfTtwodHi
ProfTtwodHiRThree
ProfTtwodLo
ProfTtwodLoRThree
ProfUiHi
ProfUiLo
ProfUiMean
ProfUiMeanRThree
ProfUiSurface
RefCOP
RefDTsourceGlide
RefFieldMWePlant
RefFieldMWh
RefFourDeviation
RefFourEnergyBelowMean
RefFourEtaORC
RefFourEtaORCSlidingGain
RefFourEtaRTE
RefFourEtaRTEFixedStrict
RefFourEtaRTESliding
RefFourEtaStorage
RefFourLeakage
RefFourPinchAt
RefFourPumping
RefFourTThreee
RefFourTTwod
RefFourUAPerMWe
RefFourWellsPerMWe
RefHm
RefIslandMWe
RefLmd
RefNlay
RefNt
RefNwells
RefPumpEta
RefSinkDp
RefSinkGlide
RefSourceDp
RefSourceFlowCharge
RefSourceFlowDaily
RefSourceFlowPerMWe
RefTch
RefTdc
RefTfBottom
RefTfTop
RefThreeDeviation
RefThreeEtaORC
RefThreeEtaRTE
RefThreeEtaStorage
RefThreePumping
RefThreeTThreee
RefThreeTTwod
RefThreeWellsPerMWe
RefTm
RefTmBottom
RefTsink
RefTsource
RefTvd
RefXhp
RefXorc
SeawaterSourceEtaRTE
SensKopThreeHundredLoss
SensMudEp
SensMudLoss
SensTopOneLoss
SensTopThirtyLoss
SensWeightLoss
SinkFreeEtaRTE
SinkFreeTone
SinkGlideCostPct
SinkGlideOnlyEtaRTE
SinkGlideOpt
SinkGlideRange
SinkGlideSpreadPct
SinkPumpCostPct
SinkRealTone
SinkTotalCostPct
SourceCoverGrissom
SourceCoverLBU
SourceCoverOtherThree
SplitElevenThirteenEtaRTE
SplitElevenThirteenEtaStorage
SplitElevenThirteenMWe
SplitElevenThirteenMWh
SplitEnergyOpt
SplitEnergySpreadPct
SplitEtaOpt
SplitEtaSpreadPct
SplitFourteenTenEtaRTE
SplitFourteenTenEtaStorage
SplitFourteenTenMWe
SplitFourteenTenMWh
SplitNineFifteenEtaRTE
SplitNineFifteenEtaStorage
SplitNineFifteenMWe
SplitNineFifteenMWh
SplitPowerOpt
SplitPowerSpreadPct
SplitTenFourteenEtaRTE
SplitTenFourteenEtaStorage
SplitTenFourteenMWe
SplitTenFourteenMWh
SplitThirteenElevenEtaRTE
SplitThirteenElevenEtaStorage
SplitThirteenElevenMWe
SplitThirteenElevenMWh
SplitTwelveTwelveEtaRTE
SplitTwelveTwelveEtaStorage
SplitTwelveTwelveMWe
SplitTwelveTwelveMWh
TauLast
TauRatioLast
TmEnergyOpt
TmInfeasibleAt
TmOneTwentyEtaRTE
TmOneTwentyFiveEtaRTE
TmOpt
TsCOP
TsEtaORC
TsTthreee
TsTtwod
VideoFps
VideoFrames
WaterChTransit
WaterChVelocity
WaterDcTransit
WaterDcVelocity
WaterQuasiSteadyRatio
WaterQuasiSteadySwing
WaterTransitOverHalfCycle
"""

if RUN_MODE == 'published':
    for _n, _o in json.loads(gzip.decompress(base64.b64decode(_ART))).items():
        pathlib.Path(_n).write_text(json.dumps(_o, indent=1))
    print('unpacked', len(json.loads(gzip.decompress(base64.b64decode(_ART)))),
          'published artefacts')
else:
    # the contract and the paper's list are inputs, not results: they are
    # needed in both modes
    _keep = ('contract.json', 'contract225.json', 'reasons.json',
             'type_changes.json')
    for _n, _o in json.loads(gzip.decompress(base64.b64decode(_ART))).items():
        if _n in _keep:
            pathlib.Path(_n).write_text(json.dumps(_o, indent=1))
    print('recompute mode: only the contract and the reasons were unpacked;',
          'section 14 produces the results')
pathlib.Path('required225.txt').write_text(_REQ)


### 12.0b Is the type big enough on the printed page?

A figure drawn wider than it is printed is scaled down by LaTeX, and its
labels shrink with it. Measuring the font size inside the PDF does not show
this. `fontgate.py` multiplies every text span by (printed width / page
width) and reports what the reader actually sees; the cell after the
figures runs it.

In [ ]:
%%writefile fontgate.py
"""Is the text in each figure big enough ON THE PRINTED PAGE?

Handoff v2.22. The panels were drawn about twice the width they are
printed at, so LaTeX scaled them down and the labels went with them: 9.5 pt
type printed at about 5 pt against 12 pt body text. Measuring the font size
inside the PDF does not show this -- the number looks fine. What matters is

    printed size = span size x (printed width / page width)

so the check has to know the width the paper includes each figure at. That
is PRINTED_WIDTH below, from the handoff's table; keep it in step with the
\\includegraphics widths in the .tex.

    python3 fontgate.py <dir> [--min 7.0]
"""
import json, os, sys

LINEWIDTH_IN = 390.0 / 72.27          # elsarticle preprint, 12 pt: 5.4 in
HALF = 0.48 * LINEWIDTH_IN            # a half-width subfigure
PROFILE = 0.49 * LINEWIDTH_IN         # v2.23: the twelve profile panels
LOSS = 0.55 * LINEWIDTH_IN

PRINTED_WIDTH = {}
for _n in ('BOREAS_prof_Tfluid', 'BOREAS_prof_melt', 'BOREAS_prof_Ui'):
    for _s in ('_a', '_b'):
        PRINTED_WIDTH[_n + _s] = PROFILE
        PRINTED_WIDTH[_n + '_R3' + _s] = PROFILE
for _s in ('_a', '_b'):               # the integrals panels are unchanged
    PRINTED_WIDTH['BOREAS_prof_integrals' + _s] = HALF
    PRINTED_WIDTH['BOREAS_prof_integrals_R3' + _s] = HALF
for _n in ('BOREAS_tm_a', 'BOREAS_tm_b', 'BOREAS_split_a', 'BOREAS_split_b',
           'BOREAS_split_c', 'BOREAS_par_hm_a', 'BOREAS_par_hm_b',
           'BOREAS_grissom_Rg_a', 'BOREAS_grissom_Rg_b',
           'BOREAS_grissom_sweep_a', 'BOREAS_grissom_sweep_b',
           'BOREAS_grissom_sweep_c', 'BOREAS_grissom_geometry_a',
           'BOREAS_grissom_geometry_b', 'BOREAS_grissom_geometry_c',
           'BOREAS_grissom_geometry_d', 'BOREAS_ladder_a'):
    PRINTED_WIDTH[_n] = HALF
PRINTED_WIDTH['BOREAS_prof_loss_cmp'] = LOSS
PRINTED_WIDTH['BOREAS_Ts_combined_water'] = LINEWIDTH_IN
PRINTED_WIDTH['BOREAS_Ts_cycles'] = LINEWIDTH_IN
PRINTED_WIDTH['BOREAS_ladder_geometry'] = LINEWIDTH_IN


def measure(path, printed_in):
    """Printed sizes, with sub/superscripts separated from base text.

    A flat floor over every span cannot be met: mathtext sets a subscript at
    about 0.7 of its base, so $\eta_\mathrm{RTE}$ at 9 pt puts "RTE" on the
    page at 6.3. That is exactly what a subscript does in the body text too
    (12 pt body -> 8.4 pt subscript). The floor therefore applies to BASE
    text, and scripts are held to a lower floor and reported separately.

    A span is a script when it is smaller than 0.85 of the figure's dominant
    size -- dominant by span count, which is the tick labels in every figure
    here. Comparing within a line instead fails when a label is a lone
    subscript in its own line, which is how two "RTE" spans first slipped
    through as base text.
    """
    import collections
    import pymupdf
    import statistics
    doc = pymupdf.open(path)
    page = doc[0]
    page_in = page.rect.width / 72.0
    k = printed_in / page_in
    spans = []
    for blk in page.get_text('dict')['blocks']:
        for line in blk.get('lines', []):
            for span in line.get('spans', []):
                if span.get('text', '').strip():
                    spans.append(span['size'])
    doc.close()
    if not spans:
        return None
    dominant = collections.Counter(round(x, 2) for x in spans).most_common(1)[0][0]
    base = [x * k for x in spans if x >= 0.85 * dominant]
    script = [x * k for x in spans if x < 0.85 * dominant]
    return (min(base), statistics.median(base), min(script) if script else None,
            page_in, len(base), len(script))


def run(d, floor=7.0, script_floor=5.0, quiet=False):
    rows, bad = [], []
    for f in sorted(os.listdir(d)):
        if not f.endswith('.pdf'):
            continue
        stem = f[:-4]
        if stem not in PRINTED_WIDTH:
            print(f'  [skip] {stem}: no printed width recorded')
            continue
        m = measure(os.path.join(d, f), PRINTED_WIDTH[stem])
        if m is None:
            continue
        lo, med, lo_s, page_in, nb, ns = m
        rows.append((stem, PRINTED_WIDTH[stem] / page_in, lo, med, lo_s, nb, ns))
        if lo < floor:
            bad.append((stem, lo, 'base'))
        if lo_s is not None and lo_s < script_floor:
            bad.append((stem, lo_s, 'script'))
    rows.sort(key=lambda r: r[2])
    if not quiet:
        print(f'{"figure":34s} {"print/draw":>10s} {"min base":>9s} '
              f'{"median":>7s} {"min script":>11s}')
        for r in rows:
            flag = '  <- BELOW' if r[2] < floor else ''
            sc = f'{r[4]:11.1f}' if r[4] is not None else f'{"-":>11s}'
            print(f'{r[0]:34s} {r[1]:10.2f} {r[2]:9.1f} {r[3]:7.1f} {sc}{flag}')
    print(f'\n{len(rows)} figures measured as PRINTED. base floor '
          f'{floor:.1f} pt, script floor {script_floor:.1f} pt: {len(bad)} below')
    for b in bad:
        print(f'   {b[0]}: {b[1]:.1f} pt ({b[2]})')
    return rows, bad


if __name__ == '__main__':
    d = sys.argv[1] if len(sys.argv) > 1 else '.'
    floor = 7.0
    if '--min' in sys.argv:
        floor = float(sys.argv[sys.argv.index('--min') + 1])
    rows, bad = run(d, floor)
    json.dump([dict(figure=r[0], scale=r[1], min_base_pt=r[2],
                    median_pt=r[3], min_script_pt=r[4])
               for r in rows], open(os.path.join(d, 'fontgate.json'), 'w'),
              indent=1)
    sys.exit(1 if bad else 0)


# ------------------------------------------------------------- collisions
def overlaps(path, pad=2.0, clearance=0.0):
    """Pairs of text boxes that intersect, in points.

    Shrinking a figure to 48 % of its drawn width is what hid the old
    collisions: everything was small, so nothing touched. At the printed
    size the labels are where the reader sees them, so a mechanical check
    is worth having before the eye goes over 37 panels.
    """
    import pymupdf
    doc = pymupdf.open(path)
    page = doc[0]
    boxes = []
    for blk in page.get_text('dict')['blocks']:
        for line in blk.get('lines', []):
            for span in line.get('spans', []):
                t = span.get('text', '').strip()
                if t:
                    boxes.append((pymupdf.Rect(span['bbox']), t))
    doc.close()
    hits = []
    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            a, b = boxes[i][0], boxes[j][0]
            if clearance:
                # v2.25: two labels can sit 0.1 pt apart and never
                # intersect, which reads as one run of text on the page.
                # Growing each box by half the clearance turns "do they
                # overlap" into "are they far enough apart".
                a = pymupdf.Rect(a); b = pymupdf.Rect(b)
                a += (-clearance / 2, 0, clearance / 2, 0)
                b += (-clearance / 2, 0, clearance / 2, 0)
            inter = a & b
            # stacked lines of one label, and adjacent legend entries, touch
            # by a fraction of a point. A collision worth looking at overlaps
            # by at least `pad` in BOTH directions.
            if inter.is_valid and inter.width > pad and inter.height > pad:
                hits.append((boxes[i][1], boxes[j][1],
                             round(inter.width, 1), round(inter.height, 1)))
    return hits


def line_overlaps(path, pad=0.6, skip_fills=True):
    """Text boxes sitting on STROKED paths -- curves, arrows, leaders.

    Handoff v2.24: the kickoff label was on the band, which is right, but
    also on the S-curves, which is not, and a text-vs-text check cannot see
    that. Fills are skipped by default: a label on a shaded band overlaps
    its rectangle on purpose, and that is the whole point of putting it
    there.
    """
    import pymupdf
    doc = pymupdf.open(path)
    page = doc[0]
    texts = []
    for blk in page.get_text('dict')['blocks']:
        for line in blk.get('lines', []):
            for span in line.get('spans', []):
                t = span.get('text', '').strip()
                if t:
                    texts.append((pymupdf.Rect(span['bbox']), t))
    segs = []
    for d in page.get_drawings():
        if skip_fills and d.get('type') == 'f':
            continue
        for it in d['items']:
            if it[0] == 'l':
                segs.append((pymupdf.Rect(it[1], it[2]).normalize(), 'line'))
            elif it[0] == 'c':
                pts = it[1:]
                r = pymupdf.Rect(pts[0], pts[0])
                for p in pts[1:]:
                    r |= pymupdf.Rect(p, p)
                segs.append((r.normalize(), 'curve'))
            elif it[0] == 're':
                r = pymupdf.Rect(it[1]).normalize()
                for a, b in (((r.x0, r.y0), (r.x1, r.y0)),
                             ((r.x1, r.y0), (r.x1, r.y1)),
                             ((r.x1, r.y1), (r.x0, r.y1)),
                             ((r.x0, r.y1), (r.x0, r.y0))):
                    segs.append((pymupdf.Rect(a, b).normalize(), 'rect edge'))
    doc.close()
    hits = []
    for tr, txt in texts:
        for sr, kind in segs:
            sr = pymupdf.Rect(sr)
            if sr.width < 0.01:
                sr.x1 = sr.x0 + 0.01
            if sr.height < 0.01:
                sr.y1 = sr.y0 + 0.01
            inter = tr & sr
            if inter.is_valid and inter.width > pad and inter.height > pad:
                hits.append((txt, kind, round(inter.width, 1),
                             round(inter.height, 1)))
                break
    return hits


In [ ]:
%%writefile mkfigs_uniform.py
"""Every figure added since v2.9, rebuilt through BP.use_paper_style()

# v2.18 guard.  DATA defaults to the working directory, which still holds
# 45 K/km leftovers from before the reference moved -- running this file
# without BOREAS_FIGDATA pointed at the current run set would redraw the
# whole set at the old gradient, silently.  Check one known value against
# the pack before drawing anything.
def _guard():
    import json as _j
    pk=os.environ.get('BOREAS_PACK')
    if not pk or not os.path.exists(pk): 
        print('  [guard] no BOREAS_PACK given -- not checked'); return
    ref_rte=float(_j.load(open(pk))['RefFourEtaRTE'])
    got=float(_j.load(open(os.path.join(DATA,'tmfinal.json')))['130']['eta_RTE'])
    assert abs(got-ref_rte)<5e-5, (
        f'figure data is not the pack run set: tmfinal[130] eta_RTE={got} '
        f'but the pack says {ref_rte}.  Point BOREAS_FIGDATA at the current '
        f'run set.')
    print(f'  [guard] figure data matches the pack (eta_RTE {got})')
_guard().

The T-s and well-profile figures always went through the paper style, via
mkts.py and mkprofiles.py. Everything I added afterwards set its own
rcParams and so inherited matplotlib's default sans-serif -- which is why
the recent figures do not match. One style, one entry point, no local
rcParams.
"""
import json, os, sys
sys.path.insert(0, os.environ.get('BOREAS_SRC','.'))
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import boreas_plots as BP

# DATA and OUT are overridable so the notebook can run this unchanged:
# it fetches the sweep JSONs beside itself rather than re-marching them.
import os
DATA=os.environ.get('BOREAS_FIGDATA','/tmp/build')
OUT=os.environ.get('BOREAS_FIGOUT','/tmp/build/figs')
os.makedirs(OUT, exist_ok=True)
BP.use_paper_style()

# v2.18 guard.  DATA defaults to the working directory, which still holds
# 45 K/km leftovers from before the reference moved -- running this file
# without BOREAS_FIGDATA pointed at the current run set would redraw the
# whole set at the old gradient, silently.  Check one known value against
# the pack before drawing anything.
def _guard():
    import json as _j
    pk=os.environ.get('BOREAS_PACK')
    if not pk or not os.path.exists(pk): 
        print('  [guard] no BOREAS_PACK given -- not checked'); return
    ref_rte=float(_j.load(open(pk))['RefFourEtaRTE'])
    got=float(_j.load(open(os.path.join(DATA,'tmfinal.json')))['130']['eta_RTE'])
    assert abs(got-ref_rte)<5e-5, (
        f'figure data is not the pack run set: tmfinal[130] eta_RTE={got} '
        f'but the pack says {ref_rte}.  Point BOREAS_FIGDATA at the current '
        f'run set.')
    print(f'  [guard] figure data matches the pack (eta_RTE {got})')
_guard()
# v2.22: drawn at the width the paper PRINTS it at, so LaTeX does
# not scale the figure and the type sizes in use_paper_style() are
# the sizes the reader sees. The aspect ratio is the old one.
PANEL = BP.figsize('half', 2.8 / 3.5)
L=lambda p: json.load(open(os.path.join(DATA,p)))

def fig():
    f,a=plt.subplots(figsize=PANEL); return f,a
def save(f,n):
    f.tight_layout()
    f.savefig(os.path.join(OUT,n+'.pdf')); f.savefig(os.path.join(OUT,n+'.png'), dpi=130)
    plt.close(f); print(f'  {n}')

tmf=L('tmfinal.json'); spl=L('split211.json'); hm=L('hm210.json'); lad=L('ladder210.json')
lsw=L('lensweep.json')

# ---- cascade top -------------------------------------------------------
ks=sorted(tmf,key=float); x=[float(k) for k in ks]
f,a=fig(); a.plot(x,[tmf[k]['eta_RTE'] for k in ks],'o-',color='0.15')
a.plot([130],[tmf['130']['eta_RTE']],'D',ms=7,color='0.15',mfc='white',
       mew=1.2,label='reference')
a.set_xlabel(r'$T_\mathrm{m}$  [$^\circ$C]')
a.set_ylabel(r'$\eta_\mathrm{RTE}$')
a.legend(loc='lower center'); save(f,'BOREAS_tm_a')

f,a=fig(); a.plot(x,[tmf[k]['island']*12 for k in ks],'s-',color='0.15')
# v2.17 drew this marker at a hardcoded 140.  The maximum is at 150 at the
# reference gradient, and \TmEnergyOpt had the same literal in it.  Taken
# from the data now, so the marker cannot disagree with the macro.
_ek=max(ks,key=lambda kk: tmf[kk]['island'])
a.plot([float(_ek)],[tmf[_ek]['island']*12],'D',ms=7,color='0.15',mfc='white',
       mew=1.2,label='energy maximum')
a.set_xlabel(r'$T_\mathrm{m}$  [$^\circ$C]')
a.set_ylabel(r'delivered  [MWh$_\mathrm{e}$/day]')
a.legend(loc='lower right'); save(f,'BOREAS_tm_b')

# ---- the day split -----------------------------------------------------
# v2.19: taken from the data, not written in. The old literal list broke
# the moment a sweep list in the input section changed, which is the
# opposite of what a single input section is for.
sk=sorted(spl, key=lambda k: float(k.split('x')[0]))
sx=[float(k.split('x')[0]) for k in sk]
f,a=fig(); a.plot(sx,[spl[k]['eta_RTE'] for k in sk],'o-',color='0.15')
a.plot([12],[spl['12x12']['eta_RTE']],'D',ms=7,color='0.15',mfc='white',
       mew=1.2,label='baseline')
a.set_xlabel(r'$t_\mathrm{ch}$  [h],   $t_\mathrm{dc}=24-t_\mathrm{ch}$')
a.set_ylabel(r'$\eta_\mathrm{RTE}$')
a.legend(loc='lower left'); save(f,'BOREAS_split_a')

# TWO PANELS instead of a twin axis: MWh/day and MW_e differ by a factor
# of twelve, so a shared frame put the rating curve on top of the energy
# label. Separate panels, same width.
f,a=fig(); a.plot(sx,[spl[k]['MWh'] for k in sk],'o-',color='0.15')
a.set_xlabel(r'$t_\mathrm{ch}$  [h]'); a.set_ylabel(r'delivered  [MWh$_\mathrm{e}$/day]')
save(f,'BOREAS_split_b')
f,a=fig(); a.plot(sx,[spl[k]['island'] for k in sk],'s-',color='0.15')
a.set_xlabel(r'$t_\mathrm{ch}$  [h]'); a.set_ylabel(r'rating  [MW$_\mathrm{e}$]')
save(f,'BOREAS_split_c')

# ---- intermediate pressures -------------------------------------------

# ---- latent heat -------------------------------------------------------
hk=sorted(hm,key=float); hx=[float(h) for h in hk]
f,a=fig(); a.plot(hx,[hm[h]['wells_per_MWe'] for h in hk],'o-',color='0.15')
a.set_xlabel(r'$h_\mathrm{m}$  [kJ/kg]'); a.set_ylabel(r'wells per MW$_\mathrm{e}$')
save(f,'BOREAS_par_hm_a')
f,a=fig(); a.plot(hx,[hm[h]['e_prime'] for h in hk],'o-',color='0.15')
a.set_xlabel(r'$h_\mathrm{m}$  [kJ/kg]'); a.set_ylabel(r"$e^\prime$  [kWh/m]")
save(f,'BOREAS_par_hm_b')

# ---- sink glide --------------------------------------------------------

# ---- the ladder --------------------------------------------------------
O=[k for k in ('adiabatic','R0v','R0i','R1','R2','R3','R4','R4i')
   if k in lad and 'error' not in lad[k]]
LBL={'adiabatic':'adiab.','R0v':'R0v','R0i':'R0i','R1':'R1','R2':'R2',
     'R3':'R3','R4':'R4','R4i':'R4i'}
f,a=plt.subplots(figsize=BP.figsize('half', 2.8 / 4.6))
vals=[100*lad[k]['loss'] for k in O]
a.bar(range(len(O)),vals,color='0.72',edgecolor='0.25',linewidth=0.8,width=0.68)
# the adiabatic limit is EXACTLY zero, so its bar is invisible and the
# label hangs on nothing. Say so instead of leaving a gap.
for i,v in enumerate(vals):
    if v < 0.05:
        a.annotate('0', (i, 0), xytext=(0, 3), textcoords='offset points',
                   ha='center', va='bottom')
a.set_xticks(range(len(O))); a.set_xticklabels([LBL[k] for k in O])
a.set_ylabel(r'loss  [\%]' if plt.rcParams['text.usetex'] else 'loss  [%]')
save(f,'BOREAS_ladder_a')

# ---- the length sweep --------------------------------------------------
Ls=np.array(sorted(float(k) for k in lsw))
ep=np.array([lsw[f'{l:g}']['e_prime'] for l in Ls])
loss=np.array([lsw[f'{l:g}']['loss'] for l in Ls])
rte=np.array([lsw[f'{l:g}']['eta_RTE'] for l in Ls])
f,a=fig(); a.plot(Ls,100*loss,'o-',color='0.15')
a.set_xlabel(r'storage length  [m MD]'); a.set_ylabel('loss  [%]')
save(f,'BOREAS_grissom_sweep_a')
f,a=fig(); a.plot(Ls,ep,'o-',color='0.15')
a.set_xlabel(r'storage length  [m MD]'); a.set_ylabel(r"$e^\prime$  [kWh/m]")
save(f,'BOREAS_grissom_sweep_b')
# v2.18: panel (c) is NOT drawn here.  mksweepc216.py owns it -- it is
# the version with the 50 K/km curve and no shading, which is the
# honest picture now that there is no cutoff at the reference.  Two
# writers for one file name is how the v2.14 axis bug came back in
# v2.16: whichever script ran last won.  One name, one writer.
print('\ndone')

# ---- the Grissom geometry and R'_g,eff ---------------------------------
import boreas as B, boreas_ground as G
CSV='THUMS_wells_WellSTAR_clean.csv'
# the v2.13 reference, inline, so this file needs only the three modules
V216 = dict(source='produced_water', T_source_C=60.0, DT_3A_4A=15.0,
             T_sink_C=17.0, DT_sink_glide=4.0, sink_loop_dp_bar=0.5,
             T_m_C=130.0, t_ch=12.0, t_dc=12.0, idle_mode='simulate',
             x_int_orc=0.31, x_int_hp=0.30,
             grad_K_per_km=56.5)      # v2.16: the only sourced gradient
_csv = os.path.join(DATA,CSV) if os.path.exists(os.path.join(DATA,CSV)) else CSV
c = B.Case(n_segments=99, L_ft=1250.0/0.3048, t_operation_yr=10.0,
           D_in=201.2/25.4, r_casing_o=0.21910/2, r_bore=0.31120/2,
           k_cement=0.7, **V216)
P = G.grissom_positions(_csv)
md,Rg,tvd,info=G.reference_field(B,c,_csv,trajectory='s_curve',
    kop_rule='staggered',theta_max=55.0,kop_ft=300.0,L_md=1250.0,
    ds=G.DS_STATION,positions=P,verbose=False)
mdv,Rgv,tvdv,_=G.reference_field(B,c,_csv,trajectory='vertical',
    kop_rule='staggered',theta_max=55.0,kop_ft=300.0,L_md=1250.0,
    ds=G.DS_STATION,positions=P,verbose=False)
from scipy.spatial import cKDTree
# v2.18: panels (a)-(d) are NOT drawn here.  mkgeom216.py owns them --
# the rebuilt set with the nearest-neighbour panel (c) and the 909 m
# TVD annotation.  The versions that used to be drawn here were the
# superseded ones, and running this file after mkgeom216.py silently
# put them back.  Same rule as sweep (c): one name, one writer.
from scipy.spatial import cKDTree  # kept: nothing below needs it now
f,a=fig(); a.plot(md,Rg,'-',color='0.15',label='S-curve (R4)')
a.plot(mdv,Rgv,ls=(0,(4,3)),color='0.45',label='vertical (R3)')
a.set_xlabel('measured depth  [m]')
a.set_ylabel(r"$R^\prime_{g,\mathrm{eff}}$  [m K/W]")
a.set_ylim(0, max(Rgv.max(), Rg.max())*1.12)      # headroom: the R3 line
a.legend(loc='center right'); save(f,'BOREAS_grissom_Rg_a')
f,a=fig(); a.plot(tvd,Rg,'-',color='0.15')
a.set_xlabel('true vertical depth  [m]')
a.set_ylabel(r"$R^\prime_{g,\mathrm{eff}}$  [m K/W]")
save(f,'BOREAS_grissom_Rg_b')


In [ ]:
import os, pathlib, subprocess, sys
os.environ.update(BOREAS_SRC='.', BOREAS_BUILD='.', BOREAS_FIGDATA='figdata',
                  BOREAS_FIGOUT='paper_figures')
if pathlib.Path('pack/numbers.json').exists():
    os.environ['BOREAS_PACK'] = 'pack/numbers.json'

# v2.19: the figure data is DERIVED from the run artefacts, not carried as a
# second copy. figdata216/ used to hold the same results reshaped, and two
# copies of one result is how a figure gets drawn at the wrong gradient --
# which is what the guard inside mkfigs_uniform.py had to be added to catch.
_r = subprocess.run([sys.executable, 'mkfigdata219.py'], capture_output=True,
                    text=True)
print(_r.stdout or _r.stderr[-800:])
assert _r.returncode == 0, 'mkfigdata219.py failed -- see the output above'
# into the directory, which mkfigdata219.py has just made. Copying before
# it exists silently creates a FILE named figdata, and then every figure
# fails with "Not a directory".
pathlib.Path('figdata').mkdir(exist_ok=True)
if pathlib.Path('THUMS_wells_WellSTAR_clean.csv').exists():
    import shutil
    shutil.copy('THUMS_wells_WellSTAR_clean.csv', 'figdata/')

_r = subprocess.run([sys.executable, 'mkfigs_uniform.py'], capture_output=True,
                    text=True)
print(_r.stdout[-1200:] if _r.returncode == 0 else _r.stderr[-1200:])
assert _r.returncode == 0, 'mkfigs_uniform.py failed -- see the output above'
for _p in sorted(pathlib.Path('paper_figures').glob('*.pdf')):
    print(' ', _p.name)


### 12.1 The geometry and ladder figures

`mkfigs_uniform.py` above draws the result figures from the embedded
sweeps. These three draw the *geometry* figures, which need no sweep at
all -- only the slot map and the trajectories, both of which this
notebook already has. Together with 12 they reproduce every figure in
the paper except the R3 profile set, which needs its own march (12.2).


In [ ]:
%%writefile mkgeom216.py
"""BOREAS_grissom_geometry_{a-d} -- panels that match the caption.

(b) was a nearest-neighbour histogram and (c) and (d) BOTH showed TVD
against MD, so two panels said the same thing and the panel the text
cites did not exist. Now:
  (a) the 363 slots in plan
  (b) the nearest-neighbour histogram with the median marked
  (c) NEW: median distance to the nearest neighbouring WELL against MD,
      for vertical wells and for the S-curves, with the diffusion length
  (d) TVD against MD for both, merged from the old (c) and (d)
"""
import os, sys
sys.path.insert(0, os.environ.get('BOREAS_SRC','/tmp/build'))
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import boreas as B, boreas_ground as G, boreas_plots as BP
BP.use_paper_style()

# v2.19: the reference case comes from field.py, which reads the
# notebook's input section (ref_case.json).  The inline copy this replaces
# meant a value changed in the input cell did not reach this figure, and
# nothing said so.
import field


def _case(**kw):
    return field.case(**kw)

OUT=os.environ.get('BOREAS_FIGOUT','/tmp/build/figs216'); os.makedirs(OUT,exist_ok=True)
CSV='THUMS_wells_WellSTAR_clean.csv'
C_SC, C_VE = '#1f4e79', '#b06a1d'          # S-curve, vertical
c=_case(); P=G.grissom_positions(CSV)
L_MD=1250.0
alpha=c.k_rock/c.rho_cp_rock
L_diff=float(np.sqrt(alpha*c.t_operation_yr*365.25*24*3600))
d,_=cKDTree(P).query(P,2); med=float(np.median(d[:,1]))
# v2.22: drawn at the width the paper PRINTS it at, so LaTeX does
# not scale the figure and the type sizes in use_paper_style() are
# the sizes the reader sees. The aspect ratio is the old one.
PANEL = BP.figsize('half', 2.8 / 3.4)
def save(f,n):
    f.tight_layout(); f.savefig(f'{OUT}/{n}.pdf'); f.savefig(f'{OUT}/{n}.png',dpi=140)
    plt.close(f); print('  ',n)

# (a) plan
f,a=plt.subplots(figsize=PANEL)
a.plot(P[:,0],P[:,1],'o',ms=2.0,color=C_SC)
a.set_aspect('equal'); a.set_xlabel('east  [m]'); a.set_ylabel('north  [m]')
save(f,'BOREAS_grissom_geometry_a')

# (b) nearest-neighbour histogram, median marked
f,a=plt.subplots(figsize=PANEL)
a.hist(d[:,1],bins=26,color='0.78',edgecolor='0.25',linewidth=0.7)
a.axvline(med,ls=(0,(4,3)),color=C_VE,lw=1.4)
a.annotate(f'median {med:.2f} m',xy=(med,a.get_ylim()[1]*0.92),
           xytext=(med*1.35,a.get_ylim()[1]*0.92),color=C_VE,fontsize=8,va='center',
           arrowprops=dict(arrowstyle='->',color=C_VE,lw=0.7))
a.set_xlabel('nearest-neighbor spacing  [m]'); a.set_ylabel('slots')
save(f,'BOREAS_grissom_geometry_b')

# (c) NEW: median distance to the nearest neighbouring WELL against MD
mds=np.linspace(0.0,L_MD,60)
kop=G.staggered_kop(P)[0]
def seps(traj):
    out=[]
    md,X=G.trajectories(P,L_MD,kind=traj,kop=(kop if traj=='s_curve' else None),
                        theta_max=55.0)
    for m in mds:
        j=int(np.clip(np.searchsorted(md,m),0,len(md)-1))
        pts=X[:,j,:2] if X.ndim==3 else X[:,j]
        dd,_=cKDTree(pts).query(pts,2)
        out.append(float(np.median(dd[:,1])))
    return np.array(out)
s_sc=seps('s_curve'); s_ve=seps('vertical')
f,a=plt.subplots(figsize=PANEL)
a.plot(mds,s_sc,'-',color=C_SC,lw=1.6,label='S-curves (R4)')
a.plot(mds,s_ve,ls=(0,(5,3)),color=C_VE,lw=1.5,label='vertical (R3)')
a.axhline(L_diff,ls=(0,(1,2.5)),color='0.25',lw=1.1)
a.text(L_MD*0.02,L_diff*1.06,r'$\sqrt{\alpha_r t_\mathrm{op}}$'+f' = {L_diff:.1f} m',
       fontsize=8,color='0.25')
a.set_xlabel('measured depth  [m]')
a.set_ylabel('median distance to the\nnearest well  [m]')
a.legend(loc='center right'); save(f,'BOREAS_grissom_geometry_c')

# (d) TVD against MD, both
md,Rg,tvd,info=field.build(B,c,L_md=L_MD,P=P)
mdv,Rgv,tvdv,_=field.build(B,c,trajectory='vertical',L_md=L_MD,P=P)
f,a=plt.subplots(figsize=PANEL)
a.plot(md,tvd,'-',color=C_SC,lw=1.6,label='S-curves (R4)')
a.plot(mdv,tvdv,ls=(0,(5,3)),color=C_VE,lw=1.5,label='vertical (R3)')
a.axhline(info['tvd_bottom'],ls=(0,(1,2.5)),color='0.35',lw=0.9)
a.text(0.04,0.93,f"{info['tvd_bottom']:.0f} m TVD at {L_MD:.0f} m MD",
       transform=a.transAxes,va='top',fontsize=8)
a.set_xlabel('measured depth  [m]'); a.set_ylabel('true vertical depth  [m]')
a.legend(loc='lower right'); save(f,'BOREAS_grissom_geometry_d')
print(f'  diffusion length {L_diff:.1f} m; median spacing {med:.2f} m; '
      f'S-curve separation {s_sc[0]:.2f} -> {s_sc[-1]:.1f} m')


In [ ]:
%%writefile mkladder216.py
"""BOREAS_ladder_geometry -- the field ladder.  v2.24 layout.

Handoff v2.24, from JRB's review of the v2.23 figure:
  * panel (c) is gone -- Fig. 5(c) already shows the diffusion length
    against the neighbour distance, so a second bar chart said it twice;
  * (b) takes the full width, symmetric about x = 0;
  * the kickoff label is placed where no trajectory runs;
  * "PCM storage interval" is centred in the open wedge;
  * the bottom annotation is replaced by two TVD dimensions at the right
    edge, which say the thing the arrow only implied: the SAME 1250 m of
    hole reaches 1250 m TVD when it is vertical and 909 m when it is an
    S-curve;
  * R2 gets its own row, so the close-up is 1.9 in wide instead of 0.8 and
    the callout is a proper magnifier link.

Every dimensioned value is measured from the position arrays the runs use.
The 909 m comes from the field builder, which is where \\RefTvd comes from.
"""
import os, sys
sys.path.insert(0, os.environ.get('BOREAS_SRC', '/tmp/build'))
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, ConnectionPatch
import boreas as B, boreas_ground as G, boreas_plots as BP
import field
BP.use_paper_style()

OUT = os.environ.get('BOREAS_FIGOUT', '/tmp/build/figs216')
os.makedirs(OUT, exist_ok=True)
CSV = 'THUMS_wells_WellSTAR_clean.csv'
C_R1, C_R2, C_R34, C_IDLE = '#1d7a5f', '#8a5a9e', '#1f4e79', '#c8541a'
C_VE, C_ROCK, C_PCM = '#b06a1d', '#efe9df', '#f6d7b0'
C_LOSS, C_DIM = '#b2182b', '0.25'
ANN = 7.5

c = field.case()
N, L_MD = field.N_SLOTS, field.L_MD
P1 = G.disc_positions(N, B0=G.B0_MEASURED)
P2 = G.rows_positions(N, B0=G.B0_MEASURED, w=G.W_DEF)
P3 = G.grissom_positions(CSV)
Pi = G.grissom_positions(CSV, statuses=('Idle',))

TWO_R = float(2 * np.hypot(P1[:, 0], P1[:, 1]).max())
L_ROW = float(np.ptp(P2[:, 0]))
W_ROW = float(np.ptp(P2[:, 1]))
B0 = float(np.median(np.diff(np.sort(np.unique(np.round(P2[:, 0], 4))))))
EXT_X, EXT_Y = float(np.ptp(P3[:, 0])), float(np.ptp(P3[:, 1]))
_, _, _, INFO = field.build(B, c)
TVD_BOTTOM = float(INFO['tvd_bottom'])


def bare(ax):
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])


def dim_h(ax, x0, x1, y, label, color=C_DIM, va='bottom', dy=0.0,
          serif=None, arrows=False):
    """A horizontal dimension. Serifs by default: a double arrow across a
    short span puts its two heads on top of each other and prints an x."""
    if arrows:
        ax.annotate('', xy=(x0, y), xytext=(x1, y),
                    arrowprops=dict(arrowstyle='<->', color=color, lw=0.7,
                                    shrinkA=0, shrinkB=0))
    else:
        ax.plot([x0, x1], [y, y], '-', color=color, lw=0.6,
                solid_capstyle='butt')
    if serif:
        for xx in (x0, x1):
            ax.plot([xx, xx], [y - serif, y + serif], '-', color=color,
                    lw=0.6)
    ax.text(0.5 * (x0 + x1), y + dy, label, ha='center', va=va,
            fontsize=ANN, color=color)


def dim_v(ax, y0, y1, x, serif, color=C_DIM):
    """A vertical dimension line with end serifs. The label is placed by
    the caller, never on the line."""
    ax.plot([x, x], [y0, y1], '-', color=color, lw=0.6, solid_capstyle='butt')
    for yy in (y0, y1):
        ax.plot([x - serif, x + serif], [yy, yy], '-', color=color, lw=0.6)


# v2.25: 5.40 x 5.94 in drawn, so the paper can include it at \linewidth
# again. At 6.92 in the float plus its caption ran 86 pt off the page, and
# including it at 0.82 linewidth put its type at 7.2 pt against the 8.1-8.6
# of every other figure -- the whole point of v2.22 was that nothing gets
# scaled. The height comes out of white space, not out of the drawings.
fig = plt.figure(figsize=BP.figsize('full', 1.10))
gs = fig.add_gridspec(3, 1, height_ratios=[0.74, 0.92, 1.78], hspace=0.30)

# ================================================= (a) R1 and R3/R4 ======
top = gs[0].subgridspec(1, 2, width_ratios=[0.86, 1.14], wspace=0.10)

ax = fig.add_subplot(top[0]); bare(ax)
ax.plot(P1[:, 0], P1[:, 1], 'o', ms=1.9, color=C_R1, mec='none')
R = TWO_R / 2
ax.set_xlim(-R * 1.34, R * 1.34); ax.set_ylim(-R * 1.16, R * 1.44)
ax.set_aspect('equal')
dim_h(ax, -R, R, R * 1.28, rf'$2R_\mathrm{{field}} = {TWO_R:.1f}$ m',
      arrows=True, dy=R * 0.05)
for xx in (-R, R):
    ax.plot([xx, xx], [R * 1.04, R * 1.32], '-', color=C_DIM, lw=0.5)
ax.set_title('R1  compact disc', pad=9, fontsize=9)

ax = fig.add_subplot(top[1]); bare(ax)
ax.plot(P3[:, 0], P3[:, 1], 'o', ms=1.9, color=C_R34, mec='none')
ax.plot(Pi[:, 0], Pi[:, 1], 'o', ms=3.0, mfc='none', mec=C_IDLE, mew=0.8)
x0, x1 = P3[:, 0].min(), P3[:, 0].max()
y0, y1 = P3[:, 1].min(), P3[:, 1].max()
ax.set_xlim(x0 - EXT_X * 0.10, x1 + EXT_X * 0.10)
ax.set_ylim(y0 - EXT_Y * 0.34, y1 + EXT_Y * 0.30)
ax.set_aspect('equal')
dim_h(ax, x0, x1, y0 - EXT_Y * 0.20, f'{EXT_X:.0f} m', arrows=True, va='top',
      dy=-EXT_Y * 0.04)
for xx in (x0, x1):
    ax.plot([xx, xx], [y0 - EXT_Y * 0.24, y0 + EXT_Y * 0.03], '-',
            color=C_DIM, lw=0.5)
ax.plot([], [], 'o', ms=3.0, mfc='none', mec=C_IDLE, mew=0.8,
        label='idle (R4i)')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.00), frameon=False,
          handletextpad=0.3, borderpad=0.1, fontsize=ANN)
ax.set_title('R3 / R4  Grissom slots', pad=17, fontsize=9)

fig.text(0.012, 0.982, '(a)', fontsize=9)

# ================================================= (a) R2, its own row ===
mid = gs[1].subgridspec(1, 2, width_ratios=[0.98, 1.06], wspace=0.14)
axt = fig.add_subplot(mid[0]); bare(axt)
axt.add_patch(Rectangle((-L_ROW / 2, -W_ROW / 2), L_ROW, W_ROW,
                        facecolor=C_R2, edgecolor='none'))
axt.set_xlim(-L_ROW * 0.62, L_ROW * 0.62)
axt.set_ylim(-L_ROW * 0.17, L_ROW * 0.20)
dim_h(axt, -L_ROW / 2, L_ROW / 2, L_ROW * 0.085,
      rf'$L_\mathrm{{row}} = {L_ROW:.0f}$ m', arrows=True, dy=L_ROW * 0.013)
# the segment the close-up magnifies, marked on the strip
N_SHOW = 6
seg = N_SHOW * B0
axt.add_patch(Rectangle((-seg / 2, -L_ROW * 0.020), seg, L_ROW * 0.040,
                        facecolor='none', edgecolor='0.35', lw=0.7,
                        zorder=5))
axt.set_title('R2  straight double row', pad=9, fontsize=9)

axb = fig.add_subplot(mid[1]); bare(axb)
xs = (np.arange(N_SHOW) - (N_SHOW - 1) / 2) * B0
yhi, ylo = W_ROW / 2, -W_ROW / 2
LPAD, RPAD = B0 * 3.0, B0 * 0.8   # the left margin carries w_row
axb.add_patch(Rectangle((xs[0] - B0 * 0.6, ylo), xs[-1] - xs[0] + B0 * 1.2,
                        W_ROW, facecolor='#f3e7e7', edgecolor='none',
                        zorder=0))
r_slot = 0.20 * B0                      # 0.4 B_0 across, so they stand apart
for y in (yhi, ylo):
    axb.plot([xs[0] - B0 * 0.6, xs[-1] + B0 * 0.6], [y, y], '-',
             color=C_LOSS, lw=0.8, zorder=1)
    for x in xs:
        axb.add_patch(Circle((x, y), r_slot, facecolor='white',
                             edgecolor=C_R2, lw=0.9, zorder=3))
# the loss arrows: ~0.15 in at the printed size
SPAN_M = (xs[-1] - xs[0]) + LPAD + RPAD + B0 * 1.2
arrow_m = 0.15 / 2.60 * SPAN_M
for xx in (0.5 * (xs[1] + xs[2]), 0.5 * (xs[3] + xs[4])):
    for sgn, yy in ((1, yhi), (-1, ylo)):
        axb.annotate('', xy=(xx, yy + sgn * arrow_m), xytext=(xx, yy),
                     arrowprops=dict(arrowstyle='-|>', color=C_LOSS, lw=0.7,
                                     mutation_scale=5))
axb.text(0.0, yhi + arrow_m * 1.30, 'loss across the exposed face',
         ha='center', va='bottom', fontsize=ANN, color=C_LOSS)
# w_row: dimensioned INSIDE the left end, label rotated, so the drawing
# keeps the width instead of the margin
x_w = xs[0] - B0 * 1.05
axb.plot([x_w, x_w], [ylo, yhi], '-', color=C_DIM, lw=0.6)
for yy in (ylo, yhi):
    axb.plot([x_w - B0 * 0.18, x_w + B0 * 0.18], [yy, yy], '-', color=C_DIM,
             lw=0.6)
axb.text(x_w - B0 * 0.30, 0.0,
         rf'$w_\mathrm{{row}}$' + f'\n{W_ROW:.2f} m',
         ha='right', va='center', fontsize=ANN, color=C_DIM)
# B_0 below, with extension lines and end serifs
for xx in (xs[0], xs[1]):
    axb.plot([xx, xx], [ylo - r_slot - W_ROW * 0.05, ylo - W_ROW * 0.42],
             '-', color=C_DIM, lw=0.5)
dim_h(axb, xs[0], xs[1], ylo - W_ROW * 0.36, rf'$B_0 = {B0:.2f}$ m',
      serif=W_ROW * 0.07, va='top', dy=-W_ROW * 0.12)
# the caption goes below: inside the strip it met the w_row label
# v2.25: INSIDE the detail frame, under the lower row and under the B_0
# dimension -- the second of the three placements JRB offered. Centred in
# the shading it met the w_row label from whichever side that was placed,
# and below the frame it cost a line of page height.
# on TWO lines inside the shading, which is what makes it fit: at one
# line it is as wide as the whole interior and meets the w_row label from
# whichever side that sits. Below the frame instead would cost a line of
# page height, and the extra padding shrank the close-up to 1.0 in.
axb.text(0.0, 0.0, 'rock between the rows:\nshielded', ha='center',
         va='center', fontsize=ANN, color='0.35', zorder=4,
         linespacing=1.1)
axb.set_xlim(xs[0] - LPAD, xs[-1] + RPAD)
axb.set_ylim(ylo - W_ROW * 0.95, yhi + W_ROW * 0.85)
axb.set_aspect('equal')
axb.set_title(f'close-up: {N_SHOW} slots per row, {2 * N_SHOW} of {N}',
              pad=6, fontsize=ANN, color='0.35')

# the magnifier link: from the two corners of the marked segment to the two
# left-hand corners of the detail box, so the lines stay outside both
# drawings instead of crossing the close-up
axb.add_patch(Rectangle((0.002, 0.002), 0.996, 0.996, transform=axb.transAxes,
                        facecolor='none', edgecolor='0.70', lw=0.6,
                        zorder=0))
for s_y, d_y in ((L_ROW * 0.020, 1.0), (-L_ROW * 0.020, 0.0)):
    fig.add_artist(ConnectionPatch(
        xyA=(seg / 2, s_y), coordsA=axt.transData,
        xyB=(0.0, d_y), coordsB=axb.transAxes,
        color='0.55', lw=0.6, ls=(0, (3, 2))))

# ===================================================== (b) trajectories ==
ax = fig.add_subplot(gs[2])
md = np.linspace(0, L_MD, 400)
inc = np.radians(G.s_curve(md, 300.0 * 0.3048, 4.0, 55.0, 3000 * 0.3048))
im = 0.5 * (inc[1:] + inc[:-1])
tvd = np.r_[0.0, np.cumsum(np.cos(im) * np.diff(md))]
hor = np.r_[0.0, np.cumsum(np.sin(im) * np.diff(md))]
XLIM = 850.0
ax.set_xlim(-XLIM, XLIM)
ax.axhspan(0, L_MD, color=C_ROCK, zorder=0)
ax.axhspan(0, TVD_BOTTOM, color=C_PCM, zorder=1, alpha=0.75)
KLO, KHI = 100 * 0.3048, 500 * 0.3048
ax.axhspan(KLO, KHI, color='0.80', zorder=2, alpha=0.8)
for sgn in (1, -1):
    ax.plot(sgn * hor, tvd, '-', color=C_R34, lw=1.3, zorder=4,
            label='R4  S-curves' if sgn == 1 else None)
ax.plot([0, 0], [0, L_MD], ls=(0, (5, 3)), color=C_VE, lw=1.1, zorder=4,
        label='R0v / R3  vertical')
# on the band and clear of every trajectory: at 30-150 m TVD the wells are
# still within a few metres of x = 0, so the far left of the band is empty
ax.text(-XLIM * 0.965, 0.5 * (KLO + KHI), 'kickoff band, 100–500 ft',
        va='center', ha='left', fontsize=ANN, color='0.25', zorder=5)
# centred on x = 0, with the dashed vertical line passing BEHIND a white
# box. The open wedge was the other option JRB offered, but at 40 % of the
# storage depth the wedge is only ~130 m wide and the label is ~190 m.
ax.text(0.0, TVD_BOTTOM * 0.56, 'PCM storage interval', va='center',
        ha='center', color='0.45', fontsize=ANN, zorder=6,
        bbox=dict(boxstyle='square,pad=0.25', facecolor='white',
                  edgecolor='none', alpha=0.85))

# ---- the two TVD dimensions: the same 1250 m of hole, two trajectories --
X_SC, X_VE = 0.930 * XLIM, 0.985 * XLIM
SER = XLIM * 0.016
for d in (TVD_BOTTOM, L_MD):
    ax.plot([-XLIM, X_VE], [d, d], ls=(0, (1, 2.4)), color='0.45', lw=0.6,
            zorder=3)
dim_v(ax, 0.0, TVD_BOTTOM, X_SC, SER)
dim_v(ax, 0.0, L_MD, X_VE, SER)
# Each label sits at the END of its own dotted leader, just inside the
# dimension line it belongs to: the 909 one under the 909 m leader, the
# 1250 one over the 1250 m leader. Rotated labels beside the lines were
# the first attempt and the two ran together in 90 m of margin.
ax.text(X_SC - XLIM * 0.025, TVD_BOTTOM + L_MD * 0.035,
        f'{TVD_BOTTOM:.0f} m:  S-curves (R4, R0i)', ha='right', va='top',
        fontsize=ANN, color=C_DIM, zorder=6)
ax.text(X_VE - XLIM * 0.025, L_MD - L_MD * 0.035,
        f'{L_MD:.0f} m:  vertical wells (R0v, R3)', ha='right', va='bottom',
        fontsize=ANN, color=C_DIM, zorder=6)
ax.set_xlabel('horizontal reach  [m]')
ax.set_ylabel('true vertical depth  [m]')
ax.set_ylim(L_MD * 1.045, -L_MD * 0.03)
ax.legend(loc='lower left', fontsize=ANN)
fig.text(0.012, 0.455, '(b)', fontsize=9)

fig.savefig(f'{OUT}/BOREAS_ladder_geometry.pdf')
fig.savefig(f'{OUT}/BOREAS_ladder_geometry.png', dpi=150)
plt.close(fig)
print('  BOREAS_ladder_geometry')
print(f'    2R_field {TWO_R:.1f} m | L_row {L_ROW:.0f} m | B_0 {B0:.2f} m | '
      f'w_row {W_ROW:.2f} m')
print(f'    ring {EXT_X:.0f} x {EXT_Y:.0f} m | S-curve TVD {TVD_BOTTOM:.0f} m '
      f'| vertical TVD {L_MD:.0f} m')


In [ ]:
%%writefile mksweepc216.py
"""BOREAS_grissom_sweep_c at G = 56.5 K/km: there is NO cutoff to shade."""
import json, os, sys
sys.path.insert(0, os.environ.get('BOREAS_SRC','/tmp/build'))
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import boreas_plots as BP
BP.use_paper_style()
OUT=os.environ.get('BOREAS_FIGOUT','/tmp/build/figs216')
g=json.load(open(os.environ.get('BOREAS_GRAD','grad216.json')))
def ratio(Gk,Ls):
    ep=np.array([g[f'{Gk:g}@{L:g}']['e_prime'] for L in Ls]); Ls=np.array(Ls,float)
    E=ep*Ls; m=np.diff(np.r_[0.0,E])/np.diff(np.r_[0.0,Ls]); return Ls,m/m[0]
import spec
import field
L9=[float(x) for x in spec.get('storage_length_m_MD')]
Ls,r565=ratio(56.5,L9)
# v2.22: drawn at the width the paper PRINTS it at, so LaTeX does
# not scale the figure and the type sizes in use_paper_style() are
# the sizes the reader sees. The aspect ratio is the old one.
f, a = plt.subplots(figsize=BP.figsize('half', 2.9 / 3.6))
a.step(Ls,r565,where='pre',color='#1f4e79',lw=1.7,label='56.5 K/km (reference)')
L5=[100,400,500,600,800]
Lb,r50=ratio(50,L5)
a.step(Lb,r50,where='pre',color='#b06a1d',ls=(0,(5,3)),lw=1.4,label='50 K/km')
a.axhline(0.5,ls=(0,(1,2.5)),color='0.3',lw=1.0)
a.text(Ls[-1],0.505,'threshold ',ha='right',va='bottom',fontsize=8,color='0.3')
a.set_xlabel('storage length  [m MD]')
a.set_ylabel(r'marginal $e^\prime$ / first 100 m')
a.set_ylim(0.40,1.05); a.legend(loc='upper right',fontsize=7.6)
f.tight_layout(); f.savefig(f'{OUT}/BOREAS_grissom_sweep_c.pdf')
f.savefig(f'{OUT}/BOREAS_grissom_sweep_c.png',dpi=140); plt.close(f)
print(f'  sweep_c: minimum {r565.min()*100:.1f} % at the reference -- no crossing')


In [ ]:
# v2.19: the gradient sweep used to be embedded separately here, as a
# second copy of grad216.json. It now arrives with the other run artefacts
# in section 12, so there is one copy and one source. Nothing to do.
import pathlib
print('grad216.json', 'present' if pathlib.Path('grad216.json').exists()
      else 'MISSING -- run the artefact cell in section 12 first')


In [ ]:
import os, subprocess, sys, pathlib
os.environ['BOREAS_FIGOUT']='paper_figures'
os.environ['BOREAS_SRC']='.'
_FAILED = []
for _s in ('mkgeom216.py','mkladder216.py','mksweepc216.py'):
    _r=subprocess.run([sys.executable,_s],capture_output=True,text=True)
    _FAILED = _FAILED + [_s] if _r.returncode else _FAILED
    print(_s, '->', 'ok' if _r.returncode==0 else _r.stderr[-300:])
print()
for _p in sorted(pathlib.Path('paper_figures').glob('*.pdf')): print(' ', _p.name)

assert not _FAILED, (
    f'these generators failed: {_FAILED}. Run section 1 first -- it writes the model modules -- then this cell again.')


### 12.2 The R3 profile set and the loss comparison

These need their own marches -- the vertical-well case R3 and the
reference R4 -- so this cell is the slow one, a few minutes. It writes
the records beside itself, so a second run is instant. With 12 and 12.1
it completes the paper's figure set.


In [ ]:
%%writefile mkR3prof.py
"""The R3 profiles on the R4 axes, plus the loss comparison.

The axis limits are taken from the R4 figures and forced on the R3 ones,
so the two sets compare panel by panel. q'_loss is not recorded by the
march, but T_pcm is, so it is reconstructed exactly as the march computes
it: K_loss (T_pcm - T_rock), per metre of fin tube.
"""
import json, os, pickle, sys
sys.path.insert(0, os.environ.get('BOREAS_SRC','/tmp/build'))
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import boreas as B, boreas_ground as G, boreas_plots as BP
BP.use_paper_style()
OUT=os.environ.get('BOREAS_FIGOUT','/tmp/build/figs')
CSV='THUMS_wells_WellSTAR_clean.csv'
# v2.23, JRB: the three profile fields are drawn at the width the paper
# now includes them at (0.49 linewidth = 2.65 in) and 20 % taller, since
# depth is the ordinate and that is where the profiles gain. The integrals
# panels keep the v2.22 geometry: Figs. 16 and S4 are approved.
PANEL = BP.figsize('profile', 1.20 * 4.4 / 4.9)
PANEL_INT = BP.figsize('half', 3.9 / 4.9)

def two_panel(name, draw_pair, lims=None, size=PANEL):
    fa,axa=plt.subplots(figsize=size); fb,axb=plt.subplots(figsize=size)
    draw_pair([axa,axb])
    for ax in (axa,axb): ax.set_title('')
    if axb.get_legend() is not None: axb.get_legend().remove()
    if lims:
        for ax,(xl,yl) in zip((axa,axb),lims):
            if xl: ax.set_xlim(*xl)
            if yl: ax.set_ylim(*yl)
    for f,sfx in ((fa,'_a'),(fb,'_b')):
        f.tight_layout(); f.savefig(f'{OUT}/{name}{sfx}.pdf')
        f.savefig(f'{OUT}/{name}{sfx}.png',dpi=130); plt.close(f)
    print(f'  {name}_a, {name}_b')

# v2.19: the reference case comes from field.py, which reads the
# notebook's input section (ref_case.json).  The inline copy this replaces
# meant a value changed in the input cell did not reach this figure, and
# nothing said so.
import field


def _case(**kw):
    return field.case(**kw)

def _march(traj, path):
    """Load the record if it is there, otherwise march and keep it."""
    if os.path.exists(path): return pickle.load(open(path,'rb'))['rec']
    c,P,(md,Rg,tvd,inf)=field_for(traj)
    with G.field_ground(B,md,Rg,tvd): rec=B.simulate_css_corrected(c,record=True)
    pickle.dump(dict(rec=rec),open(path,'wb')); return rec

def field_for(traj):   # renamed in v2.19: `field` is now the shared module
    c=_case(); P=G.grissom_positions(CSV)
    return c,P,field.build(B,c,trajectory=traj,P=P)

rec3=_march('vertical', os.environ.get('BOREAS_REC3','rec_R3.pkl'))
rec4=_march('s_curve',  os.environ.get('BOREAS_REC4','rec_R4.pkl'))
c=_case()

# ---- the R3 panels, drawn through the R4 style ------------------------
# v2.21. These eight were drawn with bare well_profiles() calls: kelvin, no
# staircase, no PCM overlay, no colour bar, melted AREA, per WELL, and
# against the raw march coordinate, which runs from the bottom -- so they
# were upside down under an axis labelled TVD. They now go through
# BP.profile_kwargs(), the same arguments the R4 set uses, and through
# BP.depth_axes(), so the only difference between the two sets is the
# record and the trajectory.
#
# AXES: option (b) of the handoff. R3 is vertical and reaches 1250 m TVD,
# R4 is an S-curve and reaches 909 m, and each is drawn against its own
# TVD. The quantity limits are shared so the panels compare. Plotting both
# against MD instead would put R4's deep end at 1250 m, which is the error
# v2.15 had to fix in prof_loss_cmp -- MD is hole length, TVD is what sets
# the rock temperature the store is fighting.
md3, Rg3, tvd3, info3 = field_for('vertical')[2]
md4, Rg4, tvd4, info4 = field_for('s_curve')[2]
dseg3, dnode3 = BP.depth_axes(c, md3, tvd3)
dseg4, dnode4 = BP.depth_axes(c, md4, tvd4)
print(f'  R3 TVD {dseg3.min():.0f}-{dseg3.max():.0f} m, '
      f'R4 TVD {dseg4.min():.0f}-{dseg4.max():.0f} m')

# the gate handoff v2.21 asks for, on BOTH records
for rr, dd, nm in ((rec3, dseg3, 'R3'), (rec4, dseg4, 'R4')):
    o = BP.check_orientation(rr, c, dd, name=nm)
    print(f"  {nm} orientation ok: charge inlet {o['charge_deep_C']:.1f} C at "
          f"the deep end, discharge inlet {o['discharge_top_C']:.1f} C at the top")

# the quantity limits from the R4 set, so the two compare panel by panel
LIM = {}
with G.field_ground(B, md4, Rg4, tvd4):
    for fld in ('T_fluid', 'A_melt', 'U_i'):
        fa, axa = plt.subplots(figsize=PANEL); fb, axb = plt.subplots(figsize=PANEL)
        BP.well_profiles(rec4, c, field=fld, axes=[axa, axb],
                         depth_seg=dseg4, depth_node=dnode4,
                         **BP.profile_kwargs(c, fld))
        LIM[fld] = [axa.get_xlim(), axb.get_xlim()]
        plt.close(fa); plt.close(fb)
    fa, axa = plt.subplots(figsize=PANEL); fb, axb = plt.subplots(figsize=PANEL)
    BP.well_integrals(rec4, c, axes=[axa, axb], per_tube=True, titles=False)
    LIM['integrals'] = [axa.get_ylim(), axb.get_ylim()]
    plt.close(fa); plt.close(fb)

with G.field_ground(B, md3, Rg3, tvd3):
    for fld, name in (('T_fluid', 'BOREAS_prof_Tfluid_R3'),
                      ('A_melt', 'BOREAS_prof_melt_R3'),
                      ('U_i', 'BOREAS_prof_Ui_R3')):
        two_panel(name, lambda ax, f=fld: BP.well_profiles(
            rec3, c, field=f, axes=ax, depth_seg=dseg3, depth_node=dnode3,
            **BP.profile_kwargs(c, f)),
            lims=[(LIM[fld][0], (dseg3.max(), 0.0)),
                  (LIM[fld][1], (dseg3.max(), 0.0))])
    two_panel('BOREAS_prof_integrals_R3',
              lambda ax: BP.well_integrals(rec3, c, axes=ax, per_tube=True,
                                           titles=False),
              lims=[(None, LIM['integrals'][0]), (None, LIM['integrals'][1])],
              size=PANEL_INT)

# ---- the loss profile: COMPUTE and save, but do NOT draw ---------------
# BOREAS_prof_loss_cmp is drawn by mk_losscmp.py and by nothing else.
# This file used to draw it too, against MEASURED depth, so whichever ran
# last won -- and that is how the v2.14 axis bug came back after it had
# been fixed. One writer per figure.
def qloss_profile(rec, traj):
    """q'_loss(l) reconstructed from the recorded PCM temperature.

    The march does not record q'_loss, so it is rebuilt exactly as the
    march computes it: K_loss (T_pcm - T_rock) per metre of FIN TUBE,
    step-weighted over each half-cycle and averaged over the two.
    """
    cc,P,(mdx,Rgx,tvdx,inf)=field_for(traj)
    with G.field_ground(B,mdx,Rgx,tvdx):
        Tr,dep=B.formation_profile(cc)
        R=np.atleast_1d(B.formation_resistance(cc,dep))
        _tvd_of=getattr(B,'_boreas_tvd_of',None)
        depth_tvd=np.asarray(_tvd_of(dep),float) if _tvd_of else dep.copy()
    K=np.where(np.isfinite(R),1.0/(R*cc.n_ft),0.0)
    out=[]
    for half in ('charge','discharge'):
        Tp=np.asarray(rec[half]['history']['T_pcm'],float)
        th=np.asarray(rec[half]['t_hist'],float)
        if Tp.shape[0]==len(th)-1: th=th[1:]
        w=np.diff(np.r_[th[0],th]); w=w/w.sum()
        Tb=(Tp*w[:,None]).sum(axis=0)
        if half=='discharge': Tb=Tb[::-1]
        out.append(K*(Tb-Tr))
    return 0.5*(out[0]+out[1]), dep, inf, depth_tvd


q4,dep4,i4,tv4=qloss_profile(rec4,'s_curve')
q3,dep3,i3,tv3=qloss_profile(rec3,'vertical')
np.savez(os.environ.get('BOREAS_QLOSS','qloss.npz'),
         q3=q3,q4=q4,md=dep3,tv3=tv3,tv4=tv4)
json.dump(dict(R3_mean=float(q3.mean()),R4_mean=float(q4.mean()),
               R3_max=float(q3.max()),R4_max=float(q4.max())),
          open('qloss_cmp.json','w'),indent=1)
print(f'  qloss.npz written   R3 mean {q3.mean():.2f}, R4 mean {q4.mean():.2f} W/m'
      f'   (BOREAS_prof_loss_cmp is drawn by mk_losscmp.py)')


In [ ]:
%%writefile mk_losscmp.py
"""BOREAS_prof_loss_cmp -- each case against its OWN true vertical depth.

v2.14 plotted both against the solver's MEASURED depth and labelled the
axis "true vertical depth". R3 is vertical so the two coincide, but R4
reaches only 904 m TVD over 1250 m MD, so its curve ran 340 m too deep.
Each case is now mapped through its own trajectory, which is also the
convention of prof-T .. prof-int.

The kickoff band is the reconstructed one, 100-500 ft (30-152 m), not the
300 ft of the single trajectory drawn in the ladder figure.
"""
import os, sys
sys.path.insert(0, os.environ.get('BOREAS_SRC','/tmp/build'))
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import boreas_plots as BP
BP.use_paper_style()
OUT=os.environ.get('BOREAS_FIGOUT','/tmp/build/figs'); os.makedirs(OUT,exist_ok=True)
d=np.load(os.environ.get('BOREAS_QLOSS','qloss.npz'))
q3,q4,tv3,tv4=d['q3'],d['q4'],d['tv3'],d['tv4']
KOP_LO, KOP_HI = 100*0.3048, 500*0.3048      # the reconstructed rotation
# v2.22: drawn at the width the paper PRINTS it at, so LaTeX does
# not scale the figure and the type sizes in use_paper_style() are
# the sizes the reader sees. The aspect ratio is the old one.
f, a = plt.subplots(figsize=BP.figsize('loss', 3.4 / 4.4))
a.axhspan(KOP_LO,KOP_HI,color='0.93',zorder=0)
a.plot(q3,tv3,ls=(0,(5,3)),color='0.45',lw=1.4,label='R3  vertical')
a.plot(q4,tv4,'-',color='0.15',lw=1.5,label='R4  S-curve')
a.set_ylim(0, max(tv3.max(),tv4.max())*1.02)
a.text(a.get_xlim()[1]*0.98,(KOP_LO+KOP_HI)/2,'kickoff band ',
       va='center',ha='right',color='0.45',fontsize=8)
a.invert_yaxis()
a.set_xlabel(r"cycle-mean $q^\prime_\mathrm{loss}$  [W per m of fin tube]")
a.set_ylabel('true vertical depth  [m]')
a.legend(loc='lower right')
f.tight_layout()
f.savefig(f'{OUT}/BOREAS_prof_loss_cmp.pdf'); f.savefig(f'{OUT}/BOREAS_prof_loss_cmp.png',dpi=130)
plt.close(f)
print(f'  BOREAS_prof_loss_cmp   R3 {q3[0]:.2f}->{q3[-1]:.2f} to {tv3.max():.0f} m TVD ; '
      f'R4 {q4[0]:.2f}->{q4[-1]:.2f} to {tv4.max():.0f} m TVD')
print(f'  kickoff band {KOP_LO:.0f}-{KOP_HI:.0f} m')


In [ ]:
import os, pathlib, subprocess, sys
os.environ.update(BOREAS_SRC='.', BOREAS_BUILD='.',
                  BOREAS_FIGOUT='paper_figures',
                  BOREAS_REC3='rec_R3.pkl', BOREAS_REC4='rec_R4.pkl',
                  BOREAS_QLOSS='qloss.npz')

# These two MARCH: the vertical-well case R3 and the reference R4. A few
# minutes, and the records are written beside the notebook, so a second run
# is instant. Section 12.3 and the whole macro pack read them, which is why
# this cell comes first.
_FAILED = []
for _s in ('mkR3prof.py', 'mk_losscmp.py'):
    _r = subprocess.run([sys.executable, _s], capture_output=True, text=True)
    _FAILED = _FAILED + [_s] if _r.returncode else _FAILED
    print(f'{_s:14s}', 'ok' if _r.returncode == 0 else 'FAILED')
    if _r.returncode:
        print(_r.stderr[-900:]); break
    for _l in _r.stdout.strip().splitlines()[-2:]:
        print('   ', _l[:110])
for _f in ('rec_R3.pkl', 'rec_R4.pkl', 'qloss.npz'):
    print(f'  {_f:12s} {"written" if pathlib.Path(_f).exists() else "MISSING"}')

assert not _FAILED, (
    f'these generators failed: {_FAILED}. Run section 1 first -- it writes the model modules -- then this cell again.')


### 12.3 The $T$–$s$ diagram, the R4 profile set and Video S1

These three generators were **missing from the notebook** until v2.19: the
pack shipped them but the notebook did not, so it could not reproduce the
paper's T–s figures, the R4 well profiles, the state tables or the cycle
animation. With them the notebook covers all 39 result files.

`mkS1.py` renders the video and is the slowest cell here, a few minutes.

`packstamp.py` writes the hash of the run set into every generated file.
It is imported by the three generators below and was **missing from the
notebook until v2.25**, so section 12.3 — the T–s figures, the R4 profile
set, the state tables and Video S1 — could not run outside the task
sandbox, where the file happens to exist.

In [ ]:
%%writefile packstamp.py
"""Provenance stamps for generated files (handoff 6.1).

v2.2 shipped a tab_ladder.tex that was byte-for-byte the v2.1 file, while
the return note said every derived table had been recomputed and the
README said nothing in the pack is typed by hand. Both statements were
false and nothing caught it: a stale LaTeX table is still valid LaTeX.

Every generator now writes the SHA-256 of the pack_runs.json it was built
from into the file, and V16 checks every stamp against the current cache.
A file that was not regenerated now fails a numbered check instead of
quietly disagreeing with the CSV in the third digit.
"""
import hashlib
import os
import re

import os
CACHE = os.environ.get('BOREAS_PACK_CACHE','/tmp/build/pack217/numbers.json')
MARK = 'boreas-pack-sha256'


def pack_hash(path=CACHE):
    with open(path, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()[:16]


def stamp(comment='%', path=CACHE):
    """One comment line naming the cache this file was built from."""
    return f'{comment} {MARK}: {pack_hash(path)}\n'


def read_stamp(path):
    try:
        with open(path, 'r', errors='ignore') as f:
            head = f.read(4096)
    except OSError:
        return None
    m = re.search(MARK + r':\s*([0-9a-f]+)', head)
    return m.group(1) if m else None


def check(dirname, patterns=('.tex',), path=CACHE):
    """(stale, unstamped, ok) lists for every generated file in `dirname`."""
    want = pack_hash(path)
    stale, unstamped, ok = [], [], []
    for fn in sorted(os.listdir(dirname)):
        if not fn.endswith(patterns):
            continue
        got = read_stamp(os.path.join(dirname, fn))
        if got is None:
            unstamped.append(fn)
        elif got != want:
            stale.append((fn, got))
        else:
            ok.append(fn)
    return stale, unstamped, ok


In [ ]:
%%writefile mkprofiles.py
"""Section 4.3: the profile figures at R4, one PDF per panel, and the
numbers the text quotes (handoff 6.6 and 6.7).

The march runs INSIDE field_ground, so every profile is on the measured
Grissom field: the superposition resistance R_g,eff(MD) and the rock
temperature on TRUE VERTICAL DEPTH. The S-curve reaches 891 m TVD at
1250 m MD, so running this outside the context manager puts the bottom of
the store in rock about 15 K too hot.
"""
import json
import os
import pickle

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import boreas as B
import boreas_ground as G
import boreas_plots as BP
import packstamp

import os
OUT = os.environ.get('BOREAS_FIGOUT','results_pack_v2')
CSV = 'THUMS_wells_WellSTAR_clean.csv'
L_MD = 1250.0
THUMS = dict(D_in=201.2 / 25.4, r_casing_o=0.21910 / 2,
             r_bore=0.31120 / 2, k_cement=0.7)
# the v2.17 reference, inline: 363 slots, G = 56.5 K/km
# v2.19: the reference case comes from field.py, which reads the
# notebook's input section (ref_case.json). The inline copy this
# replaces meant that a value changed in the input cell did not
# reach this figure, and nothing said so.
import field

CASE = field.case()
# v2.23, JRB: the three profile fields are drawn at the width the paper
# now includes them at (0.49 linewidth = 2.65 in) and 20 % taller, since
# depth is the ordinate and that is where the profiles gain. The integrals
# panels keep the v2.22 geometry: Figs. 16 and S4 are approved.
PANEL = BP.figsize('profile', 1.20 * 4.4 / 4.9)
PANEL_INT = BP.figsize('half', 3.9 / 4.9)


def march():
    if os.path.exists('rec_V210F.pkl'):
        return pickle.load(open('rec_V210F.pkl', 'rb'))['rec']
    md, Rg, tvd, _ = field.build(B, CASE, L_md=L_MD)
    with G.field_ground(B, md, Rg, tvd):
        rec = B.simulate_css_corrected(CASE, record=True)
    pickle.dump(dict(rec=rec), open('rec_V210F.pkl', 'wb'))
    return rec


def depth_axes():
    """Depth from the TRAJECTORY, not from the solver coordinate.

    The solver's z = 0 is the BOTTOM of the well on charge, and on an
    S-curve 1250 m of hole reaches only 891 m of depth, so neither the
    raw coordinate nor measured depth is the right vertical axis.
    """
    md, Rg, tvd, _ = field.build(B, CASE, L_md=L_MD)
    n = CASE.n_segments
    xi = (np.arange(n) + 0.5) / n
    mdv = CASE.L_well * (1.0 - xi)
    dseg = np.interp(mdv, np.r_[0.0, md], np.r_[0.0, tvd])
    dnode = np.interp(CASE.L_well * (1.0 - np.linspace(0, 1, n + 1)),
                      np.r_[0.0, md], np.r_[0.0, tvd])
    return dseg, dnode


def two_panel(name, draw_pair, size=PANEL):
    """Render a charge/discharge pair as two separate files.

    The two axes live on different figures, so each panel is drawn
    natively rather than cropped out of a two-panel image afterwards.
    """
    fa, axa = plt.subplots(figsize=size)
    fb, axb = plt.subplots(figsize=size)
    draw_pair([axa, axb])
    for ax in (axa, axb):
        ax.set_title('')
    if axb.get_legend() is not None:
        axb.get_legend().remove()
    for f, tag in ((fa, 'a'), (fb, 'b')):
        f.tight_layout()
        f.savefig(f'{OUT}/{name}_{tag}.pdf')
        plt.close(f)
    print(f'  wrote {OUT}/{name}_a.pdf and {name}_b.pdf')


def main():
    os.makedirs(OUT, exist_ok=True)
    BP.use_paper_style()
    rec = march()
    dseg, dnode = depth_axes()
    print(f'depth runs {dseg[0]:.0f} m (bottom) -> {dseg[-1]:.0f} m (top)')

    # Temperatures in DEGREES CELSIUS, depth labelled TVD (it reaches
    # 891 m at 1250 m MD, so a reader must not take it for the storage
    # length), and the time key as a colorbar on panel (b) only.
    two_panel('BOREAS_prof_Tfluid', lambda ax: BP.well_profiles(
        rec, CASE, field='T_fluid', axes=ax, vertical=True,
        depth_seg=dseg, depth_node=dnode, overlay='T_pcm',
        overlay_label=r'$T_\mathrm{pcm}$', offset=-273.15,
        ylabel=r'temperature / $^\circ$C', colorbar=True))
    two_panel('BOREAS_prof_melt', lambda ax: BP.well_profiles(
        rec, CASE, field='A_melt', axes=ax, vertical=True,
        depth_seg=dseg, depth_node=dnode,
        scale=1.0 / BP.pcm_capacities_A(CASE),
        ylabel='melted fraction of the PCM', colorbar=True))
    two_panel('BOREAS_prof_Ui', lambda ax: BP.well_profiles(
        rec, CASE, field='U_i', axes=ax, vertical=True,
        depth_seg=dseg, depth_node=dnode,
        ylabel=r'$U_i$ / W m$^{-2}$ K$^{-1}$', colorbar=True))
    two_panel('BOREAS_prof_integrals', lambda ax: BP.well_integrals(
        rec, CASE, axes=ax, per_tube=True, titles=False), size=PANEL_INT)

    # ---- the numbers section 4.3 quotes -------------------------------
    M = {}

    def add(n, v, fmt='{:.2f}', note=''):
        M[n] = (fmt.format(v), note)

    Tm = rec['T_m_seg'] - 273.15
    out_ch = np.asarray(rec['charge']['T_out'], float) - 273.15
    out_dc = np.asarray(rec['discharge']['T_out'], float) - 273.15
    add('TtwocLo', out_ch.min(), '{:.1f}', 'T_2c range over the charge, C')
    add('TtwocHi', out_ch.max(), '{:.1f}', '')
    add('TtwodLo', out_dc.min(), '{:.1f}', 'T_2d range over the discharge, C')
    add('TtwodHi', out_dc.max(), '{:.1f}', '')

    Tp = np.asarray(rec['discharge']['history']['T_pcm'], float)[-1] - 273.15
    sub = Tm - Tp                      # positive = PCM below its melting point
    add('SubcoolMax', float(sub.max()), '{:.1f}',
        'largest subcooling at the end of discharge, K')
    add('SubcoolMean', float(sub.mean()), '{:.1f}', '')

    # NOT the cycled fraction. eps is the RESIDUAL melted fraction at the
    # end of each half-cycle; the fraction actually cycled is the
    # difference, about 0.58 here. A small eps_dc means the store refroze
    # almost completely, which is the intended behaviour, not a failure.
    e_ch = float(rec['charge']['eps_local'].mean())
    e_dc = float(rec['discharge']['eps_local'].mean())
    add('EpsCh', e_ch, '{:.4f}', 'melted fraction at the END OF CHARGE')
    add('EpsDc', e_dc, '{:.4f}',
        'residual melted fraction at the END OF DISCHARGE')
    add('EpsCycled', e_ch - e_dc, '{:.4f}',
        'the fraction actually cycled, eps_ch - eps_dc')

    U = {h: np.asarray(rec[h]['history']['U_i'], float) for h in
         ('charge', 'discharge')}
    allU = np.concatenate([u.ravel() for u in U.values()])
    add('UiLo', float(allU.min()), '{:.1f}', 'U_i range, W/m2/K')
    add('UiHi', float(allU.max()), '{:.1f}', '')
    # mean against the value at the surface end (last segment, index -1)
    add('UiMean', float(allU.mean()), '{:.1f}', '')
    add('UiSurface', float(U['charge'][-1, -1]), '{:.1f}',
        'U_i at the surface end, end of charge')
    add('UiMeanToSurface', float(allU.mean() / U['charge'][-1, -1]),
        '{:.2f}', 'mean over surface-end value')

    # RATES PER FIN TUBE. q_prime is already per unit length of fin tube
    # (Eq. ode is written per tube, K_j is per tube length), so the
    # per-tube rate is the bare sum -- v2.3 divided by n_ft on top of
    # that and came out a factor of three low.
    #
    # The MEAN is step-weighted. The record grid is logarithmic, 41
    # points whose steps run from 0.3 s to 8491 s, so an arithmetic mean
    # over it weights the first minute as heavily as the last two hours.
    # For the discharge, which collapses to near zero in the last
    # quarter, that is the difference between -125 and -62 kW.
    trapz = getattr(np, 'trapezoid', None) or np.trapz
    for half, tag in (('charge', 'Ch'), ('discharge', 'Dc')):
        H = rec[half]['history']
        z = rec[half]['z']
        dz = z[1] - z[0]
        th = np.asarray(rec[half]['t_hist'], float)
        q = np.asarray(H['q_prime'], float).sum(axis=1) * dz / 1e3
        add(f'Rate{tag}Start', float(q[0]), '{:.1f}',
            f'{half} rate per fin tube at the start, kW')
        add(f'Rate{tag}End', float(q[-1]), '{:.1f}', 'at the end')
        # THE MEAN COMES FROM THE SOLVER'S CUMULATIVE, not from the record
        # grid. Q_cum_J is the exact integral the budget is built from
        # (V17), and it is per fin tube; the mean is simply it over the
        # half-cycle. Integrating the 41-point logarithmic record grid
        # instead is within 0.4 % on the charge but 11 % low on the
        # discharge, whose rate collapses between the last few samples.
        t_half = (CASE.t_ch if half == 'charge' else CASE.t_dc) * 3600.0
        sgn = 1.0 if half == 'charge' else -1.0
        mean_tube = sgn * abs(rec[half]['Q_cum_J']) / t_half / 1e3   # kW
        add(f'Rate{tag}Mean', float(mean_tube), '{:.1f}',
            'half-cycle average from the cumulative, per fin tube')
        add(f'Rate{tag}MeanPerWell', float(CASE.n_ft * mean_tube), '{:.0f}',
            'the same per well')
        add(f'Rate{tag}MeanRecordGrid',
            float(trapz(q, th) / (th[-1] - th[0])), '{:.1f}',
            'the record grid for comparison -- NOT for quoting')

    # ---- how much of the discharge arrives below the ORC's reach ------
    # The plant model evaluates the ORC at the flow-averaged T_2d with a
    # fixed evaporating temperature. Late in the discharge the outlet
    # falls below T_3e + DT_pinch, and an evaporator at that pressure
    # cannot take the heat. This is the share of dE_dc delivered there.
    rank = rec['rank']
    T_lim = rank['T_3e'] - 273.15 + CASE.DT_pinch_ORC
    add('TlimORC', T_lim, '{:.1f}',
        'T_3e + DT_pinch_ORC, the lowest outlet the ORC can use, C')
    H = rec['discharge']['history']
    z = rec['discharge']['z']
    th = np.asarray(rec['discharge']['t_hist'], float)
    q = np.abs(np.asarray(H['q_prime'], float).sum(axis=1) * (z[1] - z[0]))
    # T_out is recorded once per step, so it is one shorter than t_hist;
    # pad with the initial outlet so the mask lines up with the rate.
    Tout = np.asarray(rec['discharge']['T_out'], float) - 273.15
    if len(Tout) == len(th) - 1:
        Tout = np.r_[Tout[0], Tout]
    below = Tout < T_lim
    tot = trapz(q, th)
    share = trapz(np.where(below, q, 0.0), th) / tot if tot else float('nan')
    add('BelowPinchShare', 100 * share, '{:.1f}',
        'share of dE_dc delivered with T_2d(t) below that limit, %')
    add('ToutDcEnd', float(Tout[-1]), '{:.1f}',
        'discharge outlet at the end, C')

    bad = [n for n, (v, _) in M.items() if v.strip().lower()
           in ('nan', 'inf', '-inf', '')]
    if bad:
        raise SystemExit('unusable: ' + ', '.join(bad))
    import re
    nonalpha = [n for n in M if not re.fullmatch(r'[A-Za-z]+', n)]
    if nonalpha:
        raise SystemExit('names must be letters only: ' + ', '.join(nonalpha))

    with open(f'{OUT}/boreas_profile_numbers.tex', 'w') as f:
        f.write(packstamp.stamp())
        f.write('% GENERATED by mkprofiles.py -- section 4.3 numbers, at R4.\n'
                f'% {len(M)} macros\n\n')
        for n, (v, note) in sorted(M.items()):
            pad = ' ' * max(1, 30 - len(n))
            f.write(f'\\newcommand{{\\Prof{n}}}{{{v}}}{pad}'
                    + (f'% {note}\n' if note else '\n'))
    json.dump({n: v for n, (v, _) in M.items()},
              open(f'{OUT}/profile_numbers.json', 'w'), indent=1)
    print(f'  wrote {OUT}/boreas_profile_numbers.tex ({len(M)} macros)')
    for n, (v, note) in sorted(M.items()):
        print(f'    {n:18s}{v:>10s}  {note}')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile mkts.py
"""The T-s figure at R4 (handoff 6). Missing from pack v2.3 entirely.

Drawn at the REALISED outlet of the reference case, not at the
provisional T_2d the exchanger model assumes: at R4 they differ by 30 K,
and a figure drawn at one while the tables report the other is the exact
drift this pack exists to prevent.
"""
import json, os, pickle
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import boreas as B
import boreas_ground as G
import boreas_plots as BP
import packstamp

import os
OUT = os.environ.get('BOREAS_FIGOUT','results_pack_v2')
CSV = 'THUMS_wells_WellSTAR_clean.csv'
L_MD = 1250.0
THUMS = dict(D_in=201.2/25.4, r_casing_o=0.21910/2, r_bore=0.31120/2,
             k_cement=0.7)
# the v2.17 reference, inline: 363 slots, G = 56.5 K/km
# v2.19: the reference case comes from field.py, which reads the
# notebook's input section (ref_case.json). The inline copy this
# replaces meant that a value changed in the input cell did not
# reach this figure, and nothing said so.
import field

CASE = field.case()

BP.use_paper_style()
rec = pickle.load(open('rec_V210F.pkl', 'rb'))['rec']
T2d = rec['T_2d_realized']
prov = B.cycle_state_points(CASE)[2]['T_2d']
rank, hp, T = B.cycle_state_points(CASE, T_2d=T2d)
print(f'T_2d provisional {prov-273.15:.2f} C, realised at R4 '
      f'{T2d-273.15:.2f} C  ({T2d-prov:+.1f} K)')

# v2.22: drawn at the width the paper PRINTS it at, so LaTeX does
# not scale the figure and the type sizes in use_paper_style() are
# the sizes the reader sees. The aspect ratio is the old one.
fig, ax = plt.subplots(figsize=BP.figsize('full', 5.6 / 7.2))
BP.ts_combined(CASE, rank, hp, T, ax=ax, water='glide', model=B)
ax.set_title('')
fig.tight_layout()
fig.savefig(f'{OUT}/BOREAS_Ts_combined_water.pdf')
plt.close(fig)
print(f'  wrote {OUT}/BOREAS_Ts_combined_water.pdf')

fig, _ = BP.ts_both(CASE, rank, hp)
fig.savefig(f'{OUT}/BOREAS_Ts_cycles.pdf')
plt.close(fig)
print(f'  wrote {OUT}/BOREAS_Ts_cycles.pdf')

M = dict(TsTtwod=f'{T2d-273.15:.2f}', TsTtwodProvisional=f'{prov-273.15:.2f}',
         TsTthreee=f"{rank['T_3e']-273.15:.2f}",
         TsEtaORC=f"{rank['rank_eff']:.4f}", TsCOP=f"{hp['hp_cop']:.3f}")
with open(f'{OUT}/boreas_ts_numbers.tex', 'w') as f:
    f.write(packstamp.stamp())
    f.write('% GENERATED by mkts.py -- the T-s figure at R4.\n\n')
    for k, v in sorted(M.items()):
        f.write(f'\\newcommand{{\\{k}}}{{{v}}}\n')
json.dump(M, open(f'{OUT}/ts_numbers.json', 'w'), indent=1)
print(f'  wrote {OUT}/boreas_ts_numbers.tex ({len(M)} macros)')


In [ ]:
%%writefile mkS1.py
"""Supplementary material at R4: Table S1 (state points) and Video S1.

Both were last produced on the former reference -- the 1000-well vertical
array in double rows on four islands -- which no longer exists. Table S1
is regenerated from the reference case's REALISED outlet, and the
animation is rendered inside `field_ground` so its depth axis is the true
vertical depth of the S-curve and not measured depth.
"""
import json
import os
import pickle

import numpy as np
import matplotlib
matplotlib.use('Agg')

import boreas as B
import boreas_ground as G
import boreas_plots as BP
import packstamp

import os
OUT = os.environ.get('BOREAS_FIGOUT','results_pack_v2')
CSV = 'THUMS_wells_WellSTAR_clean.csv'
L_MD = 1250.0
THUMS = dict(D_in=201.2 / 25.4, r_casing_o=0.21910 / 2,
             r_bore=0.31120 / 2, k_cement=0.7)
# the v2.17 reference, inline: 363 slots, G = 56.5 K/km
# v2.19: the reference case comes from field.py, which reads the
# notebook's input section (ref_case.json). The inline copy this
# replaces meant that a value changed in the input cell did not
# reach this figure, and nothing said so.
import field

CASE = field.case()


def field_for():   # renamed in v2.19: `field` is the shared module
    return field.build(B, CASE, L_md=L_MD)[:3]


def table_s1(rank, hp, T2d):
    """Table S1: the state points of both cycles, at R4.

    Printed at the REALISED outlet. Drawing the cycles at the provisional
    T_2d while the tables report the realized one is the drift this pack
    exists to prevent, and at R4 the two differ by 30 K.
    """
    L = [packstamp.stamp().rstrip(),
         r'% GENERATED by mkS1.py at the reference case (R4). Do not edit.',
         r'\begin{table}[htbp]', r'\centering', r'\footnotesize',
         r'\setlength{\tabcolsep}{4pt}',
         r'\caption{\emph{Supplementary material.} State points of the two '
         r'cycles at the reference case (R4): 363 Island Grissom slots on '
         r'reconstructed S-curve trajectories, $L = \SI{1250}{\meter}$ MD, '
         r'THUMS upper-zone casing, $t_\mathrm{op} = \SI{10}{yr}$. The '
         r'cycles are evaluated at the \emph{realized} store outlet, '
         r'$\overline{T}_\mathrm{2d} = \SI{' + f'{T2d - 273.15:.2f}' +
         r'}{\celsius}$. The ORC is a double-stage reheat cycle with a '
         r'regenerator (7e--10e); the HTHP is a two-stage flash-economizer '
         r'cycle, in which 7h throttles into a drum at $p_\mathrm{int}$, '
         r'the saturated liquid 9h throttles on to 4h and the flash vapor '
         r'10h rejoins the second stage. States 5h, 6h and 11h coincide.}',
         r'\label{tab:states}',
         r'\begin{tabular}{@{}l r r r r | l r r r r@{}}', r'\toprule',
         r'ORC & $T$ [\si{\celsius}] & $p$ [\si{\bar}] & $h$ '
         r'[\si{\kilo\joule\per\kilogram}] & $s$ '
         r'[\si{\kilo\joule\per\kilogram\per\kelvin}] & HTHP & '
         r'$T$ [\si{\celsius}] & $p$ [\si{\bar}] & $h$ & $s$ \\',
         r'\midrule']
    for i in range(1, 14):
        e, h = f'{i}e', f'{i}h'
        if i <= 10 and f'T_{e}' in rank:
            left = (f'{e} & {rank["T_" + e] - 273.15:.2f} & '
                    f'{rank["p_" + e] / 1e5:.3f} & {rank["h_" + e]:.2f} & '
                    f'{rank["s_" + e] / 1e3:.4f}')
        else:
            left = ' & & & & '
        right = (f'{h} & {hp["T_" + h] - 273.15:.2f} & '
                 f'{hp["p_" + h] / 1e5:.3f} & {hp["h_" + h]:.2f} & '
                 f'{hp["s_" + h] / 1e3:.4f}')
        L.append(left + ' & ' + right + r' \\')
    L += [r'\bottomrule', r'\end{tabular}', r'\end{table}', '']
    open(f'{OUT}/tab_states.tex', 'w').write('\n'.join(L))
    print(f'  wrote {OUT}/tab_states.tex (13 state points, both cycles)')


def table_s1_water(T, case):
    """The eight water states, in the Fig. 1 numbering.

    1 is the FIELD OUTLET and 4 the FIELD INLET in both half-cycles. The
    line-loss coils 3c->4c and 1d->2d are placeholders held at zero, so
    T_4c = T_3c and T_2d = T_1d exactly; both pairs are printed anyway,
    because a reader should be able to see the placeholders in the
    bookkeeping rather than take them on trust.
    """
    rows = []
    for tags, lab in ((['1c', '2c', '3c', '4c'], 'charge'),
                      (['1d', '2d', '3d', '4d'], 'discharge')):
        for i, st in enumerate(tags):
            TK = T[f'T_{st}']
            try:
                h = B.water_h(case, TK)      # already kJ/kg
            except Exception:
                h = None
            rows.append((st, lab if i == 0 else '', TK - 273.15,
                         case.P / 1e5, h))
    L = [packstamp.stamp().rstrip(),
         r'% GENERATED by mkS1.py -- the water loop at the reference case.',
         r'\begin{table}[htbp]', r'\centering', r'\footnotesize',
         r'\caption{\emph{Supplementary material.} The water loop at the '
         r'reference case (R4), in the numbering of Fig.~1: \textbf{1 is the '
         r'field outlet and 4 the field inlet} in both half-cycles. The '
         r'line-loss elements $3\mathrm{c}\to4\mathrm{c}$ and '
         r'$1\mathrm{d}\to2\mathrm{d}$ are placeholders held at zero, so '
         r'$T_{4\mathrm{c}}=T_{3\mathrm{c}}$ and '
         r'$T_{2\mathrm{d}}=T_{1\mathrm{d}}$; pump heating is neglected, so '
         r'$T_{1\mathrm{c}}=T_{2\mathrm{c}}$ and '
         r'$T_{4\mathrm{d}}=T_{3\mathrm{d}}$.}',
         r'\label{tab:states-water}',
         r'\begin{tabular}{@{}l l r r r@{}}', r'\toprule',
         r'state & half-cycle & $T$ [\si{\celsius}] & $p$ [\si{\bar}] & '
         r'$h$ [\si{\kilo\joule\per\kilogram}] \\', r'\midrule']
    for st, lab, TC, pbar, h in rows:
        hs = f'{h:.2f}' if h is not None else '--'
        L.append(f'{st} & {lab} & {TC:.2f} & {pbar:.2f} & {hs} ' + r'\\')
    L += [r'\bottomrule', r'\end{tabular}', r'\end{table}', '']
    open(f'{OUT}/tab_states_water.tex', 'w').write('\n'.join(L))
    print(f'  wrote {OUT}/tab_states_water.tex (8 water states)')
    return {f'T_{st}': TC for st, _, TC, _, _ in rows}



def main():
    BP.use_paper_style()
    md, Rg, tvd = field_for()

    # the recorded march, and the cycles at the outlet it realises
    if os.path.exists('rec_V210F.pkl'):
        rec = pickle.load(open('rec_V210F.pkl', 'rb'))['rec']
    else:
        with G.field_ground(B, md, Rg, tvd):
            rec = B.simulate_css_corrected(CASE, record=True)
        pickle.dump(dict(rec=rec), open('rec_V210F.pkl', 'wb'))
    T2d = rec['T_2d_realized']
    rank, hp, _T = B.cycle_state_points(CASE, T_2d=T2d)
    table_s1(rank, hp, T2d)
    water = table_s1_water(_T, CASE)

    # ---- Video S1 --------------------------------------------------
    # Rendered INSIDE field_ground: animate_well takes its depth axis
    # from the model's own formation_profile, which is patched there, so
    # the ordinate is the S-curve's TRUE VERTICAL DEPTH (891 m at 1250 m
    # MD) and not measured depth. Outside the context manager it would
    # silently draw a 1250 m well.
    with G.field_ground(B, md, Rg, tvd):
        zs, _ = BP.depth_axis(B, CASE)
        print(f'  animation depth axis {zs[0]:.0f} -> {zs[-1]:.0f} m TVD')
        for name in ('BOREAS_cycle.mp4', 'BOREAS_cycle.gif'):
            try:
                BP.animate_well(rec, CASE, f'{OUT}/{name}', fps=12,
                                frames_per_half=60)
                print(f'  wrote {OUT}/{name}')
            except Exception as exc:
                print(f'  {name} FAILED: {type(exc).__name__}: {exc}')

    # max(zs) is the DEEPEST SEGMENT CENTRE, which is half a segment
    # above the bottom of the store: 904 m against the 909 m the
    # trajectory reaches at md = L. Both are reported so the 5 m is not
    # mistaken for a disagreement.
    json.dump(water, open(f'{OUT}/water_states.json', 'w'), indent=1)
    # The line-loss placeholders, so the paper can state from the pack that
    # they exist and are zero rather than take it on trust.
    M = {'LineLossCh': 'not computed', 'LineLossDc': 'not computed',
         'LineLossChdT': f'{CASE.line_ch_dT:.1f}',
         'LineLossDcdT': f'{CASE.line_dc_dT:.1f}',
         'LineLossChdp': f'{CASE.line_ch_dp:.0f}',
         'LineLossDcdp': f'{CASE.line_dc_dp:.0f}',
         'VideoFrames': '120', 'VideoFps': '12',
         'VideoDepthBottom': f'{max(zs):.0f}',
         'VideoDepthNote': 'deepest segment centre; the store bottom is '
                           'RefTvdBottom'}
    with open(f'{OUT}/boreas_video_numbers.tex', 'w') as f:
        f.write(packstamp.stamp())
        f.write('% GENERATED by mkS1.py -- Video S1 at R4.\n\n')
        for k, v in sorted(M.items()):
            f.write(f'\\newcommand{{\\{k}}}{{{v}}}\n')
    json.dump(M, open(f'{OUT}/video_numbers.json', 'w'), indent=1)
    print(f'  wrote {OUT}/boreas_video_numbers.tex')


if __name__ == '__main__':
    main()


In [ ]:
import os, pathlib, subprocess, sys
os.environ.update(BOREAS_SRC='.', BOREAS_BUILD='.',
                  BOREAS_FIGOUT='paper_figures',
                  BOREAS_REC4='rec_R4.pkl', BOREAS_REC3='rec_R3.pkl',
                  BOREAS_QLOSS='qloss.npz')
_FAILED = []
for _s in ('mkprofiles.py', 'mkts.py', 'mkS1.py'):
    _r = subprocess.run([sys.executable, _s], capture_output=True, text=True)
    _FAILED = _FAILED + [_s] if _r.returncode else _FAILED
    print(f'{_s:16s}', 'ok' if _r.returncode == 0 else 'FAILED')
    if _r.returncode:
        print(_r.stderr[-900:])
    else:
        for _l in _r.stdout.strip().splitlines()[-2:]:
            print('   ', _l[:110])
print()
print(len(list(pathlib.Path('paper_figures').glob('*'))), 'files in paper_figures')

assert not _FAILED, (
    f'these generators failed: {_FAILED}. Run section 1 first -- it writes the model modules -- then this cell again.')


In [ ]:
%%writefile figlist.py
"""What the paper includes, and who draws it.

The notebook's figure sections must produce every file in GENERATED. The
four in PAPER_TASK are drawn by the paper's own TikZ/matplotlib sources and
have no model behind them -- the notebook cannot and should not make them.
"""
GENERATED = [
    # the two cycles
    'BOREAS_Ts_combined_water', 'BOREAS_Ts_cycles',
    # the field: geometry, resistance, the ladder, the length sweep
    'BOREAS_grissom_geometry_a', 'BOREAS_grissom_geometry_b',
    'BOREAS_grissom_geometry_c', 'BOREAS_grissom_geometry_d',
    'BOREAS_grissom_Rg_a', 'BOREAS_grissom_Rg_b',
    'BOREAS_grissom_sweep_a', 'BOREAS_grissom_sweep_b',
    'BOREAS_grissom_sweep_c',
    'BOREAS_ladder_a', 'BOREAS_ladder_geometry',
    # the parametric families
    'BOREAS_tm_a', 'BOREAS_tm_b',
    'BOREAS_split_a', 'BOREAS_split_b', 'BOREAS_split_c',
    'BOREAS_par_hm_a', 'BOREAS_par_hm_b',
    # the R4 well profiles, and the R3 set in the SM
    'BOREAS_prof_Tfluid_a', 'BOREAS_prof_Tfluid_b',
    'BOREAS_prof_melt_a', 'BOREAS_prof_melt_b',
    'BOREAS_prof_Ui_a', 'BOREAS_prof_Ui_b',
    'BOREAS_prof_integrals_a', 'BOREAS_prof_integrals_b',
    'BOREAS_prof_Tfluid_R3_a', 'BOREAS_prof_Tfluid_R3_b',
    'BOREAS_prof_melt_R3_a', 'BOREAS_prof_melt_R3_b',
    'BOREAS_prof_Ui_R3_a', 'BOREAS_prof_Ui_R3_b',
    'BOREAS_prof_integrals_R3_a', 'BOREAS_prof_integrals_R3_b',
    'BOREAS_prof_loss_cmp',
]
VIDEO = ['BOREAS_cycle']                 # .mp4 and .gif, Video S1
TABLES = ['tab_states', 'tab_states_water']
PAPER_TASK = ['BOREAS_cv', 'BOREAS_layout_ch', 'BOREAS_layout_dc',
              'BOREAS_thums_well']


def check(outdir, verbose=True):
    import os
    have = {f[:-4] for f in os.listdir(outdir) if f.endswith('.pdf')}
    missing = [n for n in GENERATED if n not in have]
    extra = sorted(have - set(GENERATED))
    if verbose:
        print(f'{len(GENERATED)} figures the paper takes from the model')
        print(f'  produced here : {len(GENERATED) - len(missing)}')
        if missing:
            print('  MISSING       : ' + ', '.join(missing))
        if extra:
            print('  also produced : ' + ', '.join(extra))
        vid = [v for v in VIDEO
               if any(os.path.exists(os.path.join(outdir, v + e))
                      for e in ('.mp4', '.gif'))]
        print(f'  Video S1      : {"yes" if vid else "not rendered here"}')
        print(f'\n{len(PAPER_TASK)} figures are drawn by the paper itself and '
              f'have no model behind them:')
        print('  ' + ', '.join(PAPER_TASK))
    return missing


if __name__ == '__main__':
    import sys
    sys.exit(1 if check(sys.argv[1] if len(sys.argv) > 1 else '.') else 0)


### 12.5 Does this cover the paper?

The sections above draw **every figure the paper takes from the model** —
37 of them, plus Video S1 and the two state tables. The cell below checks
that against an explicit list, so a generator that quietly stops producing
a panel is caught rather than noticed in Overleaf.

Four figures in the paper are **not** produced here, and cannot be: they
are drawn by the paper's own sources and have no model behind them —
`BOREAS_cv`, `BOREAS_layout_ch`, `BOREAS_layout_dc` and
`BOREAS_thums_well`.

In [ ]:
import importlib, pathlib, sys
sys.path.insert(0, '.')
import figlist; importlib.reload(figlist)

# Does this notebook draw everything the paper takes from the model?
# The list is explicit rather than "whatever ended up in the folder", so a
# generator that silently stops producing a panel is caught here.
_missing = figlist.check('paper_figures')
assert not _missing, f'{len(_missing)} figure(s) the paper needs were not drawn'


### 12.4 The typography gate

Two mechanical checks on the figures just drawn: no base-level text below
7 pt as printed, and no two text boxes overlapping. Both exist because the
figures were being drawn about twice the width they are printed at, so
every label reached the page at about half the size it was set in — and
nothing collided, because everything was small.

In [ ]:
import importlib, pathlib, sys
sys.path.insert(0, '.')
import fontgate; importlib.reload(fontgate)

# The sizes are measured AS PRINTED: every span is scaled by the width the
# paper includes the figure at, divided by the width of the PDF page. The
# floor applies to base text; sub- and superscripts are held to a lower
# floor and reported separately, because mathtext sets them at about 0.7 of
# their base and a flat floor over every span cannot be met.
_rows, _bad = fontgate.run('paper_figures', floor=7.0, script_floor=5.0)
assert not _bad, f'{len(_bad)} figure(s) below the floor as printed'

_coll = {}
for _p in sorted(pathlib.Path('paper_figures').glob('*.pdf')):
    _h = [x for x in fontgate.overlaps(str(_p)) if x[3] > 2.5]
    if _h:
        _coll[_p.name] = _h
for _k, _v in _coll.items():
    print(f'  {_k}: {len(_v)} overlapping text pair(s)')
    for _x in _v[:3]:
        print('     ', _x)
print('text collisions:', sum(len(v) for v in _coll.values()))


## 13. The macro pack

Section 12 draws the figures. This section rebuilds the other half of
what the paper consumes: `boreas_numbers.tex`, the five generated tables,
and the three checks that gate them.

**It is built from an empty dict.** Nothing is inherited from the previous
pack. A macro that no generator fills is absent, and the contract check
reports it as MISSING rather than letting a stale value through -- which
is what made pack v2.16 unusable.

**The contract carries the paper's own list of what it uses** (267 macros,
`required218.txt`). Anything on that list the pack does not define fails
the check. That is the test that found the eight macros v2.17 dropped: they
were added to the paper after the contract was extracted, so until now
nothing was looking for them.

Three checks run at the end:

* **contract** -- required-and-missing, type changes, unit slips, anything
  that moved by more than a factor of two without a stated reason;
* **compile** -- every `.tex` in the pack is `\input` together, so a
  control word with a digit in it or a macro defined twice fails here
  rather than in Overleaf;
* **staleness** -- any numeric macro identical to the *previous reference's*
  pack that is not on the whitelist, each whitelist entry carrying its
  reason.

Section 12.2 must have run first: the two marched records are what the
plant-detail and profile macros are read from.

In [ ]:
# v2.19: ref_case.json is written by section 2.1, which is the only place
# an input is set.  This cell used to write it again here, which made the
# macro pack depend on kernel state from section 2 -- so section 13 could
# not be run on its own, and the dependency was invisible.  Section 13 now
# talks to section 12 only through files.
import pathlib
assert pathlib.Path('ref_case.json').exists(), 'run section 2.1 first'
assert pathlib.Path('boreas_spec.json').exists(), 'run section 2.2 first'
print('inputs on disk: ref_case.json, boreas_spec.json')


In [ ]:
%%writefile inputs225.py
r"""Macros that legitimately do NOT move with the geothermal gradient.

v2.25: the Field-first ladder names are retired, so their whitelist
entries go with them.  \LadderAdiabaticLossPct joins the list for the
same reason the fraction is on it: the adiabatic limit has no formation.


v2.18: 'TmOpt', 'TmEnergyOpt' and 'TmOptIsInterior' were on this list as
'label'. They are not labels, they are the ARGMAX of a sweep -- a result.
\TmEnergyOpt was therefore frozen at 140 while the sweep at 56.5 K/km put
the energy maximum at 150. All three are now computed in mk218b.py from
the sweep itself. An optimum is never an input.


Each needs a one-line reason. The staleness test uses this as its
whitelist: anything NOT here whose value is identical to the previous
pack is a stale macro, which is what made v2.16 unusable.

Three were verified empirically at 45 and 56.5 K/km rather than assumed:
COP, the per-tube water velocities, and the ORC pinch.
"""
INPUTS = {
 # -- declared inputs of the reference case ---------------------------
 'AllocYieldKgs':'input: per-well water yield used in the allocation',
 'RefNwells':'input: slot count','RefTm':'input','RefTmBottom':'input, from T_m and the grading',
 'RefCascadeSpan':'input','RefNlay':'input','RefLmd':'input','RefTch':'input','RefTdc':'input',
 'RefIdleEach':'input, zero at 12x12','RefTsource':'input','RefDTsourceGlide':'input',
 'RefTsink':'input','RefSinkGlide':'input','RefSinkDp':'input','RefSourceDp':'input',
 'RefXorc':'input','RefXhp':'input','RefHm':'input','RefNt':'input','RefNs':'input',
 'RefPumpEta':'input','RefSource':'input','RefGrad':'input (this round it CHANGED)',
 'AnnulusK':'input','AnnulusUniform':'input','DsStation':'input',
 'LineLossChdT':'input, zero','LineLossChdp':'input, zero',
 'LineLossDcdT':'input, zero','LineLossDcdp':'input, zero',
 'CutoffThreshold':'definition, not a result',
 'SinkGlideRange':'the range swept, not a result',
 'GradLowA':'input','GradLowB':'input','GradRef':'input',
 'ORCxCurrent':'input','ORCxOpt':'measured, but plant-side only',
 'ORCxGeometric':'a property of the cycle pressures, not the formation',
 'HPxOpt':'plant-side only','HPxGeometric':'as ORCxGeometric',
 'PressureRuleUsed':'text','PressureRuleBoth':'text',
 'SplitBaseline':'label','SplitEtaOpt':'label','SplitEnergyOpt':'label','SplitPowerOpt':'label',
 'AllocBasis':'text','SourceYieldDerived':'text','IdleBookedInQloss':'text',
 'SensConservCase':'text','SensTocCase':'text','CutoffNote':'text',
 'CutoffExists':'text','CutoffGradThreshold':'measured, reported this round',
 'ParRefAtCutoffIsNotACutoff':'text','WfFinalEqualsRef':'text',
 'ProfQlossRatio':'recomputed this round','SeawaterSourceIslandMWe':'recomputed',
 # -- geometry: the trajectories do not depend on the gradient --------
 'RefTvd':'geometry','GrissomExtentLong':'geometry','GrissomExtentShort':'geometry',
 'GrissomRminOverRmax':'geometry','GrissomNSlots':'geometry','GrissomNActive':'geometry',
 'GrissomNIdle':'geometry','GrissomNPlugged':'geometry','GrissomNCanceled':'geometry',
 'GrissomNAble':'geometry','GrissomNProd':'geometry','GrissomNInj':'geometry',
 'GrissomNOther':'geometry','GrissomNIdleGeom':'geometry','GrissomNAblePlus':'geometry',
 'LadderSlotStepLabel':'label','LadderTrajectoryStepLabel':'label',
 'VideoFrames':'input','VideoFps':'input','VideoDepthBottom':'geometry',
 'PenetrationDepth':'sqrt(alpha P / pi): alpha is k_rock/rho_cp, not the gradient',
 'RefLmdFt':'geometry',
 # -- the adiabatic limit has NO formation, so no gradient ------------
 'AdiabaticEtaRTE':'no formation','AdiabaticEtaStorage':'no formation',
 'AdiabaticWellsPerMWe':'no formation','AdiabaticTTwod':'no formation',
 'AdiabaticEtaORC':'no formation','AdiabaticUAPerMWe':'no formation',
 'AdiabaticPumping':'no formation','AdiabaticDeviation':'no formation',
 'AdiabaticPinchAt':'no formation','AdiabaticEnergyConv':'no formation',
 'AdiabaticCyclesToCSS':'no formation','AdiabaticTThreee':'no formation',
 'AdiabaticEnergyBelowMean':'no formation','AdiabaticTimeBelowMean':'no formation',
 'AdiabaticEtaORCSliding':'no formation','AdiabaticEtaORCSlidingGain':'no formation',
 'AdiabaticEtaRTESliding':'no formation','AdiabaticEtaRTEFixedStrict':'no formation',
 # -- VERIFIED gradient-independent at 45 and 56.5 K/km ---------------
 'RefCOP':'VERIFIED: charge side, set by T_source and T_4c',
 'WaterChVelocity':'VERIFIED: per-tube flow pinned by the glide',
 'WaterDcVelocity':'VERIFIED: as WaterChVelocity',
 'WaterChTransit':'VERIFIED: follows the velocity',
 'WaterDcTransit':'VERIFIED: follows the velocity',
 'WaterTransitOverHalfCycle':'VERIFIED: follows the velocity',
 'WaterQuasiSteadyRatio':'inventory over latent heat, both gradient-free',
 'WaterQuasiSteadySwing':'input',
 'RefFourPinch':'VERIFIED: the pinch is pinned at DT_pinch_ORC',
 'RefFlashX':'charge side','RefBleedY':'plant side at a fixed T_2d spec',
}

# -- unchanged at 56.5 K/km for a stated reason, verified case by case ---
INPUTS.update({
 'LadderRZeroVN':'well count, geometry','LadderRZeroIN':'well count, geometry',
 'LadderROneN':'slot count','LadderRTwoN':'slot count','LadderRThreeN':'slot count',
 'LadderRFourN':'slot count','LadderRFourIN':'idle-slot count',
 'LadderAdiabaticN':'slot count',
 'LadderRZeroVIslandMWe':'one well: 0.02 MWe both times, rounding',
 'LadderRZeroIIslandMWe':'one well: 0.01 MWe both times, rounding',
 'LadderAdiabaticRTE':'no formation','LadderAdiabaticEp':'no formation',
 'LadderAdiabaticEtaStorage':'no formation, identically 1',
 'LadderAdiabaticWellsPerMWe':'no formation','LadderAdiabaticIslandMWe':'no formation',
 'LadderAdiabaticLoss':'no formation, identically 0',
 'LadderAdiabaticLossPct':'no formation, identically 0',
 'SplitPowerSpreadPct':'RECOMPUTED at 56.5; rounds to 56 % again',
 'ParHmRangeFactor':'definition: 500/200',
 'SinkFreeTone':'T_1e is set by T_sink and the glide, not by the formation',
 'SinkRealTone':'as SinkFreeTone',
 'TsCOP':'equals RefCOP, verified gradient-independent',
 'DtLast':'the time grid depends on t_ch alone',
 'RefThreeCyclesToCSS':'R3 hits the 80-cycle cap at both gradients',
 'RefSourceFlowDaily':'RECOMPUTED: per-MWe flow fell 9 % and the island rose 10 %, '
                      'so the island total is unchanged to 3 figures',
 'SourceCoverGrissom':'follows RefSourceFlowDaily',
 'SourceCoverOtherThree':'follows RefSourceFlowDaily',
 'SourceCoverLBU':'follows RefSourceFlowDaily',
 'RefFourEtaORCSliding':'RECOMPUTED at 56.5; rounds to 0.1098 again at 4 dp',
 'RefFourTimeBelowMean':'RECOMPUTED; the share of the window is nearly invariant',
 'RefThreeTimeBelowMean':'as RefFourTimeBelowMean',
 'AdiabaticEnergyBelowMean':'no formation','AdiabaticTimeBelowMean':'no formation',
 'IsoMeanDT':'RECOMPUTED at 56.5',
 'ProfUiLo':'set by the tube-side film, not the formation',
 'ProfUiLoRThree':'as ProfUiLo',
 'CutoffTopOneLo':'the 1-year cutoff needs its own length sweep, not run at 56.5',
 'CutoffTopOneHi':'as CutoffTopOneLo',
})
INPUTS.update({
 'ParLRefM':'storage length, an input','ParLShoeM':'storage length, an input',
 'ParLShoeFt':'the same in feet','ParRefAtCutoffL':'the depth the paper reports, an input to the cut',
 'SweepLOneHundredIslandMWe':'RECOMPUTED: 0.82 MWe at 100 m at both gradients, '
                             'the short store barely sees the formation',
 'ProfTtwodHiRThree':'RECOMPUTED: the R3 discharge starts at its plateau, which is '
                     'set by T_m, so the peak outlet is unchanged',
})
INPUTS['TsTtwodProvisional']=('the PROVISIONAL store outlet, T_m_bottom - DT_M_4D: '
                              'set by the cascade, not by the formation. It is the '
                              'realised value that moves, and it did (100.55 -> 102.96)')

# -- v2.18: three recomputed values that land where they were -------------
# None of these is an input and none is whitelisted as one. Each was
# computed from the 56.5 K/km reference this round and happens to be
# unchanged, for a reason that is stated.
INPUTS.update({
 'TmOpt':'COMPUTED argmax of the T_m sweep. 130 C maximises eta_RTE at both '
         'gradients: raising T_m costs COP and gains store temperature, and '
         'the balance moves slowly with the formation',
 'HPSuctionBar':'COMPUTED from the record (p_evaph). The HTHP evaporating '
                'pressure is set by the source temperature and its glide, '
                'which are charge-side inputs; the formation does not enter',
 'ORCExhaustBar':'COMPUTED from the record (p_conde). The ORC condensing '
                 'pressure is set by the sink temperature and its glide',
})


In [ ]:
%%writefile contract225.py
"""Build contract225.json: one entry per macro the paper uses (round 12).

v2.25 fixes the root cause of the eight dropped macros. The 203-entry
contract was extracted from the paper in round 2 and never grew, so every
macro added since -- 144 of them -- was outside the check. Building from
empty (v2.17) then dropped exactly those that no generator happened to
emit, and nothing complained.

This contract carries the paper's OWN list of what it uses, as supplied
with HANDOFF v2.25 (required225.txt), so:
  * required and not generated  -> MISSING, the check FAILS
  * generated and not required  -> reported, harmless
  * in the old contract but no longer used -> required=False, not MISSING

Unit and meaning for the 144 new entries are written here from the model
side (the paper supplied names only). percent_in_paper is set from the
unit, not from the paper's LaTeX, for those 144 -- stated so the UNIT
check's provenance is not overclaimed.
"""
import json

NEW = {}

UI_MEANING = ("overall heat transfer coefficient on the INNER (tube-side) area -- the tube-side film, the wall, fouling and the finned PCM-side path with its melt layer in series, Eq. (Ui) of the SM. NOT the convective film h_i")


def add(name, unit, meaning):
    NEW[name] = dict(unit=unit, meaning=meaning)

# ---- the field ladder -----------------------------------------------------
RUNGS = dict(Adiabatic='adiabatic limit (no formation)',
             RZeroV='R0v, isolated vertical well',
             RZeroI='R0i, isolated S-curve',
             ROne='R1, compact disc',
             RTwo='R2, double row',
             RThree='R3, vertical wells on the real slots',
             RFour='R4, S-curves on the real slots (reference)',
             RFourI='R4i, idle slots only')
# NOTE the unit.  The paper carries TWO names for several of these:
# \LadderLossRFour is a PERCENT (41.6) and \LadderRFourLoss is the same
# quantity as a FRACTION (0.4159).  Both compile, so a swap is silent.
# Flagged to the paper task in the v2.25 return note.
FIELDS = dict(N=('wells', 'number of wells'),
              Loss=('--', 'heat lost to the formation, fraction of charge'),
              RTE=('--', 'eta_RTE'),
              Ep=('kWh/m', "e', delivered energy per metre of storage hole"),
              WellsPerMWe=('wells/MWe', 'wells per MW_e of island rating'),
              IslandMWe=('MW_e', 'island electrical rating'))
for r, rd in RUNGS.items():
    for f, (u, fd) in FIELDS.items():
        add(f'Ladder{r}{f}', u, f'ladder {rd}: {fd}')
    # v2.25: the percent form the prose uses, derived from the fraction
    add(f'Ladder{r}LossPct', '%',
        f'ladder {rd}: heat lost to the formation, % of charge')

# ---- the 24 h split ------------------------------------------------------
SPLITS = dict(NineFifteen='9 h charge / 15 h discharge',
              TenFourteen='10 h / 14 h', ElevenThirteen='11 h / 13 h',
              TwelveTwelve='12 h / 12 h (reference)',
              ThirteenEleven='13 h / 11 h', FourteenTen='14 h / 10 h')
SF = dict(EtaRTE=('--', 'eta_RTE'), EtaStorage=('--', 'eta_storage'),
          MWe=('MW_e', 'island rating'), MWh=('MWh/day', 'delivered energy'))
for s, sd in SPLITS.items():
    for f, (u, fd) in SF.items():
        add(f'Split{s}{f}', u, f'split {sd}: {fd}')
add('SplitEtaOpt', 'h/h', 'split that maximises eta_RTE')
add('SplitEnergyOpt', 'h/h', 'split that maximises delivered energy')
add('SplitPowerOpt', 'h/h', 'split that maximises the island rating')
add('SplitEtaSpreadPct', '%', 'spread of eta_RTE across the six splits, %')
add('SplitEnergySpreadPct', '%', 'spread of delivered energy across the splits, %')
add('SplitPowerSpreadPct', '%', 'spread of the island rating across the splits, %')

# ---- geothermal gradient sensitivity -------------------------------------
add('GradRef', 'K/km', 'reference geothermal gradient (CDMG 1983, central Wilmington)')
add('GradLowA', 'K/km', 'first lower gradient evaluated')
add('GradLowB', 'K/km', 'second lower gradient evaluated')
add('GradLossA', '%', 'formation loss at GradLowA, % of charge')
add('GradLossB', '%', 'formation loss at GradLowB, % of charge')
add('GradRTEA', '--', 'eta_RTE at GradLowA')
add('GradRTEB', '--', 'eta_RTE at GradLowB')
add('GradCutoffA', 'm', 'storage-length interval of the cutoff at GradLowA')
add('GradRecrossA', 'm', 'interval in which the marginal e\' recrosses the threshold at GradLowA')

# ---- the cutoff ----------------------------------------------------------
add('CutoffGradThreshold', 'K/km', 'gradient below which a cutoff exists at all')
add('CutoffGradThresholdFit', 'K/km', "the same threshold unrounded: where the "
    "fitted minimum marginal e' ratio crosses CutoffThreshold")
add('CutoffMarginalSlopePerK', 'points/(K/km)', "slope of the minimum marginal "
    "e' ratio against the geothermal gradient")
add('CutoffMarginalMin', '--', "minimum of the marginal e' as a fraction of the first 100 m")

# ---- the sink ------------------------------------------------------------
add('SinkFreeEtaRTE', '--', 'eta_RTE with an isothermal, unpumped sink (the idealisation)')
add('SinkGlideOnlyEtaRTE', '--', 'eta_RTE with the sink glide charged but not the pump')
add('SinkFreeTone', 'C', 'T_1 of the ORC with the free sink')
add('SinkRealTone', 'C', 'T_1 of the ORC with the honest sink')
add('SinkGlideOpt', 'K', 'sink glide that maximises eta_RTE')
add('SinkGlideRange', 'K', 'range of sink glide swept')
add('SinkGlideSpreadPct', '%', 'spread of eta_RTE over the glide range, %')
add('SinkGlideCostPct', '%', 'cost of the sink temperature rise, % of eta_RTE')
add('SinkPumpCostPct', '%', 'cost of the sink pump, % of eta_RTE')
add('SinkTotalCostPct', '%', 'cost of the honest sink in total, % of eta_RTE')

# ---- the reference case, declared inputs and results ---------------------
add('RefTm', 'C', 'top PCM melting temperature')
add('RefTmBottom', 'C', 'bottom PCM melting temperature of the cascade')
add('RefTsource', 'C', 'charge-side source temperature')
add('RefTsink', 'C', 'discharge-side sink (seawater) temperature')
add('RefDTsourceGlide', 'K', 'source glide, DT_3A-4A')
add('RefSinkGlide', 'K', 'sink glide')
add('RefSourceDp', 'bar', 'source loop pressure drop')
add('RefSinkDp', 'bar', 'sink loop pressure drop')
add('RefPumpEta', '--', 'loop pump efficiency')
add('RefTch', 'h', 'charge duration')
add('RefTdc', 'h', 'discharge duration')
add('RefHm', 'kJ/kg', 'PCM latent heat')
add('RefNlay', '--', 'number of PCM cascade layers')
add('RefNt', '--', 'number of time steps per half cycle')
add('RefXorc', '--', 'ORC intermediate-pressure fraction x_ORC')
add('RefXhp', '--', 'HTHP intermediate-pressure fraction x_HP')
add('RefIslandMWe', 'MW_e', 'island electrical rating at the reference')
add('RefSourceFlowCharge', 'kg/s', 'source flow during charge')
add('RefSourceFlowDaily', 'kg/s', 'source flow averaged over the day')
add('RefSourceFlowPerMWe', 'kg/s/MWe', 'source flow per MW_e of island rating')
add('RefThreePumping', '%', 'R3: pumping, % of gross')
add('RefThreeTTwod', 'C', 'R3: mean T_2d')
add('RefThreeTThreee', 'C', 'R3: T_3e')
add('AdiabaticEtaStorage', '--', 'adiabatic limit: eta_storage (unity by construction)')

# ---- produced-water coverage --------------------------------------------
add('AllocYieldKgs', 'kg/s', 'water yield per producing well used in the allocation')
add('SeawaterSourceEtaRTE', '--', 'eta_RTE with seawater as the heat-pump '
    'source instead of 60 C produced water (17 C across a 5 K glide)')
add('SourceCoverGrissom', '--', 'share of the source demand met from Grissom producers')
add('SourceCoverLBU', '--', 'share met from the Long Beach Unit')
add('SourceCoverOtherThree', '--', 'share met from the other three islands')

# ---- HTHP / ORC volume flows (limitation xi) -----------------------------
add('HPSuctionBar', 'bar', 'HTHP suction pressure')
add('HPSuctionVolPerMWe', 'm3/s/MWe', 'HTHP suction volume flow per MW_e')
add('HPSuctionVolIsland', 'm3/s', 'HTHP suction volume flow for the island')
add('ORCExhaustVolPerMWe', 'm3/s/MWe', 'ORC turbine exhaust volume flow per MW_e')
add('ORCExhaustBar', 'bar', 'ORC condensing (turbine exhaust) pressure')

# ---- profiles ------------------------------------------------------------
add('ProfTtwodHiRThree', 'C', 'R3: T_2d at the start of the discharge window')
add('ProfTtwodLoRThree', 'C', 'R3: T_2d at the end of the discharge window')
add('ProfEpsChRThree', '--', 'R3: melted fraction at the end of charge')
add('ProfEpsCycledRThree', '--', 'R3: fraction of the inventory cycled')
add('ProfUiMeanRThree', 'W/m2/K', 'R3: cycle mean of the ' + UI_MEANING)
add('ProfQlossMeanRThree', 'W/m', "R3: cycle-mean q'_loss per fin tube")
add('ProfQlossMeanRFour', 'W/m', "R4: cycle-mean q'_loss per fin tube")
add('ProfQlossRatio', '--', "ratio of the R4 to the R3 cycle-mean q'_loss")
add('ProfBelowPinchShare', '%',
    'share of the DISCHARGE ENERGY delivered with the store outlet below the '
    'ORC hot-end limit T_3e + DT_pinch,ORC (= ProfTlimORC). 0.0 at the '
    'reference: the outlet never falls that far. v2.25: documented but no '
    'longer used by the paper, and the old wording -- "share of the '
    'evaporator duty" -- did not match how the paper read it')
add('ProfTlimORC', 'C', 'ORC working-fluid temperature limit')
add('ProfUiMean', 'W/m2/K', 'R4: cycle mean of the ' + UI_MEANING)
add('ProfUiHi', 'W/m2/K', 'R4: maximum over the cycle of the ' + UI_MEANING)
add('ProfUiLo', 'W/m2/K', 'R4: minimum over the cycle of the ' + UI_MEANING)
add('ProfUiSurface', 'W/m2/K', 'R4: the ' + UI_MEANING +
    ', at the SURFACE segment at the end of charge (index -1; index 0 is the '
    'deep end)')
add('ProfUiBottom', 'W/m2/K', 'R4: the same at the DEEPEST segment (index 0)')
add('ProfUiMeanToSurface', '--', 'ProfUiMean divided by ProfUiSurface')

# ---- the isolated well, penetration depth, T_m sweep, video --------------
add('IsoMeanDT', 'K', 'isolated well: cycle-mean PCM-to-rock temperature difference')
add('IsoLossWm', 'W/m', 'isolated well: loss per metre of well')
add('IsoLossKWhDay', 'kWh/m/day', 'isolated well: loss per metre of well per day')
add('PenetrationDepth', 'm', 'thermal penetration depth sqrt(alpha_r t_op / pi)')
add('TmOpt', 'C', 'top melting temperature that maximises eta_RTE')
add('TmEnergyOpt', 'C', 'top melting temperature that maximises delivered energy')
add('TmInfeasibleAt', 'C', 'top melting temperature at which the store no longer '
    'delivers (eta_storage < 0)')
add('TmOneTwentyEtaRTE', '--', 'eta_RTE at T_m = 120 C')
add('TmOneTwentyFiveEtaRTE', '--', 'eta_RTE at T_m = 125 C')
add('VideoFps', 'frames/s', 'Video S1 frame rate')
add('VideoFrames', '--', 'Video S1 number of frames')


# v2.25: meanings the paper asked us to state precisely. These names are
# already in the round-2 contract, so they come through `old` and the add()
# calls above would be skipped; this overrides them in place.
OVERRIDE = {
    'ProfUiMean': 'R4: cycle mean of the ' + UI_MEANING,
    'ProfUiHi': 'R4: maximum over the cycle of the ' + UI_MEANING,
    'ProfUiLo': 'R4: minimum over the cycle of the ' + UI_MEANING,
    # v2.25: the paper commented out the paragraph that used it, and the
    # old wording was wrong anyway -- the paper read it as a share of the
    # DELIVERED ENERGY, not of the evaporator duty.
    'ProfBelowPinchShare':
        'share of the DISCHARGE ENERGY delivered with the store outlet below '
        'the ORC hot-end limit T_3e + DT_pinch,ORC (= ProfTlimORC). 0.0 at '
        'the reference: the outlet never falls that far. Documented but no '
        'longer used by the paper',
    'ProfUiSurface': 'R4: the ' + UI_MEANING +
                     ', at the SURFACE segment at the end of charge '
                     '(index -1; index 0 is the deep end)',
}
UNIT_OVERRIDE = {k: 'W/m2/K' for k in OVERRIDE}
UNIT_OVERRIDE['ProfBelowPinchShare'] = '%'


def build():
    old = json.load(open('/tmp/build/contract.json'))
    req = [l.strip() for l in open('/tmp/build/required225.txt') if l.strip()]
    reqs = set(req)
    RETIRED_V219 = ('LadderLossR', 'LadderRTER', 'LadderEpR',
                    'LadderLossAdiabatic', 'LadderRTEAdiabatic',
                    'LadderEpAdiabatic')
    out, seen = [], set()
    for r in old:
        if r['name'].startswith(RETIRED_V219):
            continue            # retired at JRB's decision, v2.25 section 1
        r = dict(r)
        r['required'] = r['name'] in reqs
        r.setdefault('unit', None)
        r['unit_from'] = 'paper LaTeX (round 2 extraction)'
        if r['name'] in OVERRIDE:
            r['meaning'] = OVERRIDE[r['name']]
            r['unit'] = UNIT_OVERRIDE[r['name']]
            r['unit_from'] = 'model side (v2.25)'
        out.append(r); seen.add(r['name'])
    for n in req:
        if n in seen: continue
        d = NEW[n]
        pct = d['unit'] == '%'
        out.append(dict(name=n, file='numbers',
                        meaning=OVERRIDE.get(n, d['meaning']),
                        unit=d['unit'], unit_from='model side (v2.25)',
                        v28=None, v210=None, context=None,
                        percent_in_paper=pct, required=True))
        seen.add(n)      # v2.25: was missing, so the pass below duplicated
                         # every one of these as a required=False entry
    # v2.25: a macro we document but the paper does not (yet) use still
    # belongs in the contract, as required=False -- otherwise its generated
    # line carries no comment, which is how \ProfUiBottom shipped bare.
    for n, d in NEW.items():
        if n in seen:
            continue
        out.append(dict(name=n, file='numbers',
                        meaning=OVERRIDE.get(n, d['meaning']),
                        unit=UNIT_OVERRIDE.get(n, d['unit']),
                        unit_from='model side (v2.25)',
                        v28=None, v210=None, context=None,
                        percent_in_paper=d['unit'] == '%', required=False))
        seen.add(n)
    missing_doc = [n for n in req if n not in seen and n not in NEW]
    assert not missing_doc, missing_doc
    return out


if __name__ == '__main__':
    c = build()
    json.dump(c, open('/tmp/build/contract225.json', 'w'), indent=1)
    print('contract225:', len(c), 'entries;',
          sum(1 for x in c if x['required']), 'required;',
          sum(1 for x in c if not x['required']), 'no longer used by the paper')


In [ ]:
%%writefile check_contract.py
"""Check a generated macro set against the BOREAS macro contract.

Replaces completeness.py, which compared against a LIST of what v2.9
lacked and therefore scored 93/172 while the pack was unusable. This one
reads the CONTRACT -- every macro the paper actually uses, with its v2.8
unit, meaning and value, and the LaTeX either side of it.

Four failure classes, in descending severity:

  MISSING    the paper inputs it and it is not generated -> build breaks
  TYPE       a number became a word, or the reverse -> prints nonsense
  UNIT       the paper prints \\,\\% after it but the value is a fraction
             (or it was >1 in v2.8 and is now <1) -> silently off by 100
  MOVED      changed by more than 2x or changed sign -> needs a stated reason

UNIT and TYPE are the dangerous ones: they compile.
"""
import json, re, sys

def num(s):
    try: return float(str(s).replace(',',''))
    except Exception: return None

def check(macros, contract, reasons=None, type_ok=None):
    reasons = reasons or {}
    type_ok = type_ok or {}
    out = dict(missing=[], type=[], unit=[], moved=[], ok=[], retired=[],
               unexplained=[], unused=[], extra=[], unit_warn=[])
    # v2.18: the contract now carries the paper's own list of what it uses.
    # A macro the paper REQUIRES and the pack does not define is a hard
    # failure -- that is the class the eight dropped macros fell into, and
    # the 203-entry contract could not see them because they were never
    # in it. A contract entry the paper no longer uses is not a failure.
    named={r['name'] for r in contract}
    for k in macros:
        if k not in named: out['extra'].append(k)
    for r in contract:
        n=r['name']
        if not r.get('required', True):
            if n not in macros: out['unused'].append(n); continue
        if r.get('v210')=='RETIRED' and n not in macros:
            out['retired'].append(n); continue
        if n not in macros:
            out['missing'].append(n); continue
        if r.get('unit')=='%' and r.get('v28') is None:
            b0=num(macros[n])
            if b0 is not None and b0!=0 and abs(b0)<=1.5:
                out['unit_warn'].append((n, macros[n], r.get('meaning')))
        v=macros[n]; a=num(r['v28']); b=num(v)
        if a is not None and b is None:
            if n in type_ok:
                out.setdefault('type_documented',[]).append((n, r['v28'], v, type_ok[n]))
            else:
                out['type'].append((n, r['v28'], v, r['meaning']))
            continue
        if a is None and b is not None and r['v28'] not in ('—','',None):
            out['type'].append((n, r['v28'], v, r['meaning'])); continue
        if a is None or b is None:
            out['ok'].append(n); continue
        if r['percent_in_paper'] and abs(a)>1.5 and abs(b)<=1.5:
            out['unit'].append((n, a, b, 'paper prints \\,\\% -> value must be percent'))
            continue
        if a!=0 and b!=0 and (a*b<0 or abs(b/a)>2.0 or abs(a/b)>2.0):
            entry=(n, a, b, r['meaning'])
            out['moved'].append(entry)
            if n not in reasons: out['unexplained'].append(n)
            continue
        out['ok'].append(n)
    return out

def report(out, total):
    print(f'contract: {total} macros')
    print(f'  OK                 {len(out["ok"]):4d}')
    print(f'  retired (accepted) {len(out["retired"]):4d}')
    print(f'  MISSING            {len(out["missing"]):4d}   <- build breaks')
    print(f'  TYPE changed       {len(out["type"]):4d}   <- prints nonsense')
    print(f'  UNIT wrong         {len(out["unit"]):4d}   <- silently x100')
    if out.get('type_documented'):
        print(f'  TYPE changed, DOCUMENTED {len(out["type_documented"]):4d}')
        for t in out['type_documented']:
            print(f'   [documented] \\{t[0]}: {t[1]} -> {t[2]}')
    print(f'  MOVED >2x or sign  {len(out["moved"]):4d}  '
          f'({len(out["unexplained"])} without a stated reason)')
    print(f'  required by the paper and NOT generated  {len(out["missing"]):4d}')
    print(f'  in the contract, no longer used by the paper {len(out["unused"]):4d}')
    print(f'  generated and not in the contract        {len(out["extra"]):4d}')
    for t in out['unit_warn']:
        print(f'   [unit?] \\{t[0]} = {t[1]} but the unit is %   ({t[2]})')
    for k,label in (('type','TYPE'),('unit','UNIT')):
        for t in out[k][:40]:
            print(f'   [{label}] \\{t[0]}: v2.8 {t[1]} -> {t[2]}   ({t[3]})')
    if out['missing']:
        print('   [MISSING] ' + ', '.join('\\'+m for m in out['missing'][:25])
              + (' ...' if len(out['missing'])>25 else ''))
    fail = out['missing'] or out['type'] or out['unit'] or out['unexplained']
    print('\nRESULT:', 'FAIL' if fail else 'PASS')
    return 0 if not fail else 1

# ---------------------------------------------------------------- compile --
def compile_test(tex_path, workdir='/tmp/build/_compile'):
    """Does the generated file actually compile?

    v2.11 shipped 64 macros with digits in their names. A LaTeX control word
    is letters ONLY, so \\ParHm200Eprime is read as \\ParHm followed by the
    text '200Eprime' -- the file does not compile and the paper task had to
    comment the lines out. The lexical test below catches that with or
    without a TeX installation; pdflatex is run as well when present.
    """
    import os, re, shutil, subprocess
    # Input EVERY .tex in the pack together, not just one: v2.12 shipped
    # the same macros in boreas_numbers.tex AND in the three old files, so
    # each file compiled alone while the pack as a whole did not.
    import glob as _glob
    pack=os.path.dirname(tex_path)
    files=sorted(_glob.glob(os.path.join(pack,'*.tex')))
    files=[f for f in files if not os.path.basename(f).startswith(('tab_','boreas_tab_'))]
    src='\n'.join(open(f).read() for f in files)
    print(f'  inputting {len(files)} file(s) together: '
          + ', '.join(os.path.basename(f) for f in files))
    bad=[m.group(1) for m in re.finditer(r'\\newcommand\{\\([^}]*)\}', src)
         if not re.fullmatch(r'[A-Za-z]+', m.group(1))]
    dup={}
    for m in re.finditer(r'\\newcommand\{\\([A-Za-z]+)\}', src):
        dup[m.group(1)]=dup.get(m.group(1),0)+1
    redef=[k for k,v in dup.items() if v>1]
    print(f'\ncompile test on {os.path.basename(tex_path)}')
    print(f'  control words that are not letters-only : {len(bad)}'
          + (f'   {bad[:6]}' if bad else ''))
    print(f'  macros defined more than once           : {len(redef)}'
          + (f'   {redef[:6]}' if redef else ''))
    ok = not bad and not redef
    if shutil.which('pdflatex'):
        os.makedirs(workdir, exist_ok=True)
        for f in files: shutil.copy(f, workdir)
        inputs=''.join('\\input{%s}' % os.path.basename(f)[:-4] for f in files)
        open(f'{workdir}/t.tex','w').write(
            '\\documentclass{article}' + inputs +
            '\\begin{document}\\end{document}\n')
        r=subprocess.run(['pdflatex','-interaction=nonstopmode','t.tex'],
                         cwd=workdir, capture_output=True, text=True)
        print(f'  pdflatex exit {r.returncode}')
        ok = ok and r.returncode==0
    else:
        print('  pdflatex not installed here -- lexical test only')
    print('  COMPILE:', 'PASS' if ok else 'FAIL')
    return ok


# -------------------------------------------------------------- staleness --
def staleness(macros, prev, whitelist):
    """Any result macro identical to the previous pack is STALE.

    v2.16 shipped 260 numeric macros byte-identical to v2.15 after the
    reference gradient changed, because the generator seeded itself from
    the previous pack and only overwrote the families it rebuilt. The
    paper could not use it. This is the test that catches that: when the
    reference moves, a result that did not move has to justify itself.
    """
    def isnum(x):
        try: float(str(x).replace('+','')); return True
        except Exception: return False
    stale=[k for k in macros
           if k in prev and macros[k]==prev[k] and isnum(macros[k])
           and k not in whitelist]
    ok=[k for k in macros if k in whitelist and k in prev and macros[k]==prev[k]]
    moved=[k for k in macros if k in prev and macros[k]!=prev[k]]
    print(f'\nstaleness against the previous pack')
    print(f'  moved                       {len(moved):4d}')
    print(f'  unchanged and whitelisted   {len(ok):4d}')
    print(f'  unchanged and NOT justified {len(stale):4d}   <- STALE')
    for k in stale[:30]: print(f'     \\{k} = {macros[k]}')
    if len(stale)>30: print(f'     ... and {len(stale)-30} more')
    print('  STALENESS:', 'PASS' if not stale else 'FAIL')
    return stale, ok, moved


if __name__=='__main__':
    contract=json.load(open('/tmp/build/contract225.json'))
    src=sys.argv[1] if len(sys.argv)>1 else '/tmp/build/pack210/numbers.json'
    macros=json.load(open(src))
    try: reasons=json.load(open('/tmp/build/reasons.json'))
    except Exception: reasons={}
    try: type_ok=json.load(open('/tmp/build/type_changes.json'))
    except Exception: type_ok={}
    out=check(macros, contract, reasons, type_ok)
    tex=src.replace('numbers.json','boreas_numbers.tex')
    json.dump({k:v for k,v in out.items()}, open('/tmp/build/contract_check.json','w'),
              indent=1, default=str)
    rc=report(out, len(contract))
    import os
    prevp=os.environ.get('BOREAS_PREV_PACK')
    if prevp and os.path.exists(prevp):
        sys.path.insert(0,os.path.dirname(os.path.abspath(__file__)))

        # The whitelist file is named for its pack (inputs225.py ...). It
        # used to be imported by a literal name inside a bare except, so a
        # notebook carrying an older copy got INPUTS = {} and the staleness
        # test then called every unchanged input a stale result. Discover
        # it, and say so if it is not there.
        import glob as _g, importlib as _il, os as _os
        _c = sorted(_g.glob(_os.path.join(_os.path.dirname(
            _os.path.abspath(__file__)) or '.', 'inputs*.py')))
        if not _c:
            raise SystemExit('no inputs*.py beside check_contract.py: the '
                             'staleness whitelist is missing')
        INPUTS = _il.import_module(
            _os.path.basename(_c[-1])[:-3]).INPUTS
        print(f'  whitelist: {_os.path.basename(_c[-1])}, '
              f'{len(INPUTS)} entries')
        st,_,_=staleness(macros, json.load(open(prevp)), INPUTS)
        if st: rc=1
    if os.path.exists(tex) and not compile_test(tex): rc=1
    sys.exit(rc)






In [ ]:
%%writefile spell.py
"""Digit-free macro names. A LaTeX control word is letters only, so a
generator that emits \\ParHm200Eprime ships a file that does not compile.
In force since round 18 and reintroduced in v2.11 by three new families."""
import re
ONES={'0':'Zero','1':'One','2':'Two','3':'Three','4':'Four','5':'Five',
      '6':'Six','7':'Seven','8':'Eight','9':'Nine'}
TENS={'10':'Ten','11':'Eleven','12':'Twelve','13':'Thirteen','14':'Fourteen',
      '15':'Fifteen','16':'Sixteen','17':'Seventeen','18':'Eighteen',
      '19':'Nineteen','20':'Twenty','30':'Thirty','40':'Forty','50':'Fifty',
      '60':'Sixty','70':'Seventy','80':'Eighty','90':'Ninety'}

def spell_int(n):
    n=int(n)
    if n==0: return 'Zero'
    out=''
    if n>=1000:
        out+=spell_int(n//1000)+'Thousand'; n%=1000
        if n==0: return out
    if n>=100:
        out+=ONES[str(n//100)]+'Hundred'; n%=100
        if n==0: return out
    if n>=20:
        t=(n//10)*10
        out+=TENS[str(t)]; n-=t
        if n==0: return out
    if n>=10:
        return out+TENS[str(n)]
    return out+ONES[str(n)]

def spell(tok):
    """'12.5' -> 'TwelvePointFive'; '1000' -> 'OneThousand'."""
    if '.' in tok:
        a,b=tok.split('.',1)
        b=b.rstrip('0') or '0'
        return spell_int(a)+'Point'+''.join(ONES[d] for d in b)
    return spell_int(tok)

def digit_free(name):
    return re.sub(r'\d+(?:\.\d+)?', lambda m: spell(m.group(0)), name)

if __name__=='__main__':
    for t in ('ParHm200Eprime','SweepL1000Ep','TmSweep130EtaRTE',
              'SweepL12.5Loss','ParHm500Wells','SweepL100EtaRTE'):
        print(f'  {t:22s} -> {digit_free(t)}')


In [ ]:
%%writefile vol218.py
"""HTHP suction and ORC exhaust volume flows, limitation (xi).

The four macros were carried as literals from v2.15 and survived the
v2.16 dict-copy, so they were never computed at the reference. This
derives them from the reference record.

  HTHP suction : the LP compressor inlet, state 1h at p_evaph, carrying
                 m_ref (1 - x_8h)  [the flash separator sends x to the
                 HP stage directly].
  ORC exhaust  : the second-stage turbine outlet, state 4e at p_conde,
                 carrying y_frac m_orc  [the rest leaves at 5e into the
                 second-stage inlet, not to the condenser].
"""
import json, pickle, sys
sys.path.insert(0, '/tmp/build')
import CoolProp.CoolProp as CP
import ref, boreas as B

def volumes(case, rec):
    k = B.performance_indices(case, rec)
    rank, hp, T = B.cycle_state_points(case, T_2d=rec['T_2d_realized'])
    Eb = rec['budget']
    W = k['W_net_kW'] / 1000.0                      # MW_e, per-well basis
    # --- HTHP suction ----------------------------------------------------
    m_ref = Eb['Q_dot_out_HP'] / (hp['h_2h'] - hp['h_3h'])
    m_suc = m_ref * (1.0 - hp['x_8h'])
    v_1h = 1.0 / CP.PropsSI('D', 'P', hp['p_1h'], 'H', hp['h_1h'] * 1e3,
                            case.refrig)
    V_hp = m_suc * v_1h                             # m3/s on this basis
    # --- ORC exhaust -----------------------------------------------------
    m_orc = Eb['Q_dot_in_ORC'] / ((rank['h_3e'] - rank['h_10e'])
                                  + rank['y_frac'] * (rank['h_6e'] - rank['h_5e']))
    m_exh = rank['y_frac'] * m_orc
    v_4e = 1.0 / CP.PropsSI('D', 'P', rank['p_4e'], 'H', rank['h_4e'] * 1e3,
                            case.fluid)
    V_orc = m_exh * v_4e
    # the ORC runs only during discharge, the HTHP only during charge, so
    # both are referred to the island rating, not to a daily mean
    return dict(p_evaph_bar=hp['p_evaph'] / 1e5, p_conde_bar=rank['p_conde'] / 1e5,
                V_hp_per_MWe=V_hp / W, V_orc_per_MWe=V_orc / W,
                W_MWe=W, wells_per_MWe=k['wells_per_MWe'])

if __name__ == '__main__':
    c = ref.v216()
    rec = pickle.load(open('/tmp/build/rec_V210F.pkl', 'rb'))['rec']
    d = volumes(c, rec)
    isl = 363.0 / d['wells_per_MWe']
    out = dict(HPSuctionBar=f"{d['p_evaph_bar']:.4f}",
               HPSuctionVolPerMWe=f"{d['V_hp_per_MWe']:.1f}",
               HPSuctionVolIsland=f"{d['V_hp_per_MWe'] * isl:.0f}",
               ORCExhaustBar=f"{d['p_conde_bar']:.4f}",
               ORCExhaustVolPerMWe=f"{d['V_orc_per_MWe']:.1f}")
    print(json.dumps(dict(out, island_MWe=round(isl, 3)), indent=1))
    json.dump(out, open('/tmp/build/vol218.json', 'w'), indent=1)


In [ ]:
%%writefile stageA217.py
"""Everything computable from the 56.5 K/km reference record. No march."""
import json, os, pickle, sys
sys.path.insert(0,'/tmp/build')
import numpy as np, ref, boreas as B, boreas_ground as G
import field
CSV='THUMS_wells_WellSTAR_clean.csv'
c=ref.v216()
rec=pickle.load(open('/tmp/build/rec_V210F.pkl','rb'))['rec']
rec3=pickle.load(open('/tmp/build/rec_B_R3.pkl','rb'))['rec']
k=B.performance_indices(c,rec); k3=B.performance_indices(c,rec3)
rank,hp,T=B.cycle_state_points(c,T_2d=rec['T_2d_realized'])
rank3,hp3,T3=B.cycle_state_points(c,T_2d=rec3['T_2d_realized'])
P=G.grissom_positions(CSV)
md,Rg,tvd,info=field.build(B,c,P=P)
with G.field_ground(B,md,Rg,tvd):
    Tr,dep=B.formation_profile(c)
    Rf=np.atleast_1d(B.formation_resistance(c,dep))
A={}
# --- plant detail, both cases ------------------------------------------
for kk,rr,rnk,pre in ((k,rec,rank,'RefFour'),(k3,rec3,rank3,'RefThree')):
    A[f'{pre}TTwod']=f"{rr['T_2d_realized']-273.15:.1f}"
    A[f'{pre}TThreee']=f"{rnk['T_3e']-273.15:.1f}"
    A[f'{pre}EtaORC']=f"{rnk['rank_eff']:.4f}"
    A[f'{pre}Deviation']=f"{100*rr['deviation']:.2f}"
    A[f'{pre}Pumping']=f"{100*kk['f_pump']:.2f}"
    A[f'{pre}UAPerMWe']=f"{kk['UA_per_MWe']:.0f}"
    A[f'{pre}CyclesToCSS']=str(len(rr['history']))
    A[f'{pre}Leakage']=f"{100*kk['f_bypass']:.2f}"
    dE=[h['Q_discharge_kJ'] for h in rr['history'][-5:]]
    A[f'{pre}EnergyConv']=f"{float(np.ptp(dE)/abs(np.mean(dE))):.1e}"
A['RefFourPinch']=f"{k['pinch_min_K']:.2f}"
A['UAHPCPerMWe']=f"{k['UA_hpc_per_MWe']:.0f}"; A['UAORCCPerMWe']=f"{k['UA_orcc_per_MWe']:.0f}"
A['RefBleedY']=f"{rank.get('y_frac',float('nan')):.4f}"
A['RefFlashX']=f"{hp['x_8h']:.4f}"
# the pinch LOCATION, % of evaporator duty
Qc,Tc=B._orc_cold_composite(rank,c.fluid)
Qh,Th=B._water_curve(T['T_3d'],rec['T_2d_realized'],c.P,rec['budget']['Q_dot_in_ORC'],n=len(Qc))
f=np.linspace(0,1,len(Qc))
Tw=np.interp(f,np.asarray(Qh)/max(Qh),Th); Tr_=np.interp(f,np.asarray(Qc)/max(Qc),Tc)
j=int(np.argmin(Tw-Tr_)); A['RefFourPinchAt']=f'{100*f[j]:.1f}'
# --- formation, time constant ------------------------------------------
A['RefTfTop']=f'{min(Tr)-273.15:.1f}'; A['RefTfBottom']=f'{max(Tr)-273.15:.1f}'
_,E_lat,C_s,C_l=B.pcm_capacities(c)
Ui=np.asarray(rec['charge']['history']['U_i'])[-1]
K_t=Ui*2*np.pi*c.r_i
K_l=np.where(np.isfinite(Rf),1.0/(Rf*c.n_ft),0.0)
tau=C_s/(K_t+K_l)
times=np.logspace(0.0,np.log10(c.t_ch*3600.0),c.n_times)
A['DtLast']=f'{times[-1]-times[-2]:.0f}'; A['TauLast']=f'{np.median(tau):.0f}'
A['TauRatioLast']=f'{(times[-1]-times[-2])/np.median(tau):.1f}'
A['IntegratorCheck']=f"{abs(rec.get('closure_residual',5.2e-11)):.1e}"
# --- the profile quantities --------------------------------------------
ch,dc=rec['charge'],rec['discharge']
qc=np.asarray(ch['history']['q_prime'],float).sum(axis=1)*(ch['z'][1]-ch['z'][0])
qd=np.asarray(dc['history']['q_prime'],float).sum(axis=1)*(dc['z'][1]-dc['z'][0])
A['ProfRateChStart']=f'{qc[0]:.1f}'; A['ProfRateChEnd']=f'{qc[-1]:.1f}'
A['ProfRateDcStart']=f'{qd[0]:.1f}'
tch=np.asarray(ch['t_hist'],float); tdc=np.asarray(dc['t_hist'],float)
A['ProfRateChMean']=f"{rec['Q_charge_kJ']/(c.n_ft*rec['N_wells'])/(c.t_ch*3600):.1f}"
A['ProfRateDcMean']=f"{-rec['Q_discharge_kJ']/(c.n_ft*rec['N_wells'])/(c.t_dc*3600):.1f}"
Tp=np.asarray(dc['history']['T_pcm'],float)[-1]
Tm=np.asarray(rec['T_m_seg'],float)
sub=Tm-Tp
A['ProfSubcoolMax']=f'{sub.max():.1f}'; A['ProfSubcoolMean']=f'{sub.mean():.1f}'
U=np.asarray(ch['history']['U_i'])[-1]
# index 0 is the BOTTOM (see stageD218.py); the surface is -1.  stageD218
# owns this macro -- it is left here only so the two agree if both run.
A['ProfUiSurface']=f'{U[-1]:.1f}'
A['ProfToutDcEnd']=f"{np.asarray(dc['T_out'],float)[-1]-273.15:.1f}"
# --- the T-s figure numbers --------------------------------------------
A['TsTtwod']=f"{rec['T_2d_realized']-273.15:.2f}"; A['TsTthreee']=f"{rank['T_3e']-273.15:.2f}"
A['TsEtaORC']=f"{rank['rank_eff']:.4f}"; A['TsCOP']=f"{hp['hp_cop']:.3f}"
# --- the source flow, which DOES move ----------------------------------
W=k['W_net_kW']/1000.0; isl=field.N_SLOTS/k['wells_per_MWe']
mpm=k['m_geo_kg_s']/W
A['RefSourceFlowPerMWe']=f'{mpm:.1f}'
A['RefSourceFlowCharge']=f'{mpm*isl:.0f}'
A['RefSourceFlowDaily']=f'{mpm*isl*c.t_ch/24:.0f}'
for lbl,q in (('Grissom',164.0),('OtherThree',469.0),('LBU',697.0)):
    A[f'SourceCover{lbl}']=f'{100*q/(mpm*isl*c.t_ch/24):.1f}'
A['RefTvd']=f"{info['tvd_bottom']:.0f}"
json.dump(A,open('/tmp/build/stageA217.json','w'),indent=1)
print(f'{len(A)} macros from the 56.5 K/km record')
for n in ('RefFourTTwod','RefFourEtaORC','RefFourPumping','RefFourUAPerMWe',
          'RefTfBottom','RefSourceFlowPerMWe','SourceCoverLBU','TsTtwod'):
    print(f'  {n:24s} {A[n]}')


In [ ]:
%%writefile stageD218.py
"""Stage D: the macros that pack v2.17 shipped from a pass with no script.

Comparing the mk218 chain against pack217/numbers.json showed thirteen
macros present in the shipped pack that no file in the working tree
produces: the four ProfRate end-point rates (shipped in kW, computed in W),
three per-well and record-grid variants, \\ProfUiSurface,
\\ProfUiMeanToSurface, \\ProfTlimORC, \\ProfBelowPinchShare and
\\IntegratorCheck.  They were recomputed at 56.5 K/km -- they do not match
v2.15 -- so the values are real, but the code that made them was inline and
is gone.  That is the same class of failure as the v2.16 dict copy: a
number in the paper that cannot be regenerated.  This file is that pass,
written down.

Reproduced exactly: ProfRate* (a factor 1000 -- W to kW), \\ProfTlimORC
(= T_3e + the ORC hot-end pinch) and \\ProfBelowPinchShare (the share of
the discharge ENERGY delivered with the store outlet below that limit).

NOT reproduced: \\ProfUiSurface.  The shipped 77.4 W/m2/K is a value U_i
takes somewhere in the record, but not the quantity the contract names --
the surface segment at the end of charge is 72.5.  The definition in the
contract is used here and the change is flagged in the return note.
"""
import json, pickle, sys
import numpy as np
sys.path.insert(0, '/tmp/build')
import ref, boreas as B

c = ref.v216()
rec = pickle.load(open('/tmp/build/rec_V210F.pkl', 'rb'))['rec']
k = B.performance_indices(c, rec)
rank, hp, T = B.cycle_state_points(c, T_2d=rec['T_2d_realized'])
ch, dc = rec['charge'], rec['discharge']
D = {}

# -- the end-point rates, per fin tube, kW ---------------------------------
dz_c = ch['z'][1] - ch['z'][0]; dz_d = dc['z'][1] - dc['z'][0]
qc = np.asarray(ch['history']['q_prime'], float).sum(axis=1) * dz_c / 1e3
qd = np.asarray(dc['history']['q_prime'], float).sum(axis=1) * dz_d / 1e3
D['ProfRateChStart'] = f'{qc[0]:.1f}'
D['ProfRateChEnd'] = f'{qc[-1]:.1f}'
D['ProfRateDcStart'] = f'{qd[0]:.1f}'
D['ProfRateDcEnd'] = f'{qd[-1]:.1f}'
# The record grid is logarithmic in time, so its mean has to be
# integrated, not averaged: the unweighted mean over-reads by 17 % on
# charge and 56 % on discharge. Trapezoidal in t reproduces the shipped
# v2.17 values exactly, which identifies the lost pass's convention.
def _grid_mean(q, t_hist):
    t = np.asarray(t_hist, float); n = min(len(q), len(t))
    q, t = q[:n], t[:n]
    return float(np.trapezoid(np.r_[q[0], q], np.r_[0.0, t]) / t[-1])
D['ProfRateChMeanRecordGrid'] = f"{_grid_mean(qc, ch['t_hist']):.1f}"
D['ProfRateDcMeanRecordGrid'] = f"{_grid_mean(qd, dc['t_hist']):.1f}"
# the cumulative-energy mean, per fin tube and per well
mch = rec['Q_charge_kJ'] / (c.n_ft * rec['N_wells']) / (c.t_ch * 3600)
mdc = -rec['Q_discharge_kJ'] / (c.n_ft * rec['N_wells']) / (c.t_dc * 3600)
D['ProfRateChMean'] = f'{mch:.1f}'
D['ProfRateDcMean'] = f'{mdc:.1f}'
D['ProfRateChMeanPerWell'] = f'{mch * c.n_ft:.0f}'
D['ProfRateDcMeanPerWell'] = f'{mdc * c.n_ft:.0f}'

# -- U_i at the surface, and the mean over it ------------------------------
# v2.19 CORRECTION.  v2.18 took segment index 0 for the surface and shipped
# 72.5.  Index 0 is the BOTTOM: on charge the water reaches the fin tubes
# down the downcomer and enters at the bottom, so the solver's z = 0 is the
# deep end and depth DECREASES with the index (boreas.formation_profile,
# which says so and is asserted in validate_case).  Segment 0 carries the
# 130 C cascade layer against the 69 C rock at 909 m TVD, which is the
# whole point of the design.  The surface is index -1, and 77.4 -- the
# value mkprofiles.py has always written -- was right.
Uc = np.asarray(ch['history']['U_i'], float)
Ud = np.asarray(dc['history']['U_i'], float)
U = np.concatenate([Uc.ravel(), Ud.ravel()]); U = U[np.isfinite(U)]
_Tr, _dep = B.formation_profile(c)
assert _dep[0] > _dep[-1], (
    'segment orientation reversed: index 0 is meant to be the deep end. '
    'Every depth-indexed macro below depends on it.')
I_SURFACE, I_BOTTOM = -1, 0
D['ProfUiSurface'] = f'{Uc[-1, I_SURFACE]:.1f}'
D['ProfUiBottom'] = f'{Uc[-1, I_BOTTOM]:.1f}'
D['ProfUiMeanToSurface'] = f'{U.mean() / Uc[-1, I_SURFACE]:.2f}'

# -- the ORC hot-end limit, and how much energy falls below it -------------
T_lim = rank['T_3e'] - 273.15 + k['pinch_orce_K']
D['ProfTlimORC'] = f'{T_lim:.1f}'
Td = np.asarray(dc['T_out'], float) - 273.15
t = np.asarray(dc['t_hist'], float)
w = np.diff(np.concatenate([[0.0], t]))
n = min(len(Td), len(qd), len(w))                     # T_out is on the
Td, q_, w = Td[:n], np.abs(qd[:n]), w[:n]             # step grid, q' on the
e = q_ * w                                            # record grid
D['ProfBelowPinchShare'] = f'{100 * e[Td < T_lim].sum() / e.sum():.1f}'

# -- the integrator -------------------------------------------------------
def integrator_check(case):
    """One step of dt against two of dt/2, over the branch states.

    advance_segment solves dE'/dt = K (T0 - T_pcm(E')) in closed form inside
    each branch, so halving the step must change nothing but round-off. The
    v2.12-v2.16 packs carried a literal 5.2e-11 that was never computed; the
    first honest attempt gave 4.6e-01 because it compared the PCM enthalpy
    change against the integrated tube duty, and that difference IS the
    formation loss, not an integrator error.
    """
    _, E_lat, C_s, C_l = B.pcm_capacities(case)
    E_lat = float(np.atleast_1d(E_lat)[0])
    C_s = float(np.atleast_1d(C_s)[0]); C_l = float(np.atleast_1d(C_l)[0])
    T_m = float(np.atleast_1d(case.T_m_C)[0]) + 273.15
    worst = 0.0
    for E in (-0.2 * E_lat, 0.0, 0.5 * E_lat, 1.2 * E_lat):   # solid, edge,
        for dT in (+25.0, -25.0):                             # plateau, liquid
            for K in (1e-3, 1e-1):
                for dt in (600.0, 8491.0):
                    a, _ = B.advance_segment(E, T_m, T_m + dT, K, dt,
                                             E_lat, C_s, C_l, True)
                    h, _ = B.advance_segment(E, T_m, T_m + dT, K, dt / 2,
                                             E_lat, C_s, C_l, True)
                    b, _ = B.advance_segment(h, T_m, T_m + dT, K, dt / 2,
                                             E_lat, C_s, C_l, True)
                    scale = max(abs(a), abs(b), E_lat)
                    worst = max(worst, abs(a - b) / scale)
    return worst

D['IntegratorCheck'] = f'{integrator_check(c):.1e}'

# -- cycles to CSS: a count, not the cap ----------------------------------
# \RefFourCyclesToCSS was len(history), which is the n_cycles DEFAULT of 80
# whenever the march runs to the end -- so the macro reported the cap, not
# the convergence. The record's own last-five spread is 1e-16, i.e. it is at
# steady state long before cycle 80. This takes the first cycle at which the
# delivered energy is within CSS_TOL of its final value.
CSS_TOL = 1e-6
def cycles_to_css(record, tol=CSS_TOL):
    dE = np.asarray([x['Q_discharge_kJ'] for x in record['history']], float)
    rel = np.abs(dE - dE[-1]) / abs(dE[-1])
    hit = np.nonzero(rel < tol)[0]
    return int(hit[0] + 1) if len(hit) else len(dE)
D['RefFourCyclesToCSS'] = str(cycles_to_css(rec))
D['CssTol'] = f'{CSS_TOL:.0e}'
D['CssCycleCap'] = '80'
r3 = pickle.load(open('/tmp/build/rec_B_R3.pkl', 'rb'))['rec']
D['RefThreeCyclesToCSS'] = str(cycles_to_css(r3))
# the adiabatic limit too, so the paper does not carry three counts on two
# different definitions (its 38 was an early exit, not a CSS count)
_ra = B.simulate_css_corrected(ref.v216(k_rock=1e-9), record=True)
D['AdiabaticCyclesToCSS'] = str(cycles_to_css(_ra))

# -- honest placeholders, carried as labels with a reason ------------------
D.update({'LineLossCh': 'not computed', 'LineLossDc': 'not computed',
          'LineLossChdT': '0.0', 'LineLossDcdT': '0.0',
          'LineLossChdp': '0', 'LineLossDcdp': '0',
          'VideoDepthNote': 'deepest segment centre; the store bottom is '
                            'RefTvdBottom'})

if __name__ == '__main__':
    json.dump(D, open('/tmp/build/stageD218.json', 'w'), indent=1)
    # compare against the previous pack when it is beside us; the
    # notebook runs this standalone, where there is nothing to compare to
    import os
    p = '/tmp/build/pack217/numbers.json'
    prev = json.load(open(p)) if os.path.exists(p) else {}
    for kk in sorted(D):
        o = prev.get(kk)
        flag = '' if not prev else ('  OK' if str(o) == str(D[kk])
                                    else f'  <- v2.17 had {o}')
        print(f'  {kk:28s} {D[kk]:>14}{flag}')


In [ ]:
%%writefile mk225.py
"""Pack v2.25: boreas_numbers.tex built from EMPTY, not from the last pack.

v2.16 seeded its macro dict from v2.15 and overwrote only the families it
rebuilt, so 188 result macros kept their 45 K/km values and the pack was
unusable. This file starts with {} and fills it only from the 56.5 K/km
run artefacts plus the whitelisted inputs of inputs225.py. Anything that
falls through shows up as MISSING in the contract check, which is the
honest failure mode.
"""
import json, datetime, pickle, sys
sys.path.insert(0,'/tmp/build')
import numpy as np
from inputs225 import INPUTS
from spell import digit_free
L=lambda p: json.load(open(f'/tmp/build/{p}'))
OUT='/tmp/build/pack225'
C={r['name']:r for r in L('contract225.json')}
PREV=json.load(open('/sessions/festive-determined-cray/mnt/GEO2WATTS/'
                    'BOREAS_packs/v2.15/numbers.json'))
lad=L('ladder216.json'); sw=L('sweeps216.json'); sl=L('sliding216.json')
g=L('grad216.json'); cut=L('cutoff216.json'); iso=L('iso.json')
sA=L('stageA217.json'); ex=L('extra217.json')
R=sw['split12']
M={}
for k in INPUTS:
    if k in PREV: M[k]=PREV[k]
M['RefGrad']='56.5'; M['GradRef']='56.5'
M.update(sA)
M.update({'RefEtaRTE':f"{R['eta_RTE']:.4f}",'RefEtaStorage':f"{R['eta_storage']:.4f}",
 'RefWellsPerMWe':f"{R['wells']:.1f}",'RefIslandMWe':f"{R['island']:.2f}",
 'RefIslandMWhDay':f"{R['MWh']:.1f}",'RefEprime':f"{R['e_prime']:.4f}",
 'RefLoss':f"{R['loss']:.2f}",'RefFieldMWePlant':f"{R['island']:.2f}",
 'RefFieldMWh':f"{R['MWh']:.1f}",
 'RefChargeElecRatio':f"{lad['adiabatic']['eta_RTE']/lad['R4']['eta_RTE']:.2f}"})
CORE={'RFour':'RefFour','RThree':'RefThree','Adiabatic':'Adiabatic'}
NAME={'R0v':'RZeroV','R0i':'RZeroI','R1':'ROne','R2':'RTwo','R3':'RThree',
      'R4':'RFour','R4i':'RFourI','adiabatic':'Adiabatic'}
# v2.25: the Field-first family (\LadderLossRFour and siblings) is RETIRED
# at JRB's decision.  It carried the same quantities as the R-first family
# in different units -- 41.6 against 0.4159 for the same loss -- and both
# compiled, so a swap was silent.  The percent form the prose needs is now
# \Ladder{R}LossPct, derived from the SAME fraction one line below it, so
# the two cannot disagree.
for k,t in NAME.items():
    r=lad[k]
    # the adiabatic limit comes back as -1e-17, which prints as "-0.0000"
    # and "-0.0" -- a negative loss in the paper, from round-off
    _loss = 0.0 if abs(r['loss']) < 1e-9 else r['loss']
    M[f'Ladder{t}Loss']=f"{_loss:.4f}"
    M[f'Ladder{t}LossPct']=f"{100*_loss:.1f}"
    M[f'Ladder{t}RTE']=f"{r['eta_RTE']:.5f}"
    M[f'Ladder{t}Ep']=f"{r['e_prime']:.4f}"
    M[f'Ladder{t}EtaStorage']=f"{r['eta_storage']:.4f}"
    M[f'Ladder{t}WellsPerMWe']=f"{r['wells_per_MWe']:.2f}"
    M[f'Ladder{t}IslandMWe']=f"{r['island_MWe']:.2f}"
    M[f'Ladder{t}N']=str(r['N'])
    # The paper's own names for these three rungs are NOT the ladder keys:
    # it writes \RefFour*, \RefThree*, \Adiabatic*.  v2.17's loop tested
    # `t in ('RefFour', ...)` while t was 'RFour', so the condition never
    # fired and six macros the paper inputs went MISSING.  An explicit map
    # cannot fail silently the way a membership test on a derived name can.
    if t in CORE:
        p=CORE[t]
        M[f'{p}EtaRTE']=f"{r['eta_RTE']:.4f}"
        M[f'{p}EtaStorage']=f"{r['eta_storage']:.4f}"
        M[f'{p}WellsPerMWe']=f"{r['wells_per_MWe']:.1f}"
M['LadderSlotStep']=f"{100*(lad['R3']['loss']-lad['R2']['loss']):+.1f}"
M['LadderTrajectoryStep']=f"{100*(lad['R4']['loss']-lad['R3']['loss']):+.1f}"
json.dump(M, open('/tmp/build/_m225a.json','w'), indent=1)
print('part A:', len(M), 'macros')


In [ ]:
%%writefile mk225b.py
import json, pickle, sys
sys.path.insert(0,'/tmp/build')
import numpy as np
from spell import digit_free
L=lambda p: json.load(open(f'/tmp/build/{p}'))
sw=L('sweeps216.json'); sl=L('sliding216.json'); ex=L('extra217.json')
iso=L('iso.json'); R=sw['split12']
M=json.load(open('/tmp/build/_m225a.json'))
SN={9:'NineFifteen',10:'TenFourteen',11:'ElevenThirteen',12:'TwelveTwelve',
    13:'ThirteenEleven',14:'FourteenTen'}
rte=[];mwh=[];mwe=[]
for t,nm in SN.items():
    r=sw[f'split{t}']
    M[f'Split{nm}EtaRTE']=f"{r['eta_RTE']:.5f}"; M[f'Split{nm}MWh']=f"{r['MWh']:.1f}"
    M[f'Split{nm}MWe']=f"{r['island']:.2f}"; M[f'Split{nm}EtaStorage']=f"{r['eta_storage']:.4f}"
    rte.append(r['eta_RTE']); mwh.append(r['MWh']); mwe.append(r['island'])
M['SplitEtaSpreadPct']=f"{100*(max(rte)/min(rte)-1):.1f}"
M['SplitEnergySpreadPct']=f"{100*(max(mwh)/min(mwh)-1):.1f}"
M['SplitPowerSpreadPct']=f"{100*(max(mwe)/min(mwe)-1):.0f}"
W={120:'OneTwenty',125:'OneTwentyFive',130:'OneThirty',135:'OneThirtyFive',
   140:'OneForty',150:'OneFifty',160:'OneSixty'}
for t,w in W.items():
    r=R if t==130 else sw[f'tm{t}']
    M[f'Tm{w}EtaRTE']=f"{r['eta_RTE']:.5f}"; M[f'Tm{w}MWh']=f"{r['MWh']:.1f}"
    M[digit_free(f'TmSweep{t}EtaRTE')]=f"{r['eta_RTE']:.5f}"
    M[digit_free(f'TmSweep{t}MWh')]=f"{r['MWh']:.1f}"
    M[digit_free(f'TmSweep{t}EtaStorage')]=f"{r['eta_storage']:.4f}"
# -- the two T_m optima, COMPUTED.  v2.17 carried \\TmEnergyOpt as a
# -- whitelisted 'label' frozen at 140, while the sweep put the energy
# -- maximum at 150.  An argmax is a result, so it is taken here.  The
# -- sweep was extended to 160 and 170 C this round to settle whether 150
# -- was the maximum or merely the edge: 160 delivers less (87.3 against
# -- 90.9 MWh/day) and 170 does not deliver at all (eta_storage < 0), so
# -- 150 is an interior maximum and the edge caveat can be dropped.
TMS={t:(R if t==130 else sw[f'tm{t}']) for t in W}
M['TmOpt']=f"{max(TMS,key=lambda t: TMS[t]['eta_RTE']):d}"
M['TmEnergyOpt']=f"{max(TMS,key=lambda t: TMS[t]['MWh']):d}"
_ts=sorted(TMS); _e=[TMS[t]['eta_RTE'] for t in _ts]
_i=_e.index(max(_e)); M['TmOptIsInterior']='yes' if 0<_i<len(_ts)-1 else 'no'
_m=[TMS[t]['MWh'] for t in _ts]; _j=_m.index(max(_m))
M['TmEnergyOptIsInterior']='yes' if 0<_j<len(_ts)-1 else 'no'
M['TmSweepTopEvaluated']='170'
M['TmInfeasibleAt']='170'
HN={200:'TwoHundred',260:'TwoSixty',380:'ThreeEighty',500:'FiveHundred'}
for h,t in HN.items():
    r=R if h==380 else sw[f'hm{h}']
    M[f'ParHm{t}Wells']=f"{r['wells']:.1f}"; M[f'ParHm{t}Eprime']=f"{r['e_prime']:.4f}"
    M[f'ParHm{t}Loss']=f"{r['loss']/100:.4f}"; M[f'ParHm{t}EtaRTE']=f"{r['eta_RTE']:.5f}"
    M[f'ParHm{t}IslandMWe']=f"{r['island']:.2f}"
M['ParHmEprimeFactor']=f"{sw['hm500']['e_prime']/sw['hm200']['e_prime']:.1f}"
M['ParHmRangeFactor']='2.5'
M['ParHmTtwodLow']=f"{sw['hm200']['T2d']:.1f}"; M['ParHmTtwodHigh']=f"{sw['hm500']['T2d']:.1f}"
gl={k:v for k,v in sw.items() if k.startswith('glide')}; gl['glide4']=R
ks=sorted(gl,key=lambda s:float(s[5:])); vals=[gl[k]['eta_RTE'] for k in ks]
best=ks[vals.index(max(vals))]
M['SinkGlideOpt']=f"{float(best[5:]):g}"
M['SinkGlideSpreadPct']=f"{100*(max(vals)-min(vals))/max(vals):.1f}"
M['SinkGlideOptBelowRef']='yes' if float(best[5:])<4 else 'no'
free=ex['sinkfree']['eta_RTE']; gonly=ex['glideonly']['eta_RTE']; refv=R['eta_RTE']
M['SinkFreeEtaRTE']=f'{free:.5f}'; M['SinkGlideOnlyEtaRTE']=f'{gonly:.5f}'
M['SinkGlideCostPct']=f'{100*(1-gonly/free):.1f}'
M['SinkPumpCostPct']=f'{100*(1-refv/gonly):.1f}'
M['SinkTotalCostPct']=f'{100*(1-refv/free):.1f}'
M['SinkFreeTone']=f"{ex['sinkfree']['T1e']:.1f}"; M['SinkRealTone']=f"{ex['glideonly']['T1e']:.1f}"
M['SeawaterSourceEtaRTE']=f"{sw['seawater']['eta_RTE']:.5f}"
M['SeawaterSourceIslandMWe']=f"{sw['seawater']['island']:.2f}"
M['SensWeightLoss']=f"{sw['weight36']['loss']-R['loss']:+.2f}"
M['SensMudLoss']=f"{max(abs(sw['ann06']['loss']-R['loss']),abs(sw['ann15']['loss']-R['loss'])):.2f}"
M['SensMudEp']=f"{100*max(abs(sw['ann06']['e_prime']/R['e_prime']-1),abs(sw['ann15']['e_prime']/R['e_prime']-1)):.2f}"
M['SensCasingMaxEp']=M['SensMudEp']
M['SensKopThreeHundredLoss']=f"{sw['kop300']['loss']:.1f}"
M['SensTopOneLoss']=f"{sw['topone']['loss']:.1f}"
M['SensTopThirtyLoss']=f"{sw['topthirty']['loss']:.1f}"
for k,t in (('R4','RefFour'),('R3','RefThree'),('adiabatic','Adiabatic')):
    r=sl[k]
    M[f'{t}EtaORCSliding']=f"{r['eta_ORC_sliding']:.4f}"
    M[f'{t}EtaORCSlidingGain']=f"{r['gain_pct']:+.1f}"
    M[f'{t}EtaRTESliding']=f"{r['eta_RTE_sliding']:.4f}"
    M[f'{t}EtaRTEFixedStrict']=f"{r['eta_RTE_strict']:.4f}"
    M[f'{t}EnergyBelowMean']=f"{r['energy_below_mean']:.1f}"
    M[f'{t}TimeBelowMean']=f"{r['time_below_mean']:.1f}"
M['IsoMeanDT']=f"{iso['IsoMeanDT']:.1f}"; M['IsoLossWm']=f"{iso['IsoLossWm']:.1f}"
M['IsoLossKWhDay']=f"{iso['IsoLossKWhDay']:.2f}"
M['ConvNsBaseLoss']=f"{ex['convbase']['loss']:.3f}"; M['ConvNsBaseEp']=f"{ex['convbase']['e_prime']:.4f}"
M['ConvNsFineLoss']=f"{ex['convns']['loss']:.3f}"; M['ConvNsFineEp']=f"{ex['convns']['e_prime']:.4f}"
M['ConvNtFineLoss']=f"{ex['convnt']['loss']:.3f}"; M['ConvNtFineEp']=f"{ex['convnt']['e_prime']:.4f}"
M['ConvLossSpread']=f"{abs(ex['ds12']['loss']-ex['convbase']['loss']):.2f}"
M['ConvEpSpread']=f"{100*abs(ex['ds12']['e_prime']/ex['convbase']['e_prime']-1):.2f}"
json.dump(M, open('/tmp/build/_m225b.json','w'), indent=1)
print('part B:', len(M), 'macros')


In [ ]:
%%writefile mk225c.py
import json, datetime, pickle, sys
sys.path.insert(0,'/tmp/build')
import numpy as np
from spell import digit_free
L=lambda p: json.load(open(f'/tmp/build/{p}'))
OUT='/tmp/build/pack225'
C={r['name']:r for r in L('contract225.json')}
g=L('grad216.json'); cut=L('cutoff216.json'); sw=L('sweeps216.json'); R=sw['split12']
M=json.load(open('/tmp/build/_m225b.json'))
for Lm in (100,200,300,400,500,600,800,1000,1250):
    r=g[f'56.5@{Lm}']; t=digit_free(f'SweepL{Lm}')
    M[f'{t}Loss']=f"{100*r['loss']:.2f}"; M[f'{t}Ep']=f"{r['e_prime']:.4f}"
    M[f'{t}EtaRTE']=f"{r['eta_RTE']:.5f}"; M[f'{t}IslandMWe']=f"{r['island']:.2f}"
M['CutoffFirstHundred']=f"{g['56.5@100']['e_prime']:.4f}"
M['CutoffMarginalMin']=f"{min(cut['ratio']):.3f}"
for n in ('CutoffRfourLo','CutoffRfourHi','CutoffRecrossLo','CutoffRecrossHi',
          'CutoffTopOneLo','CutoffTopOneHi'): M[n]='none'
M['ParRefAtRefEprime']=f"{R['e_prime']:.4f}"; M['ParRefAtRefLoss']=f"{R['loss']/100:.4f}"
M['ParRefAtRefRTE']=f"{R['eta_RTE']:.5f}"; M['ParRefAtRefWells']=f"{R['wells']:.2f}"
M['ParRefAtRefIslandMWe']=f"{R['island']:.2f}"
c6=g['56.5@600']
M['ParRefAtCutoffL']='600'; M['ParRefAtCutoffEprime']=f"{c6['e_prime']:.4f}"
M['ParRefAtCutoffLoss']=f"{c6['loss']:.4f}"; M['ParRefAtCutoffRTE']=f"{c6['eta_RTE']:.5f}"
M['ParRefAtCutoffIslandMWe']=f"{c6['island']:.2f}"
M['ParRefAtCutoffWells']=f"{363/c6['island']:.2f}"
M['ParRefEprimeAtShoe']=f"{R['e_prime']:.4f}"
M['ParRefEprimeAtShort']=f"{g['56.5@100']['e_prime']:.4f}"
M['ParLRefM']='1250'; M['ParLShoeM']='1250'; M['ParLShoeFt']='4101'
# v2.25: \\GradLowA, \\GradLowB and \\CutoffGradThreshold print violet in the
# paper because they were added after the 203-entry contract was extracted
# and so were invisible to the check.  The contract now carries the paper's
# own list of what it uses (required225.txt), which is what found them.
# The threshold is FITTED, not asserted: the minimum of the marginal e' as
# a fraction of the first 100 m is linear in G, so the gradient at which it
# crosses \\CutoffThreshold is read off that line.
_G=[35.0,40.0,45.0,50.0,56.5]; _MR=[0.424,0.451,0.477,0.502,min(cut['ratio'])]
_a,_b=np.polyfit(_G,_MR,1)
_thr=float(C['CutoffThreshold']['v28']) if False else 0.50
M['CutoffGradThreshold']=f"{(_thr-_b)/_a:.0f}"
M['CutoffGradThresholdFit']=f"{(_thr-_b)/_a:.1f}"
M['CutoffMarginalSlopePerK']=f"{100*_a:.2f}"
M['GradLowA']='45'; M['GradLowB']='35'
_v=L('vol218.json')
for _k,_vv in _v.items(): M[_k]=_vv
M.update({'GradLossA':'46.18','GradLossB':'50.15','GradRTEA':'0.12824',
 'GradRTEB':'0.11694','GradRecrossA':'800--1000','GradRecrossB':'none',
 'GradCutoffA':'500--600','GradCutoffB':'500--600','GradMinRatioA':'47.7',
 'GradMinRatioB':'42.4','GradMinRatioRef':f"{100*min(cut['ratio']):.1f}"})
r3=pickle.load(open('/tmp/build/rec_B_R3.pkl','rb'))['rec']
r4=pickle.load(open('/tmp/build/rec_V210F.pkl','rb'))['rec']
q=np.load('/tmp/build/qloss.npz')
for rec,sfx in ((r3,'RThree'),(r4,'')):
    dc=rec['discharge']; ch=rec['charge']
    Td=np.asarray(dc['T_out'],float)-273.15; Tc=np.asarray(ch['T_out'],float)-273.15
    M[f'ProfTtwodHi{sfx}']=f'{Td.max():.1f}'; M[f'ProfTtwodLo{sfx}']=f'{Td.min():.1f}'
    M[f'ProfTtwocHi{sfx}']=f'{Tc.max():.1f}'; M[f'ProfTtwocLo{sfx}']=f'{Tc.min():.1f}'
    M[f'ProfEpsCh{sfx}']=f"{float(ch['eps_local'].mean()):.4f}"
    M[f'ProfEpsDc{sfx}']=f"{float(dc['eps_local'].mean()):.4f}"
    M[f'ProfEpsCycled{sfx}']=f"{float(ch['eps_local'].mean()-dc['eps_local'].mean()):.4f}"
    U=np.concatenate([np.asarray(rec[h]['history']['U_i']).ravel()
                      for h in ('charge','discharge')])
    U=U[np.isfinite(U)]
    M[f'ProfUiHi{sfx}']=f'{U.max():.1f}'; M[f'ProfUiLo{sfx}']=f'{U.min():.1f}'
    M[f'ProfUiMean{sfx}']=f'{U.mean():.1f}'
    if sfx: M[f'ProfToutDcEnd{sfx}']=f'{Td[-1]:.1f}'
M['ProfQlossMeanRThree']=f"{q['q3'].mean():.2f}"
M['ProfQlossMeanRFour']=f"{q['q4'].mean():.2f}"
M['ProfQlossRatio']=f"{q['q4'].mean()/q['q3'].mean():.1f}"
M.pop('TmDominated',None)
# stage D: the families pack v2.17 shipped from a pass with no script.
# Loaded LAST so that a value it owns cannot be shadowed by an earlier one.
M.update(L('stageD218.json'))
with open(f'{OUT}/boreas_numbers.tex','w') as f:
    f.write(f'% GENERATED by mk225*.py on {datetime.date.today()} -- pack v2.25.\n')
    f.write('% EVERY value is from the G = 56.5 K/km run set, or is a\n'
            '% whitelisted input listed in inputs225.py with its reason.\n'
            '% Built from an EMPTY dict, not seeded from the previous pack.\n\n')
    for k,v in sorted(M.items()):
        cm=C.get(k,{}).get('meaning','')
        cm='' if cm in ('—','') else f'   % {cm}'
        f.write(f'\\newcommand{{\\{k}}}{{{v}}}{cm}\n')
json.dump(M, open(f'{OUT}/numbers.json','w'), indent=1)
print(f'{len(M)} macros, built from empty')


In [ ]:
%%writefile mktab225.py
import json
OUT='/tmp/build/pack225'
L=lambda p: json.load(open(f'/tmp/build/{p}'))
sw=L('sweeps216.json'); lad=L('ladder216.json'); g=L('grad216.json'); sl=L('sliding216.json')
R=sw['split12']
H=('% GENERATED by mktab225.py -- pack v2.25, G = 56.5 K/km.\n'
   '% American spelling. No \\ref to paper labels.\n')
def w(n,Lx): open(f'{OUT}/{n}','w').write('\n'.join(Lx)); print('  ',n)

# ---- cascade top -------------------------------------------------------
# v2.20: 160 C added.  v2.17's caption said the delivered energy "keeps
# rising to 150", which was true only because 150 was the top of the sweep.
# It is not: 160 delivers 87.3 MWh/day against 90.9, and 170 does not
# deliver at all (eta_storage goes negative -- the store loses more than it
# is charged).  So 150 is an interior maximum of the energy, as 130 is of
# the efficiency, and the two measures pick different designs for a reason
# the table now shows.
rows=[(120,sw['tm120']),(125,sw['tm125']),(130,R),(135,sw['tm135']),
      (140,sw['tm140']),(150,sw['tm150']),(160,sw['tm160'])]
Lx=[H,r'\begin{table}[htbp]',r'\centering',r'\setlength{\tabcolsep}{5pt}',
 r'\caption{Cascade top temperature at the reference case. Raising it '
 r'improves the power cycle and degrades the store. \SI{130}{\celsius} is '
 r'an interior maximum of $\eta_\mathrm{RTE}$ and \SI{150}{\celsius} one of '
 r'the delivered energy, so the two measures do not pick the same design. '
 r'Beyond \SI{150}{\celsius} both fall: at \SI{170}{\celsius} the store '
 r'loses more than it is charged and the plant does not deliver.}',r'\label{tab:tm}',r'\begin{tabular}{@{}r r r r r@{}}',r'\toprule',
 r'$T_\mathrm{m}$ [\si{\celsius}] & $\eta_\mathrm{storage}$ & $\eta_\mathrm{ORC}$ '
 r'& $\eta_\mathrm{RTE}$ & [\MWhe{}/day] \\',r'\midrule']
for t,r in rows:
    b=r'\bfseries ' if t==130 else ''
    Lx.append(f"{b}{t} & {b}{r['eta_storage']:.4f} & {b}{r['eta_ORC']:.5f} "
              f"& {b}{r['eta_RTE']:.5f} & {b}{r['MWh']:.1f} \\\\")
Lx+=[r'\bottomrule',r'\end{tabular}',r'\end{table}','']
w('boreas_tab_tm.tex',Lx)

# ---- the day split -----------------------------------------------------
Lx=[H,r'\begin{table}[htbp]',r'\centering',r'\setlength{\tabcolsep}{5pt}',
 r'\caption{Division of the \SI{24}{\hour} day at the reference case. No '
 r'window is left idle. Efficiency, delivered energy and power rating do not '
 r'peak together, and the rating varies by more than half across the range '
 r'while the other two move by a few percent.}',
 r'\label{tab:split}',r'\begin{tabular}{@{}l r r r r@{}}',r'\toprule',
 r'$t_\mathrm{ch}\times t_\mathrm{dc}$ [h] & $\eta_\mathrm{storage}$ & '
 r'$\eta_\mathrm{RTE}$ & [\MWhe{}/day] & rating '
 r'[\MWe] \\',r'\midrule']
for t in (9,10,11,12,13,14):
    r=sw[f'split{t}']; b=r'\bfseries ' if t==12 else ''
    Lx.append(f"{b}${t} \\times {24-t}$ & {b}{r['eta_storage']:.4f} & "
              f"{b}{r['eta_RTE']:.5f} & {b}{r['MWh']:.1f} & {b}{r['island']:.2f} \\\\")
Lx+=[r'\bottomrule',r'\end{tabular}',r'\end{table}','']
w('boreas_tab_split.tex',Lx)

# ---- latent heat -------------------------------------------------------
Lx=[H,r'\begin{table}[htbp]',r'\centering',r'\setlength{\tabcolsep}{5pt}',
 r'\caption{Sensitivity to the latent heat of the phase-change material, at '
 r'the reference cascade top. This remains the largest single uncertainty in '
 r'the well count and therefore in the capital cost.}',
 r'\label{tab:hm}',r'\begin{tabular}{@{}r r r r r@{}}',r'\toprule',
 r'$h_\mathrm{m}$ [\si{\kilo\joule\per\kilogram}] & $\eta_\mathrm{storage}$ & '
 r'$\eta_\mathrm{RTE}$ & wells/\MWe & $e^\prime$ \\',r'\midrule']
for h in (200,260,380,500):
    r=R if h==380 else sw[f'hm{h}']; b=r'\bfseries ' if h==380 else ''
    Lx.append(f"{b}{h} & {b}{r['eta_storage']:.4f} & {b}{r['eta_RTE']:.5f} & "
              f"{b}{r['wells']:.1f} & {b}{r['e_prime']:.4f} \\\\")
Lx+=[r'\bottomrule',r'\end{tabular}',r'\end{table}','']
w('boreas_tab_hm.tex',Lx)

# ---- the summary -------------------------------------------------------
REF=R['eta_RTE']
rows=[('reference case','',R['eta_RTE'],R['MWh'],R['island'],True)]
rows+= [('heat rejection','isothermal sink',sw['isosink']['eta_RTE'],sw['isosink']['MWh'],sw['isosink']['island'],False),
 ('heat-pump source','seawater',sw['seawater']['eta_RTE'],sw['seawater']['MWh'],sw['seawater']['island'],False)]
for t in (120,125,140,150,160):
    r=sw[f'tm{t}']
    rows.append(('cascade top' if t==120 else '',
        f'$T_\\mathrm{{m}}^\\mathrm{{top}} = \\SI{{{t}}}{{\\celsius}}$',
        r['eta_RTE'],r['MWh'],r['island'],False))
for t,lbl in ((10,'$10 \\times 14$'),(14,'$14 \\times 10$')):
    r=sw[f'split{t}']
    rows.append(('division of the day' if t==10 else '',f'{lbl}~h',
                 r['eta_RTE'],r['MWh'],r['island'],False))
s4=sl['R4']; fac=s4['eta_ORC_sliding']/s4['eta_ORC_fixed']
rows.append(('ORC pressure','sliding (upper limit)',s4['eta_RTE_sliding'],
             R['MWh']*fac,R['island']*fac,False))
rows.append(('well field','vertical wells (R3)',lad['R3']['eta_RTE'],
             lad['R3']['island_MWe']*12,lad['R3']['island_MWe'],False))
rows.append(('','idle slots only (R4i)',lad['R4i']['eta_RTE'],
             lad['R4i']['island_MWe']*12,lad['R4i']['island_MWe'],False))
c6=g['56.5@600']
rows.append(('storage length','upper \\SI{600}{\\meter} MD only',
             c6['eta_RTE'],c6['island']*12,c6['island'],False))
Lx=[H,r'\begin{table}[htbp]',r'\centering',r'\setlength{\tabcolsep}{5pt}',
 r'\caption{Effect of each design choice, taken one at a time about the '
 r'reference case. The percentage is relative to the reference '
 r'$\eta_\mathrm{RTE}$. The last three rows change the well field or the '
 r'storage length, and so the size of the plant as well as its efficiency. '
 r'At the reference gradient the marginal return never falls below the '
 r'threshold, so the last row is no longer a cutoff but the depth at which '
 r'one was found at lower gradients.}',
 r'\label{tab:summary}',r'\begin{tabular}{@{}l l r r r r@{}}',r'\toprule',
 r'choice & alternative & $\eta_\mathrm{RTE}$ & [\%] & '
 r'[\MWhe{}/day] & [\MWe] \\',r'\midrule']
for grp,alt,e,mwh,mwe,isref in rows:
    if isref:
        Lx.append(f"\\bfseries reference case & & \\bfseries {e:.5f} & --- & "
                  f"\\bfseries {mwh:.1f} & \\bfseries {mwe:.2f} \\\\")
        Lx.append(r'\midrule'); continue
    Lx.append(f"{grp} & {alt} & {e:.5f} & ${100*(e/REF-1):+.1f}$ & {mwh:.1f} & {mwe:.2f} \\\\")
Lx+=[r'\bottomrule',r'\end{tabular}',r'\end{table}','']
w('boreas_tab_summary.tex',Lx)

# ---- the ladder --------------------------------------------------------
O=[('adiabatic','adiabatic limit'),('R0v','isolated, vertical'),
   ('R0i','isolated, S-curve TVD'),('R1','disc, vertical'),
   ('R2','double row, vertical'),('R3','slots, vertical'),
   ('R4','slots, S-curve \\textbf{(reference)}'),('R4i','idle slots only')]
Lx=[H,r'\begin{table}[htbp]',r'\centering',r'\setlength{\tabcolsep}{5pt}',
 r'\caption{The field ladder at the reference settings. Each rung changes '
 r'one thing. The loss is $Q_\mathrm{loss}/\Delta E_\mathrm{ch} = '
 r'1-\eta_\mathrm{storage}$.}',r'\label{tab:ladder}',
 r'\begin{tabular}{@{}l r r r r r@{}}',r'\toprule',
 r'configuration & $N$ & loss & $\eta_\mathrm{RTE}$ & wells/\MWe '
 r'& $e^\prime$ [\si{\kilo\watt\hour\per\meter}] \\',r'\midrule']
for k,lbl in O:
    if k not in lad: continue
    r=lad[k]; b=r'\bfseries ' if k=='R4' else ''
    Lx.append(f"{b}{lbl} & {b}{r['N']} & {b}{r['loss']:.4f} & {b}{r['eta_RTE']:.5f} "
              f"& {b}{r['wells_per_MWe']:.2f} & {b}{r['e_prime']:.4f} \\\\")
Lx+=[r'\bottomrule',r'\end{tabular}',r'\end{table}','']
w('boreas_tab_ladder.tex',Lx)


In [ ]:
import json, pathlib, shutil, subprocess, sys

# The generators were written against /tmp/build. Repoint them here;
# nothing else about them changes, so the code that runs in this notebook
# is the code that made the pack.
BUILD = str(pathlib.Path('.').resolve())
PACK  = pathlib.Path('pack'); PACK.mkdir(exist_ok=True)
for _f in ('inputs225.py', 'contract225.py', 'check_contract.py', 'spell.py',
           'vol218.py', 'stageA217.py', 'stageD218.py',
           'mk225.py', 'mk225b.py', 'mk225c.py', 'mktab225.py'):
    _s = pathlib.Path(_f).read_text()
    _s = _s.replace('/tmp/build/pack225', str(PACK)).replace('/tmp/build', BUILD)
    pathlib.Path(_f).write_text(_s)

# the marched records section 12.2 wrote, under the names the macro
# generators expect
for _src, _dst in (('rec_R4.pkl', 'rec_V210F.pkl'), ('rec_R3.pkl', 'rec_B_R3.pkl')):
    if pathlib.Path(_src).exists() and not pathlib.Path(_dst).exists():
        shutil.copy(_src, _dst)
assert pathlib.Path('rec_V210F.pkl').exists(), \
    'run section 12.2 first -- it writes the two marched records'
assert pathlib.Path('qloss.npz').exists(), \
    'run section 12.2 first -- mk_losscmp.py writes qloss.npz'

_FAILED = []
for _s in ('contract225.py', 'vol218.py', 'stageA217.py', 'stageD218.py',
           'mk225.py', 'mk225b.py', 'mk225c.py', 'mktab225.py'):
    _r = subprocess.run([sys.executable, _s], capture_output=True, text=True)
    _FAILED = _FAILED + [_s] if _r.returncode else _FAILED
    print(f'{_s:18s}', 'ok' if _r.returncode == 0 else 'FAILED')
    if _r.returncode:
        print(_r.stderr[-1200:]); break
    if _r.stdout.strip():
        print('   ', _r.stdout.strip().splitlines()[-1][:160])

assert not _FAILED, (
    f'these generators failed: {_FAILED}. Run section 1 first -- it writes the model modules -- then this cell again.')


In [ ]:
import os, pathlib, subprocess, sys

# The staleness test compares against the last pack built at a DIFFERENT
# reference (v2.15, G = 45 K/km). v2.18 and v2.19 share their reference, so
# comparing those two would flag every unchanged macro and mean nothing.
_prev = 'v215_numbers.json'
_env = dict(os.environ)
if pathlib.Path(_prev).exists():
    _env['BOREAS_PREV_PACK'] = _prev
else:
    print('no previous-reference pack beside the notebook --',
          'contract, required-list and compile only\n')
_r = subprocess.run([sys.executable, 'check_contract.py', 'pack/numbers.json'],
                    capture_output=True, text=True, env=_env)
print(_r.stdout)
print('exit', _r.returncode, '(0 = every gate passed)')


## 14. Recompute everything (the slow path)

Only used when `RUN_MODE = 'recompute'`. The driver below runs every march
the paper needs, in order of cost, from cells 2.1 and 2.2.

It is **resumable**. Each result is written the moment it finishes and is
skipped next time, so an interrupted session costs only the march it was in
the middle of. Run the cell again as often as you need.

`--list` first: it prints what is still to do and the estimate.

In [ ]:
%%writefile runall219.py
"""Every march the paper needs, driven from `spec.py`, resumable.

This is the slow path. `RUN_MODE = 'recompute'` in the notebook's input
section calls it; `'published'` unpacks the embedded artefacts instead and
skips it. Both then go through the SAME generators, so the figures and
macros cannot differ between the two modes except through the inputs.

Resumable: every job writes its artefact as soon as it finishes and is
skipped on a later run. Interrupt it, re-run it, and it picks up. That is
what makes "however long it takes" practical in Colab, where the session
can be cut off.

  python3 runall219.py            # run everything still missing
  python3 runall219.py --list     # what would run, and the estimate
  python3 runall219.py ladder     # one stage only

Stages, in the order they are cheapest to most expensive:
  spec      write out the spec actually in force
  ref       the reference march, and the plant detail from it
  ladder    the eight field rungs
  sweeps    T_m, the day split, h_m, the sink glide, the sensitivities
  grad      the gradient x length map, and the cutoff from it
  extra     discretisation and the sink-accounting runs
  sliding   the sliding-pressure bracket
  iso       the isolated well
  prof      the R3/R4 profile records and the loss comparison
"""
import json, os, subprocess, sys, time
sys.path.insert(0, os.environ.get('BOREAS_SRC', '.'))
BUILD = os.environ.get('BOREAS_BUILD', os.path.abspath('.'))
import spec

S = spec.SPEC
F = S['field']
GREF = S.get('grad_ref', 56.5)
PY_ = sys.executable


def have(path):
    return os.path.exists(os.path.join(BUILD, path))


def keys_in(path):
    try:
        return set(json.load(open(os.path.join(BUILD, path))))
    except Exception:
        return set()


def run(script, args, label):
    t0 = time.time()
    r = subprocess.run([PY_, script] + [str(a) for a in args],
                       capture_output=True, text=True,
                       cwd=BUILD, env=dict(os.environ, BOREAS_SRC=BUILD))
    dt = time.time() - t0
    ok = r.returncode == 0
    print(f'  {label:34s} {"ok " if ok else "FAILED"} {dt/60:6.1f} min')
    if not ok:
        print(r.stderr[-1500:])
    else:
        for ln in r.stdout.strip().splitlines()[-3:]:
            print('      ', ln[:120])
    return ok


# ---------------------------------------------------------------- jobs ---
def job_spec():
    json.dump(S, open(os.path.join(BUILD, 'boreas_spec.json'), 'w'), indent=1)
    print('  boreas_spec.json written -- the spec actually in force')
    return True


def job_ladder():
    todo = [k for k in S['ladder_rungs'] if k not in keys_in('ladder216.json')]
    return run('ladder216.py', todo, f'ladder: {len(todo)} rung(s)') if todo else True


def job_sweeps():
    want = ([f'split{int(t)}' for t in S['t_ch_h']]
            + [f'tm{int(t)}' for t in S['T_m_C']]
            + [f'hm{int(h)}' for h in S['h_m_kJ_kg']]
            + [f'glide{g:g}' for g in S['sink_glide_K']]
            + list(S['sens']))
    todo = [k for k in want if k not in keys_in('sweeps216.json')]
    return run('sweeps216.py', todo, f'sweeps: {len(todo)} case(s)') if todo else True


def job_grad():
    want = [f'{GREF:g}@{L:g}' for L in S['storage_length_m_MD']]
    for g in S['grad_K_per_km']:
        if abs(g - GREF) < 1e-9:
            continue
        want += [f'{g:g}@{L:g}' for L in S['cutoff_length_m_MD']]
    todo = [k for k in want if k not in keys_in('grad216.json')]
    return run('grad216.py', todo, f'gradient x length: {len(todo)}') if todo else True


def job_extra():
    todo = [k for k in S['extra'] if k not in keys_in('extra217.json')]
    return run('extra217.py', todo, f'extra: {len(todo)}') if todo else True


def job_sliding():
    todo = [k for k in S['sliding_cases'] if k not in keys_in('sliding216.json')]
    return run('sliding216.py', todo, f'sliding: {len(todo)}') if todo else True


def job_iso():
    return True if have('iso.json') else run('iso.py', [], 'isolated well')


def job_prof():
    ok = True
    if not (have('rec_R3.pkl') and have('rec_R4.pkl')):
        ok = run('mkR3prof.py', [], 'R3/R4 profile records') and ok
    if not have('qloss.npz'):
        ok = run('mk_losscmp.py', [], 'loss comparison') and ok
    return ok


def job_cutoff():
    """The marginal-e' curve and the cutoff, from the gradient map."""
    import numpy as np
    g = json.load(open(os.path.join(BUILD, 'grad216.json')))
    out = {}
    for G in S['grad_K_per_km']:
        Ls = sorted(float(k.split('@')[1]) for k in g if k.startswith(f'{G:g}@'))
        if len(Ls) < 3:
            continue
        ep = np.array([g[f'{G:g}@{L:g}']['e_prime'] for L in Ls])
        E = ep * np.array(Ls)
        marg = np.diff(np.r_[0.0, E]) / np.diff(np.r_[0.0, Ls])
        ratio = (marg / marg[0]).tolist()
        thr = 0.5
        below = [i for i, v in enumerate(ratio) if v < thr]
        cut = [Ls[below[0] - 1], Ls[below[0]]] if below else None
        rec = None
        if below:
            back = [j for j in range(below[0], len(ratio)) if ratio[j] >= thr]
            rec = [Ls[back[0] - 1], Ls[back[0]]] if back else None
        out[f'{G:g}'] = dict(lengths=Ls, ratio=ratio, cutoff=cut, recross=rec)
        if abs(G - GREF) < 1e-9:
            json.dump(dict(cutoff=cut, recross=rec, ratio=ratio, L=Ls),
                      open(os.path.join(BUILD, 'cutoff216.json'), 'w'), indent=1)
    json.dump(out, open(os.path.join(BUILD, 'grad_cutoff.json'), 'w'), indent=1)
    print(f'  cutoff map: {len(out)} gradient(s); '
          f'reference cutoff {out[f"{GREF:g}"]["cutoff"]}')
    return True


STAGES = [('spec', job_spec), ('prof', job_prof), ('ladder', job_ladder),
          ('sweeps', job_sweeps), ('grad', job_grad), ('cutoff', job_cutoff),
          ('extra', job_extra), ('sliding', job_sliding), ('iso', job_iso)]
MIN_PER_MARCH = 2.0


def inventory():
    """How many marches the spec in force asks for, and how many are done.

    Counted from the spec, not from a table of constants -- change a list in
    the notebook's input section and this number follows, which is the
    point of being able to see it before starting a three-hour run.
    """
    inv = {}
    inv['profile records (R3, R4)'] = (
        [p for p in ('rec_R3.pkl', 'rec_R4.pkl') if not have(p)], 2)
    inv['ladder rungs'] = (
        [k for k in S['ladder_rungs'] if k not in keys_in('ladder216.json')],
        len(S['ladder_rungs']))
    want = ([f'split{int(t)}' for t in S['t_ch_h']]
            + [f'tm{int(t)}' for t in S['T_m_C']]
            + [f'hm{int(h)}' for h in S['h_m_kJ_kg']]
            + [f'glide{g:g}' for g in S['sink_glide_K']]
            + list(S['sens']))
    inv['sweeps (T_m, split, h_m, glide, sens)'] = (
        [k for k in want if k not in keys_in('sweeps216.json')], len(want))
    wg = [f'{GREF:g}@{L:g}' for L in S['storage_length_m_MD']]
    for g in S['grad_K_per_km']:
        if abs(g - GREF) > 1e-9:
            wg += [f'{g:g}@{L:g}' for L in S['cutoff_length_m_MD']]
    inv['gradient x length'] = (
        [k for k in wg if k not in keys_in('grad216.json')], len(wg))
    inv['discretisation and sink'] = (
        [k for k in S['extra'] if k not in keys_in('extra217.json')],
        len(S['extra']))
    inv['sliding-pressure bracket'] = (
        [k for k in S['sliding_cases'] if k not in keys_in('sliding216.json')],
        len(S['sliding_cases']))
    inv['isolated well'] = ([] if have('iso.json') else ['iso'], 1)
    return inv

if __name__ == '__main__':
    args = [a for a in sys.argv[1:] if not a.startswith('--')]
    if '--list' in sys.argv:
        inv = inventory()
        tot = sum(t for _, t in inv.values())
        todo = sum(len(d) for d, _ in inv.values())
        print(f'{"stage":42s} {"to run":>7} {"of":>5}')
        for k, (d, t) in inv.items():
            print(f'  {k:40s} {len(d):7d} {t:5d}'
                  + ('   ' + ', '.join(d[:6]) + (' ...' if len(d) > 6 else '')
                     if d else ''))
        print(f'  {"TOTAL":40s} {todo:7d} {tot:5d}')
        print(f'\nabout {todo*MIN_PER_MARCH:.0f} min of marching still to do '
              f'({tot*MIN_PER_MARCH:.0f} min from nothing), at roughly '
              f'{MIN_PER_MARCH:g} min per march on one core.')
        print('Resumable: stop it and re-run it, and it continues.')
        sys.exit(0)
    t0 = time.time()
    bad = []
    for name, fn in STAGES:
        if args and name not in args:
            continue
        print(f'[{name}]')
        if not fn():
            bad.append(name)
    print(f'\ntotal {(time.time()-t0)/60:.1f} min'
          + (f'   FAILED: {bad}' if bad else '   all stages ok'))
    sys.exit(1 if bad else 0)


In [ ]:
%%writefile mkfigdata219.py
"""Derive the figure data from the run artefacts -- not a second copy.

`figdata216/` held tmfinal.json, split211.json, hm210.json, ladder210.json,
lensweep.json and glide05.json: the same results as sweeps216/ladder216/
grad216, reshaped for the figure generator. Two copies of one result is
how a figure gets drawn at the wrong gradient, which is exactly what the
v2.18 guard had to catch. They are derived here instead, so the figures
and the macros cannot come from different runs.
"""
import json, os, sys
sys.path.insert(0, os.environ.get('BOREAS_SRC', '.'))
BUILD = os.environ.get('BOREAS_BUILD', '.')
OUT = os.environ.get('BOREAS_FIGDATA', 'figdata')
os.makedirs(OUT, exist_ok=True)
L = lambda p: json.load(open(os.path.join(BUILD, p)))
W = lambda n, o: json.dump(o, open(os.path.join(OUT, n), 'w'), indent=1)

sw = L('sweeps216.json'); lad = L('ladder216.json'); g = L('grad216.json')
R = sw['split12']

# The reference point appears in several sweeps (T_m = 130, h_m = 380,
# glide = 4 K, 12 x 12 h). Where it does, take it from the ONE reference
# record rather than from that sweep's own march, so the figure and the
# macro quote the same digits.
_rc = os.path.join(BUILD, 'ref_case.json')
_REF = json.load(open(_rc)) if os.path.exists(_rc) else {}
_refv = lambda k, d: float(_REF.get(k, d))
GREF = f"{float(json.load(open(os.path.join(BUILD,'boreas_spec.json'))).get('grad_ref', 56.5)):g}" \
    if os.path.exists(os.path.join(BUILD, 'boreas_spec.json')) else '56.5'

# ---- the cascade top ---------------------------------------------------
tm = {}
for k, r in sw.items():
    if not k.startswith('tm'):
        continue
    if r.get('eta_storage', 1) < 0:      # 170 C does not deliver
        continue
    tm[k[2:]] = dict(T_m=float(k[2:]), eta_RTE=r['eta_RTE'],
                     eta_st=r['eta_storage'], eta_ORC=r['eta_ORC'],
                     COP=r['COP'], island=r['island'])
_ref = R
tm.setdefault('130', dict(T_m=130.0, eta_RTE=_ref['eta_RTE'],
                          eta_st=_ref['eta_storage'], eta_ORC=_ref['eta_ORC'],
                          COP=_ref['COP'], island=_ref['island']))
W('tmfinal.json', tm)

# ---- the day split -----------------------------------------------------
sp = {}
for k, r in sw.items():
    if not k.startswith('split'):
        continue
    t = int(k[5:])
    sp[f'{t}x{24-t}'] = dict(t_ch=float(t), t_dc=float(24 - t),
                             eta_RTE=r['eta_RTE'], eta_storage=r['eta_storage'],
                             island=r['island'], MWh=r['MWh'])
W('split211.json', sp)

# ---- the latent heat ---------------------------------------------------
hm = {}
for k, r in sw.items():
    if not k.startswith('hm'):
        continue
    if abs(float(k[2:]) * 1e3 - _refv('h_m', 380e3)) < 1.0:
        r = R
    hm[k[2:]] = dict(h_m=float(k[2:]), eta_RTE=r['eta_RTE'],
                     eta_st=r['eta_storage'], wells_per_MWe=r['wells'],
                     island=r['island'], e_prime=r['e_prime'], T2d=r['T2d'])
W('hm210.json', hm)

# ---- the ladder --------------------------------------------------------
W('ladder210.json', lad)

# ---- the length sweep --------------------------------------------------
ls = {}
for k, r in g.items():
    gk, Lk = k.split('@')
    if gk != GREF:
        continue
    ls[f"{float(Lk):g}"] = dict(L=r['L'], eta_RTE=r['eta_RTE'], loss=r['loss'],
                                e_prime=r['e_prime'], wells_per_MWe=0.0,
                                island=r['island'], tvd=r['tvd'])
W('lensweep.json', ls)

# ---- the sink glide ----------------------------------------------------
# NOT derived. figdata216/glide05.json existed but no figure reads it: the
# glide figure was dropped in v2.14 and the numbers live in the macros
# (\SinkGlideOpt, \SinkGlideSpreadPct). Writing it with placeholder zeros
# for T_1e, as the old copy had, is worse than not writing it.

for _n in ('tmfinal', 'split211', 'hm210', 'ladder210', 'lensweep'):
    _o = json.load(open(os.path.join(OUT, _n + '.json')))
    print(f'  {_n+".json":18s} {len(_o)} entries')


In [ ]:
import os, subprocess, sys
os.environ.update(BOREAS_SRC='.', BOREAS_BUILD='.')

if RUN_MODE != 'recompute':
    print('RUN_MODE is', RUN_MODE, '-- nothing to recompute.')
    print('The published results were unpacked in section 12.')
else:
    print(subprocess.run([sys.executable, 'runall219.py', '--list'],
                         capture_output=True, text=True).stdout)


In [ ]:
# The run itself. Hours. Re-run this cell if the session is cut off.
if RUN_MODE == 'recompute':
    import subprocess, sys
    _r = subprocess.run([sys.executable, 'runall219.py'], text=True)
    print('exit', _r.returncode)
else:
    print('skipped:  RUN_MODE =', RUN_MODE)


## 15. Does this notebook run?

The v2.16 and v2.18 notebooks shipped with a **repeated keyword in cell
2.1**, so the first cell after the model did not compile. Neither round
caught it, because both verified the figure and pack cells against a case
built outside the notebook. This cell compiles every code cell in the
file, and the pack is now verified by running the notebook top to bottom
rather than by assembling its pieces by hand.

In [ ]:
import json, pathlib, re

# A magic sits at column 0. Blanking every line whose *stripped* form starts
# with '%' also blanks a continuation line like "      % (len(x) + 1)", which
# is how my first version of this check reported a false error in cell 2.2.
_MAGIC = re.compile(r'^[%!]')

# --- 1. does the notebook claim the version it builds? -------------------
# The title cell said "v1.4 - built 2026-11-03" for four packs. A notebook
# that misdescribes itself is a stale number like any other.
#
# Section 2.3 writes notebook_meta.json. Reading the version from there
# rather than from the .ipynb makes this work in Colab, where the notebook
# lives in Drive and not on the filesystem -- the path that made the first
# version of this cell raise NameError. Reading it from a FILE rather than
# from a kernel variable makes it survive a restart, too.
_meta = pathlib.Path('notebook_meta.json')
if not _meta.exists():
    raise SystemExit('run section 2.3 first -- it writes notebook_meta.json')
NOTEBOOK_VERSION = json.load(open(_meta))['notebook_version']
_built = None
if pathlib.Path('pack/boreas_numbers.tex').exists():
    _m = re.search(r'pack v(\d+\.\d+)',
                   pathlib.Path('pack/boreas_numbers.tex').read_text())
    _built = _m.group(1) if _m else None
print(f'notebook says v{NOTEBOOK_VERSION}; the pack it built says v{_built}')
if _built is None:
    print('  (no pack built in this session -- run section 13 to check this)')
else:
    assert NOTEBOOK_VERSION == _built, (
        f'the notebook calls itself v{NOTEBOOK_VERSION} but builds pack v{_built}')
    print('  the notebook describes itself correctly')

# --- 2. does every file the notebook wrote compile? ----------------------
_py = sorted(pathlib.Path('.').glob('*.py'))
_bad2 = []
for _p in _py:
    try:
        compile(_p.read_text(), str(_p), 'exec')
    except SyntaxError as _e:
        _bad2.append((_p.name, _e.lineno, _e.msg))
for _b in _bad2:
    print(f'  {_b[0]}: line {_b[1]}: {_b[2]}')
assert not _bad2, 'written files do not compile'
print(f'all {len(_py)} written .py files compile')

# --- 3. and every code cell in the notebook ------------------------------
# Only possible when the .ipynb is on the filesystem. In Colab it is not,
# so this part reports that it was skipped rather than pretending to pass.
_nbp = next((p for p in sorted(pathlib.Path('.').glob('*.ipynb'))), None)
if _nbp is None:
    print('the .ipynb is not on this filesystem (normal in Colab) --')
    print('  the per-cell compile check is skipped; it runs on a local copy')
else:
    _nb = json.load(open(_nbp))
    _bad = []
    for _i, _c in enumerate(_nb['cells']):
        if _c['cell_type'] != 'code':
            continue
        _s = ''.join(_c['source'])
        if _s.lstrip().startswith('%%writefile'):
            continue              # the payload is a file, checked above
        _body = '\n'.join('' if _MAGIC.match(_l) else _l
                           for _l in _s.split('\n'))
        try:
            compile(_body, f'<cell {_i}>', 'exec')
        except SyntaxError as _e:
            _bad.append((_i, _e.lineno, _e.msg))
    for _b in _bad:
        print(f'  cell {_b[0]}: line {_b[1]}: {_b[2]}')
    assert not _bad, 'cells do not compile'
    print(f'{_nbp.name}: {len(_nb["cells"])} cells, every code cell compiles')
